# Week 2: adapting pricing to the emerging market

I am Manuel Rodríguez, an actuarial student and individual participant in the IAA AI Hackathon 2026. In this notebook I explain and reconstruct my Week 2 adaptation, developed with AI assistance.

I estimate the 2025 ultimate cost for each emerging-market policy period. The developed market provides a reference; I keep joins within each market. The separate commercial premium exercise applies expense assumptions to twenty reference policies and does not change the risk-premium submission.


## Notebook guide

| Purpose | What I reconstruct |
| --- | --- |
| Selected recipe | F3c, my Week 2 submission |
| Required inputs | 11 developed-market and 13 emerging-market Parquet files from Part 2 |
| Final exports | `submission_week2_F3c.csv` and `commercial_reference_premiums.csv` |
| External model or repository download | None; source and frozen configuration are embedded |

For a first reading, I follow the [method](#modeling-and-evidence), [execution plan](#inspect-the-execution-plan) and [output checks](#check-and-save-the-risk-premium-csv). The [embedded source package](#embedded-source-package) preserves the individual programs for inspection. Its source cells are marked as hidden in JupyterLab's presentation metadata; I can expand them. GitHub may display them in full.

For execution, I attach the data, check [dependencies and storage](#dependencies-and-run-workspace) and run cells in order. The later [recipe execution](#run-the-frozen-stages) trains models and can take hours. Reading the notebook does not require starting that execution.

My resumed local reconstruction completed the 48 stages through F3c and recovered the historical F3c CSV. The twenty reference premiums are calculated separately below. I have not established a hosted Kaggle execution or an uninterrupted Run All of this adapted notebook. AI-assisted reconstruction is distinct from independent professional review.


## Scope and reproducibility

I rebuild the historical F3c risk-premium chain from the official Parquet inputs, including its developed-market T2b models, A1 transfer, public-probe calibration and F3c adjustment. I then calculate the twenty commercial reference premiums. The embedded code and frozen aggregate constants are part of this notebook; no repository checkout, previous model, prediction file or team script is required.

This is a notebook adaptation of my historical reconstruction, not an independent professional validation. Its full execution on Kaggle has not yet been verified. Numerical differences can arise from library versions and hardware; a hash mismatch must be investigated rather than presented as an exact reproduction.

Attach Part 2 and run the cells in order. The input files remain read-only. I keep intermediate data, models and logs in a fresh temporary directory outside `/kaggle/working`; only final CSVs go into `/kaggle/working`. The temporary directory does not persist with the notebook output. Keep the row-level final outputs private under the competition rules.


## Attach the competition data

Add the IAA AI-Assisted Hackathon 2026 Part 2 competition as an input. The archived listing contains `developed/` and `emerging/` folders. The developed folder is the historical reference from Part 1. If the search finds multiple matches, set the corresponding path explicitly. Local tests can set `IAA_W2_DEVELOPED` and `IAA_W2_EMERGING`.


## Optional dependency installation

Kaggle usually includes these libraries. If the environment does not match the recorded reference, set the switch to `True` and run this cell with Internet enabled. Restart the kernel after installation, set the switch back to `False`, and run from the top. Keep the same Kaggle session so the installed packages remain available. The reference versions may not support every Kaggle Python image.


In [ ]:
INSTALL_REFERENCE_PACKAGES = False
if INSTALL_REFERENCE_PACKAGES:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==2.5.3", "pandas==3.0.5", "scipy==1.18.1", "scikit-learn==1.9.1", "lightgbm==4.7.0", "pyarrow==26.0.0"])
    raise RuntimeError("Packages installed. Restart the kernel, turn installation off, and run all cells again.")


In [ ]:
import os, sys, hashlib, importlib.metadata
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

for package in ("numpy", "pandas", "pyarrow"):
    print(f"{package}: {importlib.metadata.version(package)}")
print("Python:", sys.version)

DEVELOPED_DIR = os.environ.get("IAA_W2_DEVELOPED", "")  # e.g. /kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/developed
EMERGING_DIR = os.environ.get("IAA_W2_EMERGING", "")  # e.g. /kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/emerging


In [ ]:
LINES = ("motor", "buildings", "contents")
BASE_FILES = ["addresses.parquet", "claims.parquet", "customers.parquet",
              "weather.parquet", "sample_submission.parquet"] + [
    f"{prefix}_{line}.parquet" for prefix in ("policies", "test") for line in LINES]

def find_market(market, required, override):
    if override:
        candidates = [Path(override)]
    else:
        candidates = [p for p in Path("/kaggle/input").rglob(market) if p.is_dir()]
    candidates = [p for p in candidates if all((p / name).is_file() for name in required)]
    if len(candidates) != 1:
        raise RuntimeError(f"Expected exactly one {market} folder with the required files; "
                           f"found {candidates}. Set the override above explicitly.")
    return candidates[0]

def inventory(folder, filenames):
    rows = []
    for name in filenames:
        path = folder / name
        pf = pq.ParquetFile(path)
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
        rows.append({"file": name, "rows": pf.metadata.num_rows,
                     "columns": len(pf.schema_arrow), "bytes": path.stat().st_size,
                     "sha256": digest.hexdigest()})
    return pd.DataFrame(rows)

def check_template(folder):
    sample = pd.read_parquet(folder / "sample_submission.parquet")
    test = pd.concat([pd.read_parquet(folder / f"test_{line}.parquet", columns=["row_id"])
                      for line in LINES], ignore_index=True)
    assert "row_id" in sample.columns and sample.row_id.notna().all()
    assert test.row_id.notna().all() and test.row_id.is_unique and sample.row_id.is_unique
    assert len(sample) == len(test) and sample.row_id.equals(test.row_id), "Template and test IDs/order differ"
    print(f"Template matches the concatenated test IDs in order: {len(test):,} rows")
    return sample


In [ ]:
emerging_files = BASE_FILES + ["expenses.parquet", "reference_policies.parquet"]
developed = find_market("developed", BASE_FILES, DEVELOPED_DIR)
emerging = find_market("emerging", emerging_files, EMERGING_DIR)
for name, folder, files in (("developed", developed, BASE_FILES), ("emerging", emerging, emerging_files)):
    print(name, folder)
    display(inventory(folder, files))
sample = check_template(emerging)


## Modeling and evidence

I use claims valued at 31 December 2024 as an incomplete observation of ultimate losses. T2b learns developed-market risk patterns; A1 transfers a separate developed-market structure. The F3c recipe combines the two and applies historical public-score information. Those public factors are frozen, so this is a reconstruction of the historical submission rather than a new blind evaluation. I do not recalculate leaderboard probes or select an alternative using the 2024 results.

Two parts of the historical source were restored: a global reporting curve reconstructed from the documented formula and the old motor confirmation join before the ID01 correction. They are clearly distinguished from recovered original code. Availability at quotation, case reserves and the stability of new-business results remain open limitations. The reconstruction previously matched F3c in a Windows Python 3.12 environment; it was performed within the same AI-assisted workflow and is not independent professional review.

The code below preserves the original authorship: Claude developed the source modeling lines; Codex reconstructed the execution chain and these notebook adaptations under my direction. The source files are embedded as text and written into this run's temporary workspace so that their imports and the historical stage boundaries remain intact.


## What changes in the emerging market

The emerging market has a shorter claims history than the developed market. I use the developed portfolio to learn risk patterns, then adapt their level and other components to emerging-market experience. The markets stay separate when policies, customers and claims are joined: a matching identifier across markets is not evidence of the same entity.

For each market I assign accepted claims to policy periods by line, policy and occurrence date and calculate observed net incurred cost. Claims are valued at 31 December 2024. I estimate reporting completeness to construct the effective exposure used by the models. This does not establish that open-claim reserves are adequate or that the reporting pattern transports perfectly between markets.

## Training and validation timeline

| Stage | Developed data | Emerging data | Purpose |
| --- | --- | --- | --- |
| `dev` | 2021–2023 | 2023, with customer folds | Out-of-fold predictions for local development |
| `hold` | 2021–2023 | Fit on 2023, evaluate 2024 | Temporal contrast |
| `pcv` | 2021–2024 | 2023–2024, with customer folds | Out-of-fold inputs for production calibration |
| `prod` | 2021–2024 | 2023–2024 | Predictions for 2025 |

T2b is a developed-market LightGBM Tweedie model with line-specific settings. It uses an internal customer split to choose the number of rounds, then refits on the permitted source training data. A1 is a separate transfer line; the production reconstruction uses three seeds and its recorded claims-history variant. The intermediate 2024 contrast and the production A1 recipe have different scopes. I do not call the contrast an out-of-fold replay of the entire production file.

Customer folds reduce same-customer leakage across fold boundaries; the year split tests transport across time. They address different problems. Both 2023 and 2024 contributed to development, so the resulting diagnostics are adaptive rather than an independent final validation.

## From components to F3c

The historical chain constructs a 50/50 blend of its designated T2b and A1 production components. It then applies the frozen global public-probe level, F2 adjustments and the credible final level and shape. F3c adds the recorded residual factors for gender in motor and contents, and for working from home and marketing segment in contents, followed by their historical level adjustment. The code recalculates the local factors from the newly generated inputs and uses the saved aggregate public-probe constants; it does not make new submissions.

The public calibration makes this a competition reconstruction. It would need a different source of level information in an actual pricing deployment. Gender is included because it belongs to the historical F3c specification; its use would need a jurisdiction-specific review and an assessment of differential effects before any real pricing application. New-business support, feature timing, influential claims and reserve uncertainty remain material limits. The conditional sensitivity checks do not establish that the method will perform the same way on a new portfolio.

## Reading the output

I expect one positive finite prediction for each emerging-market test row, in the sample template's order. The notebook checks that contract and compares the CSV with the historical SHA-256. An exact match is a reproducibility check for the file, not independent validation of the actuarial assumptions. A mismatch stops the final step and leaves the generated file available for investigation. The reference premium calculation below is separate from the submission and does not reapply exposure to a period risk premium.


## Interpreting the metric

For nonnegative observed cost `y` and positive prediction `mu`, Tweedie deviance at power 1.5 is `4 * (y / sqrt(mu) + sqrt(mu) - 2 * sqrt(y))`. The local D² is one minus the total model deviance divided by the deviance of the constant mean prediction. Higher is better; a negative value means worse than that constant on the evaluated rows. The observed target remains a valuation proxy for ultimate cost. I do not claim this local calculation proves every detail of Kaggle's unpublished evaluator.

When comparing models, I keep the same rows and distinguish a global D² from segment-specific D² values, which have different null predictions. I also inspect the support and uncertainty of each segment instead of treating a small score difference as conclusive.


In [ ]:
from sklearn.metrics import mean_tweedie_deviance, d2_tweedie_score
# A small synthetic calculation, including a zero-cost observation.
y_demo = np.array([0.0, 0.25, 2.0, 100.0])
mu_demo = np.array([0.1, 0.3, 1.5, 25.0])
d_demo = 4 * (y_demo / np.sqrt(mu_demo) + np.sqrt(mu_demo) - 2 * np.sqrt(y_demo))
null_demo = 4 * (y_demo / np.sqrt(y_demo.mean()) + np.sqrt(y_demo.mean()) - 2 * np.sqrt(y_demo))
assert np.isclose(d_demo.mean(), mean_tweedie_deviance(y_demo, mu_demo, power=1.5), rtol=1e-13)
assert np.isclose(1 - d_demo.sum() / null_demo.sum(), d2_tweedie_score(y_demo, mu_demo, power=1.5), rtol=1e-13)
print("The formula agrees with the local scikit-learn calculation.")


## Dependencies and run workspace

The reference reconstruction used Python 3.12.14, NumPy 2.5.3, pandas 3.0.5, PyArrow 26.0.0, SciPy 1.18.1, scikit-learn 1.9.1 and LightGBM 4.7.0. I check the installed versions before running. If the Kaggle image differs, install compatible versions in a separate setup step and restart the kernel; package installation may require enabling Internet. Exact bytes are not promised on a different platform. The historical local 57-stage reconstruction, which also included alternatives beyond F3c, took about 57 minutes excluding installation on its specific hardware. This notebook stops at F3c. [Kaggle's notebook documentation](https://www.kaggle.com/docs/notebooks), consulted on 28 October 2026, describes a 12-hour CPU session and 20 GB of saved working storage. Runtime and storage in a hosted run remain unverified.


In [ ]:
import json, shutil, subprocess, time, uuid
from datetime import datetime, timezone
from importlib import metadata
REFERENCE = {"numpy": "2.5.3", "pandas": "3.0.5", "pyarrow": "26.0.0", "scipy": "1.18.1", "scikit-learn": "1.9.1", "lightgbm": "4.7.0"}
installed = {name: metadata.version(name) for name in REFERENCE}
print("Installed:", installed)
print("Reference:", REFERENCE)
if installed != REFERENCE:
    print("Environment differs from the recorded reference; output identity must be checked.")
import tempfile
RUN_ROOT = Path(os.environ.get("IAA_NOTEBOOK_WORK", tempfile.gettempdir()))
RUN = RUN_ROOT / ("iaa_week2_" + uuid.uuid4().hex[:12])
RUN.mkdir(parents=True, exist_ok=False)
EXPORT_ROOT = Path(os.environ.get("IAA_NOTEBOOK_EXPORT", "/kaggle/working" if Path("/kaggle/working").is_dir() else str(RUN)))
EXPORT = EXPORT_ROOT / (RUN.name + "_exports")
EXPORT.mkdir(parents=True, exist_ok=False)
print("Final outputs:", EXPORT)
print("New run:", RUN)
SOURCES = {}


## Embedded source package

I keep the source files below as text assignments. Running these cells registers the code in memory; the later execution section writes it into the new workspace and starts the recipe. I retain the original program boundaries, authorship and frozen configuration. No past row-level predictions or fitted models are embedded.


### Embedded source: `ml/codigo/abrir_hold.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/abrir_hold.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/abrir_hold.py'] = '''"""Apertura única de 2024 (HOLD) para la lista congelada.

Lee entrega/lista_congelada.json; parámetros de mezcla y calibración desde el OOF de DEV; tendencia de la fase hold.
"""
import json
import numpy as np, pandas as pd
from comun import *
from evaluar import metricas, bootstrap_dif
import receta

lista = json.load(open(ENTREGA / "lista_congelada.json"))
tend = json.load(open(TRABAJO / "pred" / "hold" / "tendencia.json"))
pesos = pd.read_parquet(TRABAJO / "pesos_2025_hold.parquet")
res = {"candidatos": {}, "bootstrap": {}}
mus = {}
for nombre, spec in lista["candidatos"].items():
    params = receta.parametros("dev", spec["componentes"], spec["calib"])
    t, mu_sin = receta.aplicar("hold", params, None)
    _, mu_con = receta.aplicar("hold", params, tend)
    t = t.merge(pesos, on="row_id", how="left")
    mus[nombre] = mu_con if lista.get("tendencia", True) else mu_sin
    res["candidatos"][nombre] = {"con_tendencia": metricas(t, mu_con), "sin_tendencia": metricas(t, mu_sin), "params": params}
    print(nombre, round(res["candidatos"][nombre]["con_tendencia"]["D2"], 4), round(res["candidatos"][nombre]["sin_tendencia"]["D2"], 4), flush=True)
ref = lista["referencias"]
for nombre in lista["candidatos"]:
    for r in ref:
        if nombre != r:
            res["bootstrap"][f"{nombre}-{r}"] = bootstrap_dif(t, mus[nombre], mus[r], B=1000)
elegido = lista["elegido"]
res["elegido"] = elegido
guardar_json(res, ENTREGA / "resultados_hold.json")
print(json.dumps({k: v for k, v in res["bootstrap"].items() if k.startswith(elegido)}, indent=1))
'''


### Embedded source: `ml/codigo/adaptacion.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/adaptacion.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/adaptacion.py'] = '''"""T5: adaptación GLM con ridge sobre las relatividades transferidas de T2 (E03).

log mu = raw_T2 + log(e c) + b0 + Z b, con penalización ridge sobre b (credibilidad).
Uso: python adaptacion.py fase
"""
from __future__ import annotations
import sys, hashlib
import numpy as np, pandas as pd
from scipy.optimize import minimize
from comun import *
from variables import cargar

FASE = sys.argv[1] if len(sys.argv) > 1 else "dev"
CONF = {"dev": ("hasta2023", [2023], None), "hold": ("hasta2023", [2023], 2024), "prod": ("hasta2024", [2023, 2024], 2025),
        "pcv": ("hasta2024", [2023, 2024], None)}[FASE]
OUT = TRABAJO / "pred" / FASE
FACT = ["region", "channel", "age_band", "gender", "occupation"]
LAMBDAS = [1e2, 1e3, 3e3, 1e4, 3e4, 1e5, 3e5, 1e6, 3e6, 1e7]


def diseno(df, niveles=None):
    cols = [df.new_business.astype(float).values[:, None], df.self_employed.fillna(False).astype(float).values[:, None]]
    nombres = ["new_business", "self_employed"]
    niv = {} if niveles is None else niveles
    for f in FACT:
        v = df[f].astype(object).where(df[f].notna(), "NA").astype(str).values
        if niveles is None:
            niv[f] = sorted(set(v))
        for n in niv[f]:
            cols.append((v == n).astype(float)[:, None]); nombres.append(f"{f}={n}")
    return np.hstack(cols), nombres, niv


def ajustar(Z, y, off, lam):
    n, k = Z.shape

    def f(beta):
        eta = off + beta[0] + Z @ beta[1:]
        a = np.exp(0.5 * eta); b = np.exp(-0.5 * eta)  # mu^(2-p), mu^(1-p) con p=1.5
        loss = np.sum(2 * (y * 2 * b + 2 * a)) + lam * np.sum(beta[1:] ** 2)  # constantes de y omitidas
        g_eta = 2 * (-y * b + a)
        grad = np.concatenate([[g_eta.sum()], Z.T @ g_eta + 2 * lam * beta[1:]])
        return loss, grad
    b0 = np.log(max(np.sum(y / np.sqrt(np.exp(off))) / np.sum(np.sqrt(np.exp(off))), 1e-9))
    x0 = np.zeros(k + 1); x0[0] = b0
    r = minimize(f, x0, jac=True, method="L-BFGS-B", options={"maxiter": 2000})
    return r.x


def interno(cids):
    return np.array([int(hashlib.sha256(("mlind_inner|" + c).encode()).hexdigest()[:8], 16) % 5 == 0 for c in cids])


coefs = {}
for l in LINEAS:
    em = cargar("emerging", l, CONF[0])
    raw = pd.read_parquet(OUT / f"T2raw_{l}.parquet")
    em = em.merge(raw, on="row_id")
    tr = em[(em.conjunto == "train") & em.year.isin(CONF[1])].reset_index(drop=True)
    sets = [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)] if CONF[2] is None else [(tr, em[em.year == CONF[2]].reset_index(drop=True))]
    preds = []; lams = []
    for a, b in sets:
        Za, nom, niv = diseno(a); Zb, _, _ = diseno(b, niv)
        offa = a.raw.values + np.log(a.exposure.values * a.c_fila.values)
        offb = b.raw.values + np.log(b.exposure.values)
        ya = a.y.values
        v = interno(a.customer_id.values)
        best = None
        for lam in LAMBDAS:
            bt = ajustar(Za[~v], ya[~v], offa[~v], lam)
            mu = np.exp(offa[v] + bt[0] + Za[v] @ bt[1:])
            dev = desviacion(ya[v], mu).sum()
            if best is None or dev < best[0]:
                best = (dev, lam)
        lam = best[1]; lams.append(lam)
        bt = ajustar(Za, ya, offa, lam)
        preds.append(pd.DataFrame({"row_id": b.row_id.values, "mu": np.exp(offb + bt[0] + Zb @ bt[1:])}))
        coefs.setdefault(l, []).append({"lambda": lam, "b0": bt[0], **dict(zip(nom, bt[1:]))})
    pd.concat(preds).to_parquet(OUT / f"T5_{l}.parquet", index=False)
    print(l, "lambdas", lams, "b_NB", [round(c["new_business"], 3) for c in coefs[l]])
guardar_json(coefs, OUT / "T5_coeficientes.json")
'''


### Embedded source: `ml/codigo/anidado.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/anidado.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/anidado.py'] = '''"""A05 (2): evaluación OOF totalmente anidada desde los raw guardados.

Para cada pliegue externo j: el nivel k de cada tronco, los pesos de la mezcla, C1 y C2 se estiman solo con los
otros pliegues, y los pesos y factores con predicciones internas cuyo k tampoco usó el pliegue j (k_{-i,-j}).
Uso: python3 anidado.py <dev|pcv> <salida.json> <receta1> [receta2 ...]   receta = A+B+C:encoge (encoge 0..1)
"""
import sys, json
import numpy as np, pandas as pd
from comun import *
from evaluar import mezcla_pesos, metricas, bootstrap_dif
import receta

FASE, SALIDA, RECETAS = sys.argv[1], sys.argv[2], sys.argv[3:]
import os
FIJOS = {}
if os.environ.get("MLIND_PESOS_FIJOS"):  # JSON de e26_pesos_des24.py; pesos por posición
    _j = json.load(open(os.environ["MLIND_PESOS_FIJOS"]))
    FIJOS = {l: list(_j["ramos"][l]["pesos"].values()) for l in LINEAS}
K_NB = 200.0
t = receta.tabla(FASE).reset_index(drop=True)
folds = pliegue(t.customer_id.values)
y = t.y.values; e = t.exposure.values; cf = t.c_fila.values; nbv = t.new_business.values


def raw_de(c):
    """raw del tronco con el que se hicieron las predicciones de la fase (dev/hold comparten; pcv/prod comparten)."""
    sub = {"dev": ["dev", "hold"], "pcv": ["pcv", "prod"]}[FASE]
    out = []
    for l in LINEAS:
        for s in sub:
            f = TRABAJO / "pred" / s / f"{c}raw_{l}.parquet"
            if f.exists():
                out.append(pd.read_parquet(f)); break
        else:
            raise FileNotFoundError(f"{c}raw_{l} en {sub}")
    r = pd.concat(out, ignore_index=True)
    v = t[["row_id"]].merge(r, on="row_id", how="left").raw.values
    assert not np.isnan(v).any(), c
    return v


def k_nivel(rw, m):
    return factor_tweedie(y[m], np.exp(rw[m]) * e[m] * cf[m])


def predecir(cs, enc, entreno, objetivo, R):
    """Mezcla + C1 + C2 estimados con 'entreno' (máscara) y aplicados a 'objetivo'. Los pesos y factores usan
    predicciones internas OOF dentro de 'entreno' con k que no ven el resto."""
    pli = np.unique(folds[entreno])
    U_in = np.zeros((entreno.sum(), len(cs))); idx = np.where(entreno)[0]
    U_out = np.zeros((objetivo.sum(), len(cs))); ido = np.where(objetivo)[0]
    for l in LINEAS:  # el nivel k es por ramo, como en transfer2.salidas y gbm.py
        L = (t.line.values == l)
        for i in pli:
            mi = (folds[idx] == i) & L[idx]
            fit = entreno & (folds != i) & L
            for a, c in enumerate(cs):
                U_in[mi, a] = k_nivel(R[c], fit) * np.exp(R[c][idx[mi]]) * e[idx[mi]]
        mo = L[ido]
        for a, c in enumerate(cs):
            U_out[mo, a] = k_nivel(R[c], entreno & L) * np.exp(R[c][ido[mo]]) * e[ido[mo]]
    res = {}
    for l in LINEAS:
        il_in = (t.line.values[idx] == l); il_out = (t.line.values[objetivo] == l)
        M = U_in[il_in] * cf[idx][il_in][:, None]
        if enc < 0:  # E26: pesos fijos estimados fuera (DES24), en el orden de los componentes
            w = np.array(FIJOS[l])
        else:
            w = mezcla_pesos(y[idx][il_in], M) if len(cs) > 1 else np.ones(1)
            w = (1 - enc) * w + enc / len(cs)
        m_in = M @ w; yy = y[idx][il_in]
        k1 = factor_tweedie(yy, m_in)
        base_out = U_out[il_out] @ w
        f2 = np.zeros(il_out.sum())
        for nb in (True, False):
            s = nbv[idx][il_in] == nb
            ks = factor_tweedie(yy[s], m_in[s] * k1); n = int((yy[s] > 0).sum()); z = n / (n + K_NB)
            f2[nbv[objetivo][il_out] == nb] = k1 * (z * ks + 1 - z)
        res[l] = (il_out, base_out, k1, f2, w)
    out = {"sin": np.zeros(objetivo.sum()), "C1": np.zeros(objetivo.sum()), "C2": np.zeros(objetivo.sum())}
    for l, (il, b, k1, f2, w) in res.items():
        out["sin"][il] = b; out["C1"][il] = b * k1; out["C2"][il] = b * f2
    return out


def main():
    recs = []
    for r in RECETAS:
        cs, enc = r.split(":") if ":" in r else (r, "0")
        recs.append((cs.split("+"), -1.0 if enc == "fijo" else float(enc)))
    todos = sorted({c for cs, _ in recs for c in cs} | {"T2b"})
    R = {c: raw_de(c) for c in todos}
    if ("T2b", 0.0) not in [(tuple(cs)[0] if len(cs) == 1 else None, enc) for cs, enc in recs]:
        recs = [(["T2b"], 0.0)] + recs
    preds = {}
    for cs, enc in recs:
        nom = "+".join(cs) + ("_des24" if enc < 0 else (f"_e{int(enc * 100)}" if len(cs) > 1 else ""))
        P = {k: np.zeros(len(t)) for k in ("sin", "C1", "C2")}
        for j in np.unique(folds):
            ob = folds == j
            o = predecir(cs, enc, ~ob, ob, R)
            for k in P:
                P[k][ob] = o[k]
        preds[nom] = P
        print(nom, {k: round(metricas(t, v)["D2"], 4) for k, v in P.items()}, flush=True)
    res = {"fase": FASE, "metricas": {n: {k: metricas(t, v) for k, v in P.items()} for n, P in preds.items()}, "bootstrap_vs_T2b": {}}
    for n, P in preds.items():
        if n == "T2b":
            continue
        res["bootstrap_vs_T2b"][n] = {k: bootstrap_dif(t, P[k], preds["T2b"][k]) for k in ("C1", "C2")}
        if FASE == "pcv":
            for nb in (True, False):
                s = nbv == nb
                res["bootstrap_vs_T2b"][n]["C1_" + ("NB" if nb else "REN")] = bootstrap_dif(
                    t[s].reset_index(drop=True), P["C1"][s], preds["T2b"]["C1"][s])
        print(n, {k: (round(v["dif"], 4), [round(x, 4) for x in v["ic95"]]) for k, v in res["bootstrap_vs_T2b"][n].items()}, flush=True)
    guardar_json(res, ENTREGA / "evolucion" / SALIDA)


if __name__ == "__main__":
    main()
'''


### Embedded source: `ml/codigo/auditoria_codex.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/auditoria_codex.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/auditoria_codex.py'] = '''"""Respuesta medida a la auditoría de Codex (S2-AUD-MLIND-C0-SCOPE y COD-S2-VAL07-20261024-MLIND-AGRUPACION).

(a) c_fila con pesos por peligro solo de 2023 (sin etiquetas de 2024) frente a la versión exportada (pesos 2023-2024).
(b) pliegues por componente cliente-dirección (unión-búsqueda) para el T2b de DEV, frente a pliegues por cliente.
"""
import hashlib, json
import numpy as np, pandas as pd
from comun import *
from completitud import por_peligro
from evaluar import metricas
import receta

res = {"a_c_fila": {}, "b_agrupacion": {}}
PEL = {"motor": ["bodily_injury", "glass_hail", "own_damage", "theft", "third_party_prop"],
       "buildings": ["accidental_damage", "escape_of_water", "fire", "flood", "storm", "subsidence"],
       "contents": ["accidental_damage", "escape_of_water", "flood", "theft"]}
nuevas = {}
for l in LINEAS:
    b = pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet")
    tr = b[b.conjunto == "train"]
    comp = {p: por_peligro(l, p, b.period_start, b.period_end) for p in PEL[l]}
    def pesos(mask):
        ult = {p: tr.loc[mask, f"y_{p}"].sum() / np.average(comp[p][(b.conjunto == "train").values][mask.values], weights=tr.exposure[mask]) for p in PEL[l]}
        s = sum(ult.values()); return {p: v / s for p, v in ult.items()}
    w23 = pesos(tr.year == 2023)
    c23 = sum(w23[p] * comp[p] for p in PEL[l])
    c_old = b.c_fila.values
    r = np.where(b.conjunto == "test", 1.0, c23 / c_old)
    res["a_c_fila"][l] = {"pesos_2023": {k: round(v, 4) for k, v in w23.items()},
                          "cambio_relativo_max_2023": float(np.abs(r[(b.year == 2023).values] - 1).max()),
                          "cambio_relativo_max_2024": float(np.abs(r[(b.year == 2024).values] - 1).max()),
                          "cambio_medio_2024": float(np.average(r[(b.year == 2024).values], weights=b.exposure[(b.year == 2024).values]) - 1)}
    nuevas[l] = pd.DataFrame({"row_id": b.row_id, "c_nueva": np.where(b.conjunto == "test", 1.0, c23)})
# efecto en las métricas del modelo final
def con_c(t):
    c = pd.concat(nuevas.values()); t2 = t.merge(c, on="row_id", how="left"); t2["c_fila"] = t2.c_nueva; return t2
t, mu, _ = receta.oof_cruzado("dev", ["T2b"], "sin")
res["a_c_fila"]["DEV_T2b_sin"] = {"exportada": metricas(t, mu)["D2"], "pesos_2023": metricas(con_c(t), mu)["D2"]}
p = receta.parametros("dev", ["T2b"], "ramo")
th, mh = receta.aplicar("hold", p, None)
res["a_c_fila"]["HOLD_T2b_C1"] = {"exportada": metricas(th, mh)["D2"], "pesos_2023_solo_en_evaluacion": metricas(con_c(th), mh)["D2"]}
# (b) unión-búsqueda cliente-dirección en el emergente 2023
filas = []
for l in LINEAS:
    b = pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet", columns=["row_id", "customer_id", "address_id", "year", "conjunto", "fold"])
    filas.append(b[(b.conjunto == "train") & (b.year == 2023)].assign(line=l))
e = pd.concat(filas, ignore_index=True)
padre = {}
def f(x):
    while padre.setdefault(x, x) != x:
        padre[x] = padre[padre[x]]; x = padre[x]
    return x
for c, a in zip("C" + e.customer_id, "A" + e.address_id):
    rc, ra = f(c), f(a)
    if rc != ra: padre[ra] = rc
comp_id = np.array([f("C" + c) for c in e.customer_id])
e["fold_ca"] = [int(hashlib.sha256(("mlind_ca|" + x).encode()).hexdigest()[:16], 16) % 5 for x in comp_id]
cruce = e.groupby("address_id").fold.nunique()
res["b_agrupacion"]["direcciones_en_mas_de_un_pliegue_por_cliente"] = int((cruce > 1).sum())
res["b_agrupacion"]["componentes"] = int(len(set(comp_id)))
# T2b DEV: el único parámetro emergente es el nivel k2 por ramo; se rehace con los pliegues nuevos
raw = pd.concat([pd.read_parquet(TRABAJO / "pred" / "dev" / f"T2braw_{l}.parquet") for l in LINEAS])
bb = pd.concat([pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet", columns=["row_id", "exposure", "c_fila", "y"]) for l in LINEAS])
e = e.merge(raw, on="row_id").merge(bb, on="row_id")
mu = np.zeros(len(e))
for k in range(5):
    for l in LINEAS:
        a = ((e.fold_ca != k) & (e.line == l)).values; bm = ((e.fold_ca == k) & (e.line == l)).values
        k2 = factor_tweedie(e.y.values[a], np.exp(e.raw.values[a]) * e.exposure.values[a] * e.c_fila.values[a])
        mu[bm] = k2 * np.exp(e.raw.values[bm]) * e.exposure.values[bm]
tt = receta.tabla("dev")
mu_al = tt[["row_id"]].merge(pd.DataFrame({"row_id": e.row_id, "mu": mu}), on="row_id").mu.values
res["b_agrupacion"]["DEV_T2b_sin"] = {"pliegues_cliente": metricas(t, receta.componentes("dev", ["T2b"], t)["T2b"])["D2"],
                                       "pliegues_cliente_direccion": metricas(tt, mu_al)["D2"]}
guardar_json(res, ENTREGA / "respuesta_auditoria_codex.json")
print(json.dumps(res, indent=1, default=float))
'''


### Embedded source: `ml/codigo/bootstrap_final.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/bootstrap_final.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/bootstrap_final.py'] = '''"""Intervalos por remuestreo de clientes de las comparaciones que cita el informe."""
import json
import numpy as np, pandas as pd
from comun import *
from evaluar import bootstrap_dif
import receta

res = {}
t = receta.tabla("dev")
m = {}
for c, cal in [("T2b", "ramo"), ("R0", "ramo"), ("T1wp", "ramo"), ("G1", "ramo"), ("N1", "ramo_nb"), ("N2a", "ramo_nb"), ("T2b", "ramo_nb"), ("N3", "ramo_nb")]:
    _, mu, _ = receta.oof_cruzado("dev", [c], cal); m[(c, cal)] = mu
res["DEV_T2b_C1-R0_C1"] = bootstrap_dif(t, m[("T2b", "ramo")], m[("R0", "ramo")])
res["DEV_T2b_C1-T1wp_C1"] = bootstrap_dif(t, m[("T2b", "ramo")], m[("T1wp", "ramo")])
res["DEV_T2b_C1-G1_C1"] = bootstrap_dif(t, m[("T2b", "ramo")], m[("G1", "ramo")])
res["DEV_N2a_C2-N1_C2"] = bootstrap_dif(t, m[("N2a", "ramo_nb")], m[("N1", "ramo_nb")])
res["DEV_N3_C2-T2b_C2"] = bootstrap_dif(t, m[("N3", "ramo_nb")], m[("T2b", "ramo_nb")])
lista = json.load(open(ENTREGA / "lista_congelada.json"))
th = None; hm = {}
for nom in ["T2b_C1", "R0_C1", "T1wp_C2", "G1_C2", "N1_C2", "N2a_C2", "N3_C2", "T2b_C2"]:
    spec = lista["candidatos"][nom]
    p = receta.parametros("dev", spec["componentes"], spec["calib"])
    th, hm[nom] = receta.aplicar("hold", p, None)
res["HOLD_T2b_C1-R0_C1"] = bootstrap_dif(th, hm["T2b_C1"], hm["R0_C1"])
res["HOLD_T2b_C1-T1wp_C2"] = bootstrap_dif(th, hm["T2b_C1"], hm["T1wp_C2"])
res["HOLD_N2a_C2-N1_C2"] = bootstrap_dif(th, hm["N2a_C2"], hm["N1_C2"])
res["HOLD_N3_C2-T2b_C2"] = bootstrap_dif(th, hm["N3_C2"], hm["T2b_C2"])
res["HOLD_T2b_C2-T2b_C1"] = bootstrap_dif(th, hm["T2b_C2"], hm["T2b_C1"])
guardar_json(res, ENTREGA / "bootstrap_final.json")
for k, v in res.items():
    print(k, round(v["dif"], 4), [round(x, 4) for x in v["ic95"]], v["p_mayor"])
'''


### Embedded source: `ml/codigo/calibracion.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/calibracion.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/calibracion.py'] = '''"""Diagnóstico de calibración: descomposición de Murphy de la desviación Tweedie 1,5 con recalibración isotónica,
A/E por deciles y por segmentos. Uso: python calibracion.py fase (dev o pcv) receta.json"""
import sys, json
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression
from comun import *
import receta

plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 130})
C = {"motor": "#2a6f97", "buildings": "#c4673b", "contents": "#5b8c5a"}


def murphy(y, m, w):
    """Puntuación media S, incertidumbre UNC, discriminación DSC y descalibración MCB (S = UNC - DSC + MCB).
    Recalibración isotónica de y sobre m (tasa por exposición, con pesos de exposición)."""
    e = w
    r = m / e
    # para Tweedie 1,5 sobre importes, el bloque óptimo es la media de tasas con pesos e^0,5 (= e^(2-p))
    iso = IsotonicRegression(out_of_bounds="clip").fit(r, y / e, sample_weight=np.sqrt(e))
    mc = np.maximum(iso.predict(r) * e, 1e-9)
    ybar = np.sum(y / np.sqrt(e)) / np.sum(np.sqrt(e)) * e  # tasa constante óptima
    S = desviacion(y, m).mean(); Sc = desviacion(y, mc).mean(); Su = desviacion(y, ybar).mean()
    return {"S": S, "UNC": Su, "DSC": Su - Sc, "MCB": S - Sc, "D2": 1 - S / Su, "D2_recal": 1 - Sc / Su}


if __name__ == "__main__":
    fase = sys.argv[1]; spec = json.load(open(sys.argv[2]))
    t, mu, _ = receta.oof_cruzado(fase, spec["componentes"], spec["calib"])
    m = mu * t.c_fila.values; y = t.y.values
    out = {"murphy": {}, "deciles": {}, "segmentos": {}}
    fig, axs = plt.subplots(1, 3, figsize=(10, 3.2))
    for ax, l in zip(axs, LINEAS):
        i = (t.line == l).values
        out["murphy"][l] = murphy(y[i], m[i], t.exposure.values[i] * t.c_fila.values[i])
        r = m[i] / (t.exposure.values[i] * t.c_fila.values[i])
        dec = pd.qcut(r, 10, labels=False, duplicates="drop")
        g = pd.DataFrame({"d": dec, "y": y[i], "m": m[i], "e": t.exposure.values[i] * t.c_fila.values[i]}).groupby("d").sum()
        g["obs"] = g.y / g.e; g["pred"] = g.m / g.e; g["AE"] = g.y / g.m
        out["deciles"][l] = g[["obs", "pred", "AE"]].round(4).to_dict()
        ax.plot(g.pred, g.obs, "o-", color=C[l], ms=4, label="observado")
        lim = [0, max(g.pred.max(), g.obs.max()) * 1.05]
        ax.plot(lim, lim, color="#999", lw=0.8, ls="--", label="diagonal")
        ax.set_title(f"{l}  (MCB/S = {out['murphy'][l]['MCB'] / out['murphy'][l]['S']:.2%})")
        ax.set_xlabel("tasa predicha (CU por año)"); ax.legend(frameon=False, fontsize=7)
    axs[0].set_ylabel("tasa observada (CU por año)")
    fig.suptitle(f"Calibración por deciles de la predicción, OOF {fase}", fontsize=10)
    fig.tight_layout(); fig.savefig(ENTREGA / "figuras" / f"f05_calibracion_deciles_{fase}.png"); plt.close(fig)
    # diagnóstico de forma: exponente a de las relatividades transferidas (a = 1 si se trasladan bien)
    out["forma"] = {}
    for l in LINEAS:
        i = (t.line == l).values; e = t.exposure.values[i] * t.c_fila.values[i]; r = m[i] / e
        g = np.exp(np.average(np.log(r), weights=e))
        mejor = None; curva = {}
        for a in np.round(np.arange(0.6, 1.41, 0.05), 2):
            ma = e * g * (r / g) ** a
            ma *= factor_tweedie(y[i], ma)
            dv_ = desviacion(y[i], ma).sum(); curva[float(a)] = dv_
            if mejor is None or dv_ < mejor[1]:
                mejor = (float(a), dv_)
        base = curva[1.0]; d0 = desviacion(y[i], np.full(i.sum(), y[i].mean())).sum()
        out["forma"][l] = {"a_optimo": mejor[0], "ganancia_D2_en_muestra": (base - mejor[1]) / d0}
    for seg in ["new_business", "year"]:
        g = pd.DataFrame({"line": t.line, "s": t[seg], "y": y, "m": m}).groupby(["line", "s"]).sum()
        g["AE"] = g.y / g.m
        out["segmentos"][seg] = {f"{a}|{b}": round(float(v), 4) for (a, b), v in g.AE.items()}
    guardar_json(out, ENTREGA / f"calibracion_{fase}.json")
    print(json.dumps(out["murphy"], indent=1, default=float)); print(out["segmentos"]); print(out["forma"])
'''


### Embedded source: `ml/codigo/comparar_lineas.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/comparar_lineas.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/comparar_lineas.py'] = '''"""Comparación en base común de la línea ML independiente con las otras líneas (2024 y producción 2025).

2024: mismas 59.202 filas y mismo T0; cada modelo trae su propia completitud (predicción en escala observada).
Remuestreo por componente cliente-dirección (contrato común). 2025: totales, tasas, correlación fila a fila
y distancia esperada en D2 entre predicciones (divergencia de Bregman de Tweedie 1,5).
"""
import json, hashlib
import numpy as np, pandas as pd
from comun import *
from evaluar import tabla_eval
import receta

OTROS = RAIZ / "comparacion" / "otros"
res = {"fuentes": {}, "2024": {}, "2025": {}}
for f in sorted(OTROS.iterdir()):
    res["fuentes"][f.name] = sha256(f)

# ---------------- 2024
o = pd.read_parquet(OTROS / "predicciones_2024.parquet")
lista = json.load(open(ENTREGA / "lista_congelada.json"))
p = receta.parametros("dev", ["T2b"], "ramo")
t, mu_c1 = receta.aplicar("hold", p, None)
p2 = receta.parametros("dev", ["T2b"], "ramo_nb"); _, mu_c2 = receta.aplicar("hold", p2, None)
p3 = receta.parametros("dev", ["R0"], "ramo"); _, mu_r0 = receta.aplicar("hold", p3, None)
d = t.merge(o[["row_id", "address_id", "R0", "A1", "A2", "A3"]], on="row_id", how="inner")
assert len(d) == len(t) == len(o) == 59202
d["MLind_T2b_C1"] = mu_c1 * t.c_fila.values
d["MLind_T2b_C2"] = mu_c2 * t.c_fila.values
d["MLind_R0"] = mu_r0 * t.c_fila.values
d = d.rename(columns={"R0": "noct_R0", "A1": "noct_A1", "A2": "noct_A2", "A3": "noct_A3"})
mods = ["MLind_T2b_C1", "MLind_T2b_C2", "MLind_R0", "noct_R0", "noct_A1", "noct_A2", "noct_A3"]
y = d.y.values
# componentes cliente-dirección
padre = {}
def f(x):
    while padre.setdefault(x, x) != x:
        padre[x] = padre[padre[x]]; x = padre[x]
    return x
for c, a in zip("C" + d.customer_id, "A" + d.address_id):
    rc, ra = f(c), f(a)
    if rc != ra: padre[ra] = rc
comp = np.array([f("C" + c) for c in d.customer_id])
_, gi = np.unique(comp, return_inverse=True)
res["2024"]["filas"] = int(len(d)); res["2024"]["componentes"] = int(gi.max() + 1)
for m in mods:
    r = {"D2": d2(y, d[m].values)}
    for l in LINEAS:
        i = (d.line == l).values
        r[f"D2_{l}"] = d2(y[i], d[m].values[i]); r[f"AE_{l}"] = float(y[i].sum() / d[m].values[i].sum())
    for nb in (True, False):
        i = d.new_business.values == nb
        r[f"D2_{'NB' if nb else 'REN'}"] = d2(y[i], d[m].values[i])
    res["2024"][m] = r
rng = np.random.default_rng(20261024)
dev = {m: desviacion(y, d[m].values) for m in mods}
sy = np.sqrt(y); B = 2000
pares = [("MLind_T2b_C1", "noct_A1"), ("MLind_T2b_C1", "noct_R0"), ("MLind_T2b_C2", "noct_A1"), ("noct_A1", "noct_R0"), ("MLind_T2b_C1", "noct_A2")]
boot = {k: [] for k in pares}; boot33 = {k: [] for k in pares}
G = gi.max() + 1
for s in range(0, B, 100):
    W = rng.poisson(1.0, size=(100, G)).astype(float)[:, gi]
    # submuestra del 33 % de componentes (escala del tablero público)
    W33 = (rng.random(size=(100, G)) < 0.33).astype(float)[:, gi]
    for WW, store in ((W, boot), (W33, boot33)):
        ybar = (WW @ y) / WW.sum(1)
        dn = 2 * ((WW @ sy) / (-0.25) - (WW @ y) * ybar ** -0.5 / (-0.5) + WW.sum(1) * ybar ** 0.5 / 0.5)
        for a, b in pares:
            store[(a, b)].append((WW @ dev[b] - WW @ dev[a]) / dn)
res["2024"]["pares"] = {}
for k in pares:
    r = np.concatenate(boot[k]); r33 = np.concatenate(boot33[k])
    res["2024"]["pares"][f"{k[0]} - {k[1]}"] = {"dif": res["2024"][k[0]]["D2"] - res["2024"][k[1]]["D2"],
        "ic95": [float(np.quantile(r, .025)), float(np.quantile(r, .975))], "p_mayor": float((r > 0).mean()),
        "sd_submuestra_33": float(np.std(r33)), "p_mayor_submuestra_33": float((r33 > 0).mean())}
# correlación 2024 entre modelos
res["2024"]["corr_log_T2b_A1"] = float(np.corrcoef(np.log(d.MLind_T2b_C1), np.log(d.noct_A1))[0, 1])

# ---------------- 2025
sub = pd.read_parquet(DATOS / "emerging" / "sample_submission.parquet")[["row_id"]]
base = pd.concat([pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet", columns=["row_id", "exposure", "new_business", "conjunto"]).assign(line=l)
                  for l in LINEAS])
base = base[base.conjunto == "test"]
q = sub.merge(base, on="row_id", how="left")
arch = {"MLind_final": PRIVADO / "prediccion_2025_mlind.csv", "MLind_C2": PRIVADO / "prediccion_2025_mlind_C2.csv",
        "MLind_C1_tendencia": PRIVADO / "prediccion_2025_mlind_C1_tendencia.csv",
        "noct_A1fixF1b3n_designada": OTROS / "A1fixF1b3n_2025.csv", "noct_A1": OTROS / "A1_2025.csv", "noct_R0": OTROS / "R0_2025.csv",
        "noct_A1fixTc": OTROS / "A1fixTc_2025.csv", "noct_A1fixF1nTc": OTROS / "A1fixF1nTc_2025.csv"}
for k, fpath in arch.items():
    x = pd.read_csv(fpath, dtype={"row_id": str})
    assert (x.row_id.values == sub.row_id.values).all() and x.prediction.notna().all()
    q[k] = x.prediction.values
for k in arch:
    r = {"total": float(q[k].sum())}
    for l in LINEAS:
        i = (q.line == l).values; e = q.exposure.values[i]; nb = q.new_business.values[i]
        r[l] = {"total": float(q[k].values[i].sum()), "tasa": float(q[k].values[i].sum() / e.sum()),
                "tasa_NB": float(q[k].values[i][nb].sum() / e[nb].sum()), "tasa_REN": float(q[k].values[i][~nb].sum() / e[~nb].sum())}
    res["2025"][k] = r
# distancia entre la entrega propia y la designada
a = q.MLind_final.values; b = q.noct_A1fixF1b3n_designada.values
res["2025"]["final_vs_designada"] = {
    "corr_log": float(np.corrcoef(np.log(a), np.log(b))[0, 1]),
    "spearman": float(pd.Series(a).rank().corr(pd.Series(b).rank())),
    "cociente_p01_p50_p99": [float(x) for x in np.quantile(a / b, [0.01, 0.5, 0.99])],
    "por_ramo": {}}
# divergencia de Bregman media frente a la desviación nula esperada (escala de 2024, por año de exposición)
null24 = {}
for l in LINEAS:
    i = (d.line == l).values
    yl = y[i]; ybar = yl.mean()
    null24[l] = float(desviacion(yl, np.full(i.sum(), ybar)).sum() / t.exposure.values[i].sum())
nulo25 = sum(null24[l] * q.exposure.values[(q.line == l).values].sum() for l in LINEAS)
def breg(mt, mo):  # E_y~mt[d(y,mo) - d(y,mt)] = d(mt, mo)
    return desviacion(mt, mo)
res["2025"]["final_vs_designada"]["gap_D2_si_la_verdad_es_designada"] = float(breg(b, a).sum() / nulo25)
res["2025"]["final_vs_designada"]["gap_D2_si_la_verdad_es_final"] = float(breg(a, b).sum() / nulo25)
for l in LINEAS:
    i = (q.line == l).values
    res["2025"]["final_vs_designada"]["por_ramo"][l] = {"corr_log": float(np.corrcoef(np.log(a[i]), np.log(b[i]))[0, 1]),
                                                       "cociente_total": float(a[i].sum() / b[i].sum())}
# parte de la distancia que es solo nivel por ramo
a_niv = a.copy()
for l in LINEAS:
    i = (q.line == l).values; a_niv[i] = a[i] * b[i].sum() / a[i].sum()
res["2025"]["final_vs_designada"]["gap_D2_misma_forma_nivel_designada"] = float(breg(b, a_niv).sum() / nulo25)
guardar_json(res, RAIZ / "comparacion" / "comparacion_lineas.json")
print(json.dumps({k: v for k, v in res.items() if k != "fuentes"}, indent=1, default=float)[:9000])
'''


### Embedded source: `ml/codigo/comparar_lineas_2.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/comparar_lineas_2.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/comparar_lineas_2.py'] = '''"""Complemento: 2024 por segmento de nuevo negocio, ponderado a la mezcla de 2025 y mezcla 50/50 con A1 (exploratorio)."""
import json
import numpy as np, pandas as pd
from comun import *
import receta

OTROS = RAIZ / "comparacion" / "otros"
o = pd.read_parquet(OTROS / "predicciones_2024.parquet")
p = receta.parametros("dev", ["T2b"], "ramo"); t, mu = receta.aplicar("hold", p, None)
d = t.merge(o[["row_id", "address_id", "A1", "R0"]], on="row_id").merge(pd.read_parquet(TRABAJO / "pesos_2025_hold.parquet"), on="row_id", how="left")
d["T2b"] = mu * t.c_fila.values
for w in (0.25, 0.5, 0.75):
    d[f"mez{int(w*100)}"] = w * d.T2b + (1 - w) * d.A1
y = d.y.values
padre = {}
def f(x):
    while padre.setdefault(x, x) != x:
        padre[x] = padre[padre[x]]; x = padre[x]
    return x
for c, a in zip("C" + d.customer_id, "A" + d.address_id):
    rc, ra = f(c), f(a)
    if rc != ra: padre[ra] = rc
_, gi = np.unique([f("C" + c) for c in d.customer_id], return_inverse=True)
G = gi.max() + 1; rng = np.random.default_rng(7)
def boot(a, b, mask=None, w=None, B=2000):
    m = np.ones(len(y), bool) if mask is None else mask
    ww = np.ones(len(y)) if w is None else w
    da = desviacion(y, d[a].values); db = desviacion(y, d[b].values); sy = np.sqrt(y); out = []
    for s in range(0, B, 100):
        W = rng.poisson(1.0, size=(100, G)).astype(float)[:, gi] * ww * m
        ybar = (W @ y) / W.sum(1)
        dn = 2 * ((W @ sy) / (-0.25) - (W @ y) * ybar ** -0.5 / (-0.5) + W.sum(1) * ybar ** 0.5 / 0.5)
        out.append((W @ db - W @ da) / dn)
    r = np.concatenate(out)
    pt = d2(y[m], d[a].values[m], ww[m]) - d2(y[m], d[b].values[m], ww[m])
    return {"dif": float(pt), "ic95": [float(np.quantile(r, .025)), float(np.quantile(r, .975))], "p_mayor": float((r > 0).mean())}
res = {"D2": {}, "pares": {}}
nb = d.new_business.values; w25 = d.w2025.values
for m in ["T2b", "A1", "mez25", "mez50", "mez75", "R0"]:
    res["D2"][m] = {"todas": d2(y, d[m].values), "NB": d2(y[nb], d[m].values[nb]), "REN": d2(y[~nb], d[m].values[~nb]),
                    "pond_2025": d2(y, d[m].values, w25)}
res["pares"]["T2b-A1_NB"] = boot("T2b", "A1", mask=nb)
res["pares"]["T2b-A1_REN"] = boot("T2b", "A1", mask=~nb)
res["pares"]["T2b-A1_pond2025"] = boot("T2b", "A1", w=w25)
res["pares"]["mez50-A1"] = boot("mez50", "A1")
res["pares"]["mez50-T2b"] = boot("mez50", "T2b")
res["pares"]["mez50-A1_pond2025"] = boot("mez50", "A1", w=w25)
res["NB_filas_2024"] = int(nb.sum()); res["NB_siniestros_2024"] = int((y[nb] > 0).sum())
# nivel por ramo en NB de 2024: cociente observado/predicho
res["AE_NB"] = {m: {l: float(y[nb & (d.line == l).values].sum() / d[m].values[nb & (d.line == l).values].sum()) for l in LINEAS} for m in ["T2b", "A1"]}
guardar_json(res, RAIZ / "comparacion" / "comparacion_lineas_2.json")
print(json.dumps(res, indent=1))
'''


### Embedded source: `ml/codigo/completitud.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/completitud.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/completitud.py'] = '''"""Completitud de reporte ponderada por coste por ramo y peligro (misma estimación que preparar_datos.py)."""
import numpy as np, pandas as pd
from comun import *

D = 1095
_G = None


def _estimar():
    c = pd.read_parquet(DATOS / "developed" / "claims.parquet")
    cols = [x for x in c.columns if x != "claim_id"]
    c = c[~(c.duplicated(cols, keep=False) & (c.claim_id.str.len() == 11))]
    c = c[c.status != "repudiated"].copy()
    c["loss_date"] = pd.to_datetime(c.loss_date); c["report_date"] = pd.to_datetime(c.report_date)
    c["neto"] = c.incurred_to_date - c.recoveries
    G = {}
    for (l, p), g in c.groupby(["line", "peril"]):
        g = g[(V - g.loss_date.values.astype("datetime64[D]")).astype(int) >= D]
        dly = (g.report_date - g.loss_date).dt.days.values; w = g.neto.values; ok = dly <= D
        cum = np.clip(np.bincount(dly[ok], weights=w[ok], minlength=D + 1).cumsum() / w[ok].sum(), 0, 1)
        G[(l, p)] = np.concatenate([[0.0], np.cumsum(np.concatenate([cum, np.ones(4000 - len(cum))]))])
    return G


def por_peligro(linea, peligro, inicio, fin):
    global _G
    if _G is None:
        _G = _estimar()
    g = _G[(linea, peligro)]
    s = pd.to_datetime(inicio).values.astype("datetime64[D]"); e = pd.to_datetime(fin).values.astype("datetime64[D]")
    a = np.clip((V - e).astype(int), 0, None); b = np.clip((V - s).astype(int), 0, None)
    return (g[b + 1] - g[a]) / (b - a + 1)
'''


### Embedded source: `ml/codigo/comun.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/comun.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/comun.py'] = '''"""Utilidades comunes de la línea ML independiente (semana 2).

Rutas, lectura de datos con comprobación de huellas, métrica Tweedie y pliegues.
"""
from __future__ import annotations
import hashlib, json, os
from pathlib import Path
import numpy as np

RAIZ = Path(os.environ.get("MLIND_RAIZ", "/home/claude/ml"))
DATOS = RAIZ / "datos"
TRABAJO = RAIZ / "trabajo"
ENTREGA = RAIZ / "entrega"
PRIVADO = RAIZ / "privado"
V = np.datetime64("2024-12-31")
LINEAS = ["motor", "buildings", "contents"]
P = 1.5

HUELLAS = {
    "emerging/addresses.parquet": "4e9bf9d2a1777896e84fd4ee471096ff075aaefefd544f91b731324b25425632",
    "emerging/claims.parquet": "1a2fce8a26cb1e05de1f53fe4e39bde772a1898e845755487625355fec4cc0e9",
    "emerging/customers.parquet": "c00a0095d5fb7b2b65cd9647216add49afb3703d863d5c358b1441f0fab1fb62",
    "emerging/expenses.parquet": "9033f666a5a45f0b44501d57e40ab1cc571c51c384f23a62942b75db323b1cdf",
    "emerging/policies_buildings.parquet": "ccc948fafc11e623d5cb010c2b494f6813f6048719eae7bcde28794467598319",
    "emerging/policies_contents.parquet": "84d7f48d03933b018bec8fdc2f4bbd96df184cca7ae1971c38d3acc955720f8e",
    "emerging/policies_motor.parquet": "7c5dbb5e5b46629ebf01a66c991183d20b8585393b589bc2644fcbd6653a3ec1",
    "emerging/reference_policies.parquet": "6d4c2839e87d092fccd4b3c3ddc16a2a4c4993c35e18481e322462cd382dd8f4",
    "emerging/sample_submission.parquet": "160fc59043f1012337605c8d8446c0efbf81458e046cf02df9186d6f909d59ef",
    "emerging/test_buildings.parquet": "21f91aaa66a1b9efd216ad0e04ff12a507dbeb721084330d9b9f06c99edd0ca6",
    "emerging/test_contents.parquet": "af5ef2553e35a2d563ca091b173500daac55acc4969d1d3a96fa6b4ec3c6a012",
    "emerging/test_motor.parquet": "108d5c68e9bab5ccd90ee5502bc3543aa515a5aa07fda44a3b227cf070a28aa5",
    "emerging/weather.parquet": "645067069f68de8c4746d1ba48a454ff00155e1aa6fbbf64cdee48ae7269cbbd",
    "developed/addresses.parquet": "2f1a9a157649a73ed1b0ee9b2caa3660b4c2a11766af3c64ab5131389079c0b1",
    "developed/claims.parquet": "323adf2e0f6c5975a026479d725d4cd04f51fce3aa069f897a99f664c6e392c0",
    "developed/customers.parquet": "065458a0b3aa9d3499e80b3b25b76baf0ef6245146477b6861f6ea35edf581eb",
    "developed/policies_buildings.parquet": "eb84372c23c16ed0619d89c8eb1396ab6916e5ec46a5907ce42c3dfce4a91cbe",
    "developed/policies_contents.parquet": "8c067cc80baed583100df999e1c3d5e389295702f7bac21d5331bbd639b83fcd",
    "developed/policies_motor.parquet": "4d2d7ae03971d5d6df3a71bbaf0ebc168250a6807b27d1561d38b638434dc95e",
    "developed/weather.parquet": "b8407f0f3c7279a49297c969a4ab2905641b232910e316fb806730065987cd49",
}


def sha256(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def comprobar_huellas() -> dict:
    out = {}
    for rel, esperado in HUELLAS.items():
        obs = sha256(DATOS / rel)
        if obs != esperado:
            raise SystemExit(f"Huella distinta en {rel}: {obs}")
        out[rel] = obs
    return out


def pliegue(customer_ids) -> np.ndarray:
    return np.array([int(hashlib.sha256(("mlind_v1|" + c).encode()).hexdigest()[:16], 16) % 5
                     for c in customer_ids], dtype=np.int8)


def desviacion(y, mu, w=None, p=P):
    y = np.asarray(y, float); mu = np.asarray(mu, float)
    d = 2 * (np.power(np.maximum(y, 0), 2 - p) / ((1 - p) * (2 - p)) - y * np.power(mu, 1 - p) / (1 - p)
             + np.power(mu, 2 - p) / (2 - p))
    return d if w is None else d * w


def d2(y, mu, w=None, p=P) -> float:
    y = np.asarray(y, float); mu = np.asarray(mu, float)
    w = np.ones_like(y) if w is None else np.asarray(w, float)
    ybar = np.sum(w * y) / np.sum(w)
    return 1 - np.sum(desviacion(y, mu, w, p)) / np.sum(desviacion(y, np.full_like(y, ybar), w, p))


def factor_tweedie(y, mu, w=None):
    """Factor k que minimiza la desviación Tweedie 1,5 de k*mu."""
    w = np.ones_like(np.asarray(y, float)) if w is None else w
    return float(np.sum(w * y / np.sqrt(mu)) / np.sum(w * np.sqrt(mu)))


def guardar_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1, default=float)
'''


### Embedded source: `ml/codigo/conteos_peligro.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/conteos_peligro.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/conteos_peligro.py'] = '''"""Conteo de siniestros aceptados por peligro y fila (misma asignación que preparar_datos.py).

Salida: trabajo/conteos_{mercado}_{ramo}.parquet con row_id y n_{peligro}. Comprueba que la suma
por peligro reproduce n_sin y que el coste por peligro reproduce y_{peligro}.
"""
import numpy as np, pandas as pd
from comun import *

for m in ["developed", "emerging"]:
    c = pd.read_parquet(DATOS / m / "claims.parquet")
    if m == "developed":
        cols = [x for x in c.columns if x != "claim_id"]
        c = c[~(c.duplicated(cols, keep=False) & (c.claim_id.str.len() == 11))]
    c = c[c.status != "repudiated"].copy()
    c["loss_date"] = pd.to_datetime(c.loss_date)
    c["neto"] = c.incurred_to_date - c.recoveries
    for l in LINEAS:
        b = pd.read_parquet(TRABAJO / f"base_{m}_{l}.parquet")
        mm = c[c.line == l].merge(b[["row_id", "policy_id", "period_start", "period_end"]], on="policy_id")
        mm = mm[(mm.loss_date >= pd.to_datetime(mm.period_start)) & (mm.loss_date <= pd.to_datetime(mm.period_end))]
        out = pd.DataFrame({"row_id": b.row_id.values})
        perils = sorted(c[c.line == l].peril.unique())
        for p in perils:
            g = mm[mm.peril == p]
            out[f"n_{p}"] = out.row_id.map(g.groupby("row_id").size()).fillna(0).astype(int)
            cost = out.row_id.map(g.groupby("row_id").neto.sum()).fillna(0.0)
            tr = (b.conjunto == "train").values
            assert np.allclose(cost.values[tr], b[f"y_{p}"].values[tr]), (m, l, p)
        tr = (b.conjunto == "train").values
        assert (out[[f"n_{p}" for p in perils]].sum(1).values[tr] == b.n_sin.values[tr]).all(), (m, l)
        out.to_parquet(TRABAJO / f"conteos_{m}_{l}.parquet", index=False)
        print(m, l, {p: int(out[f"n_{p}"].values[tr].sum()) for p in perils}, flush=True)
'''


### Embedded source: `ml/codigo/credibilidad_entre_anios.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/credibilidad_entre_anios.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/credibilidad_entre_anios.py'] = '''"""Prueba cruzada entre años del factor de nuevo negocio: factor estimado en un año (OOF PCV, C1) y aplicado al otro."""
import numpy as np, pandas as pd
from comun import *
import receta
t, mu, _ = receta.oof_cruzado("pcv", ["T2b"], "ramo")
m = mu * t.c_fila.values; y = t.y.values; res = {}
for l in LINEAS:
    il = (t.line == l).values; curva = {}
    for z in np.round(np.arange(0, 1.01, 0.1), 1):
        dv = 0.0
        for a, b in [(2023, 2024), (2024, 2023)]:
            ia = il & (t.year.values == a); ib = il & (t.year.values == b)
            nb_a = ia & t.new_business.values; nb_b = ib & t.new_business.values
            ka = factor_tweedie(y[ia], m[ia]); ks = factor_tweedie(y[nb_a], m[nb_a] * ka)
            mb = m.copy(); mb[nb_b] = m[nb_b] * (1 + z * (ks - 1))
            kb = factor_tweedie(y[ib], mb[ib])
            dv += desviacion(y[ib], mb[ib] * kb).sum()
        curva[float(z)] = dv
    d0 = curva[0.0]
    res[l] = {"z_optimo": min(curva, key=curva.get), "cambio_relativo_desviacion_por_10mil": {k: round((v - d0) / d0 * 1e4, 2) for k, v in curva.items()}}
    print(l, res[l]["z_optimo"])
guardar_json(res, ENTREGA / "credibilidad_entre_anios.json")
'''


### Embedded source: `ml/codigo/e09_e12_tabla.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/e09_e12_tabla.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e09_e12_tabla.py'] = '''"""Tabla de E10-E12 frente a sus referencias en DEV (sin calibrar y C1) y HOLD (C1 desde DEV), con remuestreo por cliente."""
import json
import numpy as np, pandas as pd
from comun import *
from evaluar import metricas, bootstrap_dif
import receta
res = {}
tdev = receta.tabla("dev")
for fase in ["dev", "hold"]:
    res[fase] = {}
    for c in ["T2", "T2cap", "T2b", "T2bs4", "T2bw"]:
        if fase == "dev":
            t, mu, _ = receta.oof_cruzado("dev", [c], "ramo"); _, mu0, _ = receta.oof_cruzado("dev", [c], "sin")
            res[fase][c] = {"C1": metricas(t, mu), "sin": metricas(t, mu0)}
        else:
            t, mu = receta.aplicar("hold", receta.parametros("dev", [c], "ramo"), None)
            res[fase][c] = {"C1": metricas(t, mu)}
        res[fase][c]["_mu"] = mu
    for a, b in [("T2cap", "T2"), ("T2bs4", "T2b"), ("T2bw", "T2b")]:
        res[fase][f"{a}-{b}"] = bootstrap_dif(t, res[fase][a]["_mu"], res[fase][b]["_mu"], B=1000)
    for c in ["T2", "T2cap", "T2b", "T2bs4", "T2bw"]:
        res[fase][c].pop("_mu")
guardar_json(res, ENTREGA / "e10_e12_resultados.json")
for fase in res:
    for k, v in res[fase].items():
        if "-" in k: print(fase, k, round(v["dif"], 4), [round(x, 4) for x in v["ic95"]], v["p_mayor"])
        else: print(fase, k, {kk: round(vv["D2"], 4) for kk, vv in v.items()}, {l: round(v["C1"][f"D2_{l}"], 4) for l in LINEAS})
'''


### Embedded source: `ml/codigo/e09_nivel.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/e09_nivel.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e09_nivel.py'] = '''"""E09: sensibilidad del D2 al nivel por ramo y al nivel del nuevo negocio (T2b + C1), DEV 2023 y HOLD 2024."""
import json
import numpy as np, pandas as pd
from comun import *
import receta

res = {}
fs = [0.8, 0.9, 0.95, 1.0, 1.05, 1.1, 1.2, 1.3]
for fase in ["dev", "hold"]:
    if fase == "dev":
        t, mu, _ = receta.oof_cruzado("dev", ["T2b"], "ramo")
    else:
        t, mu = receta.aplicar("hold", receta.parametros("dev", ["T2b"], "ramo"), None)
    m = mu * t.c_fila.values; y = t.y.values; base = d2(y, m)
    r = {"base": base, "ramo": {}, "nuevo_negocio": {}, "optimo": {}}
    for l in LINEAS:
        i = (t.line == l).values
        r["ramo"][l] = {str(f): d2(y, np.where(i, m * f, m)) - base for f in fs}
        r["nuevo_negocio"][l] = {str(f): d2(y, np.where(i & t.new_business.values, m * f, m)) - base for f in fs}
        r["optimo"][l] = {"factor_tweedie": factor_tweedie(y[i], m[i]),
                          "factor_tweedie_NB": factor_tweedie(y[i & t.new_business.values], m[i & t.new_business.values])}
    res[fase] = r
guardar_json(res, ENTREGA / "e09_nivel.json")
for fase in res:
    print(fase, round(res[fase]["base"], 4))
    for l in LINEAS:
        print(" ", l, "ramo", {k: round(v, 4) for k, v in res[fase]["ramo"][l].items()})
        print(" ", l, "NB  ", {k: round(v, 4) for k, v in res[fase]["nuevo_negocio"][l].items()}, {k: round(v, 3) for k, v in res[fase]["optimo"][l].items()})
'''


### Embedded source: `ml/codigo/e11_e12.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/e11_e12.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e11_e12.py'] = '''"""E11 (promedio de 4 semillas de T2b) y E12 (desarrollado reponderado hacia la cartera emergente 2025).
Rondas fijas iguales a las de T2b en la fase dev. Salidas en pred/dev y pred/hold como T2bs4 y T2bw."""
import json
import numpy as np, pandas as pd, lightgbm as lgb
from comun import *
from variables import *
from gbm import P_DEVB

RONDAS = {"motor": 249, "buildings": 194, "contents": 169}
out = {}
for l in LINEAS:
    em = cargar("emerging", l, "hasta2023"); dv = cargar("developed", l, "hasta2023")
    dv = dv[dv.year.isin([2021, 2022, 2023])].reset_index(drop=True)
    cols = comunes(l)
    cats = categorias_union([dv, em], cols + SOLO_EMERGENTE + MARKETING + TARIFA + MAKE)
    Xd = matriz(dv, cols, cats); Xe = matriz(em, cols, cats)
    off = np.log(dv.exposure.values * dv.c_fila.values)
    # E11: cuatro semillas (la original 20261024 + tres)
    raws = []
    for s in [20261024, 11, 12, 13]:
        b = lgb.train(dict(P_DEVB[l], seed=s), lgb.Dataset(Xd, dv.y.values, init_score=off), RONDAS[l])
        raws.append(b.predict(Xe, raw_score=True))
    r_s4 = np.log(np.mean(np.exp(np.column_stack(raws)), axis=1))
    # E12: razón de densidad desarrollado -> test emergente 2025 con variables comunes sin historia
    cw = [c for c in cols if c not in ("hist_pol_12m", "hist_cli_12m")]
    te = em[em.conjunto == "test"]
    Xc = pd.concat([matriz(dv, cw, cats), matriz(te, cw, cats)], ignore_index=True)
    z = np.r_[np.zeros(len(dv)), np.ones(len(te))]
    pc = dict(objective="binary", learning_rate=0.05, num_leaves=31, min_data_in_leaf=500, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, verbose=-1, num_threads=2, seed=5)
    fo = dv.fold.values
    p = np.zeros(len(dv))
    clf = lgb.train(pc, lgb.Dataset(Xc, z), 200)
    p = clf.predict(matriz(dv, cw, cats))  # dentro de muestra: solo para pesos, no para evaluar
    w = p / (1 - p) * (len(dv) / len(te)); w = np.clip(w, 0.2, 5); w = w / w.mean()
    ess = w.sum() ** 2 / (w ** 2).sum()
    b = lgb.train(P_DEVB[l], lgb.Dataset(Xd, dv.y.values, weight=w, init_score=off), RONDAS[l])
    r_w = b.predict(Xe, raw_score=True)
    out[l] = {"ess_frac": float(ess / len(dv))}
    for nom, r in [("T2bs4", r_s4), ("T2bw", r_w)]:
        em["raw"] = r
        tr = em[(em.conjunto == "train") & (em.year == 2023)].reset_index(drop=True)
        for fase, sets in [("dev", [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)]),
                           ("hold", [(tr, em[em.year == 2024].reset_index(drop=True))])]:
            preds = []
            for a, bb in sets:
                k = factor_tweedie(a.y.values, np.exp(a.raw.values) * a.exposure.values * a.c_fila.values)
                preds.append(pd.DataFrame({"row_id": bb.row_id.values, "mu": k * np.exp(bb.raw.values) * bb.exposure.values}))
            pd.concat(preds).to_parquet(TRABAJO / "pred" / fase / f"{nom}_{l}.parquet", index=False)
    print(l, out[l], flush=True)
guardar_json(out, ENTREGA / "e11_e12_info.json")
'''


### Embedded source: `ml/codigo/e13_clima_anual.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/e13_clima_anual.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e13_clima_anual.py'] = '''"""E13: T2b con el clima del propio año de la póliza (exploratorio)."""
import numpy as np, pandas as pd, lightgbm as lgb
from comun import *
from variables import *
from gbm import P_DEVB
RONDAS = {"motor": 249, "buildings": 194, "contents": 169}
VARS = ["hail_days", "wind_exposure_index", "freezing_days", "dry_summer_index", "wildfire_index", "rainfall_mm", "mean_temperature_c"]
w = {m: pd.read_parquet(DATOS / m / "weather.parquet").rename(columns={v: "w_" + v for v in VARS}) for m in ["developed", "emerging"]}
for l in LINEAS:
    em = cargar("emerging", l, "hasta2023"); dv = cargar("developed", l, "hasta2023")
    dv = dv[dv.year.isin([2021, 2022, 2023])].reset_index(drop=True)
    dv = dv.merge(w["developed"], left_on=["address_id", "year"], right_on=["address_id", "year"], how="left")
    em = em.merge(w["emerging"], left_on=["address_id", "year"], right_on=["address_id", "year"], how="left")
    cols = [c for c in comunes(l) if not c.startswith("clima_")] + ["w_" + v for v in VARS]
    cats = categorias_union([dv, em], cols)
    off = np.log(dv.exposure.values * dv.c_fila.values)
    b = lgb.train(P_DEVB[l], lgb.Dataset(matriz(dv, cols, cats), dv.y.values, init_score=off), RONDAS[l])
    em["raw"] = b.predict(matriz(em, cols, cats), raw_score=True)
    tr = em[(em.conjunto == "train") & (em.year == 2023)].reset_index(drop=True)
    for fase, sets in [("dev", [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)]), ("hold", [(tr, em[em.year == 2024].reset_index(drop=True))])]:
        preds = []
        for a, bb in sets:
            k = factor_tweedie(a.y.values, np.exp(a.raw.values) * a.exposure.values * a.c_fila.values)
            preds.append(pd.DataFrame({"row_id": bb.row_id.values, "mu": k * np.exp(bb.raw.values) * bb.exposure.values}))
        pd.concat(preds).to_parquet(TRABAJO / "pred" / fase / f"T2bclima_{l}.parquet", index=False)
    imp = pd.Series(b.feature_importance("gain"), index=cols); imp = imp / imp.sum()
    print(l, "ganancia clima del año", round(float(imp[[c for c in cols if c.startswith("w_")]].sum()), 3), flush=True)
'''


### Embedded source: `ml/codigo/e14_eval.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/e14_eval.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e14_eval.py'] = '''"""Evaluación de E14-E18: candidatos individuales y mezclas en DEV (OOF 2023, cruzado) y HOLD (2024, exploratorio).

Uso: python3 e14_eval.py <salida.json> <cand1> [cand2 ...] [--mezclas A+B+C,...]
Cada candidato o mezcla se evalúa con calibración C1 (ramo) y C2 (ramo x nuevo negocio, K=200).
Las mezclas usan pesos convexos por ramo cruzados por pliegue (DEV) o estimados en todo DEV (HOLD),
tal cual (e0), encogidos al 50 % hacia pesos iguales (e50) y con pesos iguales (e100).
"""
from __future__ import annotations
import sys, json
import numpy as np, pandas as pd
from comun import *
from evaluar import mezcla_pesos, metricas, bootstrap_dif, calibrar_cruzado
from receta import tabla, componentes, K_NB


def pesos(t, P, comps, il, encoge):
    M = np.column_stack([P[c][il] * t.c_fila.values[il] for c in comps])
    w = mezcla_pesos(t.y.values[il], M) if len(comps) > 1 else np.ones(1)
    return (1 - encoge) * w + encoge / len(comps)


def dev_oof(t, P, comps, encoge):
    folds = pliegue(t.customer_id.values)
    if len(comps) == 1:
        return P[comps[0]], folds
    U = np.column_stack([P[c] for c in comps]); out = np.zeros(len(t))
    for k in np.unique(folds):
        te = folds == k
        for l in LINEAS:
            il = (t.line == l).values
            w = pesos(t[~te].reset_index(drop=True), {c: P[c][~te] for c in comps}, comps, il[~te], encoge)
            out[te & il] = U[te & il] @ w
    return out, folds


def hold_pred(td, Pd, th, Ph, comps, calib, encoge):
    mu = np.zeros(len(th)); info = {}
    for l in LINEAS:
        il = (td.line == l).values; ih = (th.line == l).values
        w = pesos(td, Pd, comps, il, encoge)
        m = np.column_stack([Pd[c][il] for c in comps]) @ w * td.c_fila.values[il]
        y = td.y.values[il]; kl = factor_tweedie(y, m)
        base = np.column_stack([Ph[c][ih] for c in comps]) @ w
        if calib == "ramo":
            f = np.full(ih.sum(), kl)
        else:
            f = np.zeros(ih.sum())
            for nb in (True, False):
                s = td.new_business.values[il] == nb
                ks = factor_tweedie(y[s], m[s] * kl); n = int((y[s] > 0).sum()); z = n / (n + K_NB)
                f[th.new_business.values[ih] == nb] = kl * (z * ks + 1 - z)
        mu[ih] = base * f
        info[l] = {"pesos": dict(zip(comps, np.round(w, 4).tolist())), "k_ramo": kl}
    return mu, info


def main():
    salida = sys.argv[1]
    args = sys.argv[2:]
    mezclas = []
    if "--mezclas" in args:
        i = args.index("--mezclas"); mezclas = [m.split("+") for m in args[i + 1].split(",")]; args = args[:i]
    cands = args if "T2b" in args else ["T2b"] + args
    td = tabla("dev").merge(pd.read_parquet(TRABAJO / "pesos_2025_dev.parquet"), on="row_id", how="left")
    th = tabla("hold")
    todos = sorted(set(cands) | {c for m in mezclas for c in m} | {"T2b"})
    Pd = componentes("dev", todos, td); Ph = componentes("hold", todos, th)
    recetas = [([c], 0.0) for c in cands] + [(m, e) for m in mezclas for e in (0.0, 0.5, 1.0)]
    res = {}; preds_dev = {}; preds_hold = {}
    for comps, enc in recetas:
        nom = "+".join(comps) + ("" if len(comps) == 1 else f"_e{int(enc * 100)}")
        base, folds = dev_oof(td, Pd, comps, enc)
        r = {"sin": metricas(td, base)} if len(comps) == 1 else {}
        for calib in ("ramo", "ramo_nb"):
            md = calibrar_cruzado(td, base, folds, por_nb=(calib == "ramo_nb"), K=K_NB)
            mh, info = hold_pred(td, Pd, th, Ph, comps, calib, enc)
            preds_dev[(nom, calib)] = md; preds_hold[(nom, calib)] = mh
            r[calib] = {"dev": metricas(td, md), "hold": metricas(th, mh), "pesos_hold": info}
        res[nom] = r
        print(nom, "DEV C1 %.4f C2 %.4f w25 %.4f | HOLD C1 %.4f C2 %.4f" % (
            r["ramo"]["dev"]["D2"], r["ramo_nb"]["dev"]["D2"], r["ramo"]["dev"]["D2_w2025"],
            r["ramo"]["hold"]["D2"], r["ramo_nb"]["hold"]["D2"]), flush=True)
    # bootstrap frente a T2b con la misma calibración
    boot = {}
    for (nom, calib), md in preds_dev.items():
        if nom == "T2b":
            continue
        boot[f"{nom}|{calib}"] = {
            "dev": bootstrap_dif(td, md, preds_dev[("T2b", calib)], B=1000),
            "hold": bootstrap_dif(th, preds_hold[(nom, calib)], preds_hold[("T2b", calib)], B=1000)}
    # DES24 desde los meta
    des = {}
    for c in todos:
        for l in LINEAS:
            f = TRABAJO / "pred" / "hold" / f"{c}_{l}_meta.json"
            if f.exists():
                m = json.load(open(f))
                if "DES24" in m:
                    des.setdefault(c, {})[l] = m["DES24"]["D2"]
    guardar_json({"resultados": res, "bootstrap_vs_T2b": boot, "DES24": des}, ENTREGA / salida)
    print(json.dumps({k: {kk: [round(v["dif"], 4), [round(x, 4) for x in v["ic95"]]] for kk, v in b.items()} for k, b in boot.items()}, indent=0))
    print("DES24", json.dumps({c: {l: round(v, 4) for l, v in d.items()} for c, d in des.items()}))


if __name__ == "__main__":
    main()
'''


### Embedded source: `ml/codigo/e25_severidad.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/e25_severidad.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e25_severidad.py'] = '''"""E25: rejilla de la severidad Gamma de FS elegida con el pliegue interno 0 del desarrollado 2021-2023.

Uso: python3 e25_severidad.py [lineas]
Salida: trabajo/modelos/dev/e25_rejilla_{linea}.json con la desviación de validación de cada combinación y la elegida.
No mira 2024 ni el emergente.
"""
import sys, json, itertools, time
import numpy as np, lightgbm as lgb
from comun import *
from variables import *

SEMILLA = 20261024
REJILLA = list(itertools.product([7, 15, 31, 63], [50, 200, 1000], [1.0, 10.0, 100.0]))


def dev_gamma(y, mu, w):
    return float(np.sum(w * 2 * (-np.log(y / mu) + (y - mu) / mu)) / np.sum(w))


def correr(linea):
    t0 = time.time()
    em = cargar("emerging", linea, "hasta2023"); dv = cargar("developed", linea, "hasta2023")
    dv = dv[dv.year.isin([2021, 2022, 2023])].reset_index(drop=True)
    cols = comunes(linea); cats = categorias_union([dv, em], cols + SOLO_EMERGENTE + MARKETING + TARIFA + MAKE)
    s = (dv.n_sin.values > 0) & (dv.y.values > 0)
    X = matriz(dv, cols, cats)[s].reset_index(drop=True)
    n = dv.n_sin.values[s].astype(float); ysev = dv.y.values[s] / n
    val = (dv.fold.values == 0)[s]
    msev = float(dv.y.values[s].sum() / n.sum())
    off = np.full(len(ysev), np.log(msev))
    filas = []
    for hojas, mind, l2 in REJILLA:
        p = dict(objective="gamma", learning_rate=0.03, num_leaves=hojas, min_data_in_leaf=mind, lambda_l2=l2,
                 feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, max_cat_to_onehot=8, cat_smooth=50,
                 verbose=-1, num_threads=2, seed=SEMILLA)
        dtr = lgb.Dataset(X[~val], ysev[~val], init_score=off[~val], weight=n[~val], free_raw_data=False)
        dva = lgb.Dataset(X[val], ysev[val], init_score=off[val], weight=n[val], reference=dtr)
        b = lgb.train(p, dtr, 3000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
        mu = np.exp(b.predict(X[val], raw_score=True, num_iteration=b.best_iteration) + off[val])
        filas.append({"num_leaves": hojas, "min_data_in_leaf": mind, "lambda_l2": l2, "rondas": int(b.best_iteration),
                      "desv_val": dev_gamma(ysev[val], mu, n[val])})
    base = dev_gamma(ysev[val], np.full(val.sum(), np.average(ysev[~val], weights=n[~val])), n[val])
    mejor = min(filas, key=lambda r: r["desv_val"])
    actual = [r for r in filas if (r["num_leaves"], r["min_data_in_leaf"], r["lambda_l2"]) == (31, 200, 10.0)][0]
    guardar_json({"linea": linea, "filas_val": int(val.sum()), "siniestros_val": float(n[val].sum()), "desv_constante": base,
                  "rejilla": filas, "elegida": mejor, "FS_actual": actual, "s": round(time.time() - t0)},
                 TRABAJO / "modelos" / "dev" / f"e25_rejilla_{linea}.json")
    print(linea, "constante %.5f" % base, "FS actual", actual, "elegida", mejor, round(time.time() - t0), "s", flush=True)


if __name__ == "__main__":
    for l in (sys.argv[1].split(",") if len(sys.argv) > 1 else LINEAS):
        correr(l)
'''


### Embedded source: `ml/codigo/e26_hold.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/e26_hold.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e26_hold.py'] = '''"""E26: HOLD 2024 (exploratorio) de mezclas con pesos fijos de DES24 o encogidos (e50) estimados en DEV; nivel C1 de DEV."""
import sys, json, numpy as np, pandas as pd
from comun import *
from evaluar import metricas, bootstrap_dif, mezcla_pesos
from receta import tabla, componentes
fij = json.load(open(ENTREGA / "evolucion" / "e26_pesos_des24.json"))
td = tabla("dev"); th = tabla("hold")
def pred(comps, modo):
    Pd = componentes("dev", comps, td); Ph = componentes("hold", comps, th); mu = np.zeros(len(th))
    for l in LINEAS:
        il = (td.line == l).values; ih = (th.line == l).values
        M = np.column_stack([Pd[c][il] * td.c_fila.values[il] for c in comps])
        if modo == "des24":
            w = np.array(list(fij["ramos"][l]["pesos"].values()))
        else:
            w = mezcla_pesos(td.y.values[il], M) if len(comps) > 1 else np.ones(1)
            w = 0.5 * w + 0.5 / len(comps) if len(comps) > 1 else w
        k = factor_tweedie(td.y.values[il], M @ w)
        mu[ih] = np.column_stack([Ph[c][ih] for c in comps]) @ w * k
    return mu
R = {"T2b": pred(["T2b"], "e50"), "v2": pred(["T2b", "T2h", "FS", "T2o"], "e50"),
     "v2t": pred(["T2b", "T2h", "FSt", "T2o"], "e50"), "v3": pred(["T2b", "T2h", "FSt", "T2o"], "des24"), "FSt": pred(["FSt"], "e50")}
out = {}
for k, v in R.items():
    m = metricas(th, v); out[k] = {"D2": m["D2"], **{f"D2_{l}": m[f"D2_{l}"] for l in LINEAS}}
    if k != "T2b":
        out[k]["vs_T2b"] = bootstrap_dif(th, v, R["T2b"])
    if k in ("v2t", "v3"):
        out[k]["vs_v2"] = bootstrap_dif(th, v, R["v2"])
    print(k, {a: (round(b, 4) if isinstance(b, float) else (round(b["dif"], 4), [round(x, 4) for x in b["ic95"]])) for a, b in out[k].items()})
guardar_json(out, ENTREGA / "evolucion" / "e26_hold.json")
'''


### Embedded source: `ml/codigo/e26_pesos_des24.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/e26_pesos_des24.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e26_pesos_des24.py'] = '''"""E26: pesos convexos de mezcla por ramo estimados en el desarrollado 2024 (DES24), troncos 2021-2023.

Cada tronco se renivela en 2024 (factor Tweedie del ramo) y los pesos minimizan la desviación Tweedie 1,5 de la mezcla.
Se informa también el D2 de la mezcla con pesos cruzados por pliegue de cliente dentro de 2024 (control de sobreajuste).
Uso: python3 e26_pesos_des24.py A,B,C,D salida.json
"""
import sys, json
import numpy as np, pandas as pd
from comun import *
from evaluar import mezcla_pesos

COMPS = sys.argv[1].split(","); SALIDA = sys.argv[2]
res = {"componentes": COMPS, "ramos": {}}
for l in LINEAS:
    base = None; M = []
    for c in COMPS:
        d = pd.read_parquet(TRABAJO / "pred" / "hold" / f"{c}des24_{l}.parquet")
        if base is None:
            base = d
        assert (d.row_id.values == base.row_id.values).all()
        mu = np.exp(d.raw.values) * d.exposure.values * d.c_fila.values
        M.append(mu * factor_tweedie(d.y.values, mu))
    M = np.column_stack(M); y = base.y.values
    w = mezcla_pesos(y, M)
    cust = pd.read_parquet(TRABAJO / f"base_developed_{l}.parquet", columns=["row_id", "customer_id"])
    folds = pliegue(base[["row_id"]].merge(cust, on="row_id", how="left").customer_id.values)
    cruz = np.zeros(len(y))
    for f in range(5):
        te = folds == f
        cruz[te] = M[te] @ mezcla_pesos(y[~te], M[~te])
    ind = {c: d2(y, M[:, i]) for i, c in enumerate(COMPS)}
    eq = M.mean(1)
    res["ramos"][l] = {"pesos": dict(zip(COMPS, np.round(w, 4).tolist())), "D2_individual": ind,
                       "D2_mezcla_en_muestra": d2(y, M @ w), "D2_mezcla_cruzada": d2(y, cruz),
                       "D2_pesos_iguales": d2(y, eq), "filas": int(len(y))}
    print(l, {c: round(v, 4) for c, v in ind.items()}, "pesos", dict(zip(COMPS, np.round(w, 3))),
          "mezcla %.4f cruzada %.4f iguales %.4f" % (d2(y, M @ w), d2(y, cruz), d2(y, eq)), flush=True)
guardar_json(res, ENTREGA / "evolucion" / SALIDA)
'''


### Embedded source: `ml/codigo/e27_backtest.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/e27_backtest.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e27_backtest.py'] = '''"""E27, diagnóstico preinscrito: la regla con 2023 solo aplicada a T2b en 2024 (HOLD, exploratorio).
Con un solo año no hay var_año: sigma = bootstrap por cliente de 2023 (se declara)."""
import json, numpy as np, pandas as pd
from comun import *
from receta import tabla, componentes
TAU, K = 0.07, 200.0
rows = []
for l in LINEAS:
    b = pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet", columns=["row_id", "customer_id", "year", "conjunto", "y", "n_sin", "c_fila", "exposure", "written_premium", "new_business"])
    rows.append(b.assign(line=l))
d = pd.concat(rows, ignore_index=True); d["seg"] = d.line + "|" + np.where(d.new_business, "NB", "REN"); d["P"] = d.written_premium * d.exposure
h = d[(d.conjunto == "train") & (d.year == 2023)].reset_index(drop=True); h["U"] = h.y / h.c_fila
th = tabla("hold"); mu = componentes("hold", ["T2b"], th)["T2b"]
t = th.merge(d[["row_id", "seg", "P"]], on="row_id", how="left"); t["mu"] = mu
def lrs(w):
    o = {}
    for l in LINEAS:
        il = (h.line == l).values; lr_l = np.sum(w[il] * h.U.values[il]) / np.sum(w[il] * h.P.values[il])
        for s in ("NB", "REN"):
            m = il & (h.seg == f"{l}|{s}").values
            n = np.sum(w[m] * h.n_sin.values[m]); z = n / (n + K)
            o[f"{l}|{s}"] = z * np.sum(w[m] * h.U.values[m]) / np.sum(w[m] * h.P.values[m]) + (1 - z) * lr_l
    return o
c0 = lrs(np.ones(len(h))); cli, idx = np.unique(h.customer_id.values, return_inverse=True)
rng = np.random.default_rng(20261024); sims = [lrs(rng.poisson(1.0, len(cli))[idx].astype(float)) for _ in range(500)]
f = {}; res = {}
for s in c0:
    m = (t.seg == s).values
    g = c0[s] * t.P.values[m].sum() / t.mu.values[m].sum()
    sig2 = np.var([np.log(x[s]) for x in sims]); z = TAU ** 2 / (TAU ** 2 + sig2)
    f[s] = float(np.clip(np.exp(z * np.log(g)), 0.85, 1.15))
    ae = t.y.values[m].sum() / (t.mu.values[m] * t.c_fila.values[m]).sum()
    res[s] = {"g": g, "sigma_boot": float(np.sqrt(sig2)), "Z": z, "f": f[s], "AE_T2b_2024": float(ae)}
mu2 = t.mu.values * t.seg.map(f).values
out = {"segmentos": res, "D2_T2b": d2(t.y.values, t.mu.values * t.c_fila.values), "D2_con_f": d2(t.y.values, mu2 * t.c_fila.values),
       "D2_con_g_pleno": d2(t.y.values, t.mu.values * t.seg.map({s: r["g"] for s, r in res.items()}).values * t.c_fila.values)}
for s, r in res.items(): print(s, {k: round(v, 3) for k, v in r.items()})
print("D2 2024: T2b %.4f, con f %.4f, con g sin credibilidad %.4f" % (out["D2_T2b"], out["D2_con_f"], out["D2_con_g_pleno"]))
guardar_json(out, ENTREGA / "nivel2025" / "e27_backtest_2024.json")
'''


### Embedded source: `ml/codigo/e27_nivel.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/e27_nivel.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e27_nivel.py'] = '''"""E27: nivel 2025 por ramo y segmento con la siniestralidad esperada sobre la prima escrita (preinscripción
entrega/nivel2025/PREINSCRIPCION_NIVEL.md, SHA 13d2ef2f…). Sin tablero.

Salida: entrega/nivel2025/e27_nivel.json y privado/e27_nivel_2025.csv (base × f_s).
"""
import json, hashlib
import numpy as np, pandas as pd
from comun import *
from scipy.stats import spearmanr

PRE = ENTREGA / "nivel2025" / "PREINSCRIPCION_NIVEL.md"
assert sha256(PRE).startswith("13d2ef2f1596621f"), "la preinscripción cambió"
BASE = RAIZ / "privado" / "mezcla50_designada_mlind_2025.csv"
assert sha256(BASE).startswith("12831ef4a05c1773"), "la base no es la mezcla enviada"
TAU, K, TOPE, B_BOOT, SEM = 0.07, 200.0, (0.85, 1.15), 1000, 20261024

filas = []
for l in LINEAS:
    b = pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet",
                        columns=["row_id", "customer_id", "year", "conjunto", "y", "n_sin", "c_fila", "exposure",
                                 "written_premium", "new_business"])
    filas.append(b.assign(line=l))
d = pd.concat(filas, ignore_index=True)
d["seg"] = d.line + "|" + np.where(d.new_business, "NB", "REN")
d["P"] = d.written_premium * d.exposure
h = d[(d.conjunto == "train") & d.year.isin([2023, 2024])].reset_index(drop=True)
h["U"] = h.y / h.c_fila
t = d[d.conjunto == "test"].reset_index(drop=True)
base = pd.read_csv(BASE)
t = t.merge(base, on="row_id", how="left"); assert t.prediction.notna().all() and len(t) == 209063
SEGS = sorted(h.seg.unique())


def lr_estrella(hh, w=None, k=K):
    w = np.ones(len(hh)) if w is None else w
    out, raw = {}, {}
    for l in LINEAS:
        il = (hh.line == l).values
        lr_l = np.sum(w[il] * hh.U.values[il]) / np.sum(w[il] * hh.P.values[il])
        for s in ("NB", "REN"):
            m = il & (hh.seg == f"{l}|{s}").values
            lr_s = np.sum(w[m] * hh.U.values[m]) / np.sum(w[m] * hh.P.values[m])
            n = np.sum(w[m] * hh.n_sin.values[m]); z = n / (n + k)
            out[f"{l}|{s}"] = z * lr_s + (1 - z) * lr_l; raw[f"{l}|{s}"] = (lr_s, lr_l, n, z)
    return out, raw


def factores(hh, tt, tau=TAU, k=K, boot=True):
    lrs, raw = lr_estrella(hh, k=k)
    # varianza bootstrap por cliente (pesos de Poisson)
    cli, idx = np.unique(hh.customer_id.values, return_inverse=True)
    rng = np.random.default_rng(SEM)
    sims = {s: [] for s in SEGS}
    if boot:
        for _ in range(B_BOOT):
            w = rng.poisson(1.0, len(cli))[idx].astype(float)
            o, _ = lr_estrella(hh, w, k=k)
            for s in SEGS:
                sims[s].append(np.log(o[s]))
    res = {}
    for s in SEGS:
        l = s.split("|")[0]
        yr = {}
        for a in (2023, 2024):
            m = (hh.seg == s).values & (hh.year == a).values
            yr[a] = hh.U.values[m].sum() / hh.P.values[m].sum()
        var_boot = float(np.var(sims[s])) if boot else np.nan
        var_anio = 0.5 * (np.log(yr[2024]) - np.log(yr[2023])) ** 2
        sig2 = var_boot + var_anio
        mt = (tt.seg == s).values
        P25 = tt.P.values[mt].sum(); Bs = tt.prediction.values[mt].sum()
        E = lrs[s] * P25; g = E / Bs
        z = tau ** 2 / (tau ** 2 + sig2)
        f = float(np.clip(np.exp(z * np.log(g)), *TOPE))
        res[s] = {"LR_segmento": raw[s][0], "LR_ramo": raw[s][1], "siniestros": raw[s][2], "Z_n": raw[s][3],
                  "LR_estrella": lrs[s], "LR_2023": yr[2023], "LR_2024": yr[2024], "var_boot": var_boot,
                  "var_anio": var_anio, "sigma": float(np.sqrt(sig2)), "P25": P25, "base_total": Bs, "E_tarifa": E,
                  "g": g, "Z": z, "f": f, "filas_2025": int(mt.sum())}
    return res


R = factores(h, t)
out = {"preinscripcion_sha256": sha256(PRE), "base_sha256": sha256(BASE), "tau": TAU, "K": K, "tope": TOPE, "segmentos": R}
for s, r in R.items():
    print(s, "LR %.3f (2023 %.3f, 2024 %.3f) LR* %.3f | g %.3f sigma %.3f Z %.3f -> f %.4f" % (
        r["LR_segmento"], r["LR_2023"], r["LR_2024"], r["LR_estrella"], r["g"], r["sigma"], r["Z"], r["f"]))

# sensibilidades (sin bootstrap nuevo para K: se reutiliza la varianza del caso central)
sens = {}
for tau in (0.05, 0.10):
    sens[f"tau_{tau}"] = {s: float(np.clip(np.exp(tau ** 2 / (tau ** 2 + R[s]["sigma"] ** 2) * np.log(R[s]["g"])), *TOPE)) for s in SEGS}
for k in (100.0, 400.0):
    Rk = factores(h, t, k=k, boot=True)
    sens[f"K_{int(k)}"] = {s: Rk[s]["f"] for s in SEGS}
out["sensibilidad"] = sens
print("sens", json.dumps({a: {s: round(v, 4) for s, v in b.items()} for a, b in sens.items()}))

# entrega E27: base × f_s
t["f"] = t.seg.map({s: R[s]["f"] for s in SEGS})
sub = pd.read_parquet(DATOS / "emerging" / "sample_submission.parquet")
o = sub[["row_id"]].merge(t[["row_id", "prediction", "f"]], on="row_id", how="left")
o["prediction"] = o.prediction * o.f
o = o[["row_id", "prediction"]]
assert len(o) == 209063 and (o.row_id.values == sub.row_id.values).all() and o.prediction.notna().all() and (o.prediction > 0).all()
f_out = RAIZ / "privado" / "e27_nivel_2025.csv"
o.to_csv(f_out, index=False, float_format="%.6f", lineterminator="\\n")
out["entrega"] = {"archivo": f_out.name, "sha256": sha256(f_out), "total": float(o.prediction.sum()),
                  "total_base": float(base.prediction.sum()),
                  "por_ramo": {l: float(o.prediction.values[t.set_index("row_id").loc[o.row_id, "line"].values == l].sum()) for l in LINEAS}}
print("entrega", out["entrega"])

# diagnóstico: orden de riesgo de la tarifa frente a la predicción base, nuevo negocio 2025 y 2023-2024 (T2b OOF pcv)
pcv = pd.concat([pd.read_parquet(TRABAJO / "pred" / "pcv" / f"T2b_{l}.parquet") for l in LINEAS])
hh = h.merge(pcv, on="row_id", how="left")
diag = {}
for l in LINEAS:
    for s in ("NB", "REN"):
        m25 = (t.seg == f"{l}|{s}").values; m23 = (hh.seg == f"{l}|{s}").values
        diag[f"{l}|{s}"] = {
            "spearman_2025": float(spearmanr(t.written_premium.values[m25], t.prediction.values[m25] / t.exposure.values[m25])[0]),
            "spearman_2023_24": float(spearmanr(hh.written_premium.values[m23], hh.mu.values[m23] / hh.exposure.values[m23])[0]),
            "prima_media_2025": float(t.written_premium.values[m25].mean()),
            "prima_media_2023_24": float(hh.written_premium.values[m23].mean()),
            "base_por_exposicion_2025": float(t.prediction.values[m25].sum() / t.exposure.values[m25].sum()),
            "T2b_por_exposicion_2023_24": float(hh.mu.values[m23].sum() / hh.exposure.values[m23].sum())}
out["diagnostico_orden"] = diag
print("orden", json.dumps({s: {k: round(v, 3) for k, v in d_.items()} for s, d_ in diag.items()}))
guardar_json(out, ENTREGA / "nivel2025" / "e27_nivel.json")
'''


### Embedded source: `ml/codigo/evaluar.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/evaluar.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/evaluar.py'] = '''"""Métricas de la línea ML independiente sobre predicciones guardadas."""
from __future__ import annotations
import sys
import numpy as np, pandas as pd
from comun import *

def tabla_eval(fase, anio):
    filas = []
    for l in LINEAS:
        b = pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet",
                            columns=["row_id", "y", "c_fila", "exposure", "customer_id", "year", "new_business", "conjunto"])
        b = b[(b.conjunto == "train") & (b.year == anio)]
        filas.append(b.assign(line=l))
    return pd.concat(filas, ignore_index=True)


def cargar_pred(fase, cand):
    out = []
    for l in LINEAS:
        f = TRABAJO / "pred" / fase / f"{cand}_{l}.parquet"
        if not f.exists():
            return None
        out.append(pd.read_parquet(f))
    return pd.concat(out, ignore_index=True)


def cuerpo_mask(t, q=0.995):
    m = np.ones(len(t), bool)
    for l in LINEAS:
        i = (t.line == l).values & (t.y.values > 0)
        thr = np.quantile(t.y.values[i], q)
        m &= ~((t.line == l).values & (t.y.values > thr))
    return m


def metricas(t, mu, w=None):
    y = t.y.values; m = mu * t.c_fila.values
    r = {"D2": d2(y, m, w)}
    if "w2025" in t:
        r["D2_w2025"] = d2(y, m, t.w2025.values)
    for l in LINEAS:
        i = (t.line == l).values
        r[f"D2_{l}"] = d2(y[i], m[i], None if w is None else w[i])
        r[f"AE_{l}"] = float(y[i].sum() / m[i].sum())
    cm = cuerpo_mask(t)
    mb = m.copy()
    for l in LINEAS:
        i = cm & (t.line == l).values
        mb[i] *= y[i].sum() / m[i].sum()
    r["D2_cuerpo"] = d2(y[cm], mb[cm])
    for nb in (True, False):
        i = t.new_business.values == nb
        r[f"AE_{'NB' if nb else 'REN'}"] = float(y[i].sum() / m[i].sum())
    return r


if __name__ == "__main__":
    fase = sys.argv[1]; anio = 2023 if fase == "dev" else 2024
    t = tabla_eval(fase, anio)
    filas = []
    for f in sorted((TRABAJO / "pred" / fase).glob("*_motor.parquet")):
        cand = f.name.replace("_motor.parquet", "")
        p = cargar_pred(fase, cand)
        if p is None: continue
        mu = t[["row_id"]].merge(p, on="row_id", how="left").mu.values
        if np.isnan(mu).any(): continue
        filas.append({"cand": cand, **metricas(t, mu)})
    df = pd.DataFrame(filas).set_index("cand").sort_values("D2", ascending=False)
    pd.set_option("display.width", 250)
    print(df.round(4))


def calibrar_cruzado(t, mu, folds, por_nb=False, K=200.0):
    """A01/E04: factor Tweedie por ramo (y opcionalmente por ramo x nuevo negocio con credibilidad),
    estimado en los otros pliegues y aplicado al pliegue retenido."""
    m = mu * t.c_fila.values; y = t.y.values
    out = mu.copy()
    for k in np.unique(folds):
        te = folds == k; trm = ~te
        for l in LINEAS:
            il = (t.line == l).values
            kl = factor_tweedie(y[trm & il], m[trm & il])
            if not por_nb:
                out[te & il] = mu[te & il] * kl
            else:
                for nb in (True, False):
                    s = (t.new_business.values == nb)
                    ks = factor_tweedie(y[trm & il & s], m[trm & il & s] * kl)
                    nsin = np.sum((y[trm & il & s] > 0))
                    z = nsin / (nsin + K)
                    out[te & il & s] = mu[te & il & s] * kl * (z * ks + (1 - z))
    return out


def tabla_dev(cands, por_nb=False):
    t = tabla_eval("dev", 2023)
    t = t.merge(pd.read_parquet(TRABAJO / "pesos_2025_dev.parquet"), on="row_id", how="left")
    from comun import pliegue
    folds = pliegue(t.customer_id.values)
    filas = []
    for c in cands:
        p = cargar_pred("dev", c)
        if p is None: continue
        mu = t[["row_id"]].merge(p, on="row_id", how="left").mu.values
        filas.append({"cand": c, "calib": "sin", **metricas(t, mu)})
        filas.append({"cand": c, "calib": "ramo", **metricas(t, calibrar_cruzado(t, mu, folds))})
        if por_nb:
            filas.append({"cand": c, "calib": "ramo_nb", **metricas(t, calibrar_cruzado(t, mu, folds, True))})
    return pd.DataFrame(filas)


def mezcla_pesos(Y, M):
    """Pesos convexos que minimizan la desviación Tweedie 1,5 de M @ w (M: n x k ya en escala observada)."""
    from scipy.optimize import minimize
    k = M.shape[1]
    if k == 1:
        return np.ones(1)
    def f(w):
        mu = np.maximum(M @ w, 1e-9)
        return np.sum(2 * (2 * Y / np.sqrt(mu) + 2 * np.sqrt(mu))), M.T @ (2 * (-Y / mu ** 1.5 + 1 / np.sqrt(mu)))
    cons = ({"type": "eq", "fun": lambda w: w.sum() - 1, "jac": lambda w: np.ones(k)},)
    r = minimize(f, np.ones(k) / k, jac=True, method="SLSQP", bounds=[(0, 1)] * k, constraints=cons)
    return np.clip(r.x, 0, 1) / np.clip(r.x, 0, 1).sum()


def mezcla_cruzada(t, preds, folds):
    """Pesos por ramo estimados en cuatro pliegues y aplicados al quinto. preds: dict cand -> mu (alineado con t)."""
    nombres = list(preds)
    M = np.column_stack([preds[c] * t.c_fila.values for c in nombres])
    U = np.column_stack([preds[c] for c in nombres])
    y = t.y.values; out = np.zeros(len(t)); pesos = []
    for k in np.unique(folds):
        te = folds == k
        for l in LINEAS:
            il = (t.line == l).values
            w = mezcla_pesos(y[~te & il], M[~te & il])
            out[te & il] = U[te & il] @ w
            pesos.append({"pliegue": int(k), "ramo": l, **dict(zip(nombres, np.round(w, 4)))})
    return out, pesos


def bootstrap_dif(t, mu_a, mu_b, B=1000, semilla=2026, w=None):
    """Diferencia de D2 (a - b) con remuestreo de clientes (pesos de Poisson). Devuelve media, IC95 y P(a>b)."""
    rng = np.random.default_rng(semilla)
    cli, idx = np.unique(t.customer_id.values, return_inverse=True)
    y = t.y.values; ma = mu_a * t.c_fila.values; mb = mu_b * t.c_fila.values
    base = np.ones(len(y)) if w is None else w
    da = desviacion(y, ma); db = desviacion(y, mb)
    sy = np.sqrt(y)
    res = []
    for s in range(0, B, 100):
        Wc = rng.poisson(1.0, size=(min(100, B - s), len(cli))).astype(float)
        W = Wc[:, idx] * base
        ybar = (W @ y) / W.sum(1)
        dn = 2 * ((W @ sy) / (-0.25) - (W @ y) * ybar ** -0.5 / (-0.5) + W.sum(1) * ybar ** 0.5 / 0.5)
        res.append((1 - (W @ da) / dn) - (1 - (W @ db) / dn))
    r = np.concatenate(res)
    return {"dif": float(d2(y, ma, w) - d2(y, mb, w)), "ic95": [float(np.quantile(r, 0.025)), float(np.quantile(r, 0.975))],
            "p_mayor": float(np.mean(r > 0))}
'''


### Embedded source: `ml/codigo/figura_comparacion.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/figura_comparacion.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/figura_comparacion.py'] = '''import json
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from comun import *
c1 = json.load(open(RAIZ / "comparacion" / "comparacion_lineas.json")); c2 = json.load(open(RAIZ / "comparacion" / "comparacion_lineas_2.json"))
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 130})
fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.6))
seg = [("todas", "todas las filas"), ("NB", "nuevo negocio"), ("REN", "renovación"), ("pond_2025", "ponderado a 2025")]
mods = [("R0", "Tasa por ramo", "#bbbbbb"), ("A1", "A1 (base de la designada)", "#c4673b"), ("T2b", "MLind T2b (C1)", "#2a6f97"), ("mez50", "mezcla 50/50", "#5b8c5a")]
x = np.arange(len(seg)); w = 0.2
for j, (k, lab, col) in enumerate(mods):
    axs[0].bar(x + (j - 1.5) * w, [c2["D2"][k][s] for s, _ in seg], w, label=lab, color=col)
axs[0].set_xticks(x, [s for _, s in seg]); axs[0].set_ylabel("D² Tweedie 1,5 en 2024")
axs[0].set_ylim(0.05, 0.15); axs[0].legend(frameon=False, fontsize=7, loc="upper center", ncol=2)
axs[0].set_title("2024, mismas 59.202 filas (exploratorio: las dos líneas ya lo habían visto)", fontsize=8.5)
v = c1["2025"]; ks = [("noct_R0", "R0", "#bbbbbb"), ("noct_A1fixF1b3n_designada", "designada A1fixF1b3n", "#c4673b"),
                      ("MLind_final", "MLind final", "#2a6f97"), ("MLind_C2", "MLind con C2", "#9bb7cc")]
x = np.arange(3)
for j, (k, lab, col) in enumerate(ks):
    axs[1].bar(x + (j - 1.5) * w, [v[k][l]["tasa"] for l in LINEAS], w, label=lab, color=col)
axs[1].set_xticks(x, LINEAS); axs[1].set_ylabel("tasa 2025 predicha (CU por año de exposición)")
axs[1].legend(frameon=False, fontsize=7); axs[1].set_title("Producción 2025: el nivel es lo que más separa a las líneas", fontsize=8.5)
fig.tight_layout(); fig.savefig(RAIZ / "comparacion" / "f10_comparacion_lineas.png"); plt.close(fig)
'''


### Embedded source: `ml/codigo/figura_evolucion.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/figura_evolucion.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/figura_evolucion.py'] = '''"""Figura f12: candidatos E14-E22 frente a T2b en 2023 (DEV), 2024 (HOLD) y DES24 (desarrollado 2024)."""
import json
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from comun import *

E = ENTREGA
fuentes = ["e14_individuales.json", "e14_fs.json", "e15_cb.json", "e21_fsp.json", "e22_mezcla.json"]
boot, des = {}, {}
for f in fuentes:
    r = json.load(open(E / f))
    for k, v in r["bootstrap_vs_T2b"].items():
        nom, cal = k.split("|")
        if cal == "ramo":
            boot[nom] = v
    des.update(r["DES24"])
filas = [("T2x", "T2x  +make_model y tenencia"), ("CB", "CB  CatBoost Tweedie"), ("FSp", "FSp  frec. x sev. por peligro"),
         ("T2o", "T2o  sin occupation"), ("T2c", "T2c  sin historia"), ("T2h", "T2h  historia centrada"),
         ("FS", "FS  frecuencia x severidad"), ("T2b+T2h+FS+T2o_e50", "Mezcla v2 (E22, e50)")]
C = {"bg": "#fcfcfb", "ink": "#0b0b0b", "ink2": "#52514e", "grid": "#e4e3df",
     "s1": "#2a78d6", "s2": "#eb6834", "s3": "#1baf7a"}
plt.rcParams.update({"font.size": 9.5, "axes.edgecolor": C["grid"], "axes.labelcolor": C["ink2"],
                     "xtick.color": C["ink2"], "ytick.color": C["ink"], "font.family": "DejaVu Sans"})
fig, axs = plt.subplots(1, 2, figsize=(12.5, 5.2), facecolor=C["bg"], gridspec_kw={"width_ratios": [1.15, 1]})
y = np.arange(len(filas))[::-1]
ax = axs[0]
for j, (fase, lab, col, mk) in enumerate([("dev", "2023 (DEV, OOF)", C["s1"], "o"), ("hold", "2024 (HOLD)", C["s2"], "s")]):
    for yi, (k, _) in zip(y, filas):
        if k not in boot:
            continue
        b = boot[k][fase]; off = 0.16 if j == 0 else -0.16
        ax.plot(b["ic95"], [yi + off] * 2, color=col, lw=2, solid_capstyle="round")
        ax.plot(b["dif"], yi + off, mk, color=col, ms=7, mec=C["bg"], mew=1.5, label=lab if yi == y[0] else None)
ax.axvline(0, color=C["ink2"], lw=1)
ax.axvspan(0.001, 0.02, color=C["grid"], alpha=0.35, lw=0)
ax.text(0.0012, y[0] + 0.55, "umbral E18: +0,001 en 2023", color=C["ink2"], fontsize=8.5, va="bottom")
ax.set_yticks(y); ax.set_yticklabels([n for _, n in filas])
ax.set_xlim(-0.026, 0.012); ax.set_ylim(-0.7, len(filas) - 0.1)
ax.set_xlabel("Diferencia de D² frente a T2b, calibración C1 (IC95 por cliente)")
ax.set_title("Emergente: 2023 y 2024", loc="left", color=C["ink"], fontsize=11)
ax.legend(loc="lower left", frameon=False)
ax.grid(axis="x", color=C["grid"], lw=0.8); ax.set_axisbelow(True); ax.set_facecolor(C["bg"])
ax = axs[1]
ramos = [("motor", "autos", C["s1"], "o"), ("buildings", "edificios", C["s2"], "s"), ("contents", "contenidos", C["s3"], "^")]
for j, (l, lab, col, mk) in enumerate(ramos):
    for yi, (k, _) in zip(y, filas):
        if k not in des:
            continue
        v = des[k][l] - des["T2b"][l]; off = (1 - j) * 0.2
        ax.plot([0, v], [yi + off] * 2, color=col, lw=1.2, alpha=0.6)
        ax.plot(v, yi + off, mk, color=col, ms=7, mec=C["bg"], mew=1.5, label=lab if yi == y[2] else None)
    if l == "contents":
        for yi, (k, _) in zip(y, filas):
            if k in des and des[k][l] - des["T2b"][l] < -0.012:
                ax.annotate(f"{des[k][l] - des['T2b'][l]:+.3f}".replace(".", ","), (-0.0116, yi - 0.45), color=C["ink2"],
                            fontsize=8, va="center")
ax.axvline(0, color=C["ink2"], lw=1)
ax.set_yticks(y); ax.set_yticklabels([""] * len(filas))
ax.set_xlim(-0.012, 0.009); ax.set_ylim(-0.7, len(filas) - 0.1)
ax.set_xticks([-0.01, -0.005, 0, 0.005])
ax.text(0.0, -0.55, "la mezcla no tiene DES24: es una combinación de troncos", color=C["ink2"], fontsize=8.5, ha="center")
ax.set_xlabel("Diferencia de D² frente a T2b en el desarrollado 2024 (renivelado)")
ax.set_title("DES24: desarrollado 2024, fuera de tiempo", loc="left", color=C["ink"], fontsize=11)
ax.legend(loc="upper right", frameon=False)
ax.grid(axis="x", color=C["grid"], lw=0.8); ax.set_axisbelow(True); ax.set_facecolor(C["bg"])
for a in axs:
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
    a.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:+.3f}".replace(".", ",") if v else "0"))
fig.suptitle("Troncos nuevos frente a T2b: lo que mejora en 2024 y en el desarrollado no mejora en 2023",
             x=0.01, ha="left", fontsize=12.5, color=C["ink"])
fig.tight_layout(rect=(0, 0, 1, 0.95))
out = E / "evolucion" / "f12_troncos.png"; out.parent.mkdir(exist_ok=True)
fig.savefig(out, dpi=150, facecolor=C["bg"])
print(out)
'''


### Embedded source: `ml/codigo/figura_nivel.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/figura_nivel.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/figura_nivel.py'] = '''"""Figura f13: nivel 2025 por segmento según la tarifa (E27) frente al tablero (sondeos globales)."""
import json
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from comun import *

r = json.load(open(ENTREGA / "nivel2025" / "e27_nivel.json"))["segmentos"]
SEG = ["motor|NB", "motor|REN", "buildings|NB", "buildings|REN", "contents|NB", "contents|REN"]
ET = {"motor|NB": "Autos\\nnuevo", "motor|REN": "Autos\\nrenov.", "buildings|NB": "Edificios\\nnuevo",
      "buildings|REN": "Edificios\\nrenov.", "contents|NB": "Contenidos\\nnuevo", "contents|REN": "Contenidos\\nrenov."}
C = {"bg": "#fcfcfb", "ink": "#0b0b0b", "ink2": "#52514e", "grid": "#e4e3df", "s1": "#2a78d6", "s2": "#eb6834", "s3": "#1baf7a"}
plt.rcParams.update({"font.size": 9.5, "axes.edgecolor": C["grid"], "axes.labelcolor": C["ink2"], "xtick.color": C["ink"],
                     "ytick.color": C["ink2"], "font.family": "DejaVu Sans"})
fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.9), facecolor=C["bg"], gridspec_kw={"width_ratios": [1.25, 1]})
x = np.arange(len(SEG))
ax = axs[0]
g = np.array([r[s]["g"] for s in SEG]); sg = np.array([r[s]["sigma"] for s in SEG]); f = np.array([r[s]["f"] for s in SEG])
ax.vlines(x - 0.12, g * np.exp(-sg), g * np.exp(sg), color=C["s2"], lw=2)
ax.plot(x - 0.12, g, "o", color=C["s2"], ms=7, mec=C["bg"], mew=1.5, label="Tarifa × siniestralidad 2023-24 (±1 σ)")
ax.plot(x + 0.12, f, "s", color=C["s1"], ms=7, mec=C["bg"], mew=1.5, label="Factor E27 con credibilidad")
for yv, txt, ls, va in [(1.0, "mezcla enviada (1,000)", "-", "top"), (1.1877, "tablero, nivel creíble 1,188", "--", "top"),
                        (1.2231, "tablero, óptimo público 1,223", ":", "bottom")]:
    ax.axhline(yv, color=C["ink2"], lw=1, ls=ls)
    ax.text(-0.45, yv * (0.995 if va == "top" else 1.005), txt, color=C["ink2"], fontsize=8, va=va, ha="left")
ax.set_xticks(x); ax.set_xticklabels([ET[s] for s in SEG])
ax.set_ylabel("Factor sobre la mezcla enviada")
ax.set_ylim(0.45, 2.15); ax.set_yscale("log"); ax.set_yticks([0.5, 0.7, 1.0, 1.4, 2.0])
ax.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter()); ax.set_xlim(-0.5, len(SEG) - 0.5)
ax.get_yaxis().set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ",")))
ax.set_title("Nivel 2025 que sugiere la tarifa y factor que se aplica", loc="left", color=C["ink"], fontsize=11)
ax.legend(loc="upper right", frameon=False, fontsize=8.5)
ax = axs[1]
l23 = [r[s]["LR_2023"] for s in SEG]; l24 = [r[s]["LR_2024"] for s in SEG]
ax.vlines(x, np.minimum(l23, l24), np.maximum(l23, l24), color=C["grid"], lw=3)
ax.plot(x, l23, "o", color=C["s1"], ms=7, mec=C["bg"], mew=1.5, label="2023")
ax.plot(x, l24, "s", color=C["s2"], ms=7, mec=C["bg"], mew=1.5, label="2024")
ax.set_xticks(x); ax.set_xticklabels([ET[s] for s in SEG])
ax.set_ylabel("Coste último / prima devengada")
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ",")))
ax.set_title("La siniestralidad cambia mucho de un año a otro", loc="left", color=C["ink"], fontsize=11)
ax.legend(loc="upper right", frameon=False)
for a in axs:
    a.set_facecolor(C["bg"]); a.grid(axis="y", color=C["grid"], lw=0.8); a.set_axisbelow(True)
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
fig.suptitle("La tarifa apunta en la misma dirección que el tablero, pero es demasiado inestable para fijar el nivel",
             x=0.01, ha="left", fontsize=12.5, color=C["ink"])
fig.tight_layout(rect=(0, 0, 1, 0.94))
out = ENTREGA / "nivel2025" / "f13_nivel_2025.png"
fig.savefig(out, dpi=150, facecolor=C["bg"]); print(out)
'''


### Embedded source: `ml/codigo/figuras_datos.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/figuras_datos.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/figuras_datos.py'] = '''"""Figuras de datos: completitud de reporte por peligro y desplazamiento 2023 -> 2025."""
import json
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from comun import *

FIG = ENTREGA / "figuras"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 130})
C = {"motor": "#2a6f97", "buildings": "#c4673b", "contents": "#5b8c5a"}
info = json.load(open(TRABAJO / "preparacion.json"))

# 1. completitud de reporte por peligro (desarrollado, coste) y contraste emergente
cdf = info["cdf_coste_desarrollado"]; con = info["contraste_reporte_emergente_2023"]
fig, axs = plt.subplots(1, 3, figsize=(10, 3.2), sharey=True)
for ax, l in zip(axs, LINEAS):
    for k, v in cdf.items():
        ll, p = k.split("|")
        if ll != l: continue
        xs = [0, 30, 90, 180, 365, 730]
        ax.plot(xs, [v[str(x)] if str(x) in v else v[x] for x in xs], marker="o", ms=3, label=p)
    ax.set_title(l); ax.set_xlabel("días desde la ocurrencia"); ax.axvline(365, color="#999", lw=0.6, ls=":")
    ax.legend(fontsize=7, frameon=False)
axs[0].set_ylabel("fracción del coste ya reportada")
fig.suptitle("Completitud de reporte ponderada por coste (desarrollado, cohorte 2021)", fontsize=10)
fig.tight_layout(); fig.savefig(FIG / "f01_completitud_reporte.png"); plt.close(fig)

# 2. desplazamiento de la cartera: nuevo negocio y mes de inicio, 2023/2024 frente a 2025
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2))
filas = []
for l in LINEAS:
    b = pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet", columns=["year", "new_business", "start_month", "exposure"])
    for y, g in b.groupby("year"):
        filas.append({"ramo": l, "año": y, "nb": np.average(g.new_business, weights=g.exposure),
                      "enero": np.average(g.start_month == 1, weights=g.exposure)})
d = pd.DataFrame(filas)
x = np.arange(3); w = 0.25
for i, l in enumerate(LINEAS):
    s = d[d.ramo == l]
    axs[0].bar(x + (i - 1) * w, s.nb.values, w, color=C[l], label=l)
axs[0].set_xticks(x, ["2023", "2024", "2025 (test)"]); axs[0].set_ylabel("nuevo negocio (% de exposición)")
axs[0].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1)); axs[0].legend(frameon=False)
axs[0].set_title("La cartera de 2025 es casi toda nueva")
pes = json.load(open(TRABAJO / "pesos_2025_dev.json"))
imp = pd.DataFrame({l: pes[l]["importancia_top"] for l in LINEAS}).fillna(0)
imp = imp.loc[imp.mean(1).sort_values().index]
yy = np.arange(len(imp))
for i, l in enumerate(LINEAS):
    axs[1].barh(yy + (i - 1) * 0.25, imp[l].values, 0.25, color=C[l], label=f"{l} (AUC {pes[l]['auc']:.2f})")
axs[1].set_yticks(yy, imp.index); axs[1].set_xlabel("ganancia relativa del clasificador 2023 frente a 2025")
axs[1].legend(frameon=False, fontsize=7); axs[1].set_title("Qué distingue la cartera 2025")
fig.tight_layout(); fig.savefig(FIG / "f02_desplazamiento_cartera.png"); plt.close(fig)
print(d.round(3))
'''


### Embedded source: `ml/codigo/figuras_extra.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/figuras_extra.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/figuras_extra.py'] = '''"""Diagrama del flujo (f00) e importancia de variables del modelo final (f09)."""
import json
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from comun import *

FIG = ENTREGA / "figuras"
plt.rcParams.update({"font.size": 9, "figure.dpi": 130})

# f00: flujo de la receta final
fig, ax = plt.subplots(figsize=(10, 3.6)); ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 3.6)
cajas = [
    (0.1, 2.2, "Desarrollado 2021-2024\\n2,4 M periodos\\n(datos, etiquetas T0,\\ncompletitud por peligro)", "#dce8f1"),
    (2.6, 2.2, "T2b: LightGBM Tweedie 1,5\\nesquema común de variables\\ndesplazamiento log(e·c)\\nhiperparámetros por E07", "#2a6f97"),
    (5.1, 2.2, "Transferencia al emergente\\nrelatividades del desarrollado\\n× nivel por ramo (factor\\nTweedie con 2023-2024)", "#c4673b"),
    (7.6, 2.2, "Calibración C1\\nfactor por ramo del OOF\\ncruzado 2023-2024\\nsin tendencia (A03, A04)", "#5b8c5a"),
    (5.1, 0.25, "Emergente 2023-2024\\n109.296 periodos, 9.258 siniestros\\nOOF por cliente (DEV, PCV) y 2024 (HOLD)", "#f3e3d6"),
    (7.6, 0.25, "Producción 2025\\n209.063 filas, 70,8 M CU\\nprima comercial de 20 pólizas", "#e3efe2"),
]
for x, y, txt, col in cajas:
    oscuro = col in ("#2a6f97", "#c4673b", "#5b8c5a")
    ax.add_patch(FancyBboxPatch((x, y), 2.25, 1.25, boxstyle="round,pad=0.05", fc=col, ec="#555", lw=0.6))
    ax.text(x + 1.125, y + 0.62, txt, ha="center", va="center", fontsize=7.6, color="white" if oscuro else "#222")
for (x0, y0, x1, y1) in [(2.4, 2.82, 2.6, 2.82), (4.9, 2.82, 5.1, 2.82), (7.4, 2.82, 7.6, 2.82), (6.22, 1.55, 6.22, 2.15), (8.72, 2.15, 8.72, 1.55)]:
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="->", color="#444", lw=1))
ax.text(0.1, 1.2, "Alternativas probadas y no elegidas:\\nsolo emergente (R0, G1, T1), ajuste fino por boosting (T3),\\nmodelo conjunto (T4), GLM de adaptación (T5), por peligro (T6),\\nFT-Transformer desde cero (N1), preentrenado y ajustado (N2a, N2b),\\nresidual sobre T2b (N3) y mezclas (E1-E3).", fontsize=7.4, va="center", color="#333")
fig.tight_layout(); fig.savefig(FIG / "f00_flujo_receta.png"); plt.close(fig)

# f09: importancia (ganancia) del T2b de producción
fig, axs = plt.subplots(1, 3, figsize=(10.5, 3.6))
imp_out = {}
for ax, l in zip(axs, LINEAS):
    imp = pd.read_csv(TRABAJO / "modelos" / "prod" / f"T2b_{l}_importancia.csv", index_col=0).iloc[:, 0]
    imp = (imp / imp.sum()).sort_values(ascending=False).head(10)[::-1]
    imp_out[l] = imp[::-1].round(4).to_dict()
    ax.barh(imp.index, imp.values, color={"motor": "#2a6f97", "buildings": "#c4673b", "contents": "#5b8c5a"}[l])
    ax.set_title(l); ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1))
    for s in ("top", "right"): ax.spines[s].set_visible(False)
fig.suptitle("Variables con más ganancia en el modelo del desarrollado (T2b de producción)", fontsize=10)
fig.tight_layout(); fig.savefig(FIG / "f09_importancia_T2b.png"); plt.close(fig)
guardar_json(imp_out, ENTREGA / "importancia_T2b_prod.json"); print(json.dumps(imp_out, indent=0))
'''


### Embedded source: `ml/codigo/figuras_resultados.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/figuras_resultados.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/figuras_resultados.py'] = '''"""Figuras de resultados: DEV frente a HOLD, inestabilidad del nuevo negocio, sensibilidad 2025, curvas del transformer."""
import json
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from comun import *
import receta

FIG = ENTREGA / "figuras"
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 130})
C = {"motor": "#2a6f97", "buildings": "#c4673b", "contents": "#5b8c5a"}
out = {}

# f04: DEV frente a HOLD (2024, sin tendencia) para la lista congelada
dev = pd.read_csv(ENTREGA / "tabla_dev.csv")
hold = json.load(open(ENTREGA / "resultados_hold.json"))["candidatos"]
filas = []
for k, v in hold.items():
    base, cal = k.rsplit("_", 1)
    cc = {"C1": "ramo", "C2": "ramo_nb"}[cal]
    dv = dev[(dev.cand == base) & (dev.calib == cc)].D2.values
    filas.append({"cand": k, "DEV": float(dv[0]) if len(dv) else np.nan, "HOLD": v["sin_tendencia"]["D2"]})
d = pd.DataFrame(filas).sort_values("HOLD"); out["dev_hold"] = d.round(5).to_dict("records")
fig, ax = plt.subplots(figsize=(7, 4))
y = np.arange(len(d))
ax.hlines(y, d.DEV, d.HOLD, color="#bbb", lw=2)
ax.scatter(d.DEV, y, color="#2a6f97", label="DEV: OOF 2023", zorder=3)
ax.scatter(d.HOLD, y, color="#c4673b", label="HOLD: 2024, abierto una vez", zorder=3)
ax.set_yticks(y, d.cand); ax.set_xlabel("D² Tweedie 1,5")
ax.legend(frameon=False, loc="lower right"); ax.set_title("Los modelos con transferencia ganan en los dos años; el orden entre ellos es ruido", fontsize=9)
fig.tight_layout(); fig.savefig(FIG / "f04_dev_frente_hold.png"); plt.close(fig)

# f06: A/E del nuevo negocio por ramo y año con C1 (OOF 2023-2024)
t, mu, _ = receta.oof_cruzado("pcv", ["T2b"], "ramo")
m = mu * t.c_fila.values
g = pd.DataFrame({"line": t.line, "year": t.year, "nb": t.new_business, "y": t.y, "m": m, "s": t.y > 0}).groupby(["line", "nb", "year"]).agg(
    y=("y", "sum"), m=("m", "sum"), n=("s", "sum"))
g["AE"] = g.y / g.m
out["ae_nb_anio"] = {f"{a}|{b}|{c}": {"AE": round(float(r.AE), 4), "n_sin": int(r.n)} for (a, b, c), r in g.iterrows()}
fig, ax = plt.subplots(figsize=(7, 3.4))
xs = 0
for l in LINEAS:
    for nb in (False, True):
        for yv, mk in ((2023, "o"), (2024, "s")):
            r = g.loc[(l, nb, yv)]
            ax.scatter(xs, r.AE, color=C[l], marker=mk, s=20 + r.n / 8, alpha=0.85)
        ax.text(xs, 0.3, ("NB" if nb else "REN"), ha="center", fontsize=7)
        xs += 1
    xs += 0.6
ax.axhline(1, color="#999", lw=0.8, ls="--")
ax.set_xticks([0.5, 3.1, 5.7], LINEAS); ax.set_ylabel("observado / esperado (C1)")
ax.set_ylim(0.25, 1.9)
ax.set_title("El exceso del nuevo negocio no se repite de un año a otro (círculo 2023, cuadrado 2024; área ∝ siniestros)", fontsize=8.5)
fig.tight_layout(); fig.savefig(FIG / "f06_nuevo_negocio_por_anio.png"); plt.close(fig)

# f07: sensibilidad del total 2025 y tasas frente a la historia
prod = json.load(open(ENTREGA / "produccion_2025.json")); ctx = json.load(open(ENTREGA / "contexto_2025.json"))
fig, axs = plt.subplots(1, 2, figsize=(10, 3.4))
var = ["final_C1", "C1_tendencia", "C2", "preinscrita_C2_tendencia"]
etq = ["final (C1)", "C1 + tendencia", "C2", "preinscrita\\n(C2 + tendencia)"]
tot = [prod["variantes"][v]["total"] / 1e6 for v in var]
axs[0].bar(etq, tot, color=["#2a6f97", "#9bb7cc", "#9bb7cc", "#9bb7cc"])
for i, v in enumerate(tot):
    axs[0].text(i, v + 0.5, f"{v:.1f}", ha="center", fontsize=8)
axs[0].set_ylabel("total predicho 2025 (millones de CU)"); axs[0].set_title("Sensibilidad del total a las dos reglas descartadas", fontsize=9)
x = np.arange(3); w = 0.25
for j, (lab, key) in enumerate([("2023 observado último", "2023"), ("2024 observado último", "2024"), ("2025 predicho", "2025")]):
    vals = [ctx[l][key]["tasa_ultima_obs"] if key != "2025" else ctx[l]["2025"]["tasa_predicha"] for l in LINEAS]
    axs[1].bar(x + (j - 1) * w, vals, w, label=lab, color=["#cfcfcf", "#9a9a9a", "#2a6f97"][j])
axs[1].set_xticks(x, LINEAS); axs[1].set_ylabel("CU por año de exposición"); axs[1].legend(frameon=False, fontsize=7)
axs[1].set_title("Tasa por exposición: historia y predicción", fontsize=9)
fig.tight_layout(); fig.savefig(FIG / "f07_sensibilidad_2025.png"); plt.close(fig)

# f08: curvas del transformer (preentrenamiento en el desarrollado)
fig, axs = plt.subplots(1, 3, figsize=(10, 3))
for ax, l in zip(axs, LINEAS):
    h = json.load(open(TRABAJO / "modelos" / "dev" / f"N_pre_{l}.json"))["hist"]
    ep = [r[0] for r in h]; tr = [r[1] for r in h]; vl = [r[2] for r in h]
    ax.plot(ep, tr, "o-", ms=3, color="#999", label="entrenamiento"); ax.plot(ep, vl, "o-", ms=3, color=C[l], label="validación")
    ax.set_title(l); ax.set_xlabel("época"); ax.legend(frameon=False, fontsize=7)
axs[0].set_ylabel("desviación Tweedie media (y con tope)")
fig.suptitle("Preentrenamiento del FT-Transformer en el mercado desarrollado 2021-2023", fontsize=10)
fig.tight_layout(); fig.savefig(FIG / "f08_transformer_preentrenamiento.png"); plt.close(fig)
guardar_json(out, ENTREGA / "figuras_datos.json")
print(json.dumps(out, indent=0)[:3000])
'''


### Embedded source: `ml/codigo/gbm.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/gbm.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/gbm.py'] = '''"""Modelos de árboles: R0, T1 (solo emergente), T2 (desarrollado transferido), T3 (ajuste fino por boosting).

Uso: python gbm.py fase   con fase en {dev, hold, prod}
- dev: OOF de 2023 con cinco pliegues por cliente (desarrollado 2021-2023 para preentrenar)
- hold: entrena con 2023 y predice 2024 (desarrollado 2021-2023)
- prod: entrena con 2023-2024 y predice 2025 (desarrollado 2021-2024)
"""
from __future__ import annotations
import sys, time, hashlib, os
import numpy as np, pandas as pd, lightgbm as lgb
from comun import *
from variables import *

FASE = sys.argv[1] if len(sys.argv) > 1 else "dev"
SOLO = sys.argv[2].split(",") if len(sys.argv) > 2 else None
CONF = {"dev": dict(clima="hasta2023", dev_years=[2021, 2022, 2023], em_train=[2023], em_eval=None),
        "hold": dict(clima="hasta2023", dev_years=[2021, 2022, 2023], em_train=[2023], em_eval=2024),
        "prod": dict(clima="hasta2024", dev_years=[2021, 2022, 2023, 2024], em_train=[2023, 2024], em_eval=2025),
        "pcv": dict(clima="hasta2024", dev_years=[2021, 2022, 2023, 2024], em_train=[2023, 2024], em_eval=None)}[FASE]
OUT = TRABAJO / "pred" / FASE
OUT.mkdir(parents=True, exist_ok=True)
MOD = TRABAJO / "modelos" / {"pcv": "prod", "hold": "dev"}.get(FASE, FASE)  # el preentrenamiento se comparte
MOD.mkdir(parents=True, exist_ok=True)

P_DEV = dict(objective="tweedie", tweedie_variance_power=1.5, learning_rate=0.05, num_leaves=63,
             min_data_in_leaf=1000, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0,
             max_cat_to_onehot=8, cat_smooth=50, verbose=-1, num_threads=2, seed=20261024)
P_EM = dict(objective="tweedie", tweedie_variance_power=1.5, learning_rate=0.02, num_leaves=15,
            min_data_in_leaf=100, feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0,
            max_cat_to_onehot=8, cat_smooth=50, verbose=-1, num_threads=2, seed=20261024)
P_FT = dict(P_EM, learning_rate=0.01, num_leaves=7, min_data_in_leaf=200, lambda_l2=20.0)
# E07: hiperparámetros del desarrollado elegidos por ramo con la validación interna del desarrollado
P_DEVB = {"motor": dict(P_DEV, learning_rate=0.03, lambda_l2=0.0, min_data_in_leaf=300),
          "buildings": dict(P_DEV, learning_rate=0.03, lambda_l2=0.0, min_data_in_leaf=3000),
          "contents": dict(P_DEV, learning_rate=0.03, lambda_l2=10.0, min_data_in_leaf=3000)}
BACKBONE = os.environ.get("MLIND_BACKBONE", "T2")


def interno(cids):
    return np.array([int(hashlib.sha256(("mlind_inner|" + c).encode()).hexdigest()[:8], 16) % 5 == 0 for c in cids])


def ajustar(X, y, off, params, max_rounds, es=200, val=None):
    """Ajuste con parada temprana interna por cliente (val: máscara booleana) o rondas fijas."""
    if val is not None:
        dtr = lgb.Dataset(X[~val], y[~val], init_score=off[~val], free_raw_data=False)
        dva = lgb.Dataset(X[val], y[val], init_score=off[val], reference=dtr)
        b = lgb.train(params, dtr, max_rounds, valid_sets=[dva], callbacks=[lgb.early_stopping(es, verbose=False)])
        n = max(b.best_iteration, 1)
        # reajuste con todo el entrenamiento y n rondas
        b = lgb.train(params, lgb.Dataset(X, y, init_score=off), n)
        return b, n
    b = lgb.train(params, lgb.Dataset(X, y, init_score=off), max_rounds)
    return b, max_rounds


def preentrenar(linea):
    """T2: LightGBM Tweedie en el desarrollado con el esquema común."""
    f = MOD / f"T2_{linea}.txt"
    d = cargar("developed", linea, CONF["clima"])
    d = d[d.year.isin(CONF["dev_years"])]
    cols = comunes(linea)
    return d, cols, f


def correr(linea):
    t0 = time.time()
    em = cargar("emerging", linea, CONF["clima"])
    dv = cargar("developed", linea, CONF["clima"])
    dv = dv[dv.year.isin(CONF["dev_years"])].reset_index(drop=True)
    colsC = comunes(linea)
    if BACKBONE == "T2c":  # A04: sin historia (sus NaN delatan el primer año de datos)
        colsC = [c for c in colsC if c not in ("hist_pol_12m", "hist_cli_12m")]
    colsE = colsC + SOLO_EMERGENTE
    todas = colsE + MARKETING + TARIFA + MAKE
    cats = categorias_union([dv, em], todas)
    # ---- T2: preentrenamiento en desarrollado
    Xd = matriz(dv, colsC, cats); yd = dv.y.values; offd = np.log(dv.exposure.values * dv.c_fila.values)
    vald = dv.fold.values == 0
    f2 = MOD / f"{BACKBONE}_{linea}.txt"
    pdev = P_DEV if BACKBONE == "T2" else P_DEVB[linea]  # T2b y T2c comparten E07
    if f2.exists():
        b2 = lgb.Booster(model_file=str(f2)); n2 = b2.current_iteration()
    else:
        b2, n2 = ajustar(Xd, yd, offd, pdev, 5000, es=150, val=vald)
        b2.save_model(str(f2))
    # renivelado en el propio desarrollado (solo informativo: el nivel emergente lo fija k2)
    kdev = factor_tweedie(yd, np.exp(b2.predict(Xd, raw_score=True) + offd))
    print(linea, "T2 rondas", n2, "s", round(time.time() - t0))
    imp = pd.Series(b2.feature_importance("gain"), index=colsC).sort_values(ascending=False)
    imp.to_csv(MOD / f"{BACKBONE}_{linea}_importancia.csv")
    pd.DataFrame({"row_id": em.row_id.values, "raw": b2.predict(matriz(em, colsC, cats), raw_score=True)}).to_parquet(
        OUT / f"{BACKBONE}raw_{linea}.parquet", index=False)
    # ---- emergente
    tr = em[(em.conjunto == "train") & em.year.isin(CONF["em_train"])].reset_index(drop=True)
    if CONF["em_eval"] is None:
        ev_sets = [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)]
    else:
        ev = em[em.year == CONF["em_eval"]].reset_index(drop=True)
        ev_sets = [(tr, ev)]
    variantes = {
        "T1": colsE, "T1mk": colsE + MARKETING, "T1wp": colsE + TARIFA, "T1mm": colsE + MAKE,
        "T3": colsE, "T3wp": colsE + TARIFA, "T3mk": colsE + MARKETING,
    }
    if SOLO:
        variantes = {k: v for k, v in variantes.items() if k in SOLO}
    if BACKBONE != "T2":
        variantes = {k.replace("T3", "T3b"): v for k, v in variantes.items() if k.startswith("T3")}
    res = {k: [] for k in ["R0", BACKBONE] + list(variantes)}
    rondas = {k: [] for k in variantes}
    for a, b in ev_sets:
        offa = np.log(a.exposure.values * a.c_fila.values)
        ya = a.y.values
        # R0: tasa última por ramo (factor Tweedie sobre exposición efectiva)
        k0 = factor_tweedie(ya, np.exp(offa))
        res["R0"].append(pd.DataFrame({"row_id": b.row_id.values, "mu": k0 * b.exposure.values}))
        # T2: relatividades del desarrollado x nivel emergente
        ra = b2.predict(matriz(a, colsC, cats), raw_score=True)
        rb = b2.predict(matriz(b, colsC, cats), raw_score=True)
        k2 = factor_tweedie(ya, np.exp(ra + offa))
        res[BACKBONE].append(pd.DataFrame({"row_id": b.row_id.values, "mu": k2 * np.exp(rb) * b.exposure.values}))
        val = interno(a.customer_id.values)
        for nom, cols in variantes.items():
            Xa = matriz(a, cols, cats); Xb = matriz(b, cols, cats)
            if nom.startswith("T1"):  # solo con BACKBONE T2
                base_a = np.full(len(a), np.log(k0)); base_b = np.full(len(b), np.log(k0))
                bm, n = ajustar(Xa, ya, offa + base_a, P_EM, 3000, es=200, val=val)
            else:
                base_a = ra + np.log(k2); base_b = rb + np.log(k2)
                bm, n = ajustar(Xa, ya, offa + base_a, P_FT, 2000, es=200, val=val)
            # E02: reajuste de nivel dentro del pliegue (factor Tweedie en entrenamiento)
            kin = factor_tweedie(ya, np.exp(bm.predict(Xa, raw_score=True) + base_a + offa))
            mu = kin * np.exp(bm.predict(Xb, raw_score=True) + base_b) * b.exposure.values
            rondas[nom].append(n)
            res[nom].append(pd.DataFrame({"row_id": b.row_id.values, "mu": mu}))
            if CONF["em_eval"] is not None:
                (TRABAJO / "modelos" / FASE).mkdir(parents=True, exist_ok=True)
                bm.save_model(str(TRABAJO / "modelos" / FASE / f"{nom}_{linea}.txt"))
    for k, v in res.items():
        out = pd.concat(v, ignore_index=True)
        out.to_parquet(OUT / f"{k}_{linea}.parquet", index=False)
    guardar_json({"rondas": rondas, f"{BACKBONE}_rondas": n2, "k_desarrollado": kdev, "segundos": round(time.time() - t0)},
                 OUT / f"gbm_{BACKBONE}_{linea}_meta.json")
    print(linea, "listo", rondas, round(time.time() - t0), "s")


if __name__ == "__main__":
    lineas = [x for x in LINEAS if (len(sys.argv) <= 3 or x in sys.argv[3].split(","))]
    for l in lineas:
        correr(l)
'''


### Embedded source: `ml/codigo/glm.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/glm.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/glm.py'] = '''"""G1: GLM Tweedie (log, p=1,5) solo emergente con variables categorizadas y ridge elegido por validación interna."""
import sys
import numpy as np, pandas as pd
from comun import *
from variables import *
from adaptacion import ajustar, interno

FASE = sys.argv[1] if len(sys.argv) > 1 else "dev"
CONF = {"dev": ("hasta2023", [2023], None), "hold": ("hasta2023", [2023], 2024), "prod": ("hasta2024", [2023, 2024], 2025),
        "pcv": ("hasta2024", [2023, 2024], None)}[FASE]
OUT = TRABAJO / "pred" / FASE
LAMBDAS = [1e2, 1e3, 1e4, 3e4, 1e5, 3e5, 1e6]


def diseno(df, cols, ref=None):
    """Una columna por nivel; numéricas en quintiles (cortes del entrenamiento) más nivel ausente."""
    ref = {} if ref is None else ref
    bloques = []; nombres = []
    for c in cols:
        if c in CATEGORICAS or df[c].dtype == bool or df[c].nunique() <= 6:
            v = df[c].astype(object).where(df[c].notna(), "NA").astype(str).values
            if c not in ref:
                ref[c] = ("cat", sorted(set(v)))
        else:
            x = pd.to_numeric(df[c], errors="coerce").astype(float).values
            if c not in ref:
                q = np.unique(np.nanquantile(x, [0.2, 0.4, 0.6, 0.8])) if np.isfinite(x).any() else np.array([])
                ref[c] = ("num", q)
            q = ref[c][1]
            v = np.where(np.isnan(x), "NA", np.searchsorted(q, x, side="right").astype(str)).astype(str)
            if ref[c][0] == "num" and len(ref[c]) == 2:
                ref[c] = ("num", q, sorted(set(v)))
        niveles = ref[c][1] if ref[c][0] == "cat" else ref[c][2]
        for n in niveles:
            bloques.append((v == n).astype(np.float32)); nombres.append(f"{c}={n}")
    return np.column_stack(bloques), nombres, ref


for l in LINEAS:
    em = cargar("emerging", l, CONF[0])
    cols = [c for c in comunes(l) + SOLO_EMERGENTE]
    tr = em[(em.conjunto == "train") & em.year.isin(CONF[1])].reset_index(drop=True)
    sets = [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)] if CONF[2] is None else [(tr, em[em.year == CONF[2]].reset_index(drop=True))]
    preds = []; lams = []
    for a, b in sets:
        Za, nom, ref = diseno(a, cols); Zb, _, _ = diseno(b, cols, ref)
        offa = np.log(a.exposure.values * a.c_fila.values); offb = np.log(b.exposure.values)
        ya = a.y.values; v = interno(a.customer_id.values)
        best = None
        for lam in LAMBDAS:
            bt = ajustar(Za[~v], ya[~v], offa[~v], lam)
            dv_ = desviacion(ya[v], np.exp(offa[v] + bt[0] + Za[v] @ bt[1:])).sum()
            if best is None or dv_ < best[0]:
                best = (dv_, lam)
        lams.append(best[1])
        bt = ajustar(Za, ya, offa, best[1])
        mu_a = np.exp(offa + bt[0] + Za @ bt[1:])
        kin = factor_tweedie(ya, mu_a)
        preds.append(pd.DataFrame({"row_id": b.row_id.values, "mu": kin * np.exp(offb + bt[0] + Zb @ bt[1:])}))
    pd.concat(preds).to_parquet(OUT / f"G1_{l}.parquet", index=False)
    print(l, "lambdas", lams, flush=True)
'''


### Embedded source: `ml/codigo/historia.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/historia.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/historia.py'] = '''"""Historia de avisos en ventana de 365 días por póliza y por cliente (todos los ramos)."""
import numpy as np, pandas as pd
from comun import *
INICIO = {"developed": "2021-01-01", "emerging": "2023-01-01"}
for m in ["developed", "emerging"]:
    c = pd.read_parquet(DATOS / m / "claims.parquet", columns=["claim_id", "policy_id", "line", "report_date", "loss_date"])
    c["report_date"] = pd.to_datetime(c.report_date)
    if m == "developed":  # mismos duplicados que en preparar_datos
        full = pd.read_parquet(DATOS / m / "claims.parquet")
        cols = [x for x in full.columns if x != "claim_id"]
        q = full.duplicated(cols, keep=False) & (full.claim_id.str.len() == 11)
        c = c[~q.values]
    bases = {l: pd.read_parquet(TRABAJO / f"base_{m}_{l}.parquet", columns=["row_id", "policy_id", "customer_id", "line", "period_start"]) for l in LINEAS}
    mapa = pd.concat([b[["policy_id", "line", "customer_id"]] for b in bases.values()]).drop_duplicates(["policy_id", "line"])
    cc = c.merge(mapa, on=["policy_id", "line"])
    for l, df in bases.items():
        df["period_start"] = pd.to_datetime(df.period_start)
        ok = (df.period_start - pd.Timedelta(days=365)) >= pd.Timestamp(INICIO[m])
        hp = c[c.line == l][["policy_id", "report_date"]].merge(df[["row_id", "policy_id", "period_start"]], on="policy_id")
        hp = hp[(hp.report_date < hp.period_start) & (hp.report_date >= hp.period_start - pd.Timedelta(days=365))]
        hc = cc[["customer_id", "report_date"]].merge(df[["row_id", "customer_id", "period_start"]], on="customer_id")
        hc = hc[(hc.report_date < hc.period_start) & (hc.report_date >= hc.period_start - pd.Timedelta(days=365))]
        out = pd.DataFrame({"row_id": df.row_id,
                            "hist_pol_12m": df.row_id.map(hp.groupby("row_id").size()).fillna(0).where(ok, np.nan),
                            "hist_cli_12m": df.row_id.map(hc.groupby("row_id").size()).fillna(0).where(ok, np.nan)})
        out.to_parquet(TRABAJO / f"historia_{m}_{l}.parquet", index=False)
        print(m, l, out.hist_pol_12m.notna().mean().round(3), out.hist_pol_12m.mean().round(4), out.hist_cli_12m.mean().round(4))
'''


### Embedded source: `ml/codigo/historia2.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/historia2.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/historia2.py'] = '''"""Historia v2 (A05): misma ventana de 365 días que historia.py, pero la ventana termina en
min(inicio de la póliza, V) para que no cuente como cero un tramo sin observar. Solo cambia
filas que empiezan después de V (el test 2025); el resto queda idéntico (se comprueba)."""
import numpy as np, pandas as pd
from comun import *
INICIO = {"developed": "2021-01-01", "emerging": "2023-01-01"}
FIN = pd.Timestamp(str(V))
for m in ["developed", "emerging"]:
    c = pd.read_parquet(DATOS / m / "claims.parquet", columns=["claim_id", "policy_id", "line", "report_date", "loss_date"])
    c["report_date"] = pd.to_datetime(c.report_date)
    if m == "developed":
        full = pd.read_parquet(DATOS / m / "claims.parquet")
        cols = [x for x in full.columns if x != "claim_id"]
        q = full.duplicated(cols, keep=False) & (full.claim_id.str.len() == 11)
        c = c[~q.values]
    bases = {l: pd.read_parquet(TRABAJO / f"base_{m}_{l}.parquet", columns=["row_id", "policy_id", "customer_id", "line", "period_start"]) for l in LINEAS}
    mapa = pd.concat([b[["policy_id", "line", "customer_id"]] for b in bases.values()]).drop_duplicates(["policy_id", "line"])
    cc = c.merge(mapa, on=["policy_id", "line"])
    for l, df in bases.items():
        df["period_start"] = pd.to_datetime(df.period_start)
        df["fin"] = df.period_start.where(df.period_start <= FIN, FIN)
        ok = (df.fin - pd.Timedelta(days=365)) >= pd.Timestamp(INICIO[m])
        hp = c[c.line == l][["policy_id", "report_date"]].merge(df[["row_id", "policy_id", "fin"]], on="policy_id")
        hp = hp[(hp.report_date < hp.fin) & (hp.report_date >= hp.fin - pd.Timedelta(days=365))]
        hc = cc[["customer_id", "report_date"]].merge(df[["row_id", "customer_id", "fin"]], on="customer_id")
        hc = hc[(hc.report_date < hc.fin) & (hc.report_date >= hc.fin - pd.Timedelta(days=365))]
        out = pd.DataFrame({"row_id": df.row_id,
                            "hist_pol_12m": df.row_id.map(hp.groupby("row_id").size()).fillna(0).where(ok, np.nan),
                            "hist_cli_12m": df.row_id.map(hc.groupby("row_id").size()).fillna(0).where(ok, np.nan)})
        old = pd.read_parquet(TRABAJO / f"historia_{m}_{l}.parquet")
        assert (old.row_id.values == out.row_id.values).all()
        tarde = (df.period_start > FIN).values
        for v in ["hist_pol_12m", "hist_cli_12m"]:
            a, b = old[v].values[~tarde], out[v].values[~tarde]
            assert np.array_equal(np.isnan(a), np.isnan(b)) and np.allclose(np.nan_to_num(a), np.nan_to_num(b)), (m, l, v)
        out.to_parquet(TRABAJO / f"historia2_{m}_{l}.parquet", index=False)
        if tarde.any():
            print(m, l, "filas tras V:", int(tarde.sum()), "media hist_pol antes/después:",
                  round(float(np.nanmean(old.hist_pol_12m.values[tarde])), 4), round(float(np.nanmean(out.hist_pol_12m.values[tarde])), 4),
                  "cli:", round(float(np.nanmean(old.hist_cli_12m.values[tarde])), 4), round(float(np.nanmean(out.hist_cli_12m.values[tarde])), 4), flush=True)
'''


### Embedded source: `ml/codigo/mezcla_designada.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/mezcla_designada.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/mezcla_designada.py'] = '''"""Opción preparada, sin enviar: mezcla 50/50 de la producción designada (A1fixF1b3n) y la final de MLind."""
import json
import numpy as np, pandas as pd
from comun import *
OTROS = RAIZ / "comparacion" / "otros"
sub = pd.read_parquet(DATOS / "emerging" / "sample_submission.parquet")[["row_id"]]
a = pd.read_csv(PRIVADO / "prediccion_2025_mlind.csv", dtype={"row_id": str})
b = pd.read_csv(OTROS / "A1fixF1b3n_2025.csv", dtype={"row_id": str})
assert (a.row_id.values == sub.row_id.values).all() and (b.row_id.values == sub.row_id.values).all()
m = pd.DataFrame({"row_id": sub.row_id, "prediction": 0.5 * a.prediction.values + 0.5 * b.prediction.values})
assert m.prediction.notna().all() and (m.prediction > 0).all() and len(m) == 209063 and m.row_id.is_unique
f = PRIVADO / "mezcla50_designada_mlind_2025.csv"
m.to_csv(f, index=False, float_format="%.6f", lineterminator="\\n")
pref = m.row_id.str[0]
r = {"archivo": f.name, "sha256": sha256(f), "filas": len(m), "total": float(m.prediction.sum()),
     "por_ramo": {k: float(v) for k, v in m.groupby(pref).prediction.sum().items()},
     "fuentes": {"mlind": sha256(PRIVADO / "prediccion_2025_mlind.csv"), "designada": sha256(OTROS / "A1fixF1b3n_2025.csv")}}
guardar_json(r, RAIZ / "comparacion" / "mezcla50_info.json"); print(json.dumps(r, indent=1))
'''


### Embedded source: `ml/codigo/mezcla_diagnostico.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/mezcla_diagnostico.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/mezcla_diagnostico.py'] = '''"""Diagnósticos de la combinación 50/50 designada + MLind (PREINSCRIPCION_MEZCLA.md). No cambian el archivo."""
import json
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression
from comun import *
import receta

OTROS = RAIZ / "comparacion" / "otros"; OUT = RAIZ / "mezcla"
res = {"fuentes": {f: sha256(p) for f, p in {"designada": OTROS / "A1fixF1b3n_2025.csv", "mlind": PRIVADO / "prediccion_2025_mlind.csv",
       "mezcla": PRIVADO / "mezcla50_designada_mlind_2025.csv", "pred2024_noct": OTROS / "predicciones_2024.parquet"}.items()}}
# ------------------------------------------------ 2024
o = pd.read_parquet(OTROS / "predicciones_2024.parquet")
p = receta.parametros("dev", ["T2b"], "ramo"); t, mu = receta.aplicar("hold", p, None)
d = t.merge(o[["row_id", "address_id", "A1"]], on="row_id")
assert len(d) == 59202
d["T2b"] = mu * t.c_fila.values
y = d.y.values; A = d.A1.values; T = d.T2b.values
d["ari"] = 0.5 * A + 0.5 * T; d["geo"] = np.sqrt(A * T)
padre = {}
def f(x):
    while padre.setdefault(x, x) != x:
        padre[x] = padre[padre[x]]; x = padre[x]
    return x
for c, a in zip("C" + d.customer_id, "A" + d.address_id):
    rc, ra = f(c), f(a)
    if rc != ra: padre[ra] = rc
_, gi = np.unique([f("C" + c) for c in d.customer_id], return_inverse=True); G = gi.max() + 1
ws = np.round(np.arange(0, 1.0001, 0.05), 2)
rng = np.random.default_rng(20261024)
res["curva_2024"] = {}
for l in LINEAS + ["todos"]:
    i = np.ones(len(d), bool) if l == "todos" else (d.line == l).values
    Dm = np.column_stack([desviacion(y[i], w * T[i] + (1 - w) * A[i]) for w in ws])
    nulo = desviacion(y[i], np.full(i.sum(), y[i].mean())).sum()
    curva = 1 - Dm.sum(0) / nulo
    gl = gi[i]; _, gl = np.unique(gl, return_inverse=True); Gl = gl.max() + 1
    opt = []
    for s in range(0, 1000, 100):
        Wc = rng.poisson(1.0, size=(100, Gl)).astype(float)
        S = np.zeros((100, len(ws)))
        for j in range(len(ws)):
            S[:, j] = np.bincount(gl, weights=Dm[:, j], minlength=Gl) @ Wc.T
        opt.append(ws[S.argmin(1)])
    opt = np.concatenate(opt)
    res["curva_2024"][l] = {"pesos_MLind": ws.tolist(), "D2": curva.tolist(), "w_optimo": float(ws[curva.argmax()]),
                            "w_optimo_ic95": [float(np.quantile(opt, .025)), float(np.quantile(opt, .975))],
                            "D2_w0": float(curva[0]), "D2_w05": float(curva[10]), "D2_w1": float(curva[-1]), "D2_max": float(curva.max())}
# aritmética frente a geométrica y calibración
def murphy_mcb(yy, mm, ee):
    r = mm / ee
    iso = IsotonicRegression(out_of_bounds="clip").fit(r, yy / ee, sample_weight=np.sqrt(ee))
    mc = np.maximum(iso.predict(r) * ee, 1e-9)
    return float((desviacion(yy, mm).sum() - desviacion(yy, mc).sum()) / desviacion(yy, mm).sum())
res["2024"] = {}
for m in ["A1", "T2b", "ari", "geo"]:
    r = {"D2": d2(y, d[m].values)}
    for l in LINEAS:
        i = (d.line == l).values
        r[l] = {"D2": d2(y[i], d[m].values[i]), "AE": float(y[i].sum() / d[m].values[i].sum()),
                "AE_NB": float(y[i & d.new_business.values].sum() / d[m].values[i & d.new_business.values].sum()),
                "AE_REN": float(y[i & ~d.new_business.values].sum() / d[m].values[i & ~d.new_business.values].sum()),
                "MCB_sobre_S": murphy_mcb(y[i], d[m].values[i], d.exposure.values[i])}
    res["2024"][m] = r
# ------------------------------------------------ 2025
sub = pd.read_parquet(DATOS / "emerging" / "sample_submission.parquet")[["row_id"]]
base = pd.concat([pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet", columns=["row_id", "policy_id", "exposure", "new_business", "conjunto"]).assign(line=l) for l in LINEAS])
q = sub.merge(base[base.conjunto == "test"], on="row_id", how="left")
for k, fp in {"designada": OTROS / "A1fixF1b3n_2025.csv", "mlind": PRIVADO / "prediccion_2025_mlind.csv", "mezcla": PRIVADO / "mezcla50_designada_mlind_2025.csv"}.items():
    x = pd.read_csv(fp, dtype={"row_id": str}); assert (x.row_id.values == q.row_id.values).all(); q[k] = x.prediction.values
assert np.allclose(q.mezcla, 0.5 * q.designada + 0.5 * q.mlind, rtol=1e-6, atol=1e-5)
res["2025"] = {"controles": {"filas": int(len(q)), "nulos": int(q.mezcla.isna().sum()), "min": float(q.mezcla.min()),
                             "positivas": bool((q.mezcla > 0).all()), "orden_plantilla": True,
                             "max_error_media": float(np.abs(q.mezcla - 0.5 * q.designada - 0.5 * q.mlind).max())}}
for k in ["designada", "mlind", "mezcla"]:
    r = {"total": float(q[k].sum())}
    for l in LINEAS:
        i = (q.line == l).values; e = q.exposure.values[i]; nb = q.new_business.values[i]
        r[l] = {"total": float(q[k].values[i].sum()), "tasa": float(q[k].values[i].sum() / e.sum()),
                "tasa_NB": float(q[k].values[i][nb].sum() / e[nb].sum()), "tasa_REN": float(q[k].values[i][~nb].sum() / e[~nb].sum())}
    res["2025"][k] = r
# ------------------------------------------------ prima comercial de las 20 referencias
ex = pd.read_parquet(DATOS / "emerging" / "expenses.parquet").set_index("item").emerging
ref = pd.read_parquet(DATOS / "emerging" / "reference_policies.parquet")
com = {"direct": ex["commission_direct"], "broker": ex["commission_broker"], "comparison_site": ex["commission_comparison_site"]}
exp_pol = q.groupby("policy_id").exposure.sum()
filas = []
for _, r in ref.iterrows():
    R = float(q.loc[q.row_id == r.row_id, "mezcla"].iloc[0]); e = r.exposure
    cat = ex["cat_reinsurance"] * R if r.line in ("buildings", "contents") else 0.0
    den = 1 - ex["variable_expense"] - com[r.channel] - ex["premium_tax"] - ex["target_profit_margin"]
    F = ex["fixed_expense_per_policy"] * e
    P = (R + cat + F) / den
    Fpol = ex["fixed_expense_per_policy"] * e / exp_pol[r.policy_id]  # 85 por póliza repartidos entre sus filas de 2025
    filas.append({"row_id": r.row_id, "ramo": r.line, "canal": r.channel, "nuevo_negocio": bool(r.new_business), "exposicion": e,
                  "riesgo_designada": float(q.loc[q.row_id == r.row_id, "designada"].iloc[0]),
                  "riesgo_mlind": float(q.loc[q.row_id == r.row_id, "mlind"].iloc[0]), "riesgo_mezcla": R,
                  "reaseguro_cat": cat, "gasto_fijo": F, "gasto_variable": P * ex["variable_expense"], "comision": P * com[r.channel],
                  "impuesto": P * ex["premium_tax"], "margen": P * ex["target_profit_margin"], "denominador": den,
                  "prima_comercial": P, "prima_comercial_F_por_poliza": (R + cat + Fpol) / den, "prima_escrita": r.written_premium})
pc = pd.DataFrame(filas); pc.round(2).to_csv(OUT / "prima_comercial_20_mezcla.csv", index=False)
res["prima_comercial"] = {"media_por_ramo": pc.groupby("ramo")[["riesgo_mezcla", "prima_comercial", "prima_escrita"]].mean().round(2).to_dict(),
                          "suma_componentes_ok": bool(np.allclose(pc.riesgo_mezcla + pc.reaseguro_cat + pc.gasto_fijo + pc.gasto_variable + pc.comision + pc.impuesto + pc.margen, pc.prima_comercial))}
guardar_json(res, OUT / "diagnostico_mezcla.json")
# ------------------------------------------------ figura
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 130})
C = {"motor": "#2a6f97", "buildings": "#c4673b", "contents": "#5b8c5a", "todos": "#333333"}
fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.6))
for l in LINEAS + ["todos"]:
    c = res["curva_2024"][l]; base = c["D2"][0]
    axs[0].plot(ws, np.array(c["D2"]) - base, color=C[l], lw=2 if l == "todos" else 1.2, label=f"{l} (óptimo {c['w_optimo']:.2f}, IC95 {c['w_optimo_ic95'][0]:.2f}-{c['w_optimo_ic95'][1]:.2f})")
axs[0].axvline(0.5, color="#999", ls="--", lw=0.8); axs[0].axhline(0, color="#ccc", lw=0.6)
axs[0].set_xlabel("peso de MLind (el resto, A1)"); axs[0].set_ylabel("D² 2024 menos D² de A1 sola")
axs[0].legend(frameon=False, fontsize=6.8); axs[0].set_title("2024 (ya visto por las dos líneas): autos y contenidos admiten 0,5; edificios prefiere MLind", fontsize=8)
x = np.arange(3); w = 0.26
for j, (k, lab, col) in enumerate([("designada", "designada A1fixF1b3n", "#c4673b"), ("mezcla", "mezcla 50/50", "#5b8c5a"), ("mlind", "MLind", "#2a6f97")]):
    vals = [res["2025"][k][l]["tasa"] for l in LINEAS]
    axs[1].bar(x + (j - 1) * w, vals, w, label=lab, color=col)
    for xi, v in zip(x + (j - 1) * w, vals):
        axs[1].text(xi, v + 5, f"{v:.0f}", ha="center", fontsize=7)
axs[1].set_xticks(x, LINEAS); axs[1].set_ylabel("tasa 2025 (CU por año de exposición)"); axs[1].legend(frameon=False, fontsize=7)
axs[1].set_title("Producción 2025 de la combinación", fontsize=8.5)
fig.tight_layout(); fig.savefig(OUT / "f11_mezcla.png"); plt.close(fig)
print(json.dumps({k: v for k, v in res.items() if k not in ("fuentes",)}, indent=1, default=float)[:7000])
'''


### Embedded source: `ml/codigo/pcv_v2.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/pcv_v2.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/pcv_v2.py'] = '''"""OOF 2023-2024 (fase pcv, troncos 2021-2024) de MLind v2 frente a v1 (T2b), pesos y nivel cruzados por pliegue."""
import json, numpy as np, pandas as pd
from comun import *
from evaluar import mezcla_pesos, metricas, bootstrap_dif
import receta
t = receta.tabla("pcv"); comps = ["T2b", "T2h", "FS", "T2o"]; P = receta.componentes("pcv", comps, t)
folds = pliegue(t.customer_id.values)
def oof(cs, enc):
    out = np.zeros(len(t))
    for l in LINEAS:
        il = (t.line == l).values
        for f in range(5):
            te = folds == f; tr = ~te & il
            M = np.column_stack([P[c][tr] * t.c_fila.values[tr] for c in cs])
            w = mezcla_pesos(t.y.values[tr], M) if len(cs) > 1 else np.ones(1)
            w = (1 - enc) * w + enc / len(cs)
            b = np.column_stack([P[c] for c in cs]) @ w
            k = factor_tweedie(t.y.values[tr], b[tr] * t.c_fila.values[tr])
            out[te & il] = b[te & il] * k
    return out
v1 = oof(["T2b"], 0); res = {"v1_T2b": metricas(t, v1)}
for enc in (0, 0.5, 1):
    v2 = oof(comps, enc); nom = f"v2_e{int(enc*100)}"
    res[nom] = metricas(t, v2)
    res[nom + "_vs_v1"] = {"todo": bootstrap_dif(t, v2, v1)}
    for a in (2023, 2024):
        s = (t.year == a).values
        res[nom + "_vs_v1"][str(a)] = bootstrap_dif(t[s].reset_index(drop=True), v2[s], v1[s])
    for nb in (True, False):
        s = (t.new_business == nb).values
        res[nom + "_vs_v1"]["NB" if nb else "REN"] = bootstrap_dif(t[s].reset_index(drop=True), v2[s], v1[s])
    print(nom, "D2 %.4f" % res[nom]["D2"], {k: (round(v["dif"], 4), [round(x, 4) for x in v["ic95"]]) for k, v in res[nom + "_vs_v1"].items()})
print("v1 D2 %.4f" % res["v1_T2b"]["D2"], {k: round(v, 4) for k, v in res["v1_T2b"].items() if k.startswith("D2")})
print("v2 e50", {k: round(v, 4) for k, v in res["v2_e50"].items() if k.startswith("D2") or k.startswith("AE")})
guardar_json(res, ENTREGA / "evolucion" / "pcv_v2_frente_v1.json")
# ponderado a la cartera 2025 (pesos de razón de densidad, solo filas 2023)
s = (t.year == 2023).values
t23 = t[s].reset_index(drop=True).merge(pd.read_parquet(TRABAJO / "pesos_2025_dev.parquet"), on="row_id", how="left")
assert t23.w2025.notna().all()
v2 = oof(comps, 0.5)
res["w2025_2023"] = {"v1": d2(t23.y.values, v1[s] * t23.c_fila.values, t23.w2025.values),
                     "v2_e50": d2(t23.y.values, v2[s] * t23.c_fila.values, t23.w2025.values),
                     "dif": bootstrap_dif(t23, v2[s], v1[s], w=t23.w2025.values)}
print("w2025 2023", {k: (round(v, 4) if isinstance(v, float) else (round(v["dif"], 4), [round(x, 4) for x in v["ic95"]])) for k, v in res["w2025_2023"].items()})
guardar_json(res, ENTREGA / "evolucion" / "pcv_v2_frente_v1.json")
'''


### Embedded source: `ml/codigo/peligros.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/peligros.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/peligros.py'] = '''"""T6 (E05): transferencia por peligro con nivel emergente por peligro y credibilidad.

Uso: python peligros.py fase
"""
from __future__ import annotations
import sys, time
import numpy as np, pandas as pd, lightgbm as lgb
from comun import *
from variables import *
from completitud import por_peligro
from gbm import P_DEV, ajustar, CONF, MOD, OUT

PELIGROS = {"motor": ["bodily_injury", "glass_hail", "own_damage", "theft", "third_party_prop"],
            "buildings": ["accidental_damage", "escape_of_water", "fire", "flood", "storm", "subsidence"],
            "contents": ["accidental_damage", "escape_of_water", "flood", "theft"]}
KCRED = 100.0
FASE = sys.argv[1] if len(sys.argv) > 1 else "dev"

for l in LINEAS:
    t0 = time.time()
    em = cargar("emerging", l, CONF["clima"]); dv = cargar("developed", l, CONF["clima"])
    dv = dv[dv.year.isin(CONF["dev_years"])].reset_index(drop=True)
    cols = comunes(l); cats = categorias_union([dv, em], cols + SOLO_EMERGENTE + MARKETING + TARIFA + MAKE)
    Xd = matriz(dv, cols, cats); Xe = matriz(em, cols, cats)
    raw = {}; rondas = {}; kdevs = {}
    for p in PELIGROS[l]:
        f = MOD / f"T6_{l}_{p}.txt"
        cd = por_peligro(l, p, dv.period_start, dv.period_end)
        offd = np.log(dv.exposure.values * np.maximum(cd, 1e-4))
        if f.exists():
            b = lgb.Booster(model_file=str(f)); n = b.current_iteration()
        else:
            b, n = ajustar(Xd, dv[f"y_{p}"].values, offd, P_DEV, 4000, es=100, val=dv.fold.values == 0)
            b.save_model(str(f))
        rondas[p] = n
        # reajuste del nivel de cada peligro en el propio desarrollado (la L2 con Tweedie lo sesga a la baja)
        kdev = factor_tweedie(dv[f"y_{p}"].values, np.exp(b.predict(Xd, raw_score=True) + offd))
        kdevs[p] = kdev
        raw[p] = b.predict(Xe, raw_score=True) + np.log(kdev)
        em[f"c_{p}"] = np.where(em.conjunto == "test", 1.0, por_peligro(l, p, em.period_start, em.period_end))
    for p in PELIGROS[l]:
        em[f"raw_{p}"] = raw[p]
    em[[c for c in em.columns if c.startswith("raw_") or c == "row_id"]].to_parquet(OUT / f"T6raw_{l}.parquet", index=False)
    tr = em[(em.conjunto == "train") & em.year.isin(CONF["em_train"])].reset_index(drop=True)
    sets = [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)] if CONF["em_eval"] is None else \\
        [(tr, em[em.year == CONF["em_eval"]].reset_index(drop=True))]
    preds = []; niveles = []
    for a, b in sets:
        # nivel común del ramo
        mu_a = sum(np.exp(a[f"raw_{p}"].values) * a.exposure.values * a[f"c_{p}"].values for p in PELIGROS[l])
        kl = factor_tweedie(a.y.values, mu_a)
        kp = {}
        for p in PELIGROS[l]:
            m = np.exp(a[f"raw_{p}"].values) * a.exposure.values * a[f"c_{p}"].values * kl
            kem = factor_tweedie(a[f"y_{p}"].values, m)
            n = int((a[f"y_{p}"].values > 0).sum()); z = n / (n + KCRED)
            kp[p] = kl * (z * kem + (1 - z))
        niveles.append({"k_ramo": kl, **{p: kp[p] for p in kp}})
        mu = sum(kp[p] * np.exp(b[f"raw_{p}"].values) for p in PELIGROS[l]) * b.exposure.values
        preds.append(pd.DataFrame({"row_id": b.row_id.values, "mu": mu}))
    pd.concat(preds).to_parquet(OUT / f"T6_{l}.parquet", index=False)
    guardar_json({"rondas": rondas, "k_desarrollado": kdevs, "niveles": niveles, "s": round(time.time() - t0)}, OUT / f"T6_{l}_meta.json")
    print(l, rondas, [{k: round(v, 3) for k, v in n.items()} for n in niveles][:2], round(time.time() - t0), "s", flush=True)
'''


### Embedded source: `ml/codigo/pesos_2025.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/pesos_2025.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/pesos_2025.py'] = '''"""W01: pesos de razón de densidad (2023 DEV frente a test 2025) para el D2 ponderado a la mezcla de 2025."""
import sys
import numpy as np, pandas as pd, lightgbm as lgb
from comun import *
from variables import *

ANIO = int(sys.argv[1]) if len(sys.argv) > 1 else 2023
CLIMA = "hasta2023"
resumen = {}
salida = []
for l in LINEAS:
    em = cargar("emerging", l, CLIMA)
    cols = [c for c in comunes(l) + SOLO_EMERGENTE if c not in ("hist_pol_12m", "hist_cli_12m")]
    a = em[(em.conjunto == "train") & (em.year == ANIO)]
    b = em[em.conjunto == "test"]
    d = pd.concat([a.assign(z=0), b.assign(z=1)], ignore_index=True)
    cats = categorias_union([d], cols)
    X = matriz(d, cols, cats); z = d.z.values
    fo = pliegue(d.customer_id.values)
    p = np.zeros(len(d))
    params = dict(objective="binary", learning_rate=0.05, num_leaves=31, min_data_in_leaf=200, feature_fraction=0.8,
                  bagging_fraction=0.8, bagging_freq=1, verbose=-1, num_threads=2, seed=7)
    for k in range(5):
        m = lgb.train(params, lgb.Dataset(X[fo != k], z[fo != k]), 300)
        p[fo == k] = m.predict(X[fo == k])
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(z, p)
    imp = pd.Series(m.feature_importance("gain"), index=cols).sort_values(ascending=False)
    pa = np.clip(p[z == 0], 1e-6, 1 - 1e-6)
    w = pa / (1 - pa) * (len(a) / len(b))
    w = np.clip(w, 0.1, 10); w = w / w.mean()
    ess = w.sum() ** 2 / (w ** 2).sum()
    resumen[l] = {"auc": round(float(auc), 4), "n_dev": int(len(a)), "n_test": int(len(b)), "ess": round(float(ess), 1),
                  "nb_dev": round(float(a.new_business.mean()), 4),
                  "nb_ponderado": round(float(np.average(a.new_business, weights=w)), 4),
                  "nb_test": round(float(b.new_business.mean()), 4),
                  "importancia_top": {k: round(float(v / imp.sum()), 4) for k, v in imp.head(8).items()}}
    salida.append(pd.DataFrame({"row_id": a.row_id.values, "w2025": w}))
    print(l, resumen[l])
suf = "dev" if ANIO == 2023 else "hold"
pd.concat(salida).to_parquet(TRABAJO / f"pesos_2025_{suf}.parquet", index=False)
guardar_json(resumen, TRABAJO / f"pesos_2025_{suf}.json")
'''


### Embedded source: `ml/codigo/preparar_datos.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/preparar_datos.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/preparar_datos.py'] = '''"""Etiquetas, completitud de reporte y esquema común de variables (línea ML independiente).

Salida: trabajo/base_{mercado}_{ramo}.parquet y trabajo/preparacion.json.
Todo se calcula desde los Parquet originales con huella comprobada.
"""
from __future__ import annotations
import numpy as np, pandas as pd
from comun import *

TRABAJO.mkdir(parents=True, exist_ok=True)
info = {"huellas": comprobar_huellas()}

UMBRAL_CAPITAL = {"buildings": 5_000_000, "contents": 2_000_000}
CLIMA = ["wind_exposure_index", "wildfire_index", "rainfall_mm", "mean_temperature_c"]
CONSTR = {"brick": "brick", "brk": "brick", "stone": "stone", "stn": "stone", "t": "timber", "w": "timber",
          "wod": "timber", "non_standard": "non_standard", "mix": "non_standard"}


def banda_edad(a):
    return pd.cut(a, [-1, 24, 34, 49, 64, 200], labels=["17-24", "25-34", "35-49", "50-64", "65+"]).astype(str)


def leer_polizas(m):
    out = {}
    for l in LINEAS:
        tr = pd.read_parquet(DATOS / m / f"policies_{l}.parquet")
        tr["conjunto"] = "train"
        if m == "emerging":
            te = pd.read_parquet(DATOS / m / f"test_{l}.parquet"); te["conjunto"] = "test"
            tr = pd.concat([tr, te], ignore_index=True)
        tr["line"] = l
        out[l] = tr
    return out


def siniestros(m):
    c = pd.read_parquet(DATOS / m / "claims.parquet")
    n0 = len(c)
    if m == "developed":
        cols = [x for x in c.columns if x != "claim_id"]
        dup = c.duplicated(cols, keep=False)
        quitar = dup & (c.claim_id.str.len() == 11)
        c = c[~quitar].copy()
        info[f"{m}_duplicados_retirados"] = int(quitar.sum())
    c["loss_date"] = pd.to_datetime(c.loss_date); c["report_date"] = pd.to_datetime(c.report_date)
    c["neto"] = c.incurred_to_date - c.recoveries
    c["aceptado"] = c.status != "repudiated"
    info[f"{m}_siniestros"] = {"leidos": n0, "usados": len(c), "rechazados": int((~c.aceptado).sum()),
                                "neto_negativo": int((c.neto < 0).sum())}
    return c


# ---------- 1. completitud de reporte ponderada por coste (desarrollado, cohortes maduras) ----------
cd = siniestros("developed")
ce = siniestros("emerging")
D = 1095
F = {}
resumen_cdf = {}
for (l, p), g in cd[cd.aceptado].groupby(["line", "peril"]):
    g = g[(V - g.loss_date.values.astype("datetime64[D]")).astype(int) >= D]
    dly = (g.report_date - g.loss_date).dt.days.values
    w = g.neto.values
    ok = dly <= D
    tot = w[ok].sum()
    cum = np.bincount(dly[ok], weights=w[ok], minlength=D + 1).cumsum() / tot
    F[(l, p)] = np.clip(cum, 0, 1)
    resumen_cdf[f"{l}|{p}"] = {k: round(float(cum[k]), 4) for k in [0, 30, 90, 180, 365, 730]}
    resumen_cdf[f"{l}|{p}"]["n"] = int(ok.sum())
info["cdf_coste_desarrollado"] = resumen_cdf
# contraste emergente: cohorte 2023 con al menos 365 días, cociente F(d)/F(365)
contraste = {}
for (l, p), g in ce[ce.aceptado & (ce.loss_date.dt.year == 2023)].groupby(["line", "peril"]):
    g = g[(V - g.loss_date.values.astype("datetime64[D]")).astype(int) >= 365]
    dly = (g.report_date - g.loss_date).dt.days.values
    w = g.neto.values; ok = dly <= 365
    if ok.sum() < 20: continue
    em = [float(w[ok & (dly <= d)].sum() / w[ok].sum()) for d in (30, 90, 180)]
    dv = [float(F[(l, p)][d] / F[(l, p)][365]) for d in (30, 90, 180)]
    contraste[f"{l}|{p}"] = {"n": int(ok.sum()), "emergente_30_90_180": [round(x, 4) for x in em],
                              "desarrollado_30_90_180": [round(x, 4) for x in dv]}
info["contraste_reporte_emergente_2023"] = contraste
G = {k: np.concatenate([[0.0], np.cumsum(np.concatenate([v, np.ones(4000 - len(v))]))]) for k, v in F.items()}


def completitud_media(l, p, s, e):
    """Media de F(V - t) para t en [s, e]; G[x+1] = sum_{d<=x} F(d)."""
    g = G[(l, p)]
    a = (V - e).astype(int); b = (V - s).astype(int)
    a = np.clip(a, 0, None); b = np.clip(b, 0, None)
    return (g[b + 1] - g[a]) / (b - a + 1)


# ---------- 2. bases por mercado ----------
for m in ["developed", "emerging"]:
    c = cd if m == "developed" else ce
    pol = leer_polizas(m)
    cu = pd.read_parquet(DATOS / m / "customers.parquet")
    ad = pd.read_parquet(DATOS / m / "addresses.parquet")
    we = pd.read_parquet(DATOS / m / "weather.parquet")
    # pesos por peligro (coste aceptado de la cohorte más madura del mercado, corregido por completitud media)
    for l in LINEAS:
        df = pol[l].copy()
        n_in = len(df)
        if m == "developed" and l == "motor":
            df["customer_id"] = df.customer_id.str.zfill(8)
        df["period_start"] = pd.to_datetime(df.period_start); df["period_end"] = pd.to_datetime(df.period_end)
        df["exposure_original"] = df.exposure
        df["exposure"] = ((df.period_end - df.period_start).dt.days + 1) / 365.0
        df["year"] = df.period_start.dt.year
        # capital
        if l in UMBRAL_CAPITAL:
            df["sum_insured_raw"] = df.sum_insured
            adj = df.sum_insured > UMBRAL_CAPITAL[l]
            df["audit_capital_div1000"] = adj
            df["sum_insured"] = np.where(adj, df.sum_insured / 1000, df.sum_insured)
            info[f"{m}_{l}_capital_div1000"] = {"train": int((adj & (df.conjunto == "train")).sum()),
                                                  "test": int((adj & (df.conjunto == "test")).sum())}
        # asignación de siniestros
        cl = c[c.line == l]
        mm = cl.merge(df[["row_id", "policy_id", "period_start", "period_end"]], on="policy_id", how="inner")
        mm = mm[(mm.loss_date >= mm.period_start) & (mm.loss_date <= mm.period_end)]
        dup = mm.claim_id.duplicated(keep=False).sum()
        asign = mm.claim_id.nunique()
        info[f"{m}_{l}_asignacion"] = {"siniestros_ramo": int(len(cl)), "asignados": int(asign),
                                        "ambiguos": int(dup), "sin_periodo": int(len(cl) - asign)}
        acc = mm[mm.aceptado]
        y = acc.groupby("row_id").neto.sum()
        df["y"] = df.row_id.map(y).fillna(0.0)
        df.loc[df.conjunto == "test", "y"] = np.nan
        df["n_sin"] = df.row_id.map(acc.groupby("row_id").size()).fillna(0)
        for p_ in sorted(cl.peril.unique()):
            df[f"y_{p_}"] = df.row_id.map(acc[acc.peril == p_].groupby("row_id").neto.sum()).fillna(0.0)
        # completitud por fila
        s = df.period_start.values.astype("datetime64[D]"); e = df.period_end.values.astype("datetime64[D]")
        perils = sorted(cl.peril.unique())
        comp = {p_: completitud_media(l, p_, s, e) for p_ in perils}
        # pesos: coste aceptado del mercado en años completos de entrenamiento, dividido por completitud media
        tr = df.conjunto == "train"
        cost = {p_: df.loc[tr, f"y_{p_}"].sum() for p_ in perils}
        ult = {p_: cost[p_] / max(np.average(comp[p_][tr], weights=df.exposure[tr]), 1e-6) for p_ in perils}
        tot = sum(ult.values())
        wpk = {p_: ult[p_] / tot for p_ in perils}
        df["c_fila"] = sum(wpk[p_] * comp[p_] for p_ in perils)
        df.loc[df.conjunto == "test", "c_fila"] = 1.0
        info[f"{m}_{l}_pesos_peligro"] = {k: round(v, 4) for k, v in wpk.items()}
        info[f"{m}_{l}_completitud_media_por_anio"] = df[tr].groupby("year").apply(
            lambda g: float(np.average(g.c_fila, weights=g.exposure))).round(4).to_dict()
        # historia (avisos reportados antes del inicio, cualquier estado)
        hp = c[c.line == l][["policy_id", "report_date"]].merge(df[["row_id", "policy_id", "period_start"]], on="policy_id")
        hp = hp[hp.report_date < hp.period_start]
        df["hist_pol_n"] = df.row_id.map(hp.groupby("row_id").size()).fillna(0)
        df["prev_rows_pol"] = df.groupby("policy_id").period_start.rank(method="first") - 1
        out_rows = df
        pol[l] = out_rows
        assert len(df) == n_in
    # historia de cliente en todos los ramos
    mapa = pd.concat([pol[l][["policy_id", "customer_id"]].assign(line=l) for l in LINEAS]).drop_duplicates(["policy_id", "line"])
    cc = c[["policy_id", "line", "report_date"]].merge(mapa, on=["policy_id", "line"])
    for l in LINEAS:
        df = pol[l]
        h = cc.merge(df[["row_id", "customer_id", "period_start"]], on="customer_id")
        h = h[h.report_date < h.period_start]
        df["hist_cli_n"] = df.row_id.map(h.groupby("row_id").size()).fillna(0)
    # maestros
    cu = cu.copy()
    if m == "emerging":
        occ = cu.occupation.str.lower().str.strip()
        cu["self_employed"] = occ.str.startswith("self-employed")
        cu["occupation"] = occ.str.replace("self-employed ", "", regex=False).str.strip()
    ad = ad.copy()
    ad["energy_rating"] = ad.energy_rating
    for l in LINEAS:
        df = pol[l].merge(cu, on="customer_id", how="left", validate="m:1").merge(ad, on="address_id", how="left", validate="m:1")
        df["tenure_years"] = df.year - df.customer_since_year
        df["property_age"] = df.year - df.year_built
        df["start_month"] = df.period_start.dt.month
        df["start_doy"] = df.period_start.dt.dayofyear
        df["full_year"] = (df.exposure >= 0.99).astype(int)
        df["log_days_quote"] = np.log1p(df.days_quote_to_inception.clip(lower=0))
        wp = df.written_premium / df.exposure
        med = wp.groupby([df.year]).transform("median")
        df["log_wp_rel"] = np.log(wp / med)
        if l == "motor":
            if m == "developed":
                df["driver_age_band"] = banda_edad(df.driver_age)
            df["age_band"] = df.driver_age_band
        else:
            if m == "developed":
                df["policyholder_age_band"] = banda_edad(df.policyholder_age)
            df["age_band"] = df.policyholder_age_band
            df["log_si"] = np.log(df.sum_insured)
        if l == "buildings":
            df["construction"] = df.construction_type.str.lower().map(CONSTR)
            assert df.construction.notna().all(), df.construction_type[df.construction.isna()].unique()
        df["market"] = m
        df["fold"] = pliegue(df.customer_id.values)
        df.to_parquet(TRABAJO / f"base_{m}_{l}.parquet", index=False)
        info[f"{m}_{l}_filas"] = df.conjunto.value_counts().to_dict()
        info[f"{m}_{l}_T0_por_anio"] = df[df.conjunto == "train"].groupby("year").y.sum().round(2).to_dict()
    # clima persistente: medias por dirección para cada ventana de años
    for nombre, anios in ({"developed": {"hasta2023": [2021, 2022, 2023], "hasta2024": [2021, 2022, 2023, 2024]},
                           "emerging": {"hasta2023": [2023], "hasta2024": [2023, 2024]}}[m]).items():
        wm = we[we.year.isin(anios)].groupby("address_id")[CLIMA].mean().add_prefix("clima_")
        wm.reset_index().to_parquet(TRABAJO / f"clima_{m}_{nombre}.parquet", index=False)
guardar_json(info, TRABAJO / "preparacion.json")
print(json.dumps({k: v for k, v in info.items() if k != "huellas" and "cdf" not in k}, indent=0, default=str)[:6000])
'''


### Embedded source: `ml/codigo/prima_comercial.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/prima_comercial.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/prima_comercial.py'] = '''"""Prima comercial de las 20 pólizas de referencia a partir de la prima de riesgo predicha (producción 2025).

P = [R (1 + cat) + F e] / (1 - gasto variable - comisión del canal - impuesto - margen),
cat = 0,12 solo en hogar (porcentaje de la prima de riesgo de propiedad), F = 85 CU por póliza y año repartido por exposición.
Sensibilidad: F entero por fila.
"""
import json
import numpy as np, pandas as pd
from comun import *

ex = pd.read_parquet(DATOS / "emerging" / "expenses.parquet").set_index("item").emerging
ref = pd.read_parquet(DATOS / "emerging" / "reference_policies.parquet")
pred = pd.read_csv(ENTREGA / "prediccion_2025_mlind.csv", dtype={"row_id": str})
d = ref.merge(pred, on="row_id", how="left")
bal = json.load(open(ENTREGA / "produccion_2025.json"))["balance_oof_pcv"]
assert d.prediction.notna().all()
com = {"direct": ex["commission_direct"], "broker": ex["commission_broker"], "comparison_site": ex["commission_comparison_site"]}
filas = []
for _, r in d.iterrows():
    R0 = r.prediction; e = r.exposure
    R = R0 * bal[r.line]  # prima de riesgo a nivel de balance (media), no al factor que minimiza la desviación
    cat = ex["cat_reinsurance"] * R if r.line in ("buildings", "contents") else 0.0
    c = com[r.channel]
    den = 1 - ex["variable_expense"] - c - ex["premium_tax"] - ex["target_profit_margin"]
    F = ex["fixed_expense_per_policy"] * e
    P = (R + cat + F) / den
    P_alt = (R + cat + ex["fixed_expense_per_policy"]) / den
    filas.append({"row_id": r.row_id, "ramo": r.line, "canal": r.channel, "exposicion": e, "nuevo_negocio": bool(r.new_business),
                  "prediccion_kaggle": R0, "factor_balance": bal[r.line], "prima_riesgo": R, "reaseguro_cat": cat, "gasto_fijo": F, "carga_neta": R + cat + F, "denominador": den,
                  "gasto_variable": P * ex["variable_expense"], "comision": P * c, "impuesto": P * ex["premium_tax"],
                  "margen": P * ex["target_profit_margin"], "prima_comercial": P, "prima_comercial_F_entero": P_alt,
                  "prima_escrita": r.written_premium, "razon_comercial_escrita": P / r.written_premium})
out = pd.DataFrame(filas)
out.round(2).to_csv(ENTREGA / "prima_comercial_20.csv", index=False)
print(out[["row_id", "ramo", "canal", "exposicion", "prima_riesgo", "prima_comercial", "prima_escrita", "razon_comercial_escrita"]].round(2).to_string())
'''


### Embedded source: `ml/codigo/produccion.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/produccion.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/produccion.py'] = '''"""Producción 2025 de la línea ML independiente.

Receta final (A03): T2b (LightGBM Tweedie preentrenado en el desarrollado 2021-2024) x nivel por ramo del emergente
2023-2024 x factor C1 del OOF cruzado de 2023-2024, sin tendencia.
Sensibilidades: C2 (ramo y negocio), tendencia del desarrollado, y la receta preinscrita (C2 + tendencia).
"""
import json
import numpy as np, pandas as pd
from comun import *
import receta

params_c2 = receta.parametros("pcv", ["T2b"], "ramo_nb")
params_c1 = dict(params_c2, calib="ramo")
tend = json.load(open(TRABAJO / "pred" / "prod" / "tendencia.json"))
p2c = receta.parametros("pcv", ["T2c"], "ramo")
variantes = {"final_C1": (params_c1, None), "C2": (params_c2, None), "C1_tendencia": (params_c1, tend),
             "preinscrita_C2_tendencia": (params_c2, tend), "T2c_C1": (p2c, None)}
sub = pd.read_parquet(DATOS / "emerging" / "sample_submission.parquet")
res = {"params": params_c2, "tendencia": tend, "variantes": {}}
for nom, (p, tt) in variantes.items():
    t, mu = receta.aplicar("prod", p, tt)
    out = sub[["row_id"]].merge(pd.DataFrame({"row_id": t.row_id.values, "prediction": mu}), on="row_id", how="left")
    ctl = {"filas": int(len(out)), "orden_igual_plantilla": bool((out.row_id.values == sub.row_id.values).all()),
           "nulos": int(out.prediction.isna().sum()), "min": float(out.prediction.min()),
           "positivas_estrictas": bool((out.prediction > 0).all()), "ids_unicos": bool(out.row_id.is_unique)}
    assert ctl["filas"] == 209063 and ctl["nulos"] == 0 and ctl["positivas_estrictas"] and ctl["orden_igual_plantilla"], ctl
    f = ENTREGA / ("prediccion_2025_mlind.csv" if nom == "final_C1" else f"prediccion_2025_mlind_{nom}.csv")
    out.to_csv(f, index=False, float_format="%.6f", lineterminator="\\n")
    ctl["sha256"] = sha256(f); ctl["archivo"] = f.name
    tot = {}
    for l in LINEAS:
        i = (t.line == l).values; e = t.exposure.values[i]; nb = t.new_business.values[i]
        tot[l] = {"filas": int(i.sum()), "exposicion": float(e.sum()), "total": float(mu[i].sum()), "tasa": float(mu[i].sum() / e.sum()),
                  "tasa_NB": float(mu[i][nb].sum() / e[nb].sum()), "tasa_REN": float(mu[i][~nb].sum() / e[~nb].sum())}
    res["variantes"][nom] = {"controles": ctl, "totales": tot, "total": float(mu.sum())}
    print(nom, round(mu.sum()), {l: round(v["tasa"], 1) for l, v in tot.items()}, ctl["sha256"][:12])
# factor de balance (sum y / sum mu c) por ramo en el OOF cruzado 2023-2024 de la receta final: para la prima comercial
t, mu, _ = receta.oof_cruzado("pcv", ["T2b"], "ramo")
res["balance_oof_pcv"] = {l: float(t.y[(t.line == l).values].sum() / (mu * t.c_fila.values)[(t.line == l).values].sum()) for l in LINEAS}
guardar_json(res, ENTREGA / "produccion_2025.json")
print("balance", res["balance_oof_pcv"])
'''


### Embedded source: `ml/codigo/produccion2.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/produccion2.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/produccion2.py'] = '''"""Producción 2025 de MLind v2 (E14-E21): mezcla convexa por ramo de troncos transferidos, nivel C1, sin tendencia.

Uso: python3 produccion2.py <nombre> <encoge> comp1 [comp2 ...]
- Pesos por ramo con el OOF de 2023-2024 (fase pcv), encogidos hacia pesos iguales en la fracción <encoge>.
- Nivel C1: factor Tweedie por ramo de la mezcla en el OOF pcv (igual que produccion.py).
- Escribe privado/prediccion_2025_<nombre>.csv, la combinación 50/50 con la designada y entrega/produccion_<nombre>.json.
"""
import sys
import numpy as np, pandas as pd
from comun import *
from evaluar import mezcla_pesos, metricas
import receta

NOM, ENC, COMPS = sys.argv[1], float(sys.argv[2]), sys.argv[3:]
PRIV = RAIZ / "privado"
DESIGNADA = RAIZ / "comparacion" / "otros" / "A1fixF1b3n_2025.csv"
SHA_DESIGNADA = "f786deb3"


def pesos(t, P, il):
    M = np.column_stack([P[c][il] * t.c_fila.values[il] for c in COMPS])
    w = mezcla_pesos(t.y.values[il], M) if len(COMPS) > 1 else np.ones(1)
    return (1 - ENC) * w + ENC / len(COMPS)


def controles(out, sub):
    c = {"filas": int(len(out)), "orden_igual_plantilla": bool((out.row_id.values == sub.row_id.values).all()),
         "nulos": int(out.prediction.isna().sum()), "min": float(out.prediction.min()),
         "positivas_estrictas": bool((out.prediction > 0).all()), "ids_unicos": bool(out.row_id.is_unique)}
    assert c["filas"] == 209063 and c["nulos"] == 0 and c["positivas_estrictas"] and c["orden_igual_plantilla"], c
    return c


tp = receta.tabla("pcv"); Pp = receta.componentes("pcv", COMPS, tp)
tt = receta.tabla("prod"); Pt = receta.componentes("prod", COMPS, tt)
mu = np.zeros(len(tt)); params = {}
# OOF cruzado de la receta en 2023-2024 (pesos y nivel fuera del pliegue) para medir y para el balance
folds = pliegue(tp.customer_id.values); oof = np.zeros(len(tp))
for l in LINEAS:
    il = (tp.line == l).values; it = (tt.line == l).values
    w = pesos(tp, Pp, il)
    b = np.column_stack([Pp[c][il] for c in COMPS]) @ w
    k = factor_tweedie(tp.y.values[il], b * tp.c_fila.values[il])
    mu[it] = (np.column_stack([Pt[c][it] for c in COMPS]) @ w) * k
    params[l] = {"pesos": dict(zip(COMPS, np.round(w, 4).tolist())), "k_ramo": k}
    for f in range(5):
        te = folds == f
        tr_ = tp[~te].reset_index(drop=True)
        wf = pesos(tr_, {c: Pp[c][~te] for c in COMPS}, il[~te])
        bf = np.column_stack([Pp[c] for c in COMPS]) @ wf
        kf = factor_tweedie(tp.y.values[~te & il], bf[~te & il] * tp.c_fila.values[~te & il])
        oof[te & il] = bf[te & il] * kf
sub = pd.read_parquet(DATOS / "emerging" / "sample_submission.parquet")
out = sub[["row_id"]].merge(pd.DataFrame({"row_id": tt.row_id.values, "prediction": mu}), on="row_id", how="left")
res = {"componentes": COMPS, "encoge": ENC, "params": params, "controles": controles(out, sub)}
f = PRIV / f"prediccion_2025_{NOM}.csv"
out.to_csv(f, index=False, float_format="%.6f", lineterminator="\\n")
res["controles"]["sha256"] = sha256(f); res["controles"]["archivo"] = f.name
tot = {}
for l in LINEAS:
    i = (tt.line == l).values; e = tt.exposure.values[i]; nb = tt.new_business.values[i]
    tot[l] = {"total": float(mu[i].sum()), "tasa": float(mu[i].sum() / e.sum()),
              "tasa_NB": float(mu[i][nb].sum() / e[nb].sum()), "tasa_REN": float(mu[i][~nb].sum() / e[~nb].sum())}
res["totales"] = tot; res["total"] = float(mu.sum())
res["oof_pcv"] = metricas(tp, oof)
res["balance_oof_pcv"] = {l: float(tp.y[(tp.line == l).values].sum() / (oof * tp.c_fila.values)[(tp.line == l).values].sum()) for l in LINEAS}
# combinación 50/50 con la designada (misma regla que MEZCLA.md)
assert sha256(DESIGNADA).startswith(SHA_DESIGNADA)
d = pd.read_csv(DESIGNADA)
assert (d.row_id.values == sub.row_id.values).all()
m50 = pd.DataFrame({"row_id": sub.row_id.values, "prediction": 0.5 * d.prediction.values + 0.5 * out.prediction.values})
f50 = PRIV / f"mezcla50_designada_{NOM}_2025.csv"
m50.to_csv(f50, index=False, float_format="%.6f", lineterminator="\\n")
res["mezcla50"] = {**controles(m50, sub), "sha256": sha256(f50), "archivo": f50.name, "total": float(m50.prediction.sum())}
lm = np.log(out.prediction.values); ld = np.log(d.prediction.values)
res["frente_designada"] = {"corr_log": float(np.corrcoef(lm, ld)[0, 1]), "cociente_p01_p99": [
    float(np.quantile(np.exp(lm - ld), 0.01)), float(np.quantile(np.exp(lm - ld), 0.99))]}
v1 = pd.read_csv(PRIV / "prediccion_2025_mlind.csv")
lv = np.log(v1.prediction.values)
res["frente_mlind_v1"] = {"corr_log": float(np.corrcoef(lm, lv)[0, 1]), "total_v1": float(v1.prediction.sum())}
guardar_json(res, ENTREGA / f"produccion_{NOM}.json")
print(NOM, round(res["total"]), {l: round(v["tasa"], 1) for l, v in tot.items()}, res["controles"]["sha256"][:12],
      "OOF pcv D2 %.4f" % res["oof_pcv"]["D2"], "corr designada %.4f v1 %.4f" % (res["frente_designada"]["corr_log"], res["frente_mlind_v1"]["corr_log"]))
'''


### Embedded source: `ml/codigo/receta.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/receta.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/receta.py'] = '''"""Recetas finales: mezcla convexa por ramo + calibración de nivel (C1 por ramo, C2 por ramo y negocio) + tendencia.

- parametros(fase_oof, comps, calib): pesos y factores con todo el OOF (DEV para HOLD, PCV para PROD).
- aplicar(fase, comps, params, tendencia): predicción final de la fase de evaluación.
"""
from __future__ import annotations
import json
import numpy as np, pandas as pd
from comun import *
from evaluar import tabla_eval, cargar_pred, mezcla_pesos, mezcla_cruzada, calibrar_cruzado, metricas, bootstrap_dif

K_NB = 200.0


def tabla(fase):
    if fase == "dev":
        t = tabla_eval("dev", 2023)
    elif fase == "hold":
        t = tabla_eval("hold", 2024)
    elif fase == "pcv":
        t = pd.concat([tabla_eval("dev", 2023), tabla_eval("dev", 2024)], ignore_index=True)
    else:  # prod: filas de test
        filas = []
        for l in LINEAS:
            b = pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet",
                                columns=["row_id", "y", "c_fila", "exposure", "customer_id", "year", "new_business", "conjunto"])
            filas.append(b[b.conjunto == "test"].assign(line=l))
        t = pd.concat(filas, ignore_index=True)
    return t.reset_index(drop=True)


def componentes(fase, comps, t):
    out = {}
    for c in comps:
        p = cargar_pred(fase, c)
        if p is None:
            raise FileNotFoundError(f"{fase}/{c}")
        mu = t[["row_id"]].merge(p, on="row_id", how="left").mu.values
        if np.isnan(mu).any():
            raise ValueError(f"faltan filas en {fase}/{c}")
        out[c] = mu
    return out


def parametros(fase_oof, comps, calib="ramo_nb"):
    t = tabla(fase_oof)
    P = componentes(fase_oof, comps, t)
    y = t.y.values; res = {"componentes": comps, "calib": calib, "ramos": {}}
    for l in LINEAS:
        il = (t.line == l).values
        M = np.column_stack([P[c][il] * t.c_fila.values[il] for c in comps])
        w = mezcla_pesos(y[il], M)
        m = M @ w
        kl = factor_tweedie(y[il], m)
        r = {"pesos": dict(zip(comps, w.round(6).tolist())), "k_ramo": kl, "k_nb": {}}
        for nb in (True, False):
            s = t.new_business.values[il] == nb
            ks = factor_tweedie(y[il][s], m[s] * kl); n = int((y[il][s] > 0).sum()); z = n / (n + K_NB)
            r["k_nb"][str(nb)] = {"k_seg": ks, "n_sin": n, "Z": z, "factor": kl * (z * ks + 1 - z)}
        res["ramos"][l] = r
    return res


def aplicar(fase, params, tend=None):
    t = tabla(fase)
    comps = params["componentes"]; P = componentes(fase, comps, t)
    mu = np.zeros(len(t))
    for l in LINEAS:
        il = (t.line == l).values; r = params["ramos"][l]
        U = np.column_stack([P[c][il] for c in comps]); w = np.array([r["pesos"][c] for c in comps])
        base = U @ w
        if params["calib"] == "ramo":
            f = np.full(il.sum(), r["k_ramo"])
        else:
            nb = t.new_business.values[il]
            f = np.where(nb, r["k_nb"]["True"]["factor"], r["k_nb"]["False"]["factor"])
        tf = 1.0 if tend is None else tend[l]["factor"]
        mu[il] = base * f * tf
    return t, mu


def oof_cruzado(fase, comps, calib="ramo_nb"):
    """Predicción OOF de la receta con pesos y calibración cruzados por pliegue (DEV o PCV)."""
    t = tabla(fase); P = componentes(fase, comps, t)
    folds = pliegue(t.customer_id.values)
    if len(comps) > 1:
        mu, pesos = mezcla_cruzada(t, P, folds)
    else:
        mu, pesos = P[comps[0]], []
    mu = calibrar_cruzado(t, mu, folds, por_nb=(calib == "ramo_nb"), K=K_NB) if calib != "sin" else mu
    return t, mu, pesos
'''


### Embedded source: `ml/codigo/red.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/red.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/red.py'] = '''"""FT-Transformer con pérdida Tweedie 1,5 y desplazamiento de exposición (línea ML independiente).

Preentrenamiento en el desarrollado y ajuste fino en el emergente.
Uso:
  python red.py pre FASE LINEA        -> preentrena en el desarrollado (dev/hold comparten 2021-2023; prod 2021-2024)
  python red.py fino FASE LINEA VAR   -> ajuste fino/variantes en el emergente (VAR en N1, N2a, N2b, N3)
"""
from __future__ import annotations
import sys, time, math, hashlib
import numpy as np, pandas as pd, torch, torch.nn as nn
from comun import *
from variables import *

torch.set_num_threads(2)
SEMILLA = 20261024


class Tokenizador(nn.Module):
    def __init__(self, n_num, cards, d):
        super().__init__()
        self.w = nn.Parameter(torch.randn(n_num, d) * 0.1)
        self.b = nn.Parameter(torch.zeros(n_num, d))
        self.miss = nn.Parameter(torch.randn(n_num, d) * 0.1)
        self.emb = nn.ModuleList([nn.Embedding(c + 1, d) for c in cards])  # índice 0 = desconocido/NaN
        for e in self.emb:
            nn.init.normal_(e.weight, std=0.1)

    def forward(self, xn, mn, xc):
        t = xn.unsqueeze(-1) * self.w + self.b
        t = torch.where(mn.unsqueeze(-1), self.miss.expand_as(t), t)
        if len(self.emb):
            tc = torch.stack([e(xc[:, j]) for j, e in enumerate(self.emb)], 1)
            t = torch.cat([t, tc], 1)
        return t


class Bloque(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__()
        self.n1 = nn.LayerNorm(d); self.att = nn.MultiheadAttention(d, h, dropout=drop, batch_first=True)
        self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 2 * d * 2), nn.GLU(), nn.Dropout(drop), nn.Linear(2 * d, d))
        self.dr = nn.Dropout(drop)

    def forward(self, x):
        z = self.n1(x); x = x + self.dr(self.att(z, z, z, need_weights=False)[0])
        return x + self.dr(self.ff(self.n2(x)))


class FTT(nn.Module):
    def __init__(self, n_num, cards, d=32, h=2, L=2, drop=0.1):
        super().__init__()
        self.tok = Tokenizador(n_num, cards, d)
        self.extra = None  # tokens nuevos del emergente
        self.cls = nn.Parameter(torch.randn(1, 1, d) * 0.1)
        self.bloques = nn.ModuleList([Bloque(d, h, drop) for _ in range(L)])
        self.nf = nn.LayerNorm(d); self.cab = nn.Linear(d, 1)
        self.sesgo = nn.Parameter(torch.zeros(1))

    def forward(self, xn, mn, xc, xe=None):
        t = self.tok(xn, mn, xc)
        if self.extra is not None and xe is not None:
            t = torch.cat([t, torch.stack([e(xe[:, j]) for j, e in enumerate(self.extra)], 1)], 1)
        x = torch.cat([self.cls.expand(len(t), -1, -1), t], 1)
        for b in self.bloques:
            x = b(x)
        return self.cab(self.nf(x[:, 0])).squeeze(-1) + self.sesgo


def perdida(eta, y, off, w):
    """Desviación Tweedie 1,5 (sin términos constantes) con mu = exp(eta + off)."""
    s = eta + off
    return torch.sum(w * 2 * (2 * y * torch.exp(-0.5 * s) + 2 * torch.exp(0.5 * s))) / torch.sum(w)


class Prepro:
    """Transformación cuantil-normal de numéricas (ajustada en el desarrollado) y códigos de categorías."""

    def __init__(self, df, num, cat, cats):
        self.num, self.cat, self.cats = num, cat, cats
        self.q = {}
        for c in num:
            v = pd.to_numeric(df[c], errors="coerce").astype(float).values
            v = v[~np.isnan(v)]
            qs = np.unique(np.quantile(v, np.linspace(0, 1, 201))) if len(v) else np.array([0.0, 1.0])
            self.q[c] = qs

    def __call__(self, df):
        from scipy.stats import norm
        XN = np.zeros((len(df), len(self.num)), np.float32); MN = np.zeros_like(XN, dtype=bool)
        for j, c in enumerate(self.num):
            v = pd.to_numeric(df[c], errors="coerce").astype(float).values if c in df else np.full(len(df), np.nan)
            m = np.isnan(v); qs = self.q[c]
            if len(qs) < 2:
                z = np.zeros(len(v))
            else:
                r = np.interp(v, qs, np.linspace(0, 1, len(qs)))
                z = norm.ppf(np.clip(r, 0.005, 0.995))
            XN[:, j] = np.where(m, 0, z); MN[:, j] = m
        XC = np.zeros((len(df), len(self.cat)), np.int64)
        for j, c in enumerate(self.cat):
            mapa = {v: i + 1 for i, v in enumerate(self.cats[c])}
            s = df[c].astype(object).where(df[c].notna(), None) if c in df else pd.Series([None] * len(df))
            XC[:, j] = [mapa.get(str(x) if x is not None else None, 0) for x in s.values]
        return XN, MN, XC


def entrenar(model, datos, val, lr, epocas, lote=1024, paciencia=3, wd=1e-5, l2sp=None, params=None, log=""):
    """datos/val: tuplas de tensores (xn, mn, xc, xe, y, off, w)."""
    params = params if params is not None else [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    n = len(datos[4]); g = torch.Generator().manual_seed(SEMILLA)
    mejor, estado, sin_mejora, hist = math.inf, None, 0, []
    ref = None
    if l2sp is not None:
        ref = {k: v.detach().clone() for k, v in model.named_parameters()}
    for ep in range(epocas):
        model.train(); perm = torch.randperm(n, generator=g); t0 = time.time(); tot = 0.0
        for i in range(0, n, lote):
            idx = perm[i:i + lote]
            xn, mn, xc, xe, y, off, w = (d[idx] if d is not None else None for d in datos)
            loss = perdida(model(xn, mn, xc, xe), y, off, w)
            if l2sp is not None:
                pen = sum(((p - ref[k]) ** 2).sum() for k, p in model.named_parameters() if k in ref and p.requires_grad)
                loss = loss + l2sp * pen
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += float(loss.detach()) * len(idx)
        dv = evaluar(model, val)
        hist.append((ep, tot / n, dv, round(time.time() - t0, 1)))
        print(log, "época", ep, "entren", round(tot / n, 4), "val", round(dv, 5), "s", round(time.time() - t0, 1), flush=True)
        if dv < mejor - 1e-6:
            mejor, estado, sin_mejora = dv, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0
        else:
            sin_mejora += 1
            if sin_mejora >= paciencia:
                break
    model.load_state_dict(estado)
    return mejor, hist


@torch.no_grad()
def evaluar(model, val, lote=8192):
    model.eval(); tot = 0.0; ws = 0.0
    n = len(val[4])
    for i in range(0, n, lote):
        sl = slice(i, i + lote)
        xn, mn, xc, xe, y, off, w = (d[sl] if d is not None else None for d in val)
        tot += float(perdida(model(xn, mn, xc, xe), y, off, w)) * float(w.sum()); ws += float(w.sum())
    return tot / ws


@torch.no_grad()
def predecir(model, xn, mn, xc, xe=None, lote=8192):
    model.eval(); out = []
    for i in range(0, len(xn), lote):
        sl = slice(i, i + lote)
        out.append(model(xn[sl], mn[sl], xc[sl], None if xe is None else xe[sl]))
    return torch.cat(out).numpy()


def tens(XN, MN, XC, XE, y, off, w):
    return (torch.tensor(XN), torch.tensor(MN), torch.tensor(XC), None if XE is None else torch.tensor(XE),
            torch.tensor(y, dtype=torch.float32), torch.tensor(off, dtype=torch.float32), torch.tensor(w, dtype=torch.float32))


# ----------------------------------------------------------------- ejecución
FASES = {"dev": dict(clima="hasta2023", dev_years=[2021, 2022, 2023], em_train=[2023], em_eval=None, pre="dev"),
         "hold": dict(clima="hasta2023", dev_years=[2021, 2022, 2023], em_train=[2023], em_eval=2024, pre="dev"),
         "prod": dict(clima="hasta2024", dev_years=[2021, 2022, 2023, 2024], em_train=[2023, 2024], em_eval=2025, pre="prod"),
         "pcv": dict(clima="hasta2024", dev_years=[2021, 2022, 2023, 2024], em_train=[2023, 2024], em_eval=None, pre="prod")}
EXTRA = ["region", "gender", "self_employed"]


def columnas(linea):
    cols = comunes(linea)
    return [c for c in cols if c not in CATEGORICAS], [c for c in cols if c in CATEGORICAS]


def submuestra(df, frac=0.1, sal="nn"):
    h = np.array([int(hashlib.sha256((sal + r).encode()).hexdigest()[:8], 16) / 2 ** 32 for r in df.row_id.values])
    keep = (df.y.values > 0) | (h < frac)
    w = np.where(df.y.values > 0, 1.0, 1.0 / frac)
    return keep, w


def preentrenar(fase, linea):
    import pickle
    cf = FASES[fase]; dirm = TRABAJO / "modelos" / cf["pre"]; dirm.mkdir(parents=True, exist_ok=True)
    f = dirm / f"N_pre_{linea}.pt"
    if f.exists():
        print("ya existe", f); return
    torch.manual_seed(SEMILLA); np.random.seed(SEMILLA)
    dv = cargar("developed", linea, cf["clima"]); em = cargar("emerging", linea, cf["clima"])
    dv = dv[dv.year.isin(cf["dev_years"])].reset_index(drop=True)
    num, cat = columnas(linea)
    cats = categorias_union([dv, em], num + cat + EXTRA)
    pp = Prepro(dv, num, cat, cats)
    tope = float(np.quantile(dv.y[dv.y > 0], 0.999))
    keep, w = submuestra(dv)
    dv = dv[keep].reset_index(drop=True); w = w[keep]
    XN, MN, XC = pp(dv)
    y = np.minimum(dv.y.values, tope); off = np.log(dv.exposure.values * dv.c_fila.values)
    base = math.log(np.sum(w * y) / np.sum(w * np.exp(off)))
    va = dv.fold.values == 0
    T = tens(XN, MN, XC, None, y, off, w)
    tr = tuple(None if t is None else t[torch.tensor(~va)] for t in T)
    vl = tuple(None if t is None else t[torch.tensor(va)] for t in T)
    m = FTT(len(num), [len(cats[c]) for c in cat])
    with torch.no_grad():
        m.sesgo.fill_(base)
    print(linea, "filas", len(dv), "val", int(va.sum()), "tope", round(tope), flush=True)
    mejor, hist = entrenar(m, tr, vl, lr=1e-3, epocas=6, lote=1024, paciencia=2, log=f"pre {linea}")
    torch.save(m.state_dict(), f)
    with open(dirm / f"N_pre_{linea}.pkl", "wb") as fh:
        pickle.dump({"prepro": pp, "num": num, "cat": cat, "cats": cats, "tope": tope, "hist": hist}, fh)
    guardar_json({"val": mejor, "hist": hist, "tope": tope, "filas": int(len(dv))}, dirm / f"N_pre_{linea}.json")


def interno(cids):
    return np.array([int(hashlib.sha256(("mlind_inner|" + c).encode()).hexdigest()[:8], 16) % 5 == 0 for c in cids])


def fino(fase, linea, var):
    import pickle
    cf = FASES[fase]; dirm = TRABAJO / "modelos" / cf["pre"]
    meta = pickle.load(open(dirm / f"N_pre_{linea}.pkl", "rb"))
    pp, num, cat, cats, tope = meta["prepro"], meta["num"], meta["cat"], meta["cats"], meta["tope"]
    em = cargar("emerging", linea, cf["clima"])
    XN, MN, XC = pp(em)
    XE = np.zeros((len(em), len(EXTRA)), np.int64)
    for c in EXTRA:
        if c not in cats:
            cats[c] = sorted(set(em[c].dropna().astype(str)))
    for j, c in enumerate(EXTRA):
        mapa = {v: i + 1 for i, v in enumerate(cats[c])}
        s = em[c].astype(object).where(em[c].notna(), None)
        XE[:, j] = [mapa.get(str(x) if x is not None else None, 0) for x in s.values]
    raw2 = pd.read_parquet(TRABAJO / "pred" / fase / f"T2braw_{linea}.parquet")
    em = em.reset_index(drop=True)
    r2 = em[["row_id"]].merge(raw2, on="row_id", how="left").raw.values
    tr_mask = ((em.conjunto == "train") & em.year.isin(cf["em_train"])).values
    if cf["em_eval"] is None:
        sets = [(tr_mask & (em.fold.values != k), tr_mask & (em.fold.values == k)) for k in range(5)]
    else:
        sets = [(tr_mask, (em.year == cf["em_eval"]).values)]
    off_all = np.log(em.exposure.values * np.where(em.conjunto == "test", 1.0, em.c_fila.values))
    preds = []; info = []
    for a, b in sets:
        torch.manual_seed(SEMILLA)
        ya = em.y.values[a]
        k0 = factor_tweedie(ya, np.exp(off_all[a]))
        extra_off = np.zeros(len(em))
        if var == "N3":
            k2 = factor_tweedie(ya, np.exp(off_all[a] + r2[a]))
            extra_off = r2 + math.log(k2)
        m = FTT(len(num), [len(cats[c]) for c in cat])
        if var in ("N2a", "N2b", "N3"):
            m.load_state_dict(torch.load(dirm / f"N_pre_{linea}.pt"))
        d = m.cls.shape[-1]
        m.extra = nn.ModuleList([nn.Embedding(len(cats[c]) + 1, d) for c in EXTRA])
        for e in m.extra:
            nn.init.normal_(e.weight, std=0.02)
        with torch.no_grad():
            if var == "N3":
                m.cab.weight.zero_(); m.cab.bias.zero_(); m.sesgo.zero_()
            else:
                # nivel de partida: mantener relatividades y fijar el nivel emergente
                eta0 = predecir(m, *[torch.tensor(x[a]) for x in (XN, MN, XC)], torch.tensor(XE[a]))
                m.sesgo += math.log(factor_tweedie(ya, np.exp(eta0 + off_all[a])))
        params = None; l2sp = None; lr = 2e-4
        if var == "N1":
            lr = 1e-3
        elif var == "N2b":
            for p in m.parameters():
                p.requires_grad = False
            for mod in [m.cab, m.nf, m.extra]:
                for p in mod.parameters():
                    p.requires_grad = True
            m.sesgo.requires_grad = True
            lr = 1e-3
        else:  # N2a, N3: ajuste completo con L2-SP
            l2sp = 1e-2
        idx_a = np.where(a)[0]; v = interno(em.customer_id.values[idx_a])
        it, iv = idx_a[~v], idx_a[v]
        yc = np.minimum(em.y.values, tope)
        T = lambda ix: tens(XN[ix], MN[ix], XC[ix], XE[ix], yc[ix], off_all[ix] + extra_off[ix], np.ones(len(ix)))
        mejor, hist = entrenar(m, T(it), T(iv), lr=lr, epocas=20, lote=512, paciencia=4, l2sp=l2sp, log=f"{var} {linea}")
        # reajuste de nivel en entrenamiento con y sin tope
        eta_a = predecir(m, torch.tensor(XN[a]), torch.tensor(MN[a]), torch.tensor(XC[a]), torch.tensor(XE[a]))
        kin = factor_tweedie(ya, np.exp(eta_a + off_all[a] + extra_off[a]))
        eta_b = predecir(m, torch.tensor(XN[b]), torch.tensor(MN[b]), torch.tensor(XC[b]), torch.tensor(XE[b]))
        mu = kin * np.exp(eta_b + extra_off[b]) * em.exposure.values[b]
        preds.append(pd.DataFrame({"row_id": em.row_id.values[b], "mu": mu}))
        info.append({"val": mejor, "epocas": len(hist), "kin": kin})
        if cf["em_eval"] is not None:
            (TRABAJO / "modelos" / fase).mkdir(parents=True, exist_ok=True)
            torch.save(m.state_dict(), TRABAJO / "modelos" / fase / f"{var}_{linea}.pt")
    out = TRABAJO / "pred" / fase; out.mkdir(parents=True, exist_ok=True)
    pd.concat(preds).to_parquet(out / f"{var}_{linea}.parquet", index=False)
    guardar_json(info, out / f"{var}_{linea}_meta.json")


if __name__ == "__main__":
    modo, fase, linea = sys.argv[1], sys.argv[2], sys.argv[3]
    if modo == "pre":
        preentrenar(fase, linea)
    else:
        for var in sys.argv[4].split(","):
            t0 = time.time(); fino(fase, linea, var); print(var, linea, "listo", round(time.time() - t0), "s", flush=True)
'''


### Embedded source: `ml/codigo/ruido_pareado.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/ruido_pareado.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/ruido_pareado.py'] = '''"""Ruido de la diferencia pareada en una submuestra del 33 % (como la parte pública) entre la mezcla 50/50 enviada
(T2b + A1 en 2024, A1 como sustituto de la designada) y la misma mezcla con MLind v2 o v2.1. Exploratorio sobre 2024."""
import json, numpy as np, pandas as pd
from comun import *
from receta import tabla, componentes
from evaluar import mezcla_pesos
td = tabla("dev"); th = tabla("hold")
def pred(comps):
    Pd = componentes("dev", comps, td); Ph = componentes("hold", comps, th); mu = np.zeros(len(th))
    for l in LINEAS:
        il = (td.line == l).values; ih = (th.line == l).values
        M = np.column_stack([Pd[c][il] * td.c_fila.values[il] for c in comps])
        w = mezcla_pesos(td.y.values[il], M) if len(comps) > 1 else np.ones(1)
        w = 0.5 * w + 0.5 / len(comps) if len(comps) > 1 else w
        mu[ih] = np.column_stack([Ph[c][ih] for c in comps]) @ w * factor_tweedie(td.y.values[il], M @ w)
    return mu * th.c_fila.values
o = pd.read_parquet(RAIZ / "comparacion" / "otros" / "predicciones_2024.parquet")
d = th[["row_id", "y"]].merge(o[["row_id", "A1"]], on="row_id", how="left").merge(
    pd.read_parquet(TRABAJO / "pesos_2025_hold.parquet"), on="row_id", how="left")
assert d.A1.notna().all() and (d.row_id.values == th.row_id.values).all()
y = d.y.values
m = {"v1": 0.5 * pred(["T2b"]) + 0.5 * d.A1.values, "v2": 0.5 * pred(["T2b", "T2h", "FS", "T2o"]) + 0.5 * d.A1.values,
     "v21": 0.5 * pred(["T2b", "T2h", "FSt", "T2o"]) + 0.5 * d.A1.values}
rng = np.random.default_rng(20261024); res = {}
for k in ("v2", "v21"):
    r = {"corr_log": float(np.corrcoef(np.log(m[k]), np.log(m["v1"]))[0, 1]), "dif_total": d2(y, m[k]) - d2(y, m["v1"]),
         "dif_pond2025": d2(y, m[k], d.w2025.values) - d2(y, m["v1"], d.w2025.values)}
    sims = []
    for _ in range(1000):
        s = rng.random(len(y)) < 0.33
        sims.append(d2(y[s], m[k][s]) - d2(y[s], m["v1"][s]))
    r["sd_33"] = float(np.std(sims)); r["p_negativa_33"] = float(np.mean(np.array(sims) < 0))
    res[k] = r
    print(k, {a: round(b, 5) for a, b in r.items()})
guardar_json(res, ENTREGA / "evolucion" / "ruido_pareado_v2.json")
'''


### Embedded source: `ml/codigo/seg_eval.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/seg_eval.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/seg_eval.py'] = '''"""D2 por segmento (nuevo negocio / renovación) y por ramo, DEV (C1 cruzado) y HOLD (C1), renivelado por ramo y segmento para medir solo estructura."""
import sys, numpy as np, pandas as pd
from comun import *
from receta import tabla, componentes
from evaluar import calibrar_cruzado
cands = sys.argv[1:]
td = tabla("dev").merge(pd.read_parquet(TRABAJO / "pesos_2025_dev.parquet"), on="row_id", how="left"); th = tabla("hold")
Pd = componentes("dev", cands, td); Ph = componentes("hold", cands, th)
folds = pliegue(td.customer_id.values)
filas = []
for c in cands:
    for fase, t, mu in (("dev", td, calibrar_cruzado(td, Pd[c], folds)), ("hold", th, Ph[c])):
        m = mu * t.c_fila.values; y = t.y.values
        r = {"cand": c, "fase": fase}
        for nb in (True, False):
            s = t.new_business.values == nb
            mm = m.copy()
            for l in LINEAS:  # renivelado por ramo dentro del segmento
                i = s & (t.line == l).values; mm[i] *= factor_tweedie(y[i], m[i])
            r["NB" if nb else "REN"] = d2(y[s], mm[s])
        filas.append(r)
df = pd.DataFrame(filas).pivot(index="cand", columns="fase", values=["NB", "REN"])
print(df.round(4))
'''


### Embedded source: `ml/codigo/sonda_nivel_s2.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/sonda_nivel_s2.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/sonda_nivel_s2.py'] = '''"""Sonda de nivel 2025 por ramo para la semana 2, sobre la mezcla enviada (público 0,15390), con credibilidad.

Misma forma cerrada que D100: para Tweedie 1,5, multiplicar por c las predicciones de un ramo cambia el D² público en
    D²(c) - D²(1) = -4 [ S1 (c^-1/2 - 1) + S2 (c^1/2 - 1) ] / N,  S1 = sum y/sqrt(mu), S2 = sum sqrt(mu) (filas públicas del ramo)
Dos sondas por ramo (x0,90 y x1,10) identifican S1/N y S2/N; el óptimo del público es c* = S1/S2.
Lección de D103: el ruido se mide frente a la parte privada, a escala del test 2025 (69 mil filas públicas, 140 mil privadas).
Factor final: log c_Z = Z log c*, con Z = tau² / (tau² + sigma²), tau fijada antes de ver las sondas.

Uso:  python sonda_nivel_s2.py generar
      python sonda_nivel_s2.py resolver --base 0.15390 --motor P090 P110 --buildings P090 P110 --contents P090 P110 [--simular]
"""
import argparse, json, datetime as dt, sys
import numpy as np, pandas as pd
from comun import *
import receta

BASE_SHA = "12831ef4a05c1773824abe5fa139d9503dd2989b4874e7d93cfc000406cfdb4a"
BASE_PUB = 0.15390
C_SONDA = {"x090": 0.90, "x110": 1.10}
PREF = {"M": "motor", "B": "buildings", "C": "contents"}
# Incertidumbre previa del nivel 2025 (desviación típica en logaritmo), fijada antes de ver las sondas a partir del
# abanico de niveles defendibles ya documentados (MLind produccion_2025.json, CLA-S2-20261024-012/017):
#  autos: C1 1,00 / C1+tendencia 1,08 / C2 1,28 / C2+tendencia 1,38 -> 0,14
#  edificios: C1 1,00 / tendencia 1,09 / nivel solo 2023 1,31 / solo 2024 0,81 -> 0,20
#  contenidos: C1 1,00 / tendencia 1,06 / C2 1,05 / C2+tendencia 1,12 -> 0,06
TAU = {"motor": 0.14, "buildings": 0.20, "contents": 0.06}
DIR = RAIZ / "sondas"


def leer_base():
    f = PRIVADO / "mezcla50_designada_mlind_2025.csv"
    assert sha256(f) == BASE_SHA, "la base no es la mezcla enviada"
    b = pd.read_csv(f, dtype={"row_id": str})
    b["line"] = b.row_id.str[0].map(PREF)
    assert b.line.notna().all()
    return b


def ruido(reps=2000, semilla=20261024):
    """sigma por ramo: desviación típica de log(c*_público / c*_privado) remuestreando componentes cliente-dirección
    del OOF 2023-2024 (T2b+C1, escala observada) a los tamaños de 2025 (público 1/3, privado 2/3 de 209.063 filas)."""
    t, mu, _ = receta.oof_cruzado("pcv", ["T2b"], "ramo")
    b = pd.concat([pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet", columns=["row_id", "address_id"]) for l in LINEAS])
    t = t.merge(b, on="row_id", how="left")
    m = mu * t.c_fila.values; y = t.y.values
    padre = {}
    def f(x):
        while padre.setdefault(x, x) != x:
            padre[x] = padre[padre[x]]; x = padre[x]
        return x
    for c, a in zip("C" + t.customer_id, "A" + t.address_id):
        rc, ra = f(c), f(a)
        if rc != ra: padre[ra] = rc
    _, gi = np.unique([f("C" + c) for c in t.customer_id], return_inverse=True); G = gi.max() + 1
    lam_pub = 209063 / 3 / len(t); lam_priv = 209063 * 2 / 3 / len(t)
    rng = np.random.default_rng(semilla); out = {}
    for l in LINEAS:
        i = (t.line == l).values
        a1 = np.bincount(gi[i], weights=y[i] / np.sqrt(m[i]), minlength=G); a2 = np.bincount(gi[i], weights=np.sqrt(m[i]), minlength=G)
        Wp = rng.poisson(lam_pub, size=(reps, G)); Wq = rng.poisson(lam_priv, size=(reps, G))
        cp = (Wp @ a1) / (Wp @ a2); cq = (Wq @ a1) / (Wq @ a2)
        out[l] = {"sd_log_c_publico": float(np.std(np.log(cp))), "sd_log_c_privado": float(np.std(np.log(cq))),
                  "sigma_pub_menos_priv": float(np.std(np.log(cp) - np.log(cq)))}
    return out


def generar(_):
    DIR.mkdir(parents=True, exist_ok=True)
    b = leer_base(); sig = ruido()
    Z = {l: TAU[l] ** 2 / (TAU[l] ** 2 + sig[l]["sigma_pub_menos_priv"] ** 2) for l in LINEAS}
    archivos = {}
    for l in LINEAS:
        for k, c in C_SONDA.items():
            x = b[["row_id", "prediction"]].copy()
            x.loc[b.line == l, "prediction"] *= c
            f = DIR / f"S2sonda_{l}_{k}.csv"
            x.to_csv(f, index=False, float_format="%.6f", lineterminator="\\n")
            archivos[f.name] = {"sha256": sha256(f), "ramo": l, "factor": c, "total": float(x.prediction.sum()), "filas": len(x)}
    reg = {"id": "S2-SONDA-NIVEL-MLIND", "utc": dt.datetime.now(dt.UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
           "base": {"archivo": "mezcla50_designada_mlind_2025.csv", "sha256": BASE_SHA, "publico": BASE_PUB, "envio": 57599788},
           "forma": "D2(c)-D2(1) = -4[S1(c^-1/2-1)+S2(c^1/2-1)]/N por ramo; c* = S1/S2",
           "tau_log": TAU, "ruido": sig, "Z": Z, "sondas": archivos,
           "regla": ["Enviar las seis sondas tal cual; ninguna se selecciona nunca.",
                     "Con las seis puntuaciones públicas y la base, resolver S1/N y S2/N por ramo; si alguno sale <= 0, ese ramo no se ajusta.",
                     "Si c* sale fuera de [0,5; 2,0], ese ramo no se ajusta (la forma supuesta no se cumple).",
                     "Candidato: base x c_Z por ramo, log c_Z = Z log c*. Se envía una vez; su público esperado se fija antes y una desviación > 0,0001 indica que el evaluador no tiene la forma supuesta.",
                     "Selección final: el candidato si su público >= el de la base; si no, la base. La decisión de enviar y seleccionar es del participante; envía Codex."]}
    guardar_json(reg, DIR / "preinscripcion_sonda_nivel_s2.json")
    print(json.dumps({k: reg[k] for k in ["ruido", "Z", "tau_log"]}, indent=1))
    for k, v in archivos.items(): print(k, v["sha256"][:12], round(v["total"]))


def resolver(a):
    reg = json.load(open(DIR / "preinscripcion_sonda_nivel_s2.json"))
    b = leer_base(); out = {"entradas": {"base": a.base}, "ramos": {}}
    for l in LINEAS:
        p90, p110 = getattr(a, l); out["entradas"][l] = [p90, p110]
        # d(c) = D2(c) - D2(1) = -4[S1 (c^-1/2-1) + S2 (c^1/2-1)]/N -> sistema lineal en s1=4 S1/N, s2=4 S2/N
        A = np.array([[-(0.9 ** -0.5 - 1), -(0.9 ** 0.5 - 1)], [-(1.1 ** -0.5 - 1), -(1.1 ** 0.5 - 1)]])
        s1, s2 = np.linalg.solve(A, np.array([p90 - a.base, p110 - a.base]))
        r = {"s1": float(s1), "s2": float(s2)}
        if s1 <= 0 or s2 <= 0:
            r.update({"valido": False, "motivo": "S1 o S2 no positivos", "c_Z": 1.0})
        else:
            cs = s1 / s2; z = reg["Z"][l]
            ok = bool(0.5 <= cs <= 2.0)
            cz = float(np.exp(z * np.log(cs))) if ok else 1.0
            ganancia_pub = lambda c: -(s1 * (c ** -0.5 - 1) + s2 * (c ** 0.5 - 1))
            r.update({"valido": ok, "c_estrella": float(cs), "Z": z, "c_Z": cz, "ganancia_publica_esperada": float(ganancia_pub(cz)),
                      "ganancia_publica_si_c_estrella": float(ganancia_pub(cs))})
        out["ramos"][l] = r
    out["publico_esperado_candidato"] = a.base + sum(r.get("ganancia_publica_esperada", 0.0) for r in out["ramos"].values())
    print(json.dumps(out, indent=1, default=float))
    if a.simular:
        return
    x = b[["row_id", "prediction"]].copy()
    for l in LINEAS:
        x.loc[b.line == l, "prediction"] *= out["ramos"][l]["c_Z"]
    f = DIR / "S2_nivel_credibilidad.csv"
    x.to_csv(f, index=False, float_format="%.6f", lineterminator="\\n")
    out["candidato"] = {"archivo": f.name, "sha256": sha256(f), "total": float(x.prediction.sum()),
                        "por_ramo": {l: float(x.prediction[b.line == l].sum()) for l in LINEAS}}
    guardar_json(out, DIR / "resolucion_sonda_nivel_s2.json")
    print(out["candidato"])


if __name__ == "__main__":
    ap = argparse.ArgumentParser(); sp = ap.add_subparsers(dest="cmd", required=True)
    sp.add_parser("generar")
    r = sp.add_parser("resolver"); r.add_argument("--base", type=float, default=BASE_PUB); r.add_argument("--simular", action="store_true")
    for l in LINEAS: r.add_argument(f"--{l}", type=float, nargs=2, required=True)
    a = ap.parse_args()
    {"generar": generar, "resolver": resolver}[a.cmd](a)
'''


### Embedded source: `ml/codigo/t2_rejilla.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/t2_rejilla.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/t2_rejilla.py'] = '''"""E07: rejilla del modelo del desarrollado con validación interna del desarrollado."""
import sys, itertools, time
import numpy as np, pandas as pd, lightgbm as lgb
from comun import *
from variables import *
from gbm import P_DEV, CONF, MOD, OUT, FASE

res = {}
for l in LINEAS:
    em = cargar("emerging", l, CONF["clima"]); dv = cargar("developed", l, CONF["clima"])
    dv = dv[dv.year.isin(CONF["dev_years"])].reset_index(drop=True)
    cols = comunes(l); cats = categorias_union([dv, em], cols + SOLO_EMERGENTE + MARKETING + TARIFA + MAKE)
    X = matriz(dv, cols, cats); y = dv.y.values; off = np.log(dv.exposure.values * dv.c_fila.values)
    va = dv.fold.values == 0
    dtr = lgb.Dataset(X[~va], y[~va], init_score=off[~va], free_raw_data=False)
    dva = lgb.Dataset(X[va], y[va], init_score=off[va], reference=dtr)
    filas = []
    for l2, md in itertools.product([0.0, 10.0], [300, 1000, 3000]):
        t0 = time.time()
        p = dict(P_DEV, learning_rate=0.03, lambda_l2=l2, min_data_in_leaf=md)
        b = lgb.train(p, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(150, verbose=False)])
        mu = np.exp(b.predict(X[va], raw_score=True, num_iteration=b.best_iteration) + off[va])
        k = factor_tweedie(y[~va], np.exp(b.predict(X[~va], raw_score=True, num_iteration=b.best_iteration) + off[~va]))
        dev_v = desviacion(y[va], mu * k).sum()
        filas.append({"lambda_l2": l2, "min_data": md, "rondas": b.best_iteration, "k_entren": k,
                      "D2_val": d2(y[va], mu * k), "s": round(time.time() - t0)})
        print(l, filas[-1], flush=True)
    res[l] = filas
    guardar_json(res, OUT / "T2_rejilla.json")
'''


### Embedded source: `ml/codigo/t2cap.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/t2cap.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/t2cap.py'] = '''"""E06: T2 con tope por fila (percentil 99,9 de las filas positivas del ramo) en el entrenamiento del desarrollado."""
import sys
import numpy as np, pandas as pd, lightgbm as lgb
from comun import *
from variables import *
from gbm import P_DEV, ajustar, CONF, MOD, OUT, FASE

for l in LINEAS:
    em = cargar("emerging", l, CONF["clima"]); dv = cargar("developed", l, CONF["clima"])
    dv = dv[dv.year.isin(CONF["dev_years"])].reset_index(drop=True)
    cols = comunes(l); cats = categorias_union([dv, em], cols + SOLO_EMERGENTE + MARKETING + TARIFA + MAKE)
    tope = float(np.quantile(dv.y[dv.y > 0], 0.999))
    f = MOD / f"T2cap_{l}.txt"
    Xd = matriz(dv, cols, cats); offd = np.log(dv.exposure.values * dv.c_fila.values)
    if f.exists():
        b = lgb.Booster(model_file=str(f)); n = b.current_iteration()
    else:
        b, n = ajustar(Xd, np.minimum(dv.y.values, tope), offd, P_DEV, 4000, es=100, val=dv.fold.values == 0)
        b.save_model(str(f))
    r = b.predict(matriz(em, cols, cats), raw_score=True)
    em["raw"] = r
    tr = em[(em.conjunto == "train") & em.year.isin(CONF["em_train"])].reset_index(drop=True)
    sets = [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)] if CONF["em_eval"] is None else \\
        [(tr, em[em.year == CONF["em_eval"]].reset_index(drop=True))]
    preds = []
    for a, bb in sets:
        k = factor_tweedie(a.y.values, np.exp(a.raw.values) * a.exposure.values * a.c_fila.values)
        preds.append(pd.DataFrame({"row_id": bb.row_id.values, "mu": k * np.exp(bb.raw.values) * bb.exposure.values}))
    pd.concat(preds).to_parquet(OUT / f"T2cap_{l}.parquet", index=False)
    pd.DataFrame({"row_id": em.row_id.values, "raw": r}).to_parquet(OUT / f"T2capraw_{l}.parquet", index=False)
    print(l, "tope", round(tope), "rondas", n, flush=True)
'''


### Embedded source: `ml/codigo/t4.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/t4.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/t4.py'] = '''"""T4: modelo conjunto desarrollado + emergente con indicador de mercado (preinscrito).

Desarrollado submuestreado al 35 % por cliente (cómputo), peso 5 para las filas emergentes,
parada temprana con la validación interna emergente, renivelado dentro del pliegue con filas emergentes.
"""
import sys, hashlib
import numpy as np, pandas as pd, lightgbm as lgb
from comun import *
from variables import *
from gbm import P_DEVB, interno, CONF, FASE, OUT, MOD

PESO_EM = 5.0
for l in LINEAS:
    em = cargar("emerging", l, CONF["clima"]); dv = cargar("developed", l, CONF["clima"])
    dv = dv[dv.year.isin(CONF["dev_years"])]
    h = np.array([int(hashlib.sha256(("t4|" + c).encode()).hexdigest()[:8], 16) / 2 ** 32 for c in dv.customer_id.values])
    dv = dv[h < 0.35].reset_index(drop=True)
    cols = comunes(l) + SOLO_EMERGENTE + ["market"]
    CATEGORICAS.add("market")
    cats = categorias_union([dv, em], cols)
    tr = em[(em.conjunto == "train") & em.year.isin(CONF["em_train"])].reset_index(drop=True)
    sets = [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)] if CONF["em_eval"] is None else [(tr, em[em.year == CONF["em_eval"]].reset_index(drop=True))]
    Xd = matriz(dv, cols, cats); offd = np.log(dv.exposure.values * dv.c_fila.values)
    preds = []; rondas = []
    for a, b in sets:
        Xa = matriz(a, cols, cats); offa = np.log(a.exposure.values * a.c_fila.values)
        v = interno(a.customer_id.values)
        X = pd.concat([Xd, Xa[~v]], ignore_index=True)
        y = np.concatenate([dv.y.values, a.y.values[~v]]); off = np.concatenate([offd, offa[~v]])
        w = np.concatenate([np.ones(len(dv)), np.full((~v).sum(), PESO_EM)])
        p = dict(P_DEVB[l])
        dtr = lgb.Dataset(X, y, weight=w, init_score=off, free_raw_data=False)
        dva = lgb.Dataset(Xa[v], a.y.values[v], init_score=offa[v], reference=dtr)
        bst = lgb.train(p, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(150, verbose=False)])
        n = max(bst.best_iteration, 1); rondas.append(n)
        X2 = pd.concat([Xd, Xa], ignore_index=True)
        y2 = np.concatenate([dv.y.values, a.y.values]); off2 = np.concatenate([offd, offa])
        w2 = np.concatenate([np.ones(len(dv)), np.full(len(a), PESO_EM)])
        bst = lgb.train(p, lgb.Dataset(X2, y2, weight=w2, init_score=off2), n)
        kin = factor_tweedie(a.y.values, np.exp(bst.predict(Xa, raw_score=True) + offa))
        mu = kin * np.exp(bst.predict(matriz(b, cols, cats), raw_score=True)) * b.exposure.values
        preds.append(pd.DataFrame({"row_id": b.row_id.values, "mu": mu}))
        if CONF["em_eval"] is not None:
            (TRABAJO / "modelos" / FASE).mkdir(parents=True, exist_ok=True)
            bst.save_model(str(TRABAJO / "modelos" / FASE / f"T4_{l}.txt"))
    pd.concat(preds).to_parquet(OUT / f"T4_{l}.parquet", index=False)
    print(l, "rondas", rondas, flush=True)
'''


### Embedded source: `ml/codigo/tabla_dev_final.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/tabla_dev_final.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/tabla_dev_final.py'] = '''"""Tabla final de DEV: todos los candidatos y mezclas, calibración C1/C2 cruzadas, intervalos frente a T2b."""
import json
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from comun import *
from evaluar import metricas, bootstrap_dif
import receta

SIMPLES = ["R0", "G1", "T1wp", "T2", "T2b", "T3bwp", "T3bmk", "T4", "T5", "T6", "N1", "N2a", "N2b", "N3"]
MEZCLAS = {"E1": ["T2b", "T3bwp", "N3"], "E2": ["T2b", "T3bwp", "N3", "T4", "G1"],
           "E3": ["G1", "T1wp", "T2", "T2b", "T3bwp", "T3bmk", "T4", "T5", "T6", "N1", "N2a", "N2b", "N3"]}
t0 = receta.tabla("dev").merge(pd.read_parquet(TRABAJO / "pesos_2025_dev.parquet"), on="row_id", how="left")
filas = []; mus = {}; pesos_mezcla = {}
for nombre in SIMPLES + list(MEZCLAS):
    comps = MEZCLAS.get(nombre, [nombre])
    for calib in ["sin", "ramo", "ramo_nb"]:
        if nombre in MEZCLAS and calib == "sin":
            continue
        t, mu, pesos = receta.oof_cruzado("dev", comps, calib)
        assert (t.row_id.values == t0.row_id.values).all()
        mus[(nombre, calib)] = mu
        if pesos: pesos_mezcla[f"{nombre}|{calib}"] = pesos
        filas.append({"cand": nombre, "calib": calib, **metricas(t0, mu)})
df = pd.DataFrame(filas)
ref = mus[("T2b", "ramo_nb")]
bs = {}
for (n, c), mu in mus.items():
    if c == "ramo_nb" and n != "T2b":
        bs[n] = bootstrap_dif(t0, mu, ref, B=1000)
df["dif_T2b_C2"] = df.apply(lambda r: bs[r.cand]["dif"] if r.calib == "ramo_nb" and r.cand in bs else np.nan, axis=1)
df["ic95_lo"] = df.apply(lambda r: bs[r.cand]["ic95"][0] if r.calib == "ramo_nb" and r.cand in bs else np.nan, axis=1)
df["ic95_hi"] = df.apply(lambda r: bs[r.cand]["ic95"][1] if r.calib == "ramo_nb" and r.cand in bs else np.nan, axis=1)
df.to_csv(ENTREGA / "tabla_dev.csv", index=False)
guardar_json({"pesos_mezcla": pesos_mezcla, "bootstrap_vs_T2b_C2": bs}, ENTREGA / "dev_mezclas_bootstrap.json")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(df[df.calib != "sin"].sort_values("D2", ascending=False).round(4).to_string())
# figura: D2 calibrado C2 por candidato con IC frente a T2b
d = df[df.calib == "ramo_nb"].set_index("cand").sort_values("D2")
fig, ax = plt.subplots(figsize=(7.5, 4.2))
col = ["#7a7a7a" if c in ("R0", "G1", "T1wp", "T4") else "#2a6f97" if c.startswith("T") else "#8e5ea2" if c.startswith("N") else "#c4673b" for c in d.index]
ax.barh(d.index, d.D2, color=col)
for i, (c, r) in enumerate(d.iterrows()):
    ax.text(r.D2 + 0.001, i, f"{r.D2:.4f}", va="center", fontsize=7)
ax.set_xlabel("D² Tweedie 1,5, OOF 2023, calibración por ramo y negocio cruzada")
ax.set_title("Desarrollo 2023: gris = solo emergente, azul = árboles con transferencia,\\nmorado = redes, naranja = mezclas", fontsize=9)
fig.tight_layout(); fig.savefig(ENTREGA / "figuras" / "f03_dev_candidatos.png"); plt.close(fig)
'''


### Embedded source: `ml/codigo/tendencia.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/tendencia.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/tendencia.py'] = '''"""Regla §6.2: tendencia anual por ramo desde el desarrollado y desplazamiento temporal Δt.

Modelo sin año ni historia (la ventana de la historia delata el año 2021) ajustado en el desarrollado de la fase;
A/E por año con etiquetas ajustadas por completitud; pendiente log-lineal ponderada por el esperado.
Uso: python tendencia.py fase   (hold: 2021-2023 -> Δt de 2023 a 2024; prod: 2021-2024 -> Δt de 2023-2024 a 2025)
"""
import sys
import numpy as np, pandas as pd, lightgbm as lgb
from comun import *
from variables import *
from gbm import P_DEVB, ajustar, CONF, FASE, MOD, OUT

res = {}
for l in LINEAS:
    dv = cargar("developed", l, CONF["clima"]); dv = dv[dv.year.isin(CONF["dev_years"])].reset_index(drop=True)
    em = cargar("emerging", l, CONF["clima"])
    cols = [c for c in comunes(l) if c not in ("hist_pol_12m", "hist_cli_12m")]
    cats = categorias_union([dv, em], cols)
    X = matriz(dv, cols, cats); off = np.log(dv.exposure.values * dv.c_fila.values)
    b, n = ajustar(X, dv.y.values, off, P_DEVB[l], 5000, es=150, val=dv.fold.values == 0)
    mu = np.exp(b.predict(X, raw_score=True) + off)
    g = pd.DataFrame({"year": dv.year, "y": dv.y, "m": mu}).groupby("year").sum()
    g["ae"] = g.y / g.m
    x = g.index.values.astype(float); w = g.m.values
    A = np.vstack([np.ones_like(x), x - x.mean()]).T
    beta = np.linalg.lstsq(A * np.sqrt(w)[:, None], np.log(g.ae.values) * np.sqrt(w), rcond=None)[0]
    tau = float(np.exp(beta[1]))
    # Δt: fecha media de cobertura (ponderada por exposición) del entrenamiento emergente y del objetivo
    def fecha_media(d):
        mid = d.period_start + (d.period_end - d.period_start) / 2
        anio = mid.dt.year + (mid.dt.dayofyear - 0.5) / 365.25
        return float(np.average(anio, weights=d.exposure))
    tr = em[(em.conjunto == "train") & em.year.isin(CONF["em_train"])]
    ob = em[em.year == CONF["em_eval"]]
    dt = fecha_media(ob) - fecha_media(tr)
    res[l] = {"ae_por_anio": g.ae.round(4).to_dict(), "tau_anual": tau, "dt": dt, "factor": tau ** dt, "rondas": n}
    print(l, res[l], flush=True)
guardar_json(res, OUT / "tendencia.json")
'''


### Embedded source: `ml/codigo/transfer2.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/transfer2.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/transfer2.py'] = '''"""E14-E17: troncos alternativos transferidos desde el desarrollado (línea ML independiente).

Uso: python3 transfer2.py <variante> <devhold|prodpcv> [lineas]
  E23: T2bi, T2ci, T2oi, FSi, T2hi = mismos troncos con el boosting arrancando en el nivel medio (ajustar0).
  A05: con MLIND_HISTORIA=2 la historia de las filas 2025 usa la ventana que termina en min(inicio, V).
  variantes: CB (CatBoost Tweedie), FS (frecuencia x severidad), T2x (+make_model y tenencia multirramo),
             T2o (sin occupation), T2h (T2c + etapa de historia centrada), T2b / T2c (solo DES24 con el modelo guardado)
  devhold: tronco 2021-2023 (clima hasta2023) -> pred/dev (OOF 2023) y pred/hold (2023 -> 2024) + DES24
  prodpcv: tronco 2021-2024 (clima hasta2024) -> pred/pcv (OOF 2023-2024) y pred/prod (2023-2024 -> 2025)
El nivel emergente se fija igual que T2b en gbm.py: factor Tweedie en el entrenamiento de cada pliegue.
"""
from __future__ import annotations
import sys, time, hashlib
import numpy as np, pandas as pd, lightgbm as lgb
from comun import *
from variables import *

VAR = sys.argv[1]
FASE = sys.argv[2]
CONF = {"devhold": dict(clima="hasta2023", dev_years=[2021, 2022, 2023], mod="dev", oof=("dev", [2023]),
                        ev=("hold", [2023], 2024), des=2024),
        "prodpcv": dict(clima="hasta2024", dev_years=[2021, 2022, 2023, 2024], mod="prod", oof=("pcv", [2023, 2024]),
                        ev=("prod", [2023, 2024], 2025), des=None)}[FASE]
MOD = TRABAJO / "modelos" / CONF["mod"]
HIST = ["hist_pol_12m", "hist_cli_12m"]
TEN = ["n_lineas", "tiene_motor", "tiene_buildings", "tiene_contents"]
SEMILLA = 20261024
INICIO0 = VAR in ("T2bi", "T2ci", "T2oi", "FSi", "T2hi")  # E23: el boosting arranca en el nivel medio
BASE = VAR[:-1] if INICIO0 else VAR
import os
SUF = os.environ.get("MLIND_SUFIJO", "")  # E24: sufijo de las salidas (los modelos guardados no cambian de nombre)
P_DEV = dict(objective="tweedie", tweedie_variance_power=1.5, learning_rate=0.05, num_leaves=63,
             min_data_in_leaf=1000, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0,
             max_cat_to_onehot=8, cat_smooth=50, verbose=-1, num_threads=2, seed=SEMILLA)
P_DEVB = {"motor": dict(P_DEV, learning_rate=0.03, lambda_l2=0.0, min_data_in_leaf=300),
          "buildings": dict(P_DEV, learning_rate=0.03, lambda_l2=0.0, min_data_in_leaf=3000),
          "contents": dict(P_DEV, learning_rate=0.03, lambda_l2=10.0, min_data_in_leaf=3000)}


def ajustar(X, y, off, params, max_rounds, es=150, val=None, w=None):
    """Parada temprana con el pliegue interno (val) y reajuste con todo el entrenamiento."""
    wt = None if w is None else w[~val]
    dtr = lgb.Dataset(X[~val], y[~val], init_score=off[~val], weight=wt, free_raw_data=False)
    dva = lgb.Dataset(X[val], y[val], init_score=off[val], weight=None if w is None else w[val], reference=dtr)
    b = lgb.train(params, dtr, max_rounds, valid_sets=[dva], callbacks=[lgb.early_stopping(es, verbose=False)])
    n = max(b.best_iteration, 1)
    b = lgb.train(params, lgb.Dataset(X, y, init_score=off, weight=w), n)
    return b, n


def ajustar0(X, y, off, params, max_rounds, es=150, val=None, w=None):
    """E23: igual que ajustar, pero el init_score incluye el log de la tasa media (LightGBM no parte del
    promedio cuando recibe init_score). Devuelve el modelo, las rondas y la constante a sumar al raw."""
    ww = np.ones(len(y)) if w is None else w
    b0 = float(np.log(np.sum(ww * y) / np.sum(ww * np.exp(off))))
    b, n = ajustar(X, y, off + b0, params, max_rounds, es=es, val=val, w=w)
    return b, n, b0


def _guardar_base(f, b0):
    guardar_json({"base": b0}, str(f) + ".base.json")


def _leer_base(f):
    import json
    return json.load(open(str(f) + ".base.json"))["base"]


def tenencia(mercado):
    b = {l: pd.read_parquet(TRABAJO / f"base_{mercado}_{l}.parquet", columns=["customer_id", "year"]) for l in LINEAS}
    cl = pd.concat([d.assign(line=l) for l, d in b.items()]).drop_duplicates()
    t = cl.assign(v=1).pivot_table(index=["customer_id", "year"], columns="line", values="v", fill_value=0)
    t.columns = [f"tiene_{c}" for c in t.columns]
    t["n_lineas"] = t.sum(axis=1)
    return t.reset_index()


def columnas(linea):
    c = comunes(linea)
    if BASE in ("T2c", "T2h"):
        c = [x for x in c if x not in HIST]
    if BASE == "T2o":
        c = [x for x in c if x != "occupation"]
    if VAR == "T2x":
        c = c + (["make_model"] if linea == "motor" else []) + TEN
    return c


# ---------------------------------------------------------------- troncos
def tronco_lgb(dv, cols, cats, linea, nombre):
    f = MOD / f"{nombre}_{linea}.txt"
    Xd = matriz(dv, cols, cats); offd = np.log(dv.exposure.values * dv.c_fila.values)
    if f.exists():
        b = lgb.Booster(model_file=str(f)); n = b.current_iteration()
        b0 = _leer_base(f) if INICIO0 else 0.0
    elif INICIO0:
        b, n, b0 = ajustar0(Xd, dv.y.values, offd, P_DEVB[linea], 5000, val=dv.fold.values == 0)
        b.save_model(str(f)); _guardar_base(f, b0)
    else:
        b, n = ajustar(Xd, dv.y.values, offd, P_DEVB[linea], 5000, val=dv.fold.values == 0); b0 = 0.0
        b.save_model(str(f))
    return (lambda df: b.predict(matriz(df, cols, cats), raw_score=True) + b0), {"rondas": n, "base": b0}


def tronco_cb(dv, cols, cats, linea):
    from catboost import CatBoostRegressor, Pool
    catc = [c for c in cols if c in CATEGORICAS]

    def X(df):
        o = pd.DataFrame(index=df.index)
        for c in cols:
            if c in CATEGORICAS:
                o[c] = df[c].astype(str).where(df[c].notna(), "NA")
            else:
                o[c] = pd.to_numeric(df[c], errors="coerce").astype(float)
        return o
    f = MOD / f"CB_{linea}.cbm"
    m = CatBoostRegressor()
    if f.exists():
        m.load_model(str(f)); n = m.tree_count_
    else:
        offd = np.log(dv.exposure.values * dv.c_fila.values); val = dv.fold.values == 0
        Xd = X(dv)
        prm = dict(loss_function="Tweedie:variance_power=1.5", learning_rate=0.08, depth=6, l2_leaf_reg=3.0,
                   one_hot_max_size=8, thread_count=2, random_seed=SEMILLA, verbose=0, allow_writing_files=False)
        m1 = CatBoostRegressor(iterations=3000, od_type="Iter", od_wait=100, **prm)
        m1.fit(Pool(Xd[~val], dv.y.values[~val], baseline=offd[~val], cat_features=catc),
               eval_set=Pool(Xd[val], dv.y.values[val], baseline=offd[val], cat_features=catc), use_best_model=True)
        n = max(m1.get_best_iteration() + 1, 1)
        m = CatBoostRegressor(iterations=n, **prm)
        m.fit(Pool(Xd, dv.y.values, baseline=offd, cat_features=catc))
        m.save_model(str(f))
    return (lambda df: m.predict(Pool(X(df), cat_features=catc), prediction_type="RawFormulaVal")), {"rondas": int(n)}


def tronco_fs(dv, cols, cats, linea):
    fF = MOD / f"{VAR}f_{linea}.txt"; fS = MOD / f"{VAR}s_{linea}.txt"
    if VAR == "FSt":  # E25: frecuencia de FS sin cambios y severidad con la rejilla elegida en el pliegue interno
        fF = MOD / f"FSf_{linea}.txt"
    Xd = matriz(dv, cols, cats); offd = np.log(dv.exposure.values * dv.c_fila.values); val = dv.fold.values == 0
    s = (dv.n_sin.values > 0) & (dv.y.values > 0)
    msev = float(dv.y.values[s].sum() / dv.n_sin.values[s].sum())
    if VAR == "FSt":
        import json
        el = json.load(open(TRABAJO / "modelos" / "dev" / f"e25_rejilla_{linea}.json"))["elegida"]
        bf = lgb.Booster(model_file=str(fF)); nf = bf.current_iteration(); b0 = 0.0
        if fS.exists():
            bs = lgb.Booster(model_file=str(fS))
        else:
            ps = dict(objective="gamma", learning_rate=0.03, num_leaves=el["num_leaves"],
                      min_data_in_leaf=el["min_data_in_leaf"], lambda_l2=el["lambda_l2"], feature_fraction=0.8,
                      bagging_fraction=0.8, bagging_freq=1, max_cat_to_onehot=8, cat_smooth=50, verbose=-1,
                      num_threads=2, seed=SEMILLA)
            Xs = Xd[s].reset_index(drop=True); n_ = dv.n_sin.values[s].astype(float)
            bs, _ = ajustar(Xs, dv.y.values[s] / n_, np.full(s.sum(), np.log(msev)), ps, 3000, es=100, val=val[s], w=n_)
            bs.save_model(str(fS))
        ns = bs.current_iteration()
    elif fF.exists():
        bf = lgb.Booster(model_file=str(fF)); bs = lgb.Booster(model_file=str(fS))
        nf, ns = bf.current_iteration(), bs.current_iteration()
        b0 = _leer_base(fF) if INICIO0 else 0.0
    else:
        pf = {k: v for k, v in P_DEVB[linea].items() if k != "tweedie_variance_power"}
        pf["objective"] = "poisson"
        if INICIO0:
            bf, nf, b0 = ajustar0(Xd, dv.n_sin.values.astype(float), offd, pf, 5000, val=val)
            _guardar_base(fF, b0)
        else:
            bf, nf = ajustar(Xd, dv.n_sin.values, offd, pf, 5000, val=val); b0 = 0.0
        ps = dict(objective="gamma", learning_rate=0.03, num_leaves=31, min_data_in_leaf=200, feature_fraction=0.8,
                  bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0, max_cat_to_onehot=8, cat_smooth=50,
                  verbose=-1, num_threads=2, seed=SEMILLA)
        Xs = Xd[s].reset_index(drop=True); n_ = dv.n_sin.values[s]
        bs, ns = ajustar(Xs, dv.y.values[s] / n_, np.full(s.sum(), np.log(msev)), ps, 5000, val=val[s], w=n_)
        bf.save_model(str(fF)); bs.save_model(str(fS))

    def raw(df):
        X = matriz(df, cols, cats)
        return bf.predict(X, raw_score=True) + b0 + bs.predict(X, raw_score=True) + np.log(msev)
    return raw, {"rondas_frec": nf, "rondas_sev": ns, "sev_media": msev, "base_frec": b0}


def tronco_fsp(dv, cols, cats, linea):
    """E21 FSp: frecuencia Poisson y severidad Gamma por peligro en el desarrollado; un solo nivel emergente por ramo."""
    from completitud import por_peligro
    Xd = matriz(dv, cols, cats); val = dv.fold.values == 0
    perils = sorted(c[2:] for c in dv.columns if c.startswith("n_") and c != "n_sin" and c != "n_lineas")
    modelos = {}; meta = {}
    pf = {k: v for k, v in P_DEVB[linea].items() if k != "tweedie_variance_power"}
    pf["objective"] = "poisson"
    ps = dict(objective="gamma", learning_rate=0.03, num_leaves=15, min_data_in_leaf=200, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0, max_cat_to_onehot=8, cat_smooth=50,
              verbose=-1, num_threads=2, seed=SEMILLA)
    for p in perils:
        fF = MOD / f"FSpf_{linea}_{p}.txt"; fS = MOD / f"FSps_{linea}_{p}.txt"
        n = dv[f"n_{p}"].values; yp = dv[f"y_{p}"].values
        s = (n > 0) & (yp > 0)
        msev = float(yp[s].sum() / n[s].sum())
        if fF.exists():
            bf = lgb.Booster(model_file=str(fF)); bs = lgb.Booster(model_file=str(fS))
        else:
            cp = np.maximum(por_peligro(linea, p, dv.period_start, dv.period_end), 1e-4)
            bf, _ = ajustar(Xd, n.astype(float), np.log(dv.exposure.values * cp), pf, 5000, val=val)
            bs, _ = ajustar(Xd[s].reset_index(drop=True), yp[s] / n[s], np.full(s.sum(), np.log(msev)), ps, 3000,
                            val=val[s], w=n[s].astype(float))
            bf.save_model(str(fF)); bs.save_model(str(fS))
        modelos[p] = (bf, bs, msev)
        meta[p] = {"rondas_frec": bf.current_iteration(), "rondas_sev": bs.current_iteration(), "sev_media": msev,
                   "siniestros": int(n.sum())}
        print("  ", linea, p, meta[p], flush=True)

    def raw(df):
        X = matriz(df, cols, cats)
        tot = np.zeros(len(df))
        for p, (bf, bs, msev) in modelos.items():
            tot += np.exp(bf.predict(X, raw_score=True) + bs.predict(X, raw_score=True)) * msev
        return np.log(tot)
    return raw, {"peligros": meta}


def tronco_h(dv, cats, linea):
    """T2h: T2c + etapa de historia centrada (multiplicador 1 donde la ventana no se observa)."""
    colsC = [x for x in comunes(linea) if x not in HIST]
    f1 = MOD / ("T2ci_%s.txt" % linea if INICIO0 else "T2c_%s.txt" % linea)
    b1 = lgb.Booster(model_file=str(f1)); base1 = _leer_base(f1) if INICIO0 else 0.0
    H = HIST + ["new_business", "renewal_count", "prior_claims_5y"]
    f = MOD / f"{VAR}_{linea}.txt"
    r1 = b1.predict(matriz(dv, colsC, cats), raw_score=True) + base1
    offd = np.log(dv.exposure.values * dv.c_fila.values)
    obs = dv.hist_pol_12m.notna().values
    XH = matriz(dv, H, cats)
    if f.exists():
        b2 = lgb.Booster(model_file=str(f)); n2 = b2.current_iteration()
    else:
        p2 = dict(objective="tweedie", tweedie_variance_power=1.5, learning_rate=0.03, num_leaves=7,
                  min_data_in_leaf=2000, lambda_l2=10.0, verbose=-1, num_threads=2, seed=SEMILLA)
        b2, n2 = ajustar(XH[obs].reset_index(drop=True), dv.y.values[obs], r1[obs] + offd[obs], p2, 3000,
                         val=(dv.fold.values == 0)[obs])
        b2.save_model(str(f))
    mu1 = np.exp(r1[obs] + offd[obs]); g = b2.predict(XH[obs], raw_score=True)
    cen = float(np.log(np.sum(mu1 * np.exp(g)) / np.sum(mu1)))

    def raw(df):
        r = b1.predict(matriz(df, colsC, cats), raw_score=True) + base1
        gg = b2.predict(matriz(df, H, cats), raw_score=True) - cen
        return r + np.where(df.hist_pol_12m.notna().values, gg, 0.0)
    return raw, {"rondas_etapa2": n2, "centrado": cen, "importancia": dict(zip(H, b2.feature_importance("gain").round(1).tolist()))}


# ---------------------------------------------------------------- salidas
def nivel(a):
    return factor_tweedie(a.y.values, np.exp(a.raw.values) * a.exposure.values * a.c_fila.values)


def salidas(em, linea):
    meta = {}
    fo, anios = CONF["oof"]
    tr = em[(em.conjunto == "train") & em.year.isin(anios)]
    out, ks = [], []
    for k in range(5):
        a, b = tr[tr.fold != k], tr[tr.fold == k]
        kk = nivel(a); ks.append(kk)
        out.append(pd.DataFrame({"row_id": b.row_id.values, "mu": kk * np.exp(b.raw.values) * b.exposure.values}))
    (TRABAJO / "pred" / fo).mkdir(parents=True, exist_ok=True)
    pd.concat(out, ignore_index=True).to_parquet(TRABAJO / "pred" / fo / f"{VAR}{SUF}_{linea}.parquet", index=False)
    fe, anios_tr, anio_ev = CONF["ev"]
    a = em[(em.conjunto == "train") & em.year.isin(anios_tr)]; b = em[em.year == anio_ev]
    kk = nivel(a)
    (TRABAJO / "pred" / fe).mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"row_id": b.row_id.values, "mu": kk * np.exp(b.raw.values) * b.exposure.values}).to_parquet(
        TRABAJO / "pred" / fe / f"{VAR}{SUF}_{linea}.parquet", index=False)
    em[["row_id", "raw"]].to_parquet(TRABAJO / "pred" / fe / f"{VAR}{SUF}raw_{linea}.parquet", index=False)
    meta.update({"k_oof": ks, "k_eval": kk})
    return meta


def correr(linea):
    t0 = time.time()
    em = cargar("emerging", linea, CONF["clima"])
    dva = cargar("developed", linea, CONF["clima"])
    if VAR == "T2x":
        em = em.merge(tenencia("emerging"), on=["customer_id", "year"], how="left")
        dva = dva.merge(tenencia("developed"), on=["customer_id", "year"], how="left")
    if VAR == "FSp":
        dva = dva.merge(pd.read_parquet(TRABAJO / f"conteos_developed_{linea}.parquet"), on="row_id", how="left")
    dv = dva[dva.year.isin(CONF["dev_years"])].reset_index(drop=True)
    cols = columnas(linea)
    cats = categorias_union([dv, em], cols + HIST + ["new_business", "renewal_count", "prior_claims_5y"])
    if VAR in ("T2x", "T2o", "T2b", "T2c", "T2bi", "T2ci", "T2oi"):
        raw, meta = tronco_lgb(dv, cols, cats, linea, VAR)
    elif VAR == "CB":
        raw, meta = tronco_cb(dv, cols, cats, linea)
    elif VAR in ("FS", "FSi", "FSt"):
        raw, meta = tronco_fs(dv, cols, cats, linea)
    elif VAR == "FSp":
        raw, meta = tronco_fsp(dv, cols, cats, linea)
    elif VAR == "F0":  # diagnóstico: frecuencia de FS x severidad media constante
        bf = lgb.Booster(model_file=str(MOD / f"FSf_{linea}.txt"))
        s = (dv.n_sin.values > 0) & (dv.y.values > 0)
        lm = float(np.log(dv.y.values[s].sum() / dv.n_sin.values[s].sum()))
        raw, meta = (lambda df: bf.predict(matriz(df, cols, cats), raw_score=True) + lm), {"rondas_frec": bf.current_iteration()}
    elif VAR in ("T2h", "T2hi"):
        raw, meta = tronco_h(dv, cats, linea)
    else:
        raise ValueError(VAR)
    meta["s_tronco"] = round(time.time() - t0)
    # DES24: estructura fuera de tiempo en el desarrollado, renivelada en el propio año
    if CONF["des"] is not None:
        d4 = dva[dva.year == CONF["des"]].reset_index(drop=True)
        mu = np.exp(raw(d4)) * d4.exposure.values * d4.c_fila.values
        k4 = factor_tweedie(d4.y.values, mu)
        meta["DES24"] = {"D2": d2(d4.y.values, k4 * mu), "k": k4, "n": len(d4)}
        if os.environ.get("MLIND_GUARDAR_DES24"):  # E26: raw del desarrollado 2024 para estimar pesos de mezcla
            pd.DataFrame({"row_id": d4.row_id.values, "raw": np.log(mu / (d4.exposure.values * d4.c_fila.values)),
                          "y": d4.y.values, "exposure": d4.exposure.values, "c_fila": d4.c_fila.values}).to_parquet(
                TRABAJO / "pred" / "hold" / f"{VAR}{SUF}des24_{linea}.parquet", index=False)
    em = em.assign(raw=raw(em))
    if VAR not in ("T2b", "T2c") or SUF:
        meta.update(salidas(em, linea))
    meta["s"] = round(time.time() - t0)
    guardar_json(meta, TRABAJO / "pred" / CONF["ev"][0] / f"{VAR}{SUF}_{linea}_meta.json")
    print(VAR, FASE, linea, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in meta.items() if k != "importancia"}, flush=True)


if __name__ == "__main__":
    lineas = sys.argv[3].split(",") if len(sys.argv) > 3 else LINEAS
    for l in lineas:
        correr(l)
'''


### Embedded source: `ml/codigo/variables.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/variables.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/variables.py'] = '''"""Esquema común de variables y carga de matrices por fase (línea ML independiente)."""
from __future__ import annotations
import numpy as np, pandas as pd
from comun import *

POLIZA = ["voluntary_excess", "channel", "payment_method", "missed_payments_12m", "new_business", "renewal_count",
          "log_days_quote", "discount_applied", "start_month", "full_year", "hist_pol_12m", "hist_cli_12m"]
CLIENTE = ["marital_status", "household_size", "prior_claims_5y", "licence_points", "years_at_address",
           "tenure_years", "age_band", "occupation"]
MARKETING = ["email_domain", "marketing_consent", "document_preference", "previous_insurer", "campaign_code",
             "branch_id", "marketing_segment"]
DIRECCION = ["urban_density", "flood_zone", "distance_to_river_km", "elevation_m", "soil_type", "property_age",
             "property_type", "roof_type", "basement", "trees_within_5m", "listed_building", "bedrooms",
             "energy_rating", "solar_panels", "clima_wind_exposure_index", "clima_wildfire_index",
             "clima_rainfall_mm", "clima_mean_temperature_c"]
RAMO = {
    "motor": ["years_licensed", "ncb_years", "vehicle_value_group", "vehicle_age", "engine_size_cc", "body_type",
              "transmission", "seats", "fuel_type", "driver_assist", "cover_type", "overnight_parking",
              "vehicle_use", "named_young_driver"],
    "buildings": ["log_si", "construction", "occupancy", "days_unoccupied", "smoke_alarms",
                  "accidental_damage_cover", "mortgage"],
    "contents": ["log_si", "tenure", "security_level", "high_value_items", "accidental_damage_cover",
                 "works_from_home", "lodgers", "bicycle_cover"],
}
SOLO_EMERGENTE = ["gender", "self_employed", "region"]
TARIFA = ["log_wp_rel"]
MAKE = ["make_model"]

CATEGORICAS = {"channel", "payment_method", "marital_status", "age_band", "occupation", "email_domain",
               "document_preference", "previous_insurer", "campaign_code", "marketing_segment", "urban_density",
               "soil_type", "property_type", "roof_type", "energy_rating", "body_type", "transmission", "fuel_type",
               "cover_type", "overnight_parking", "vehicle_use", "construction", "occupancy", "tenure",
               "security_level", "gender", "region", "make_model", "branch_id"}


def comunes(linea, marketing=False):
    return POLIZA + CLIENTE + (MARKETING if marketing else []) + DIRECCION + RAMO[linea]


def _historia(df, claims, inicio_datos):
    """Avisos (cualquier estado) de la póliza y del cliente reportados en los 365 días previos al inicio.
    NaN si la ventana no está observada completa en el mercado."""
    lo = df.period_start - pd.Timedelta(days=365)
    ok = lo >= pd.Timestamp(inicio_datos)
    hp = claims[["policy_id", "line", "report_date"]].merge(
        df[["row_id", "policy_id", "line", "period_start"]], on=["policy_id", "line"])
    hp = hp[(hp.report_date < hp.period_start) & (hp.report_date >= hp.period_start - pd.Timedelta(days=365))]
    pol = df.row_id.map(hp.groupby("row_id").size()).fillna(0)
    return pol.where(ok, np.nan)


_cache = {}


def cargar(mercado, linea, ventana_clima):
    """Base con variables derivadas. ventana_clima: 'hasta2023' o 'hasta2024'."""
    key = (mercado, linea, ventana_clima)
    if key in _cache:
        return _cache[key].copy()
    df = pd.read_parquet(TRABAJO / f"base_{mercado}_{linea}.parquet")
    cl = pd.read_parquet(TRABAJO / f"clima_{mercado}_{ventana_clima}.parquet")
    df = df.merge(cl, on="address_id", how="left")
    if "occupation" in df:
        df["occupation"] = df.occupation.str.lower().str.strip()
    import os  # A05: MLIND_HISTORIA=2 usa la ventana que termina en min(inicio, V) (solo cambia filas 2025)
    pref = "historia2" if os.environ.get("MLIND_HISTORIA") == "2" else "historia"
    hist = pd.read_parquet(TRABAJO / f"{pref}_{mercado}_{linea}.parquet")
    df = df.merge(hist, on="row_id", how="left")
    _cache[key] = df
    return df.copy()


def matriz(df, cols, categorias):
    X = pd.DataFrame(index=df.index)
    for c in cols:
        if c in CATEGORICAS:
            v = df[c].astype(str).where(df[c].notna(), None) if c in df else pd.Series(None, index=df.index)
            X[c] = pd.Categorical(v, categories=categorias[c])
        else:
            X[c] = pd.to_numeric(df[c], errors="coerce").astype(float) if c in df else np.nan
    return X


def categorias_union(frames, cols):
    out = {}
    for c in cols:
        if c in CATEGORICAS:
            vals = set()
            for f in frames:
                if c in f:
                    vals |= set(f[c].dropna().astype(str).unique())
            out[c] = sorted(vals)
    return out
'''


### Embedded source: `ml/codigo/verificar_informe.py`

Historical source: `src/semana2/lineas/ml_independiente_20261023/codigo/verificar_informe.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/verificar_informe.py'] = '''"""Comprueba que cada cifra del informe sale de su artefacto. Solo biblioteca estándar más pandas para el CSV."""
import json, re, sys, hashlib
from pathlib import Path
import pandas as pd

R = Path(__file__).resolve().parents[1]
E = R / "entrega" if (R / "entrega").exists() else R
T = Path(__file__).resolve().parents[1] / "trabajo"
if not T.exists():  # copia en el equipo: las evidencias de trabajo viajan dentro de entrega
    T = None
txt = (E / "INFORME.md").read_text(encoding="utf-8")
J = lambda f: json.load(open(E / f, encoding="utf-8"))


def es(x, d):
    s = f"{x:,.{d}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")


def pct(x, d=1):
    return es(100 * x, d) + " %"


dev = pd.read_csv(E / "tabla_dev.csv")
D = lambda c, k, col="D2": float(dev[(dev.cand == c) & (dev.calib == k)][col].iloc[0])
hold = J("resultados_hold.json")["candidatos"]
H = lambda c: hold[c]["sin_tendencia"]["D2"]
bs = J("bootstrap_final.json"); prod = J("produccion_2025.json"); cal = J("calibracion_pcv.json"); ctx = J("contexto_2025.json")
_t = (lambda f, alt=None: json.load(open((T / f) if T else (E / (alt or Path(f).name)), encoding="utf-8")))
prep = _t("preparacion.json"); pesos = _t("pesos_2025_dev.json")
hb = J("hold_prueba_tendencia.json"); cred = J("credibilidad_entre_anios.json"); fig = J("figuras_datos.json")
tend_h = _t("pred/hold/tendencia.json", "tendencia_hold.json")
V = prod["variantes"]
pc = pd.read_csv(E / "prima_comercial_20.csv").groupby("ramo")[["prima_comercial", "prima_escrita"]].mean()
ic = lambda k: f"[{'+' if bs[k]['ic95'][0] >= 0 else '−'}{es(abs(bs[k]['ic95'][0]), 4)}; {'+' if bs[k]['ic95'][1] >= 0 else '−'}{es(abs(bs[k]['ic95'][1]), 4)}]"
checks = [
    ("total 2025", es(V["final_C1"]["total"], 0)),
    ("autos 2025", es(V["final_C1"]["totales"]["motor"]["total"], 0)),
    ("edificios 2025", es(V["final_C1"]["totales"]["buildings"]["total"], 0)),
    ("contenidos 2025", es(V["final_C1"]["totales"]["contents"]["total"], 0)),
    ("tasa autos", es(V["final_C1"]["totales"]["motor"]["tasa"], 1)),
    ("tasa edificios", es(V["final_C1"]["totales"]["buildings"]["tasa"], 1)),
    ("tasa contenidos", es(V["final_C1"]["totales"]["contents"]["tasa"], 1)),
    ("sha", V["final_C1"]["controles"]["sha256"][:16]),
    ("DEV T2b C1", es(D("T2b", "ramo"), 4)), ("HOLD T2b C1", es(H("T2b_C1"), 4)),
    ("DEV R0 C1", es(D("R0", "ramo"), 4)), ("HOLD R0", es(H("R0_C1"), 4)),
    ("dif DEV T2b-R0", "+" + es(bs["DEV_T2b_C1-R0_C1"]["dif"], 4)), ("IC DEV T2b-R0", ic("DEV_T2b_C1-R0_C1")),
    ("dif HOLD T2b-R0", "+" + es(bs["HOLD_T2b_C1-R0_C1"]["dif"], 4)), ("IC HOLD T2b-R0", ic("HOLD_T2b_C1-R0_C1")),
    ("dif DEV T2b-T1wp", "+" + es(bs["DEV_T2b_C1-T1wp_C1"]["dif"], 4)), ("IC DEV T2b-T1wp", ic("DEV_T2b_C1-T1wp_C1")),
    ("dif HOLD T2b-T1wp", "+" + es(bs["HOLD_T2b_C1-T1wp_C2"]["dif"], 4)), ("IC HOLD T2b-T1wp", ic("HOLD_T2b_C1-T1wp_C2")),
    ("N2a-N1 DEV", "+" + es(bs["DEV_N2a_C2-N1_C2"]["dif"], 4)), ("IC N2a-N1 DEV", ic("DEV_N2a_C2-N1_C2")),
    ("N2a-N1 HOLD", "+" + es(bs["HOLD_N2a_C2-N1_C2"]["dif"], 4)), ("IC N2a-N1 HOLD", ic("HOLD_N2a_C2-N1_C2")),
    ("N3-T2b DEV", "−" + es(abs(bs["DEV_N3_C2-T2b_C2"]["dif"]), 4)), ("IC N3-T2b DEV", ic("DEV_N3_C2-T2b_C2")),
    ("C2-C1 HOLD", es(abs(bs["HOLD_T2b_C2-T2b_C1"]["dif"]), 4)), ("IC C2-C1 HOLD", ic("HOLD_T2b_C2-T2b_C1")),
    ("T0 emergente", es(sum(sum(prep[f"emerging_{l}_T0_por_anio"].values()) for l in ("motor", "buildings", "contents")), 2)),
    ("compl 2024 autos", es(prep["emerging_motor_completitud_media_por_anio"]["2024"], 4)),
    ("compl 2024 edif", es(prep["emerging_buildings_completitud_media_por_anio"]["2024"], 4)),
    ("compl 2024 cont", es(prep["emerging_contents_completitud_media_por_anio"]["2024"], 4)),
    ("compl 2023 autos", es(prep["emerging_motor_completitud_media_por_anio"]["2023"], 4)),
    ("BI 90 días", pct(prep["cdf_coste_desarrollado"]["motor|bodily_injury"]["90"], 0)),
    ("BI 365 días", pct(prep["cdf_coste_desarrollado"]["motor|bodily_injury"]["365"], 0)),
    ("capital edif train", str(prep["emerging_buildings_capital_div1000"]["train"])),
    ("capital cont test", str(prep["emerging_contents_capital_div1000"]["test"])),
    ("duplicados", str(prep["developed_duplicados_retirados"])),
    ("AUC autos", es(pesos["motor"]["auc"], 2)), ("AUC hogar", es(pesos["buildings"]["auc"], 2)),
    ("incendio emergente", pct(prep["emerging_buildings_pesos_peligro"]["fire"], 1)),
    ("incendio desarrollado", pct(prep["developed_buildings_pesos_peligro"]["fire"], 1)),
    ("DEV T3bwp C2", es(D("T3bwp", "ramo_nb"), 4)), ("DEV T2b C2", es(D("T2b", "ramo_nb"), 4)),
    ("DEV N2a C2", es(D("N2a", "ramo_nb"), 4)), ("DEV N1 C2", es(D("N1", "ramo_nb"), 4)),
    ("DEV N3 C2", es(D("N3", "ramo_nb"), 4)), ("DEV E1 C2", es(D("E1", "ramo_nb"), 4)),
    ("DEV T4 C2", es(D("T4", "ramo_nb"), 4)), ("DEV T6 C2", es(D("T6", "ramo_nb"), 4)), ("DEV T5 C2", es(D("T5", "ramo_nb"), 4)),
    ("DEV T2b cuerpo", es(D("T2b", "ramo_nb", "D2_cuerpo"), 4)), ("DEV T2b w2025", es(D("T2b", "ramo_nb", "D2_w2025"), 4)),
    ("DEV T1wp w2025", es(D("T1wp", "ramo_nb", "D2_w2025"), 4)), ("DEV G1 C1", es(D("G1", "ramo"), 4)),
    ("HOLD N2a", es(H("N2a_C2"), 4)), ("HOLD N1", es(H("N1_C2"), 4)), ("HOLD T1wp", es(H("T1wp_C2"), 4)), ("HOLD T3bwp", es(H("T3bwp_C2"), 4)),
    ("tendencia hold autos", pct(tend_h["motor"]["tau_anual"] - 1, 1).replace(" %", "")),
    ("tend C2", "−" + es(abs(hb["T2b_C2"]["dif"]), 4)), ("tend C1", "−" + es(abs(hb["T2b_C1"]["dif"]), 4)),
    ("NB autos 2023", es(fig["ae_nb_anio"]["motor|True|2023"]["AE"], 2)), ("NB autos 2024", es(fig["ae_nb_anio"]["motor|True|2024"]["AE"], 2)),
    ("cred autos", es(cred["motor"]["z_optimo"], 1)),
    ("MCB autos", pct(cal["murphy"]["motor"]["MCB"] / cal["murphy"]["motor"]["S"], 1)),
    ("MCB edif", pct(cal["murphy"]["buildings"]["MCB"] / cal["murphy"]["buildings"]["S"], 1)),
    ("MCB cont", pct(cal["murphy"]["contents"]["MCB"] / cal["murphy"]["contents"]["S"], 1)),
    ("forma edif", es(cal["forma"]["buildings"]["ganancia_D2_en_muestra"], 4)),
    ("balance autos", pct(prod["balance_oof_pcv"]["motor"], 1)), ("balance cont", pct(prod["balance_oof_pcv"]["contents"], 1)),
    ("total C2", es(V["C2"]["total"], 0)), ("total preinscrita", es(V["preinscrita_C2_tendencia"]["total"], 0)),
    ("total tendencia", es(V["C1_tendencia"]["total"], 0)), ("total T2c", es(V["T2c_C1"]["total"], 0)),
    ("C2 sube", pct(V["C2"]["total"] / V["final_C1"]["total"] - 1, 1)),
    ("tasa obs 2023 autos", es(ctx["motor"]["2023"]["tasa_ultima_obs"], 1)), ("tasa obs 2024 edif", es(ctx["buildings"]["2024"]["tasa_ultima_obs"], 1)),
    ("razón edificios", es(ctx["buildings"]["2025"]["razon_riesgo_prima_escrita"], 2)),
    ("prima comercial autos", es(pc.loc["motor", "prima_comercial"], 1)), ("prima escrita edif", es(pc.loc["buildings", "prima_escrita"], 1)),
]
fallos = [(n, s) for n, s in checks if s not in txt]
# controles del CSV
cands = [E / "prediccion_2025_mlind.csv", E.parent / "privado" / "prediccion_2025_mlind.csv",
         E.parents[2] / "Repositorio_IAA2026" / "privado" / "semana2" / "claude_ml_independiente_20261023" / "prediccion_2025_mlind.csv"]
csv = next((c for c in cands if c.exists()), None)
if csv is None:
    print("aviso: CSV de predicción no encontrado junto al informe; se omite su huella")
else:
    h = hashlib.sha256(csv.read_bytes()).hexdigest()
    if h != V["final_C1"]["controles"]["sha256"]:
        fallos.append(("sha del CSV", h))
for n, s in fallos:
    print("FALLA", n, s)
print(f"{len(checks) - len(fallos)} de {len(checks)} cifras del informe ligadas a su artefacto" + ("" if not fallos else " (hay fallos)"))
sys.exit(1 if fallos else 0)
'''


### Embedded source: `noche/codigo/base_mercado.py`

Historical source: `src/semana2/lineas/validacion_nocturna_20261023/codigo/base_mercado.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/base_mercado.py'] = '''"""Base independiente del mercado emergente (semana 2), desde los Parquet originales.

Reglas, fijadas antes de medir su efecto:
  EXP-N1  exposure_model = round(días inclusivos / 365, 4) en todas las filas (convención del test 2025).
  ASG-N1  un siniestro se asigna a la fila con mismo policy_id, mismo ramo y period_start <= loss_date <= period_end.
          Categorías: unica, sin_fila (ninguna fila contiene la fecha), ambigua (más de una).
  OBJ-N1  T0 = suma por fila de (incurred_to_date - recoveries) de siniestros no rechazados asignados de forma única.
          No se vuelve a restar la franquicia (paid e incurred ya son netos). Rechazados fuera.
  CAP-N1  capital de hogar: se marca la escala dudosa con la prueba de prima por mil (prima escrita / capital * 1000)
          y se crea sum_insured_model = sum_insured / 1000 solo si la fila supera el umbral del ramo y su prima por mil
          queda por debajo del mínimo de la población normal; sum_insured_raw intacto.
Salida: salidas/s2/base_{ramo}.parquet (train 2023-2024 y test 2025) y salidas/s2/asignacion.parquet.
"""
import json, hashlib, pathlib
import numpy as np, pandas as pd

R = pathlib.Path('__RUN_ROOT__/noche')
import sys as _s
MK = _s.argv[1] if len(_s.argv) > 1 else 's2'
D = R / f'datos/{MK}'
O = R / f'salidas/{MK}'; O.mkdir(parents=True, exist_ok=True)
LINES = ['motor', 'buildings', 'contents']
PERILS = {'motor': ['own_damage', 'third_party_prop', 'bodily_injury', 'theft', 'glass_hail'],
          'buildings': ['storm', 'flood', 'escape_of_water', 'subsidence', 'fire', 'accidental_damage'],
          'contents': ['theft', 'escape_of_water', 'accidental_damage', 'flood']}
CUT = pd.Timestamp('2024-12-31')
log = {}

def sha(p):
    return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()

claims = pd.read_parquet(D / 'claims.parquet')
if MK == 's1':  # DUP-N1: 604 copias identicas salvo claim_id (11 caracteres); se conserva el de 10
    claims = claims[claims.claim_id.str.len() == 10].reset_index(drop=True)
claims['ld'] = pd.to_datetime(claims.loss_date)
claims['rd'] = pd.to_datetime(claims.report_date)
claims['net'] = claims.incurred_to_date - claims.recoveries
claims['accepted'] = claims.status != 'repudiated'

asg_all = []
for ln in LINES:
    tr = pd.read_parquet(D / f'policies_{ln}.parquet'); tr['split'] = 'train'
    te = pd.read_parquet(D / f'test_{ln}.parquet'); te['split'] = 'test'
    df = pd.concat([tr, te], ignore_index=True)
    df['ps'] = pd.to_datetime(df.period_start); df['pe'] = pd.to_datetime(df.period_end)
    df['year'] = df.ps.dt.year
    days = (df.pe - df.ps).dt.days + 1
    df['days'] = days
    df['exposure_raw'] = df.exposure
    df['exposure_model'] = (days / 365).round(4)
    df['exp_changed'] = ~np.isclose(df.exposure_raw, df.exposure_model, atol=6e-5)
    log[f'{ln}_exp_changed_train'] = int(df.exp_changed[df.split == 'train'].sum())
    log[f'{ln}_exp_changed_test'] = int(df.exp_changed[df.split == 'test'].sum())

    c = claims[claims.line == ln].copy()
    m = c[['claim_id', 'policy_id', 'ld']].merge(df.loc[df.split == 'train', ['row_id', 'policy_id', 'ps', 'pe']], on='policy_id', how='left')
    m['inside'] = (m.ld >= m.ps) & (m.ld <= m.pe)
    n_in = m.groupby('claim_id').inside.sum()
    c['n_rows'] = c.claim_id.map(n_in).fillna(0).astype(int)
    c['category'] = np.select([c.n_rows == 1, c.n_rows == 0], ['unica', 'sin_fila'], 'ambigua')
    first = df[df.split == 'train'].groupby('policy_id').ps.min()
    c['policy_first_start'] = c.policy_id.map(first)
    c['prehistoria'] = (c.category == 'sin_fila') & (c.ld < c.policy_first_start)
    rid = m[m.inside].drop_duplicates('claim_id').set_index('claim_id').row_id
    c['row_id'] = c.claim_id.map(rid)
    asg_all.append(c)

    ok = c[(c.category == 'unica') & c.accepted]
    g = ok.groupby('row_id')
    agg = pd.DataFrame({'t0': g.net.sum(), 'paid': g.paid_to_date.sum(), 'incurred': g.incurred_to_date.sum(),
                        'recov': g.recoveries.sum(), 'n_claims': g.size(),
                        'n_open': g.apply(lambda x: (x.status == 'open').sum(), include_groups=False),
                        'case_reserve': g.apply(lambda x: (x.incurred_to_date - x.paid_to_date).sum(), include_groups=False)})
    for p in PERILS[ln]:
        sp = ok[ok.peril == p].groupby('row_id')
        agg[f'n_{p}'] = sp.size(); agg[f't0_{p}'] = sp.net.sum()
    df = df.merge(agg, left_on='row_id', right_index=True, how='left')
    num = [x for x in agg.columns]
    df.loc[df.split == 'train', num] = df.loc[df.split == 'train', num].fillna(0)
    log[f'{ln}_claims'] = {k: int(v) for k, v in c.groupby(['category']).size().items()}
    log[f'{ln}_accepted_unica'] = int(len(ok)); log[f'{ln}_t0'] = round(float(ok.net.sum()), 2)
    log[f'{ln}_prehistoria'] = int(c.prehistoria.sum())
    log[f'{ln}_sin_fila_no_prehist'] = int(((c.category == 'sin_fila') & ~c.prehistoria).sum())

    if ln in ('buildings', 'contents'):
        df['sum_insured_raw'] = df.sum_insured
        df['ppm'] = df.written_premium / df.sum_insured * 1000
        log[f'{ln}_si_null'] = int(df.sum_insured.isna().sum())
    df.drop(columns=['ps', 'pe']).to_parquet(O / f'base_{ln}.parquet', index=False)

asg = pd.concat(asg_all, ignore_index=True)
asg.drop(columns=['ld', 'rd']).to_parquet(O / 'asignacion.parquet', index=False)
log['t0_total'] = round(sum(log[f'{l}_t0'] for l in LINES), 2)
json.dump(log, open(O / 'log_base.json', 'w'), indent=1, default=str)
print(json.dumps(log, indent=1, default=str))
'''


### Embedded source: `noche/codigo/reporte.py`

Historical source: `src/semana2/lineas/validacion_nocturna_20261023/codigo/reporte.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/reporte.py'] = '''"""Completitud de reporte por peligro con el estimador de Lynden-Bell (datos truncados a la derecha).

Supuesto REP-N1 (fijado antes de medir): el retraso de reporte de un siniestro aceptado depende del peligro y no del
mercado ni del año; se estima con los dos mercados juntos y se contrasta mercado contra mercado. F(t) es la probabilidad
de estar reportado t días después de la ocurrencia, normalizada a 1 en el mayor retraso observable (1.460 días); la cola
más allá no es identificable con una sola valoración y se declara como límite.
Coste: se contrasta si la severidad depende del retraso (cohortes maduras del desarrollado). Si no depende, la
completitud de importes es igual a la de conteos.
"""
import json, pathlib
import numpy as np, pandas as pd

R = pathlib.Path('__RUN_ROOT__/noche')
CUT = pd.Timestamp('2024-12-31')
B = 7  # días por cubeta


def load(mk):
    c = pd.read_parquet(R / f'datos/{mk}/claims.parquet')
    if mk == 's1':  # duplicados idénticos salvo claim_id: se conserva el de 10 caracteres
        c = c[c.claim_id.str.len() == 10]
    c = c[c.status != 'repudiated'].copy()
    c['ld'] = pd.to_datetime(c.loss_date); c['rd'] = pd.to_datetime(c.report_date)
    c['delay'] = (c.rd - c.ld).dt.days; c['window'] = (CUT - c.ld).dt.days
    c['market'] = mk
    c['net'] = c.incurred_to_date - c.recoveries
    return c


def lynden_bell(delay, window, tmax):
    """F(t) en cubetas de B días; delay<=window por construcción."""
    db = (delay // B).astype(int); wb = (window // B).astype(int)
    nb = tmax // B + 1
    d = np.bincount(db, minlength=nb)[:nb]
    # riesgo en tiempo inverso: casos con db <= s <= wb
    start = np.bincount(db, minlength=nb + 1)[:nb]
    end = np.bincount(np.minimum(wb, nb - 1) + 1, minlength=nb + 1)[:nb]
    risk = np.cumsum(start) - np.cumsum(end)  # casos con db<=s y wb>=s
    h = np.where(risk > 0, d / np.maximum(risk, 1), 0.0)
    # F(s) = prod_{u>s} (1-h_u)
    surv = np.cumprod((1 - h)[::-1])[::-1]
    F = np.concatenate([surv[1:], [1.0]])
    return F  # F[s] = P(delay bucket <= s | delay <= tmax)


def main():
    c = pd.concat([load('s1'), load('s2')], ignore_index=True)
    tmax = int(c.window.max())
    out = {}; tabla = {}
    for p, g in c.groupby('peril'):
        F = lynden_bell(g.delay.values, g.window.values, tmax)
        Fe = lynden_bell(g[g.market == 's2'].delay.values, g[g.market == 's2'].window.values, 730) if (g.market == 's2').sum() > 30 else None
        Fd = lynden_bell(g[g.market == 's1'].delay.values, g[g.market == 's1'].window.values, tmax)
        tabla[p] = F.tolist()
        k = {lag: round(float(F[min(lag // B, len(F) - 1)]), 4) for lag in (30, 91, 182, 365, 730)}
        kd = {lag: round(float(Fd[min(lag // B, len(Fd) - 1)]), 4) for lag in (30, 91, 182, 365, 730)}
        ke = {lag: round(float(Fe[min(lag // B, len(Fe) - 1)] * Fd[min(730 // B, len(Fd) - 1)]), 4) for lag in (30, 91, 182, 365, 730)} if Fe is not None else None
        out[p] = {'n': int(len(g)), 'n_emerging': int((g.market == 's2').sum()), 'F_pooled': k, 'F_developed': kd,
                  'F_emerging_rescaled_to_dev730': ke}
    json.dump({'bucket_days': B, 'tmax_days': tmax, 'F': tabla}, open(R / 'salidas/reporte_F.json', 'w'))
    json.dump(out, open(R / 'salidas/reporte_resumen.json', 'w'), indent=1)
    # severidad por retraso en cohortes maduras del desarrollado (2021-2022): razón media tardío / temprano
    m = c[(c.market == 's1') & (c.ld.dt.year <= 2022)]
    sev = {}
    for p, g in m.groupby('peril'):
        q = g.delay.quantile([0.5, 0.9])
        sev[p] = {'media_hasta_p50': round(float(g[g.delay <= q[0.5]].net.mean()), 1),
                  'media_p50_p90': round(float(g[(g.delay > q[0.5]) & (g.delay <= q[0.9])].net.mean()), 1),
                  'media_sobre_p90': round(float(g[g.delay > q[0.9]].net.mean()), 1), 'n': int(len(g))}
    json.dump(sev, open(R / 'salidas/severidad_por_retraso.json', 'w'), indent=1)
    print(json.dumps(out, indent=1)); print(json.dumps(sev, indent=1))


if __name__ == '__main__':
    main()
'''


### Embedded source: `noche/codigo/exposicion_efectiva.py`

Historical source: `src/semana2/lineas/validacion_nocturna_20261023/codigo/exposicion_efectiva.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/exposicion_efectiva.py'] = '''"""Exposición efectiva por peligro y por coste para cada fila de entrenamiento.

e_ef_p = (1/365) * suma_{d en el periodo} F_p(corte - d), con F_p de reporte.py (conteos) o ponderada por coste.
e_ef_coste = suma_p s_p * e_ef_p / suma_p s_p, con s_p la participación del peligro en el coste último del ramo.
Para 2025 (test) la exposición a usar al predecir el último es la nominal.
"""
import json, pathlib
import numpy as np, pandas as pd

R = pathlib.Path('__RUN_ROOT__/noche')
CUT = pd.Timestamp('2024-12-31')
PERILS = {'motor': ['own_damage', 'third_party_prop', 'bodily_injury', 'theft', 'glass_hail'],
          'buildings': ['storm', 'flood', 'escape_of_water', 'subsidence', 'fire', 'accidental_damage'],
          'contents': ['theft', 'escape_of_water', 'accidental_damage', 'flood']}


def daily_G(F, B, n):
    Fd = np.repeat(np.asarray(F), B)[:n]
    if len(Fd) < n:
        Fd = np.concatenate([Fd, np.ones(n - len(Fd))])
    return np.concatenate([[0.0], np.cumsum(Fd)])  # G[k] = sum_{t<k} F(t)


def add_effective(df, line, Ftab, B, shares):
    ps = pd.to_datetime(df.period_start); pe = pd.to_datetime(df.period_end)
    w_hi = (CUT - ps).dt.days.values  # mayor ventana (primer día)
    w_lo = (CUT - pe).dt.days.values  # menor ventana (último día)
    train = (w_lo >= 0)
    tot = np.zeros(len(df)); wsum = 0.0
    for p in PERILS[line]:
        G = daily_G(Ftab[p], B, 4000)
        e = np.where(train, (G[np.clip(w_hi, 0, 3999) + 1] - G[np.clip(w_lo, 0, 3999)]) / 365.0, np.nan)
        df[f'eef_{p}'] = e
        tot += shares[p] * np.nan_to_num(e); wsum += shares[p]
    df['eef_coste'] = np.where(train, tot / wsum, np.nan)
    df['comp_coste'] = df.eef_coste / df.exposure_model
    return df
'''


### Embedded source: `noche/codigo/ultimo_mercado.py`

Historical source: `src/semana2/lineas/validacion_nocturna_20261023/codigo/ultimo_mercado.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/ultimo_mercado.py'] = '''"""Exposición efectiva, último estimado y recuperaciones pendientes del emergente.

REP-N2: para el coste se usa la F ponderada por coste (neto recortado al p99,5 del peligro).
Participaciones s_p del peligro en el coste último del ramo: emergente 2023-2024, último_p = T0_p * e / e_ef_p.
REC-N1 (diagnóstico, no se aplica): ningún abierto tiene recuperaciones; se mide cuánto bajaría el neto si los abiertos
recuperaran como los cerrados del mismo peligro.
"""
import json, pathlib, sys
import numpy as np, pandas as pd
sys.path.insert(0, str(pathlib.Path(__file__).parent))
from exposicion_efectiva import add_effective, PERILS

R = pathlib.Path('__RUN_ROOT__/noche')
import sys as _s
MK = _s.argv[1] if len(_s.argv) > 1 else 's2'
O = R / f'salidas/{MK}'
rep = json.load(open(R / 'salidas/reporte_F.json'))
F = rep['F_coste']; B = rep['bucket_days']
res = {}
for ln in PERILS:
    b = pd.read_parquet(O / f'base_{ln}.parquet')
    tr = b.split == 'train'
    tmp = add_effective(b.copy(), ln, F, B, {p: 1.0 for p in PERILS[ln]})
    ult = {}
    for p in PERILS[ln]:
        ult[p] = float(b.loc[tr, f't0_{p}'].sum() * tmp.loc[tr, 'exposure_model'].sum() / tmp.loc[tr, f'eef_{p}'].sum())
    tot = sum(ult.values()); shares = {p: ult[p] / tot for p in ult}
    b = add_effective(b, ln, F, B, shares)
    b.to_parquet(O / f'base_{ln}.parquet', index=False)
    r = {'shares': {p: round(v, 4) for p, v in shares.items()}}
    for y in sorted(b.loc[tr,'year'].unique()):
        s = b[tr & (b.year == y)]
        r[str(y)] = {'exposure': round(s.exposure_model.sum(), 2), 'eef_coste': round(s.eef_coste.sum(), 2),
                     'comp': round(s.eef_coste.sum() / s.exposure_model.sum(), 4), 't0': round(s.t0.sum(), 2),
                     'rate_t0': round(s.t0.sum() / s.exposure_model.sum(), 2),
                     'rate_ult': round(s.t0.sum() / s.eef_coste.sum(), 2),
                     'freq_obs': round(s.n_claims.sum() / s.exposure_model.sum(), 4)}
    res[ln] = r
# recuperaciones pendientes
a = pd.read_parquet(O / 'asignacion.parquet')
a = a[(a.category == 'unica') & a.accepted]
rec = {}
for (ln, p), g in a.groupby(['line', 'peril']):
    cl = g[g.status == 'closed']; op = g[g.status == 'open']
    rate = cl.recoveries.sum() / cl.incurred_to_date.sum() if cl.incurred_to_date.sum() > 0 else 0
    if rate > 0:
        rec[f'{ln}/{p}'] = {'rate_closed': round(rate, 4), 'open_incurred': round(op.incurred_to_date.sum(), 2),
                            'implied_recov': round(rate * op.incurred_to_date.sum(), 2), 'n_open': int(len(op))}
for ln in PERILS:
    t0 = a[a.line == ln].net.sum()
    imp = sum(v['implied_recov'] for k, v in rec.items() if k.startswith(ln + '/'))
    res[ln]['recov_open_implied_share_t0'] = round(imp / t0, 4)
res['recoveries'] = rec
json.dump(res, open(O / 'ultimo_resumen.json', 'w'), indent=1)
print(json.dumps(res, indent=1))
'''


### Embedded source: `noche/codigo/conf_insumos.py`

Historical source: `src/semana2/lineas/validacion_nocturna_20261023/codigo/conf_insumos.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/conf_insumos.py'] = '''"""Insumos de la confirmación 2023 -> 2024 sin etiquetas de 2024 de ningún mercado (regla de SEL01).

1. Completitud de reporte F (conteos y coste) con siniestros de ocurrencia <= 2023 de los dos mercados; el truncamiento
   sigue siendo el 31-12-2024 (fecha de valoración común).
2. Participaciones s_p con el emergente 2023 y el desarrollado 2021-2023.
3. e_ef_coste de las filas 2023 (ajuste) y 2024 (vínculo de observación para evaluar) del emergente, y de 2021-2023 del
   desarrollado.
Salida: salidas/conf/base_{mk}_{ramo}.parquet con las columnas de base + eef_* recalculadas.
"""
import json, pathlib, sys
import numpy as np, pandas as pd
sys.path.insert(0, str(pathlib.Path(__file__).parent))
import reporte as RP
from exposicion_efectiva import add_effective, PERILS

R = pathlib.Path('__RUN_ROOT__/noche'); O = R / 'salidas/conf'; O.mkdir(parents=True, exist_ok=True)
c = pd.concat([RP.load('s1'), RP.load('s2')], ignore_index=True)
c = c[c.ld.dt.year <= 2023]
tmax = int(c.window.max())
Fc = {}
for p, g in c.groupby('peril'):
    cap = g.net.quantile(0.995)
    w = np.minimum(g.net.values, cap)
    db = (g.delay.values // RP.B).astype(int); wb = (g.window.values // RP.B).astype(int); nb = tmax // RP.B + 1
    d = np.bincount(db, weights=w, minlength=nb)[:nb]
    st = np.bincount(db, weights=w, minlength=nb + 1)[:nb]
    en = np.bincount(np.minimum(wb, nb - 1) + 1, weights=w, minlength=nb + 1)[:nb]
    risk = np.cumsum(st) - np.cumsum(en)
    h = np.where(risk > 0, d / np.maximum(risk, 1e-9), 0)
    surv = np.cumprod((1 - h)[::-1])[::-1]
    Fc[p] = np.concatenate([surv[1:], [1.0]]).tolist()
json.dump({'bucket_days': RP.B, 'F_coste': Fc, 'nota': 'ocurrencia <= 2023'}, open(O / 'reporte_F_conf.json', 'w'))
res = {}
for mk, years in (('s2', [2023, 2024]), ('s1', [2021, 2022, 2023])):
    for ln in PERILS:
        b = pd.read_parquet(R / f'salidas/{mk}/base_{ln}.parquet')
        b = b[(b.split == 'train') & b.year.isin(years)].reset_index(drop=True)
        fit_years = [y for y in years if y <= 2023]
        tmp = add_effective(b.copy(), ln, Fc, RP.B, {p: 1.0 for p in PERILS[ln]})
        f = tmp.year.isin(fit_years) if mk == 's1' else tmp.year == 2023
        ult = {p: float(b.loc[f, f't0_{p}'].sum() * tmp.loc[f, 'exposure_model'].sum() / tmp.loc[f, f'eef_{p}'].sum()) for p in PERILS[ln]}
        tot = sum(ult.values()); shares = {p: v / tot for p, v in ult.items()}
        b = add_effective(b, ln, Fc, RP.B, shares)
        b.to_parquet(O / f'base_{mk}_{ln}.parquet', index=False)
        res[f'{mk}_{ln}'] = {'shares': {p: round(v, 4) for p, v in shares.items()},
                             'comp_by_year': b.groupby('year').apply(lambda x: x.eef_coste.sum() / x.exposure_model.sum(), include_groups=False).round(4).to_dict()}
json.dump(res, open(O / 'conf_insumos.json', 'w'), indent=1, default=str)
print(json.dumps(res, indent=1, default=str))
'''


### Embedded source: `noche/codigo/desarrollado_gbm.py`

Historical source: `src/semana2/lineas/validacion_nocturna_20261023/codigo/desarrollado_gbm.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/desarrollado_gbm.py'] = '''"""Relatividades del mercado desarrollado con el esquema común (TRF-N1).

LightGBM Tweedie 1,5 por ramo, entrenado con 2021-2024 del desarrollado (offset log e_ef_coste), solo con variables que
existen en los dos mercados (edad en bandas, sin región, sin crédito, kilometraje ni telemática) más la exposición.
Las etiquetas emergentes no intervienen: aplicar el modelo al emergente no filtra información del desarrollo 2023.
Guarda el modelo y la predicción log-relativa g_dev para cada fila emergente (train y test) y para el desarrollado.
Uso: python desarrollado_gbm.py ramo [--años 2021,2022,2023,2024]
"""
import json, pathlib, sys, time
import numpy as np, pandas as pd, lightgbm as lgb
sys.path.insert(0, str(pathlib.Path(__file__).parent))
from variables import features
import marco as M

R = pathlib.Path('__RUN_ROOT__/noche'); O = R / 'salidas/transfer'; O.mkdir(parents=True, exist_ok=True)
line = sys.argv[1]
years = [2021, 2022, 2023, 2024]
if '--años' in sys.argv:
    years = [int(x) for x in sys.argv[sys.argv.index('--años') + 1].split(',')]
tag = 'dev' + ''.join(str(y)[-2:] for y in years) + (sys.argv[sys.argv.index('--sufijo') + 1] if '--sufijo' in sys.argv else '')
LR = float(sys.argv[sys.argv.index('--lr') + 1]) if '--lr' in sys.argv else 0.05
PAT = int(sys.argv[sys.argv.index('--pat') + 1]) if '--pat' in sys.argv else 200
SEED = int(sys.argv[sys.argv.index('--seed') + 1]) if '--seed' in sys.argv else 11
t = time.time()
CONF = '--conf' in sys.argv  # confirmación: bases sin etiquetas de 2024
b1 = pd.read_parquet(R / (f'salidas/conf/base_s1_{line}.parquet' if CONF else f'salidas/s1/base_{line}.parquet'))
b1 = b1[(b1.split == 'train') & b1.year.isin(years)].reset_index(drop=True)
X1, cols1, _ = features(b1, line, 's1')
X1['exposure_model'] = b1.exposure_model.values
b2 = pd.read_parquet(R / (f'salidas/conf/base_s2_{line}.parquet' if CONF else f'salidas/s2/base_{line}.parquet'))
X2, cols2, _ = features(b2, line, 's2')
X2['exposure_model'] = b2.exposure_model.values
DROPC = sys.argv[sys.argv.index('--sin') + 1].split(',') if '--sin' in sys.argv else []
CLIMA = ['wind_exposure_index', 'wildfire_index', 'rainfall_mm', 'mean_temperature_c']
if '--clima' in sys.argv:  # CLIMA-N1: clima persistente de la dirección medido en los otros años (nunca el del propio año)
    def otros(base, mk, yrs):
        w = pd.read_parquet(R / f'datos/{mk}/weather.parquet'); w = w[w.year.isin(yrs)]
        pv = w.pivot_table(index='address_id', columns='year', values=CLIMA)
        out = pd.DataFrame(index=base.index)
        for v in CLIMA:
            vals = np.stack([base.address_id.map(pv[(v, y)]).values for y in yrs], axis=1)
            mask = np.stack([(base.year.values != y) for y in yrs], axis=1)
            out[f'cl_{v}'] = np.nansum(np.where(mask, vals, 0), axis=1) / np.maximum((mask & ~np.isnan(vals)).sum(axis=1), 1)
        return out
    c1x = otros(b1, 's1', [2021, 2022, 2023, 2024]); c2x = otros(b2, 's2', [2023, 2024])
    for c in c1x: X1[c] = c1x[c].values; X2[c] = c2x[c].values
if '--f1' in sys.argv:  # F1-N1: historia propia de la póliza conocida al inicio (reportada antes de period_start)
    def hist(base, mk):
        a = pd.read_parquet(R / f'salidas/{mk}/asignacion.parquet'); a = a[(a.line == line) & a.accepted][['policy_id', 'report_date']]
        a['rd'] = pd.to_datetime(a.report_date)
        bb = base[['row_id', 'policy_id', 'period_start']].copy(); bb['ps'] = pd.to_datetime(bb.period_start)
        mm = bb.merge(a, on='policy_id'); mm = mm[mm.rd < mm.ps]
        n = bb.row_id.map(mm.groupby('row_id').size()).fillna(0).values
        first = bb.groupby('policy_id').ps.min()
        yrs = ((bb.ps - bb.policy_id.map(first)).dt.days / 365.25).values
        return n, yrs
    n1, y1 = hist(b1, 's1'); n2, y2 = hist(b2, 's2')
    X1['f1_siniestros_previos'] = n1; X1['f1_anios_observados'] = y1
    X2['f1_siniestros_previos'] = n2; X2['f1_anios_observados'] = y2
cols = [c for c in X1.columns if c in X2.columns and c not in DROPC]
# categorías alineadas: las del desarrollado; las emergentes nuevas pasan a 'missing'
for c in cols:
    if str(X1[c].dtype) == 'category':
        cats = list(X1[c].cat.categories)
        v = X2[c].astype('string')
        X2[c] = pd.Categorical(v.where(v.isin(cats), 'missing'), categories=cats)
        unseen = int((~v.isin(cats)).sum())
        if unseen: print('categoría no vista', c, unseen)
y = b1.t0.values; off = np.log(b1.eef_coste.values)
if '--cap' in sys.argv:  # ROB-C1: relatividades con siniestros topados a CAP; el exceso lo absorbe el nivel emergente por balance
    CAP = float(sys.argv[sys.argv.index('--cap') + 1])
    asg = pd.read_parquet(R / 'salidas/s1/asignacion.parquet'); asg = asg[(asg.category == 'unica') & asg.accepted & (asg.line == line)]
    full_t0 = b1.row_id.map(asg.groupby('row_id').net.sum()).fillna(0.0).values
    assert np.allclose(full_t0, y, atol=0.01), 'el T0 de la base no cuadra con la asignación'
    y = b1.row_id.map(asg.assign(c=np.minimum(asg.net, CAP)).groupby('row_id').c.sum()).fillna(0.0).values
base = np.log(y.sum() / np.exp(off).sum())
p = dict(objective='tweedie', tweedie_variance_power=1.5, learning_rate=LR, num_leaves=31, min_data_in_leaf=1000,
         feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0, min_data_per_group=1000,
         cat_smooth=50, verbose=-1, seed=SEED, deterministic=True, force_row_wise=True, num_threads=2)
if '--reg' in sys.argv:  # ROB-C2: árboles más pequeños y más regularizados
    p.update(num_leaves=15, min_data_in_leaf=3000, lambda_l2=50.0, feature_fraction=0.7)
if '--params' in sys.argv:  # ROB-C4: rejilla elegida solo con la validación interna del desarrollado
    p.update(json.loads(sys.argv[sys.argv.index('--params') + 1]))
inner = M.fold_of(b1.customer_id.values, salt='noche_dev', k=5) == 0
dtr = lgb.Dataset(X1.loc[~inner, cols], y[~inner], init_score=off[~inner] + base, free_raw_data=False)
dva = lgb.Dataset(X1.loc[inner, cols], y[inner], init_score=off[inner] + base, reference=dtr)
bst = lgb.train(p, dtr, num_boost_round=4000, valid_sets=[dva], callbacks=[lgb.early_stopping(PAT, verbose=False)])
nb = bst.best_iteration
full = lgb.train(p, lgb.Dataset(X1[cols], y, init_score=off + base), num_boost_round=nb)
full.save_model(str(O / f'gbm_{tag}_{line}.txt'))
g1 = full.predict(X1[cols], raw_score=True); g2 = full.predict(X2[cols], raw_score=True)
c1 = np.log(np.average(np.exp(g1), weights=b1.exposure_model))  # centrar a relatividad media 1 en el desarrollado
pd.DataFrame({'row_id': b2.row_id, 'lg_dev': g2 - c1}).to_parquet(O / f'lg_{tag}_{line}_em.parquet', index=False)
pd.DataFrame({'row_id': b1.row_id, 'lg_dev': g1 - c1}).to_parquet(O / f'lg_{tag}_{line}_dev.parquet', index=False)
imp = pd.Series(full.feature_importance('gain'), index=cols).sort_values(ascending=False)
info = {'line': line, 'years': years, 'rows': int(len(b1)), 'rounds': int(nb), 'valid_d2_inner': None,
        'seconds': round(time.time() - t, 1), 'importance_gain_share': (imp / imp.sum()).round(4).head(25).to_dict(),
        'cols': cols}
mu_va = np.exp(bst.predict(X1.loc[inner, cols], raw_score=True) + off[inner] + base)
info['valid_d2_inner'] = round(float(M.d2(y[inner], mu_va)), 5)
json.dump(info, open(O / f'info_{tag}_{line}.json', 'w'), indent=1)
print(json.dumps(info, indent=1))
'''


### Embedded source: `noche/codigo/variables.py`

Historical source: `src/semana2/lineas/validacion_nocturna_20261023/codigo/variables.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/variables.py'] = '''"""Variables F0 por ramo, para los dos mercados, con el mismo esquema común.

Excluidas por contrato (decididas antes de medir): renewal_decision (solo en entrenamiento), customer_service_contacts
(mide el siniestro del propio periodo), written_premium y discount_applied (tarifa vigente, F4, solo ablación),
identificadores, surname, importes de siniestros, clima del propio año.
Normalizaciones: ocupación a minúsculas sin espacios sobrantes ni prefijo «self-employed » (marca aparte);
códigos de construcción emergentes a la nomenclatura del desarrollado; capital de hogar con la regla CAP-N1;
edad del desarrollado a las bandas emergentes (17-24, 25-34, 35-49, 50-64, 65+).
"""
import pathlib
import numpy as np, pandas as pd

R = pathlib.Path('__RUN_ROOT__/noche')
# CON-N1: códigos locales emergentes y alias t/w del desarrollado a una sola nomenclatura. Evidencia: proporciones
# brick 70 %, t+w 12,6 %, non_standard 9,9 %, stone 8,0 % (desarrollado) frente a BRK 70 %, WOD 12,0 %, MIX 10,1 %, STN 7,9 %.
CONS = {'BRK': 'brick', 'STN': 'stone', 'WOD': 'timber', 'MIX': 'non_standard', 'T': 'timber', 'W': 'timber'}
CAP_THR = {'buildings': 5_000_000, 'contents': 2_000_000}  # dentro de los huecos medidos (894k-60M y 283k-7,6M)
BANDS = [(17, 24, '17-24'), (25, 34, '25-34'), (35, 49, '35-49'), (50, 64, '50-64'), (65, 200, '65+')]

POL_COMMON = ['start_month', 'voluntary_excess', 'compulsory_excess', 'channel', 'payment_method', 'missed_payments_12m',
              'new_business', 'renewal_count', 'days_quote_to_inception']
# VAR-N1: fuera previous_insurer, branch_id, email_domain, marketing_consent, document_preference y la marca
# self_employed: sin señal de frecuencia en el desarrollado (chi2/gl entre 0,15 y 2,1 con 1,9 millones de filas; ver bitácora).
CUST = ['occupation_n', 'marital_status', 'household_size', 'prior_claims_5y', 'licence_points',
        'years_at_address', 'cust_tenure', 'has_campaign', 'marketing_segment']
ADDR = ['region', 'urban_density', 'flood_zone', 'distance_to_river_km', 'elevation_m', 'soil_type', 'prop_age',
        'property_type', 'roof_type', 'basement', 'trees_within_5m', 'listed_building', 'bedrooms', 'energy_rating',
        'solar_panels']
LINE = {'motor': ['age_band', 'years_licensed', 'ncb_years', 'vehicle_value_group', 'vehicle_age', 'make_model',
                  'engine_size_cc', 'body_type', 'transmission', 'seats', 'vehicle_colour', 'fuel_type',
                  'driver_assist', 'cover_type', 'overnight_parking', 'vehicle_use', 'named_young_driver'],
        'buildings': ['age_band', 'log_si', 'construction_type', 'occupancy', 'days_unoccupied', 'smoke_alarms',
                      'accidental_damage_cover', 'mortgage'],
        'contents': ['age_band', 'log_si', 'tenure', 'security_level', 'high_value_items', 'accidental_damage_cover',
                     'works_from_home', 'lodgers', 'bicycle_cover']}
CATEG = {'channel', 'payment_method', 'occupation_n', 'marital_status', 'email_domain', 'document_preference',
         'previous_insurer', 'branch_id', 'marketing_segment', 'region', 'urban_density', 'soil_type', 'property_type',
         'roof_type', 'energy_rating', 'age_band', 'make_model', 'body_type', 'transmission', 'vehicle_colour',
         'fuel_type', 'cover_type', 'overnight_parking', 'vehicle_use', 'construction_type', 'occupancy',
         'smoke_alarms', 'tenure', 'security_level'}
WEATHER = ['hail_days', 'wind_exposure_index', 'freezing_days', 'dry_summer_index', 'wildfire_index', 'rainfall_mm',
           'mean_temperature_c']


def band(age):
    out = pd.Series(pd.NA, index=age.index, dtype='object')
    for lo, hi, lab in BANDS:
        out[(age >= lo) & (age <= hi)] = lab
    return out


def features(base, line, market):
    """base: tabla de filas (de s2_base o equivalente) con year y exposure_model. Devuelve (X, cols)."""
    d = pd.read_parquet(R / f'datos/{market}/customers.parquet')
    a = pd.read_parquet(R / f'datos/{market}/addresses.parquet')
    occ = d.occupation.astype('string').str.strip().str.lower()
    d['self_employed'] = occ.str.startswith('self-employed').fillna(False)
    d['occupation_n'] = occ.str.replace(r'^self-employed\\s+', '', regex=True).str.replace(r'\\s+', ' ', regex=True)
    d['has_campaign'] = d.campaign_code.notna()
    # ID01 (D32): en autos del desarrollado el customer_id perdió los ceros iniciales; se rellena a ocho posiciones solo para cruzar.
    # Error propio corregido el 23-10 a las 21:25: la primera versión de la noche no lo hacía y 98.110 + 25.554 filas de autos
    # del desarrollado quedaban sin variables de cliente.
    key = base.customer_id.astype(str) if __import__('os').environ.get('A1_PRE_ID01') else base.customer_id.astype(str).str.zfill(8)
    df = base.assign(_cid=key.values).merge(d.drop(columns=['surname'], errors='ignore').rename(columns={'customer_id': '_cid'}),
                                            on='_cid', how='left', validate='m:1').drop(columns=['_cid'])
    assert __import__('os').environ.get('A1_PRE_ID01') or df.customer_since_year.notna().all(), 'cruce de clientes incompleto'
    df = df.merge(a, on='address_id', how='left', validate='m:1')
    df['cust_tenure'] = df.year - df.customer_since_year
    df['start_month'] = pd.to_datetime(df.period_start).dt.month  # NB-N2: el calendario separa parte del efecto del nuevo negocio
    df['prop_age'] = df.year - df.year_built
    if line == 'motor':
        df['age_band'] = df['driver_age_band'] if 'driver_age_band' in df else band(df['driver_age'])
    else:
        df['age_band'] = df['policyholder_age_band'] if 'policyholder_age_band' in df else band(df['policyholder_age'])
        si = df['sum_insured'].astype(float)
        scale = si > CAP_THR[line]
        df['cap_scaled'] = scale
        df['log_si'] = np.log(np.where(scale, si / 1000.0, si))
    if line == 'buildings':
        df['construction_type'] = df.construction_type.astype('string').str.strip()
        df['construction_type'] = df.construction_type.map(lambda v: CONS.get(v.upper(), v.lower()) if isinstance(v, str) else v)
    if 'smoke_alarms' in df:
        df['smoke_alarms'] = df.smoke_alarms.map({True: 'yes', False: 'no'}).fillna('missing')
    cols = POL_COMMON + CUST + ADDR + LINE[line]
    if market == 's1':
        cols = [c for c in cols if c != 'region']
    X = df[cols].copy()
    for c in cols:
        if c in CATEG:
            X[c] = X[c].astype('string').fillna('missing').astype('category')
        elif X[c].dtype == bool or str(X[c].dtype) == 'boolean':
            X[c] = X[c].astype(float)
        else:
            X[c] = pd.to_numeric(X[c], errors='coerce').astype(float)
    return X, cols, df


def weather_other_year(df, market, vars_, mode):
    """mode='base' nada; 'otro': clima de la dirección en el otro año de entrenamiento (2023↔2024; 2025 = media 23-24)."""
    w = pd.read_parquet(R / f'datos/{market}/weather.parquet')
    w = w[w.year.isin([2023, 2024])].pivot_table(index='address_id', columns='year', values=vars_)
    out = pd.DataFrame(index=df.index)
    for v in vars_:
        a23 = df.address_id.map(w[(v, 2023)]); a24 = df.address_id.map(w[(v, 2024)])
        out[f'w_{v}'] = np.where(df.year == 2023, a24, np.where(df.year == 2024, a23, (a23 + a24) / 2))
    return out
'''


### Embedded source: `noche/codigo/marco.py`

Historical source: `src/semana2/lineas/validacion_nocturna_20261023/codigo/marco.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/marco.py'] = '''"""Marco común de la escalera de semana 2: particiones, métricas, bootstrap por componentes y modelos estándar.

Particiones propias (fijadas antes de medir, independientes de las de Codex): fold = int(sha256('noche_v1|'+customer_id)[:16],16) % 5.
Métrica: D² Tweedie p=1,5 sobre importes por fila, global (tres ramos concatenados) y por ramo; nulo = media de y del conjunto.
En desarrollo y confirmación el objetivo es T0 y la predicción es la de T0: mu = tasa último * e_ef_coste.
"""
import hashlib, json, pathlib, sys, time
import numpy as np, pandas as pd

R = pathlib.Path('__RUN_ROOT__/noche')
sys.path.insert(0, str(R / 'codigo'))
P = 1.5


def fold_of(ids, salt='noche_v1', k=5):
    return np.array([int(hashlib.sha256(f'{salt}|{x}'.encode()).hexdigest()[:16], 16) % k for x in ids])


def dev(y, mu, p=P):
    y = np.asarray(y, float); mu = np.asarray(mu, float)
    return 2 * (np.power(y, 2 - p) / ((1 - p) * (2 - p)) - y * np.power(mu, 1 - p) / (1 - p) + np.power(mu, 2 - p) / (2 - p))


def d2(y, mu):
    y = np.asarray(y, float)
    return 1 - dev(y, mu).sum() / dev(y, np.full_like(y, y.mean())).sum()


def components(cust, addr):
    """Componentes conexas cliente-dirección (union-find)."""
    keys = pd.Index(pd.unique(np.concatenate([('c' + pd.Series(cust).astype(str)).values, ('a' + pd.Series(addr).astype(str)).values])))
    par = np.arange(len(keys))
    def find(i):
        while par[i] != i:
            par[i] = par[par[i]]; i = par[i]
        return i
    ci = keys.get_indexer('c' + pd.Series(cust).astype(str)); ai = keys.get_indexer('a' + pd.Series(addr).astype(str))
    for x, z in zip(ci, ai):
        rx, rz = find(x), find(z)
        if rx != rz: par[rx] = rz
    roots = np.array([find(i) for i in ci])
    return pd.factorize(roots)[0]


def boot_d2_diff(y, mus, comp, n=10000, seed=2026102301, batch=250):
    """Bootstrap de Poisson por componente. mus: dict nombre->mu. Devuelve dict de arrays de D² por réplica."""
    y = np.asarray(y, float); K = comp.max() + 1
    agg = lambda v: np.bincount(comp, weights=v, minlength=K)
    Sy = agg(y); Sy2 = agg(np.power(y, 2 - P)); Sn = agg(np.ones_like(y))
    Sd = {k: agg(dev(y, m)) for k, m in mus.items()}
    rng = np.random.default_rng(seed); out = {k: [] for k in mus}
    for s in range(0, n, batch):
        W = rng.poisson(1.0, size=(min(batch, n - s), K)).astype(float)
        ny = W @ Sy; nn = W @ Sn; c = ny / nn; ny2 = W @ Sy2
        null = 2 * (ny2 / ((1 - P) * (2 - P)) - ny * np.power(c, 1 - P) / (1 - P) + nn * np.power(c, 2 - P) / (2 - P))
        for k in mus:
            out[k].append(1 - (W @ Sd[k]) / null)
    return {k: np.concatenate(v) for k, v in out.items()}


# ---------------------------------------------------------------- modelos estándar

def design_glm(Xtr, Xte, cols, categ, nbins=6, min_share=0.005, w=None):
    """Numéricas a cuantiles (bordes del entrenamiento) más 'missing'; categorías raras (<0,5 % de filas) a 'otros'."""
    A = {}; B = {}
    for c in cols:
        if c in categ or Xtr[c].nunique(dropna=True) <= 12:  # binarias y discretas cortas como categorías
            s_tr = Xtr[c].astype('string').fillna('missing'); s_te = Xte[c].astype('string').fillna('missing')
            share = s_tr.value_counts(normalize=True)
            keep = set(share[share >= min_share].index)
            A[c] = s_tr.where(s_tr.isin(keep), 'otros'); B[c] = s_te.where(s_te.isin(keep), 'otros')
        else:
            v = Xtr[c].astype(float)
            qs = np.unique(np.nanquantile(v, np.linspace(0, 1, nbins + 1)[1:-1])) if v.notna().any() else np.array([])
            def cut(x):
                x = x.astype(float)
                lab = np.searchsorted(qs, x.values, side='right').astype(str)
                lab = np.where(np.isnan(x.values), 'missing', lab)
                return pd.Series(lab, index=x.index)
            A[c] = cut(Xtr[c]); B[c] = cut(Xte[c])
    A = pd.DataFrame(A); B = pd.DataFrame(B)
    for c in cols:
        cats = sorted(set(A[c]) | {'otros'})
        A[c] = pd.Categorical(A[c], categories=cats); B[c] = pd.Categorical(B[c].where(B[c].isin(cats), 'otros'), categories=cats)
    return A, B


def fit_glm_tweedie(Xtr, ytr, off_tr, Xte, off_te, alpha):
    from glum import GeneralizedLinearRegressor, TweedieDistribution
    m = GeneralizedLinearRegressor(family=TweedieDistribution(P), link='log', alpha=alpha, l1_ratio=0.0,
                                   fit_intercept=True, drop_first=False, max_iter=200)
    m.fit(Xtr, ytr, offset=off_tr)
    return m.predict(Xte, offset=off_te), m


def fit_glm_fs(Xtr, ntr, t0tr, off_tr, Xte, off_te, alpha_f, alpha_s):
    from glum import GeneralizedLinearRegressor
    f = GeneralizedLinearRegressor(family='poisson', link='log', alpha=alpha_f, l1_ratio=0.0, drop_first=False, max_iter=200)
    f.fit(Xtr, ntr, offset=off_tr)
    pos = (ntr > 0) & (t0tr > 0)
    s = GeneralizedLinearRegressor(family='gamma', link='log', alpha=alpha_s, l1_ratio=0.0, drop_first=False, max_iter=200)
    s.fit(Xtr[pos], (t0tr[pos] / ntr[pos]), sample_weight=ntr[pos])
    return f.predict(Xte, offset=off_te) * s.predict(Xte), (f, s)


def fit_gbm_tweedie(Xtr, ytr, off_tr, Xte, off_te, ids_tr, params=None, seed=7):
    import lightgbm as lgb
    p = dict(objective='tweedie', tweedie_variance_power=P, learning_rate=0.02, num_leaves=7, min_data_in_leaf=400,
             feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0, min_data_per_group=500,
             cat_smooth=50, max_cat_to_onehot=8, verbose=-1, seed=seed, deterministic=True, force_row_wise=True,
             num_threads=2)
    if params: p.update(params)
    base = np.log(ytr.sum() / np.exp(off_tr).sum())  # punto de partida: tasa media del entrenamiento
    off_tr = off_tr + base; off_te = off_te + base
    inner = fold_of(ids_tr, salt='noche_inner', k=5) == 0
    dtr = lgb.Dataset(Xtr[~inner], ytr[~inner], init_score=off_tr[~inner], free_raw_data=False)
    dva = lgb.Dataset(Xtr[inner], ytr[inner], init_score=off_tr[inner], reference=dtr)
    b = lgb.train(p, dtr, num_boost_round=3000, valid_sets=[dva], callbacks=[lgb.early_stopping(150, verbose=False)])
    nb = b.best_iteration
    full = lgb.train(p, lgb.Dataset(Xtr, ytr, init_score=off_tr), num_boost_round=max(nb, 1))
    return np.exp(full.predict(Xte, raw_score=True) + off_te), (full, nb)
'''


### Embedded source: `ml/codigo/fsp_light.py`

Historical source: `src/semana2/lineas/mision017_20261025/exp26_20261026/ml2/codigo/fsp_light.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/fsp_light.py'] = '''"""Frecuencia (Poisson) y severidad (Gamma) por peligro entrenadas en el desarrollado 2021-2023 (esquema común T2b),
predichas para todas las filas emergentes (2023, 2024, 2025; clima hasta2023). Versión ligera (lr 0,08).
Salida: trabajo/pred/fsp/{linea}.parquet con row_id, f_{p} (siniestros esperados con desarrollo completo) y s_{p}."""
import sys, time, json
import numpy as np, pandas as pd, lightgbm as lgb
from comun import *
from variables import *
from completitud import por_peligro
linea = sys.argv[1]
HIST = ["hist_pol_12m", "hist_cli_12m"]
cols = comunes(linea)
t0 = time.time()
em = cargar("emerging", linea, "hasta2023")
dva = cargar("developed", linea, "hasta2023")
dv = dva[dva.year.isin([2021, 2022, 2023])].reset_index(drop=True); del dva
cn = pd.read_parquet(TRABAJO / f"conteos_developed_{linea}.parquet")
dv = dv.merge(cn, on="row_id", how="left")
cats = categorias_union([dv, em], cols + HIST + ["new_business", "renewal_count", "prior_claims_5y"])
Xd = matriz(dv, cols, cats); Xe = matriz(em, cols, cats); val = dv.fold.values == 0
perils = sorted(c[2:] for c in cn.columns if c.startswith("n_"))
PF = dict(objective="poisson", learning_rate=0.08, num_leaves=31, min_data_in_leaf=1000, feature_fraction=0.8, bagging_fraction=0.8,
          bagging_freq=1, lambda_l2=10.0, max_cat_to_onehot=8, cat_smooth=50, verbose=-1, num_threads=2, seed=20261026)
PS = dict(objective="gamma", learning_rate=0.05, num_leaves=15, min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8,
          bagging_freq=1, lambda_l2=10.0, max_cat_to_onehot=8, cat_smooth=50, verbose=-1, num_threads=2, seed=20261026)
def fit(X, y, off, p, val, w=None, rounds=2000):
    dtr = lgb.Dataset(X[~val], y[~val], init_score=off[~val], weight=None if w is None else w[~val], free_raw_data=False)
    dvl = lgb.Dataset(X[val], y[val], init_score=off[val], weight=None if w is None else w[val], reference=dtr)
    b = lgb.train(p, dtr, rounds, valid_sets=[dvl], callbacks=[lgb.early_stopping(50, verbose=False)])
    n = max(b.best_iteration, 1); b = lgb.train(p, lgb.Dataset(X, y, init_score=off, weight=w), n); return b, n
out = pd.DataFrame({"row_id": em.row_id.values}); meta = {}
for p in perils:
    n = dv[f"n_{p}"].values.astype(float); yp = dv[f"y_{p}"].values
    cp = np.maximum(por_peligro(linea, p, dv.period_start, dv.period_end), 1e-4)
    off = np.log(dv.exposure.values * cp); b0 = float(np.log(n.sum() / np.exp(off).sum())); off = off + b0
    bf, nf = fit(Xd, n, off, PF, val)
    (TRABAJO / "modelos/fsp_dev").mkdir(parents=True, exist_ok=True)
    bf.save_model(str(TRABAJO / "modelos/fsp_dev" / f"f_{linea}_{p}.txt"))
    s = (n > 0) & (yp > 0); msev = float(yp[s].sum() / n[s].sum())
    bs, ns = fit(Xd[s].reset_index(drop=True), yp[s] / n[s], np.full(s.sum(), np.log(msev)), PS, val[s], w=n[s])
    bs.save_model(str(TRABAJO / "modelos/fsp_dev" / f"s_{linea}_{p}.txt"))
    guardar_json({"b0": b0, "msev": msev, "nf": nf, "ns": ns}, TRABAJO / "modelos/fsp_dev" / f"meta_{linea}_{p}.json")
    out[f"f_{p}"] = np.exp(bf.predict(Xe, raw_score=True) + b0) * em.exposure.values
    out[f"s_{p}"] = np.exp(bs.predict(Xe, raw_score=True) + np.log(msev))
    out[f"cp_{p}"] = por_peligro(linea, p, em.period_start, em.period_end)
    meta[p] = {"b0": b0, "rondas_f": nf, "rondas_s": ns, "n_dev": int(n.sum()), "sev_media": msev}
    print(linea, p, meta[p], round(time.time() - t0), flush=True)
(TRABAJO / "pred" / "fsp").mkdir(parents=True, exist_ok=True)
out.to_parquet(TRABAJO / "pred" / "fsp" / f"{linea}.parquet", index=False)
json.dump(meta, open(TRABAJO / "pred" / "fsp" / f"{linea}_meta.json", "w"), indent=1)
print("fin", linea, round(time.time() - t0))
'''


### Embedded source: `ml/codigo/fsp_light_prod.py`

Historical source: `src/semana2/lineas/mision017_20261025/exp26_20261026/ml2/codigo/fsp_light_prod.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/fsp_light_prod.py'] = '''"""Frecuencia (Poisson) y severidad (Gamma) por peligro entrenadas en el desarrollado 2021-2023 (esquema común T2b),
predichas para todas las filas emergentes (2023, 2024, 2025; clima hasta2023). Versión ligera (lr 0,08).
Salida: trabajo/pred/fsp/{linea}.parquet con row_id, f_{p} (siniestros esperados con desarrollo completo) y s_{p}."""
import sys, time, json
import numpy as np, pandas as pd, lightgbm as lgb
from comun import *
from variables import *
from completitud import por_peligro
linea = sys.argv[1]
VENT, ANIOS, SUFX = "hasta2024", [2021, 2022, 2023, 2024], "_prod"
HIST = ["hist_pol_12m", "hist_cli_12m"]
cols = comunes(linea)
t0 = time.time()
em = cargar("emerging", linea, VENT)
dva = cargar("developed", linea, VENT)
dv = dva[dva.year.isin(ANIOS)].reset_index(drop=True); del dva
cn = pd.read_parquet(TRABAJO / f"conteos_developed_{linea}.parquet")
dv = dv.merge(cn, on="row_id", how="left")
cats = categorias_union([dv, em], cols + HIST + ["new_business", "renewal_count", "prior_claims_5y"])
Xd = matriz(dv, cols, cats); Xe = matriz(em, cols, cats); val = dv.fold.values == 0
perils = sorted(c[2:] for c in cn.columns if c.startswith("n_"))
PF = dict(objective="poisson", learning_rate=0.08, num_leaves=31, min_data_in_leaf=1000, feature_fraction=0.8, bagging_fraction=0.8,
          bagging_freq=1, lambda_l2=10.0, max_cat_to_onehot=8, cat_smooth=50, verbose=-1, num_threads=2, seed=20261026)
PS = dict(objective="gamma", learning_rate=0.05, num_leaves=15, min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8,
          bagging_freq=1, lambda_l2=10.0, max_cat_to_onehot=8, cat_smooth=50, verbose=-1, num_threads=2, seed=20261026)
def fit(X, y, off, p, val, w=None, rounds=2000):
    dtr = lgb.Dataset(X[~val], y[~val], init_score=off[~val], weight=None if w is None else w[~val], free_raw_data=False)
    dvl = lgb.Dataset(X[val], y[val], init_score=off[val], weight=None if w is None else w[val], reference=dtr)
    b = lgb.train(p, dtr, rounds, valid_sets=[dvl], callbacks=[lgb.early_stopping(50, verbose=False)])
    n = max(b.best_iteration, 1); b = lgb.train(p, lgb.Dataset(X, y, init_score=off, weight=w), n); return b, n
out = pd.DataFrame({"row_id": em.row_id.values}); meta = {}
for p in perils:
    n = dv[f"n_{p}"].values.astype(float); yp = dv[f"y_{p}"].values
    cp = np.maximum(por_peligro(linea, p, dv.period_start, dv.period_end), 1e-4)
    off = np.log(dv.exposure.values * cp); b0 = float(np.log(n.sum() / np.exp(off).sum())); off = off + b0
    bf, nf = fit(Xd, n, off, PF, val)
    (TRABAJO / "modelos/fsp_prod").mkdir(parents=True, exist_ok=True)
    bf.save_model(str(TRABAJO / "modelos/fsp_prod" / f"f_{linea}_{p}.txt"))
    s = (n > 0) & (yp > 0); msev = float(yp[s].sum() / n[s].sum())
    bs, ns = fit(Xd[s].reset_index(drop=True), yp[s] / n[s], np.full(s.sum(), np.log(msev)), PS, val[s], w=n[s])
    bs.save_model(str(TRABAJO / "modelos/fsp_prod" / f"s_{linea}_{p}.txt"))
    guardar_json({"b0": b0, "msev": msev, "nf": nf, "ns": ns}, TRABAJO / "modelos/fsp_prod" / f"meta_{linea}_{p}.json")
    out[f"f_{p}"] = np.exp(bf.predict(Xe, raw_score=True) + b0) * em.exposure.values
    out[f"s_{p}"] = np.exp(bs.predict(Xe, raw_score=True) + np.log(msev))
    out[f"cp_{p}"] = por_peligro(linea, p, em.period_start, em.period_end)
    meta[p] = {"b0": b0, "rondas_f": nf, "rondas_s": ns, "n_dev": int(n.sum()), "sev_media": msev}
    print(linea, p, meta[p], round(time.time() - t0), flush=True)
(TRABAJO / "pred" / "fsp").mkdir(parents=True, exist_ok=True)
out.to_parquet(TRABAJO / "pred" / "fsp" / f"{linea}{SUFX}.parquet", index=False)
json.dump(meta, open(TRABAJO / "pred" / "fsp" / f"{linea}{SUFX}_meta.json", "w"), indent=1)
print("fin", linea, round(time.time() - t0))
'''


### Embedded source: `m017/forma_local.py`

Historical source: `src/semana2/lineas/mision017_20261025/codigo/forma_local.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/forma_local.py'] = '''import numpy as np, pandas as pd, json
P = 1.5
def dev(y, m):
    return 2 * (np.maximum(y, 0) ** (2 - P) / ((1 - P) * (2 - P)) - y * m ** (1 - P) / (1 - P) + m ** (2 - P) / (2 - P))
def kopt(y, m):  # escalar Tweedie 1,5 óptimo
    return np.sum(y / np.sqrt(m)) / np.sum(np.sqrt(m))
b = pd.read_parquet("base_etiquetas.parquet").set_index("row_id")
def cargar(conj, mod):
    d = pd.concat([pd.read_parquet(f"oof/pred/{conj}/{mod}_{l}.parquet") for l in ["motor", "buildings", "contents"]]).set_index("row_id")
    t = b.loc[d.index].copy(); t["m"] = d.mu.values * t.c_fila.values; return t
p24 = pd.read_parquet("in/predicciones_2024.parquet").set_index("row_id")
sets = {"DEV23_T2b": cargar("dev", "T2b"), "HOLD24_T2b": cargar("hold", "T2b")}
for c in ["A1", "R0"]:
    t = b.loc[p24.index].copy(); t["m"] = p24[c].values; sets[f"HOLD24_{c}"] = t
for k, t in sets.items():
    print(k, len(t), "y", round(t.y.sum()), "m", round(t.m.sum()), "ratio", round(t.y.sum() / t.m.sum(), 3))
A = np.round(np.arange(0.6, 1.41, 0.05), 2)
res = {}
for k, t in sets.items():
    t = t.copy(); t["cel"] = t.line + "_" + t.new_business.astype(int).astype(str)
    y = t.y.values; null = dev(y, np.full(len(y), y.mean())).sum(); res[k] = {}
    base_dev = 0.0; best_dev = 0.0
    for c, g in t.groupby("cel"):
        yy = g.y.values; e = g.exposure.values * g.c_fila.values; r = g.m.values / e; gm = np.sum(g.m.values) / np.sum(e)
        devs = []
        for a in A:
            m = e * gm * (r / gm) ** a; m = m * kopt(yy, m); devs.append(dev(yy, m).sum())
        devs = np.array(devs); i1 = int(np.where(A == 1.0)[0][0]); ib = int(devs.argmin())
        base_dev += devs[i1]; best_dev += devs[ib]
        res[k][c] = {"n": len(g), "positivos": int((yy > 0).sum()), "a_opt": float(A[ib]), "ganancia_D2": round(float((devs[i1] - devs[ib]) / null), 5)}
    res[k]["_total_ganancia_D2"] = round(float((base_dev - best_dev) / null), 5)
print(json.dumps(res, indent=1))
'''


### Embedded source: `m017/forma_local3.py`

Historical source: `src/semana2/lineas/mision017_20261025/codigo/forma_local3.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/forma_local3.py'] = '''import numpy as np, pandas as pd, json
exec(open("forma_local.py").read().split("A = np.round")[0].replace('print(k, len(t)', '0 and print(k, len(t)'))
te = b[b.conjunto == "test"].copy(); te["cel"] = te.line + "_" + te.new_business.astype(int).astype(str); n25 = te.cel.value_counts(normalize=True)
# mezcla proxy 2024: media de A1 y T2b, cada uno llevado al mismo total por ramo
h = sets["HOLD24_T2b"].copy(); a1 = sets["HOLD24_A1"].m.reindex(h.index)
tot = h.groupby("line").m.transform("sum") / a1.groupby(h.line).transform("sum")
h["m"] = 0.5 * h.m + 0.5 * a1 * tot; sets["HOLD24_mezcla"] = h
def aplicar(t, acel):
    t = t.copy(); t["cel"] = t.line + "_" + t.new_business.astype(int).astype(str); out = t.m.values.copy()
    for c, a in acel.items():
        i = (t.cel == c).values; e = t.exposure.values[i] * t.c_fila.values[i]; r = t.m.values[i] / e; gm = t.m.values[i].sum() / e.sum()
        mm = e * gm * (r / gm) ** a; out[i] = mm * t.m.values[i].sum() / mm.sum()
    return out
def d2w(t, m, w):
    y = t.y.values; return 1 - (dev(y, m) * w).sum() / (dev(y, np.full(len(y), y.mean())) * w).sum()
REGLA = {"motor_1": 0.5, "motor_0": 0.85}
out = {}
for k, t in sets.items():
    if k == "HOLD24_R0": continue
    cel = t.line + "_" + t.new_business.astype(int).astype(str); n = cel.value_counts(normalize=True); w = cel.map(n25 / n).values
    # nivel óptimo por celda en ambos (solo forma)
    def nivel(m):
        m = m.copy()
        for c in cel.unique():
            i = (cel == c).values; m[i] = m[i] * kopt(t.y.values[i], m[i])
        return m
    m0 = nivel(t.m.values); m1 = nivel(aplicar(t, REGLA))
    out[k] = {"D2_base_comp2025": round(d2w(t, m0, w), 5), "D2_forma_comp2025": round(d2w(t, m1, w), 5),
              "ganancia_comp2025": round(d2w(t, m1, w) - d2w(t, m0, w), 5),
              "ganancia_comp_propia": round(d2w(t, m1, np.ones(len(t))) - d2w(t, m0, np.ones(len(t))), 5)}
print(json.dumps(out, indent=1))
# mecanismo: A/E por NCB en autos, nuevo negocio frente a renovación (2023+2024, T2b)
pm = pd.concat([pd.read_parquet(f"datos/emerging/policies_motor.parquet", columns=["row_id", "ncb_years", "years_licensed", "driver_age_band"])]).set_index("row_id")
t = pd.concat([sets["DEV23_T2b"], sets["HOLD24_T2b"]]); t = t[t.line == "motor"].join(pm)
t["ncb"] = pd.cut(t.ncb_years, [-1, 0, 2, 5, 9, 99], labels=["0", "1-2", "3-5", "6-9", "10+"])
t["dec"] = t.groupby("new_business").m.transform(lambda s: pd.qcut(s / t.loc[s.index, "exposure"], 5, labels=False))
for nb in (False, True):
    g = t[t.new_business == nb]; k = g.y.sum() / g.m.sum()
    print("NB" if nb else "REN", "A/E por quintil de tasa predicha:", (g.groupby("dec").y.sum() / g.groupby("dec").m.sum() / k).round(2).tolist())
    print("   A/E por NCB:", (g.groupby("ncb", observed=True).y.sum() / g.groupby("ncb", observed=True).m.sum() / k).round(2).to_dict(), g.groupby("ncb", observed=True).size().to_dict())
'''


### Embedded source: `m017/forma_v2.py`

Historical source: `src/semana2/lineas/mision017_20261025/codigo/forma_v2.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/forma_v2.py'] = '''"""Forma en autos con el escalar Tweedie correcto y la composición de 2025 por celda × banda de exposición."""
import numpy as np, pandas as pd, json
exec(open("forma_local3.py").read().split("REGLA =")[0])
def banda(e): return np.where(e < 0.5, "c", np.where(e < 0.9, "m", "l"))
te = b[b.conjunto == "test"]; k25 = (te.line + "_" + te.new_business.astype(int).astype(str) + "_" + banda(te.exposure.values)).value_counts(normalize=True)
def pesos(t):
    k = t.line + "_" + t.new_business.astype(int).astype(str) + "_" + banda(t.exposure.values); n = k.value_counts(normalize=True)
    return k.map(k25 / n).fillna(0).values
A = np.round(np.arange(0.2, 1.31, 0.05), 2)
def nivel_celda(t, m):
    cel = (t.line + "_" + t.new_business.astype(int).astype(str)).values; m = m.copy()
    for c in np.unique(cel): i = cel == c; m[i] *= kopt(t.y.values[i], m[i])
    return m
def d2w(t, m, w):
    y = t.y.values; mu0 = np.average(y, weights=w); return 1 - (dev(y, m) * w).sum() / (dev(y, np.full(len(y), mu0)) * w).sum()
out = {}
for k in ["DEV23_T2b", "HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]:
    t = sets[k]; w = pesos(t); r = {}
    for c in ["motor_1", "motor_0"]:
        cel = (t.line + "_" + t.new_business.astype(int).astype(str)).values; i = cel == c; s = t[i]; ww = w[i]
        yy = s.y.values; e = (s.exposure * s.c_fila).values; rr = s.m.values / e; gm = s.m.values.sum() / e.sum()
        def curva(mask, wt):
            return np.array([(dev(yy[mask], (lambda mm: mm * kopt(yy[mask], mm))(e[mask] * gm * (rr[mask] / gm) ** a)) * wt[mask]).sum() for a in A])
        todo = curva(np.ones(len(s), bool), np.ones(len(s))); largo = curva(s.exposure.values >= 0.9, np.ones(len(s))); rep = curva(np.ones(len(s), bool), ww)
        r[c] = {"a_opt_todas": float(A[todo.argmin()]), "a_opt_expo>=0.9": float(A[largo.argmin()]), "a_opt_reponderado2025": float(A[rep.argmin()]),
                "positivos_expo>=0.9": int(((yy > 0) & (s.exposure.values >= 0.9)).sum())}
    base = d2w(t, nivel_celda(t, t.m.values), w)
    for a1, a0 in ((0.5, 0.85), (0.6, 0.9), (0.7, 0.9), (0.75, 0.9), (0.8, 0.9), (0.85, 0.9)):
        r[f"ganancia_reponderada_a{a1}_{a0}"] = round(d2w(t, nivel_celda(t, aplicar(t, {"motor_1": a1, "motor_0": a0})), w) - base, 5)
    out[k] = r
print(json.dumps(out, indent=1)); json.dump(out, open("forma_v2.json", "w"), indent=1)
'''


### Embedded source: `m017/eval_v2.py`

Historical source: `src/semana2/lineas/mision017_20261025/codigo/eval_v2.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/eval_v2.py'] = '''import numpy as np, pandas as pd, json
exec(open("forma_v2.py").read().split("out = {}")[0])
exec(open("ncb_local.py").read().split("def evalua")[0].split("exec(")[0] if False else "")
pm = pd.read_parquet("datos/emerging/policies_motor.parquet", columns=["row_id", "ncb_years"]).set_index("row_id")
tm = pd.read_parquet("datos/emerging/test_motor.parquet", columns=["row_id", "ncb_years"]).set_index("row_id")
ncb = pd.concat([pm, tm]).ncb_years
cu = pd.read_parquet("datos/emerging/customers.parquet").set_index("customer_id")
def bnd(v): return pd.cut(v, [-1, 0, 2, 5, 9, 99], labels=["0", "1-2", "3-5", "6-9", "10+"]).astype(str)
def celda(t): return (t.line + "_" + t.new_business.astype(int).astype(str)).values
def fac(t, clave, k=300):
    c = celda(t); f = {}
    for cc in np.unique(c):
        i = c == cc; base = t.y.values[i].sum() / t.m.values[i].sum(); kk = clave(t)[i]
        for v in np.unique(kk):
            j = kk == v; n = (t.y.values[i][j] > 0).sum(); ae = t.y.values[i][j].sum() / t.m.values[i][j].sum() / base
            f[(cc, v)] = 1 + n / (n + k) * (ae - 1)
    return f
def aplica(t, f, clave, m=None):
    m = t.m.values.copy() if m is None else m.copy(); c = celda(t); kk = clave(t); m0 = m.copy()
    m = m * np.array([f.get((a, v), 1.0) for a, v in zip(c, kk)])
    for cc in np.unique(c): i = c == cc; m[i] *= m0[i].sum() / m[i].sum()
    return m
k_ncb = lambda t: np.where(t.line.values == "motor", bnd(ncb.reindex(t.index).fillna(0).values), "x")
k_pri = lambda t: np.minimum(cu.prior_claims_5y.reindex(t.customer_id.values).fillna(0).values, 2).astype(int).astype(str)
res = {}
for tr_k, te_k in (("DEV23_T2b", "HOLD24_T2b"), ("DEV23_T2b", "HOLD24_A1"), ("DEV23_T2b", "HOLD24_mezcla"), ("HOLD24_T2b", "DEV23_T2b")):
    tr, te = sets[tr_k], sets[te_k]; w = pesos(te)
    f1 = fac(tr, k_ncb); f2 = fac(tr, k_pri)
    m_n = aplica(te, f1, k_ncb); t2 = te.copy(); t2["m"] = m_n; m_np = aplica(t2, f2, k_pri)
    base = d2w(te, nivel_celda(te, te.m.values), w); r = {"base": round(base, 5)}
    r["ncb"] = round(d2w(te, nivel_celda(te, m_n), w) - base, 5)
    r["ncb+prior"] = round(d2w(te, nivel_celda(te, m_np), w) - base, 5)
    t3 = te.copy(); t3["m"] = m_np
    for a1 in (0.6, 0.7, 0.8, 0.9, 1.0):
        r[f"ncb+prior+forma({a1},0.85)"] = round(d2w(te, nivel_celda(te, aplicar(t3, {"motor_1": a1, "motor_0": 0.85})), w) - base, 5)
    res[f"{tr_k}->{te_k}"] = r
print(json.dumps(res, indent=1)); json.dump(res, open("eval_v2.json", "w"), indent=1)
'''


### Embedded source: `m017/f3_eval.py`

Historical source: `src/semana2/lineas/mision017_20261025/codigo/f3_eval.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/f3_eval.py'] = '''"""F3: factores residuales de variables propias del emergente o excluidas de T2b (género, aseguradora previa, ocupación,
segmento de marketing...), sobre la transformación de F2. Fuera de tiempo, composición 2025 por celda × banda."""
import numpy as np, pandas as pd, json, sys
exec(open("eval_v2.py").read().split("res = {}")[0])
DE = "datos/emerging"
pol = pd.concat([pd.concat([pd.read_parquet(f"{DE}/policies_{l}.parquet"), pd.read_parquet(f"{DE}/test_{l}.parquet")]) for l in ["motor","buildings","contents"]], ignore_index=True).set_index("row_id")
ad = pd.read_parquet(f"{DE}/addresses.parquet").set_index("address_id")
def col(t, c):
    return np.array([str(x) for x in _col(t, c)], dtype=object)
def _col(t, c):
    if c in cu.columns: return cu[c].reindex(t.customer_id.values).astype(str).values
    if c in pol.columns: return pol[c].reindex(t.index).astype(str).values
    if c in ad.columns: return ad[c].reindex(pol.address_id.reindex(t.index).values).astype(str).values
    if c == "occ": return cu.occupation.reindex(t.customer_id.values).astype(str).str.lower().str.replace("self-employed ", "", regex=False).str.strip().values
    raise KeyError(c)
FORMA = {"motor_0": 0.9}
def f2_fit(pool):
    f1 = fac(pool, k_ncb); t2 = pool.copy(); t2["m"] = aplica(pool, f1, k_ncb); f2 = fac(t2, k_pri); return f1, f2
def f2_apply(t, f1, f2):
    m = aplica(t, f1, k_ncb); tt = t.copy(); tt["m"] = m; m = aplica(tt, f2, k_pri); tt["m"] = m
    return aplicar(tt, FORMA)
def fac_linea(t, clave, lines, k=300):
    """factor por línea (agrupa NB y REN), relativo a la base de cada celda"""
    c = celda(t); kk = clave(t); f = {}
    rel = t.m.values.copy()
    for cc in np.unique(c):
        i = c == cc; rel[i] = rel[i] * t.y.values[i].sum() / t.m.values[i].sum()
    for l in lines:
        i = t.line.values == l
        for v in np.unique(kk[i]):
            j = i & (kk == v); n = (t.y.values[j] > 0).sum(); ae = t.y.values[j].sum() / rel[j].sum()
            f[(l, v)] = 1 + n / (n + k) * (ae - 1)
    return f
def aplica_linea(t, f, clave, m):
    c = celda(t); kk = clave(t); m0 = m.copy()
    m = m * np.array([f.get((l, v), 1.0) for l, v in zip(t.line.values, kk)])
    for cc in np.unique(c): i = c == cc; m[i] *= m0[i].sum() / m[i].sum()
    return m
K = 300
CANDS = {"gender": ["motor", "contents"], "gender_b": ["buildings"], "previous_insurer": ["motor"], "occ": ["motor"],
         "marketing_segment": ["contents"], "licence_points": ["buildings"], "policyholder_age_band": ["contents"],
         "marketing_consent": ["contents"], "payment_method": ["buildings"], "region": ["motor"], "campaign_code": ["contents", "buildings"]}
def clave_de(v):
    vv = "gender" if v == "gender_b" else v
    return lambda t: col(t, vv)
res = {}
for tr_k, te_k in (("DEV23_T2b", "HOLD24_T2b"), ("DEV23_T2b", "HOLD24_A1"), ("DEV23_T2b", "HOLD24_mezcla"), ("HOLD24_T2b", "DEV23_T2b")):
    tr, te = sets[tr_k].copy(), sets[te_k].copy(); w = pesos(te)
    f1, f2 = f2_fit(tr); tr["m"] = f2_apply(tr, f1, f2); te["m"] = f2_apply(te, f1, f2)
    base = d2w(te, nivel_celda(te, te.m.values), w); r = {"base_F2": round(base, 5)}
    for v, lines in CANDS.items():
        cl = clave_de(v); f = fac_linea(tr, cl, lines, K); m = aplica_linea(te, f, cl, te.m.values)
        r[v] = round(d2w(te, nivel_celda(te, m), w) - base, 5)
    # secuencial: gender -> previous_insurer -> occ -> marketing_segment
    seq = ["gender", "previous_insurer", "occ", "marketing_segment"]
    trs, m_te = tr.copy(), te.m.values.copy()
    for v in seq:
        cl = clave_de(v); f = fac_linea(trs, cl, CANDS[v], K); trs["m"] = aplica_linea(trs, f, cl, trs.m.values); m_te = aplica_linea(te, f, cl, m_te)
        r["seq+" + v] = round(d2w(te, nivel_celda(te, m_te), w) - base, 5)
    res[f"{tr_k}->{te_k}"] = r; print(f"{tr_k}->{te_k}", r, flush=True)
json.dump(res, open("exp26/f3_eval.json", "w"), indent=1)
'''


### Embedded source: `m017/construir_F2.py`

Historical source: `src/semana2/lineas/mision017_20261025/codigo/construir_F2.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/construir_F2.py'] = '''"""F2: F (mezcla × 1,1877) con factor NCB en autos y factor de siniestros previos declarados (prior_claims_5y, 0/1/2+) en las seis celdas,
ambos por tipo de negocio, estimados con los OOF de T2b de 2023 y 2024 juntos (credibilidad n/(n+300)), total de celda conservado;
forma 0,9 en renovación de autos; razón de nivel Tweedie (escalar correcto) medida en cuatro series y aplicada por celda."""
import numpy as np, pandas as pd, json, hashlib
exec(open("eval_v2.py").read().split("res = {}")[0])
FORMA = {"motor_0": 0.9}
pool = pd.concat([sets["DEV23_T2b"], sets["HOLD24_T2b"]]); f1 = fac(pool, k_ncb)
t2 = pool.copy(); t2["m"] = aplica(pool, f1, k_ncb); f2 = fac(t2, k_pri)
def transformar(t):
    m = aplica(t, f1, k_ncb); tt = t.copy(); tt["m"] = m; m = aplica(tt, f2, k_pri); tt["m"] = m
    return aplicar(tt, FORMA)
cels = ["motor_0", "motor_1", "buildings_0", "buildings_1", "contents_0", "contents_1"]
raz = {}
for k in ["DEV23_T2b", "HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]:
    t = sets[k]; m2 = transformar(t); c = celda(t)
    raz[k] = {cc: kopt(t.y.values[c == cc], m2[c == cc]) / kopt(t.y.values[c == cc], t.m.values[c == cc]) for cc in cels}
R = {cc: float(np.mean([raz[k][cc] for k in raz])) for cc in cels}
F = pd.read_csv("in/final_sonda_global_2025.csv", dtype={"row_id": str}); pass
t = b.loc[F.row_id].copy(); t["m"] = F.prediction.values; t["c_fila"] = 1.0
m2 = transformar(t); c = celda(t)
for cc, r in R.items(): m2[c == cc] *= r
out = F.copy(); out["prediction"] = m2
assert len(out) == 209063 and np.isfinite(out.prediction).all() and (out.prediction > 0).all() and (out.row_id.values == F.row_id.values).all()
p = "F2_ncb_previos_2025.csv"; out.to_csv(p, index=False, float_format="%.6f", lineterminator="\\n")
info = {"archivo": p, "sha256": hashlib.sha256(open(p, "rb").read()).hexdigest(), "base": "F 42659dbb…", "forma": FORMA,
        "factores_ncb": {f"{a}|{v}": round(x, 4) for (a, v), x in f1.items() if a.startswith("motor")},
        "factores_previos": {f"{a}|{v}": round(x, 4) for (a, v), x in f2.items()},
        "razon_nivel": {k: round(v, 4) for k, v in R.items()}, "razon_por_serie": {k: {c_: round(v, 4) for c_, v in d.items()} for k, d in raz.items()},
        "total": round(float(m2.sum()), 2), "total_F": round(float(F.prediction.sum()), 2),
        "total_celda": {cc: round(float(m2[c == cc].sum()), 2) for cc in cels}, "total_F_celda": {cc: round(float(F.prediction.values[c == cc].sum()), 2) for cc in cels},
        "evidencia_fuera_de_tiempo": "eval_v2.json: NCB+previos de 2023 aplicados en 2024 con la composición 2025 por celda y banda de exposición: +0,0029 a +0,0031 (T2b, A1, mezcla); de 2024 en 2023: +0,0093"}
json.dump(info, open("F2_info.json", "w"), indent=1, ensure_ascii=False); print(json.dumps(info, indent=1, ensure_ascii=False))
'''


### Embedded source: `m017/construir_final.py`

Historical source: `src/semana2/lineas/mision017_20261025/codigo/construir_final.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/construir_final.py'] = '''"""Construye el final de la misión 0,17 a partir de las puntuaciones públicas (recibos por SHA).
Uso: python construir_final.py recibos.json [--prueba]
- Nivel por celda: sonda por celda sobre F (álgebra de resolver_celda.py, paradas preinscritas).
- Forma por celda (motor_1, buildings_1, contents_1): parábola en a por (a_bajo, a_alto y 1,0 = F2); a_final = 1 + Z (a* − 1) si es cóncava
  y a* ∈ [0,3; 1,8]; si no, 1. Z de sigma_forma.json (τ 0,25).
- Final = F2 con la forma de cada celda (razón de nivel Tweedie local) × factor de nivel de la celda relativo a F (f_s / g).
Escribe final_creible_2025.csv (selección) y final_pleno_2025.csv (información), y final_info.json."""
import sys, json, hashlib
import numpy as np, pandas as pd
exec(open("construir_F2.py").read().split("F = pd.read_csv")[0])
pre = json.load(open("in/preinscripcion.json")); B = pre["base"]; CEL = pre["celdas"]; DED = pre["celda_deducida"]
SF = json.load(open("sondas_F2_info.json")); SIG = json.load(open("sigma_forma.json"))
K, LO, HI = 1.2, 0.75, 1.33
nombre_sha = {B["sha256"]: ("F", None)}; nombre_sha.update({v["sha256"]: ("celda", v["celda"]) for v in pre["sondas"].values()})
nombre_sha[SF["base"]["sha256"]] = ("F2", None); nombre_sha.update({v["sha256"]: ("forma", (v["celda"], v["a"])) for v in SF["sondas"].values()})
def resolver(sc):
    bA, bB, g, cg = B["beta_A"], B["beta_B"], B["g"], B["c_optimo_global"]; dF = sc["F"]
    b_s = {s: bB * CEL[s]["cuota_raiz_mu"] for s in CEL}; a_s = {}
    for s in CEL:
        if s != DED: a_s[s] = (dF - sc[("celda", s)] - b_s[s] * ((g * K) ** 0.5 - g ** 0.5)) / ((g * K) ** -0.5 - g ** -0.5)
    a_s[DED] = bA - sum(a_s.values())
    assert all(v > 0 for v in a_s.values()), ("a_s no positivo", a_s)
    cs = {s: a_s[s] / b_s[s] for s in CEL}
    grande = [s for s in CEL if abs(np.log(cs[s] / cg)) > max(0.7, 3 * CEL[s]["sigma"])]
    cred = {s: float(np.clip((cs[s] / cg) ** CEL[s]["Z"], LO, HI)) * g for s in CEL}
    pleno = {s: float(np.clip(cs[s], g * LO, g * HI)) for s in CEL}
    prev = lambda f: dF - sum(a_s[s] * (f[s] ** -0.5 - g ** -0.5) + b_s[s] * (f[s] ** 0.5 - g ** 0.5) for s in CEL)
    return {"c_opt": cs, "rel_global": {s: cs[s] / cg for s in CEL}, "cred": cred, "pleno": pleno, "grande": grande,
            "d2_F_celda_cred": prev(cred), "d2_F_celda_pleno": prev(pleno), "g": g}
def parabola(pts):
    a = np.array([p[0] for p in pts]); d = np.array([p[1] for p in pts]); c2, c1, c0 = np.polyfit(a, d, 2)
    if c2 >= 0: return None, (c2, c1, c0)
    v = -c1 / (2 * c2); return (v if 0.3 <= v <= 1.8 else None), (c2, c1, c0)
def construir(sc, base_df, t, sets_eval=None):
    lv = resolver(sc); out = {"nivel": lv, "forma": {}}
    for cc in ["motor_1", "buildings_1", "contents_1"]:
        pts = [(1.0, sc["F2"])] + [(a, sc[("forma", (cc, a))]) for (c_, a) in [k[1] for k in sc if isinstance(k, tuple) and k[0] == "forma"] if c_ == cc]
        if len(pts) < 3: out["forma"][cc] = {"a_final": 1.0, "nota": "faltan puntos"}; continue
        v, coef = parabola(pts); Z = SIG[cc]["Z_tau0.25"]
        out["forma"][cc] = {"puntos": pts, "vertice": v, "coef": coef, "Z": Z,
                            "a_cred": 1.0 if v is None else 1 + Z * (v - 1), "a_pleno": 1.0 if v is None else v,
                            "d2_vertice": None if v is None else float(np.polyval(coef, v))}
    res = {}
    for modo in ("cred", "pleno"):
        m = base_df.prediction.values.copy(); c = celda(t)
        for cc, fz in out["forma"].items():
            a = fz.get(f"a_{modo}", 1.0)
            if a != 1.0:
                tt = t.copy(); tt["m"] = m; m2 = aplicar(tt, {cc: a}); i = c == cc
                rs = []
                for k_, s_ in ser.items():
                    cs_ = celda(s_); j = cs_ == cc; mm = aplicar(s_, {cc: a}); rs.append(kopt(s_.y.values[j], mm[j]) / kopt(s_.y.values[j], s_.m.values[j]))
                m[i] = m2[i] * float(np.mean(rs)); fz[f"razon_{modo}"] = float(np.mean(rs))
        f = lv[modo]
        for cc in CEL: m[c == cc] *= f[cc] / lv["g"]
        res[modo] = m
    return out, res
ser = {}
for k_ in ["DEV23_T2b", "HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]:
    s_ = sets[k_].copy(); s_["m"] = transformar(s_); ser[k_] = s_
if __name__ == "__main__" and "--prueba" not in sys.argv:
    sc = {}
    for r in json.load(open(sys.argv[1])):
        if r["sha256"] not in nombre_sha: print("AVISO: SHA desconocido, se ignora:", r["sha256"][:12]); continue
        tipo, cual = nombre_sha[r["sha256"]]; sc[tipo if cual is None else (tipo, cual)] = float(r["d2"])
    falta = ({"F", "F2"} | {("celda", s) for s in CEL if s != DED}) - set(sc)
    if falta: sys.exit(f"PARADA: faltan puntuaciones obligatorias {falta}")
    falta_forma = {("forma", (v["celda"], v["a"])) for v in SF["sondas"].values()} - set(sc)
    if falta_forma: print("AVISO: faltan sondas de forma (esa celda queda con a = 1):", falta_forma)
    lo, hi = pre["d2_F_intervalo_redondeo"]
    if sc["F"] + 5e-6 < lo - 1e-9 or sc["F"] - 5e-6 > hi + 1e-9: sys.exit(f"PARADA preinscrita: D2(F) = {sc['F']} fuera de [{lo}, {hi}]")
    pass
    for v in SF["sondas"].values():
        cc_, a_ = v["celda"], v["a"]; rs_ = [kopt(s_.y.values[celda(s_) == cc_], aplicar(s_, {cc_: a_})[celda(s_) == cc_]) / kopt(s_.y.values[celda(s_) == cc_], s_.m.values[celda(s_) == cc_]) for s_ in ser.values()]
        assert abs(np.mean(rs_) - v["razon_nivel"]) < 6e-5, ("razón de forma distinta de la de las sondas", cc_, a_, np.mean(rs_))
    F2 = pd.read_csv("F2_ncb_previos_2025.csv", dtype={"row_id": str}); t = b.loc[F2.row_id].copy(); t["m"] = F2.prediction.values; t["c_fila"] = 1.0
    info, res = construir(sc, F2, t)
    if info["nivel"]["grande"] and "--forzar" not in sys.argv:
        print(json.dumps(info, indent=1, default=str)); sys.exit("PARADA preinscrita de nivel en " + str(info["nivel"]["grande"]))
    for modo, m in res.items():
        out = F2.copy(); out["prediction"] = m; assert len(out) == 209063 and np.isfinite(m).all() and (m > 0).all()
        p = f"final_{'creible' if modo == 'cred' else 'pleno'}_2025.csv"; out.to_csv(p, index=False, float_format="%.6f", lineterminator="\\n")
        info[p] = {"sha256": hashlib.sha256(open(p, "rb").read()).hexdigest(), "total": round(float(m.sum()), 2)}
    json.dump(info, open("final_info.json", "w"), indent=1, default=str, ensure_ascii=False); print(json.dumps(info, indent=1, default=str, ensure_ascii=False))
'''


### Embedded source: `m017/exp26/lib26.py`

Historical source: `src/semana2/lineas/mision017_20261025/exp26_20261026/m017/exp26/lib26.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/exp26/lib26.py'] = '''import numpy as np, pandas as pd, json
exec(open("__RUN_ROOT__/m017/f3_eval.py").read().split("K = 300")[0])
W = pd.read_parquet("__RUN_ROOT__/ml/datos/emerging/weather.parquet").set_index(["address_id", "year"])
def wvar(t, v, year=None):
    a = pol.address_id.reindex(t.index).values; yr = t.year.values.astype(int) if year is None else np.full(len(t), year)
    return W[v].reindex(pd.MultiIndex.from_arrays([a, yr])).values
def d2u(t, m):
    y = t.y.values; return 1 - dev(y, m).sum() / dev(y, np.full(len(y), y.mean())).sum()
def rel_base(t):
    """m llevado al nivel Tweedie-óptimo de su celda"""
    return nivel_celda(t, t.m.values)
def fac_tw(t, clave, lines, k=300, by_nb=False):
    """factor Tweedie-óptimo por nivel, relativo a la celda, con credibilidad n/(n+k) en log"""
    c = celda(t); kk = clave(t); m = rel_base(t); y = t.y.values; f = {}
    for l in lines:
        il = t.line.values == l
        grupos = [("all", il)] if not by_nb else [(nb, il & (t.new_business.values == nb)) for nb in (False, True)]
        for g, i in grupos:
            for v in np.unique(kk[i]):
                j = i & (kk == v); n = (y[j] > 0).sum()
                if n == 0: continue
                kv = np.sum(y[j] / np.sqrt(m[j])) / np.sum(np.sqrt(m[j]))
                f[(l, g, v)] = np.exp(n / (n + k) * np.log(max(kv, 1e-3)))
    return f
def aplica_tw(t, f, clave, m, by_nb=False):
    c = celda(t); kk = clave(t); m0 = m.copy()
    g = t.new_business.values if by_nb else np.array(["all"] * len(t), dtype=object)
    m = m * np.array([f.get((l, gg, v), 1.0) for l, gg, v in zip(t.line.values, g, kk)])
    for cc in np.unique(c): i = c == cc; m[i] *= m0[i].sum() / m[i].sum()
    return m
'''


### Embedded source: `m017/exp26/fsp_blend_eval.py`

Historical source: `src/semana2/lineas/mision017_20261025/exp26_20261026/m017/exp26/fsp_blend_eval.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/exp26/fsp_blend_eval.py'] = '''exec(open("__RUN_ROOT__/m017/exp26/lib26.py").read())
import sys
T = "__RUN_ROOT__/ml/trabajo"
KP = 100.0
def fsp_tables():
    out = {}
    for l in ["motor", "buildings", "contents"]:
        f = pd.read_parquet(f"{T}/pred/fsp/{l}.parquet").set_index("row_id")
        per = [c[2:] for c in f.columns if c.startswith("f_")]
        cn = pd.read_parquet(f"{T}/conteos_emerging_{l}.parquet").set_index("row_id")
        be = pd.read_parquet(f"{T}/base_emerging_{l}.parquet", columns=["row_id", "y", "year", "conjunto"] + [f"y_{p}" for p in per]).set_index("row_id")
        out[l] = (f, per, cn, be)
    return out
FT = fsp_tables()
def kappas(l, years):
    f, per, cn, be = FT[l]; tr = be[(be.conjunto == "train") & be.year.astype(int).isin(years)].index
    ef = {p: (f.loc[tr, f"f_{p}"] * f.loc[tr, f"cp_{p}"]).values for p in per}; es = {p: f.loc[tr, f"s_{p}"].values for p in per}
    single = sum(ef[p] * es[p] for p in per); kg = be.loc[tr, "y"].sum() / single.sum(); kap = {}
    for p in per:
        n = cn.loc[tr, f"n_{p}"].values; kf = n.sum() / ef[p].sum(); ks = be.loc[tr, f"y_{p}"].sum() / max((n * es[p]).sum(), 1e-9)
        z = n.sum() / (n.sum() + KP); kap[p] = float(np.exp(z * np.log(max(kf * ks / kg, 1e-3))))
    return kap
def fsp_pred(idx, l, kap, ultimate=False):
    f, per, cn, be = FT[l]
    return sum(kap[p] * f.loc[idx, f"f_{p}"].values * (1.0 if ultimate else f.loc[idx, f"cp_{p}"].values) * f.loc[idx, f"s_{p}"].values for p in per)
def blend(t, m, w, kap_by_line):
    out = m.copy()
    for l in ["motor", "buildings", "contents"]:
        i = (t.line == l).values
        if w[l] == 0: continue
        r = fsp_pred(t.index[i], l, kap_by_line[l])
        r = r * m[i].sum() / r.sum()
        out[i] = np.exp((1 - w[l]) * np.log(m[i]) + w[l] * np.log(r))
    c = celda(t)
    for cc in np.unique(c): j = c == cc; out[j] *= m[j].sum() / out[j].sum()
    return out
SEQ = [("gender", ["motor", "contents"]), ("works_from_home", ["contents"]), ("marketing_segment", ["contents"])]
res = {}
for tr_k, te_k, tr_y in (("DEV23_T2b", "HOLD24_T2b", [2023]), ("DEV23_T2b", "HOLD24_mezcla", [2023]), ("DEV23_T2b", "HOLD24_A1", [2023]), ("HOLD24_T2b", "DEV23_T2b", [2024])):
    tr, te = sets[tr_k].copy(), sets[te_k].copy(); w25 = pesos(te)
    f1, f2 = f2_fit(tr); tr["m"] = f2_apply(tr, f1, f2); te["m"] = f2_apply(te, f1, f2)
    kap = {l: kappas(l, tr_y) for l in ["motor", "buildings", "contents"]}
    # factores estimados en el año de entrenamiento sobre la base F2
    FAC = []
    trs = tr.copy()
    for v, lines in SEQ:
        cl = (lambda vv: (lambda t: col(t, vv)))(v); f = fac_tw(trs, cl, lines, 300); trs["m"] = aplica_tw(trs, f, cl, trs.m.values); FAC.append((cl, f))
    def facs(t, m):
        for cl, f in FAC: m = aplica_tw(t, f, cl, m)
        return m
    b0 = te.m.values
    r = {}
    for nm, W in (("w0", 0.0), ("w25", 0.25), ("w40", 0.4), ("w50", 0.5), ("w60", 0.6), ("w75", 0.75)):
        for usefac in (False, True):
            m = blend(te, b0, {"motor": W, "buildings": W, "contents": W}, kap)
            if usefac: m = facs(te, m)
            key = nm + ("+fac" if usefac else "")
            mm = nivel_celda(te, m)
            r[key + "|ALLu"] = round(d2u(te, mm) - d2u(te, nivel_celda(te, b0)), 5)
            r[key + "|w2025"] = round(d2w(te, mm, w25) - d2w(te, nivel_celda(te, b0), w25), 5)
            for l in ["motor", "buildings", "contents"]:
                for seg, s in (("REN", ~te.new_business.values.astype(bool)), ("NB", te.new_business.values.astype(bool))):
                    j = (te.line == l).values & s; tt = te[j]
                    r[f"{key}|{l}_{seg}"] = round(d2u(tt, nivel_celda(tt, m[j])) - d2u(tt, nivel_celda(tt, b0[j])), 5)
    res[f"{tr_k[:5]}->{te_k[:9]}"] = r
    print(tr_k, te_k, flush=True)
df = pd.DataFrame(res); pd.set_option("display.width", 250); pd.set_option("display.max_rows", 500)
df.to_csv("__RUN_ROOT__/m017/exp26/fsp_blend_eval.csv")
print(df.to_string())
'''


### Embedded source: `m017/construir_F3b.py`

Historical source: `src/semana2/lineas/mision017_20261025/exp26_20261026/m017/construir_F3b.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/construir_F3b.py'] = '''"""F3b (candidato, sin enviar). Revisión de F3 con las auditorías I y J (26-10):
- base: el archivo F2 publicado (público 0,16033), que ya lleva sus razones de nivel;
- forma de motor_1: a = 1 + Z·(v − 1), v = vértice sin fuga de nivel (exp26/fuga_celdas.json), aplicada con Σ√μ fijo en la celda 2025
  (la misma normalización con la que se corrigió el vértice);
- factores residuales validados por I: género en autos; género, teletrabajo y segmento de marketing en contenidos
  (se retiran aseguradora previa y ocupación: no superan su nulo con el número correcto de niveles);
  Tweedie-óptimos, credibilidad n/(n+K), estimados con los OOF de T2b 2023+2024 después de la transformación de F2, total de celda conservado;
  razón de nivel local de los factores con los dos años pesados por igual;
- nivel creíble por celda (sondas sobre F) relativo a F: × cred_s/g.
Uso: python construir_F3b.py [K] [--sin-factores] [--sin-forma] [--seq=gender,works_from_home,...]"""
import sys, json, hashlib
argv = sys.argv[:]; sys.argv = ["x", "--prueba"]
import numpy as np, pandas as pd
exec(open("exp26/lib26.py").read())
exec(open("construir_final.py").read())
sys.argv = argv
pos = [a for a in sys.argv[1:] if not a.startswith("--")]
KC = float(pos[0]) if pos else 300.0
SIN_F = "--sin-factores" in sys.argv; SIN_A = "--sin-forma" in sys.argv
SEQ_ALL = {"gender": ["motor", "contents"], "works_from_home": ["contents"], "marketing_segment": ["contents"],
           "previous_insurer": ["motor"], "occ": ["motor"]}
seqarg = [a for a in sys.argv if a.startswith("--seq=")]
SEQN = seqarg[0][6:].split(",") if seqarg else ["gender", "works_from_home", "marketing_segment"]
SEQ = [] if SIN_F else [(v, SEQ_ALL[v]) for v in SEQN]
FUGA = json.load(open("exp26/fuga_celdas.json"))
A_M1 = 1.0 if SIN_A else 1 + SIG["motor_1"]["Z_tau0.25"] * (FUGA["motor_1"]["sin_nivel"]["vertice"] - 1)
cels = ["motor_0", "motor_1", "buildings_0", "buildings_1", "contents_0", "contents_1"]
# factores sobre el pool local (transformación F2 + forma local con Σμ fijo)
pool = pd.concat([sets["DEV23_T2b"], sets["HOLD24_T2b"]]).copy()
pt = pool.copy(); pt["m"] = transformar(pool); pool["m"] = aplicar(pt, {"motor_1": A_M1}) if A_M1 != 1.0 else pt.m.values
FAC = []
for v, lines in SEQ:
    cl = (lambda vv: (lambda t: col(t, vv)))(v); f = fac_tw(pool, cl, lines, KC)
    pool["m"] = aplica_tw(pool, f, cl, pool.m.values); FAC.append((v, cl, f))
def factores(t, m):
    for v, cl, f in FAC: m = aplica_tw(t, f, cl, m)
    return m
# razón de nivel local de los factores (antes/después, sobre la base F2 + forma), dos años con igual peso
raz = {}
for k_ in ["DEV23_T2b", "HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]:
    s_ = sets[k_].copy(); tt = s_.copy(); tt["m"] = transformar(s_); m0 = aplicar(tt, {"motor_1": A_M1}) if A_M1 != 1.0 else tt.m.values
    m1 = factores(s_, m0); c_ = celda(s_)
    raz[k_] = {cc: kopt(s_.y.values[c_ == cc], m1[c_ == cc]) / kopt(s_.y.values[c_ == cc], m0[c_ == cc]) for cc in cels}
Rf = {cc: 0.5 * raz["DEV23_T2b"][cc] + 0.5 * np.mean([raz[k][cc] for k in ["HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]]) for cc in cels}
# nivel creíble
sc = {}
for r in json.load(open("recibos_reales_20261026.json")):
    if r["sha256"] in nombre_sha:
        tipo, cual = nombre_sha[r["sha256"]]; sc[tipo if cual is None else (tipo, cual)] = float(r["d2"])
lv = resolver(sc)
F2 = pd.read_csv("F2_ncb_previos_2025.csv", dtype={"row_id": str}); pass
F = pd.read_csv("in/final_sonda_global_2025.csv", dtype={"row_id": str}); assert (F.row_id.values == F2.row_id.values).all()
t = b.loc[F2.row_id].copy(); t["m"] = F2.prediction.values; t["c_fila"] = 1.0; c = celda(t)
m = F2.prediction.values.copy()
if A_M1 != 1.0:
    i = c == "motor_1"; ms = aplicar(t.assign(m=m), {"motor_1": A_M1})
    m[i] = ms[i] * (np.sqrt(m[i]).sum() / np.sqrt(ms[i]).sum()) ** 2     # Σ√μ fijo en la celda
m = factores(t, m)
for cc in cels: m[c == cc] *= Rf[cc] * lv["cred"][cc] / lv["g"]
out = F2.copy(); out["prediction"] = m
assert len(out) == 209063 and np.isfinite(m).all() and (m > 0).all()
tag = "F3b" + ("_sinfac" if SIN_F else "") + ("_sinforma" if SIN_A else "") + (f"_K{int(KC)}" if KC != 300 else "") + ("" if not seqarg else "_" + "-".join(SEQN))
p = f"exp26/{tag}_2025.csv"; out.to_csv(p, index=False, float_format="%.6f", lineterminator="\\n")
info = {"archivo": p, "sha256": hashlib.sha256(open(p, "rb").read()).hexdigest(), "K": KC, "a_motor_1": A_M1, "seq": [v for v, _ in SEQ],
        "factores": {v: {f"{a}|{vv}": round(x, 4) for (a, g_, vv), x in f.items()} for v, cl, f in FAC},
        "razon_nivel_factores": {k: round(v, 4) for k, v in Rf.items()}, "razon_por_serie": {k: {c_: round(v, 4) for c_, v in d.items()} for k, d in raz.items()},
        "nivel_cred_rel_F": {k: round(v / lv["g"], 4) for k, v in lv["cred"].items()},
        "total": round(float(m.sum()), 2), "total_celda": {cc: round(float(m[c == cc].sum()), 2) for cc in cels},
        "total_F2_celda": {cc: round(float(F2.prediction.values[c == cc].sum()), 2) for cc in cels},
        "total_F_celda": {cc: round(float(F.prediction.values[c == cc].sum()), 2) for cc in cels}}
json.dump(info, open(f"exp26/{tag}_info.json", "w"), indent=1, ensure_ascii=False)
print(json.dumps({k: v for k, v in info.items() if k not in ("razon_por_serie",)}, indent=1, ensure_ascii=False))
'''


### Embedded source: `m017/construir_F4b.py`

Historical source: `src/semana2/lineas/mision017_20261025/exp26_20261026/m017/construir_F4b.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/construir_F4b.py'] = '''"""F4 (candidato, sin enviar): F3b + mezcla geométrica con un modelo de frecuencia × severidad por peligro transferido del
desarrollado (FSp ligero, recalibrado por peligro en el emergente con credibilidad n/(n+100)), pesos por ramo
(autos 0,25, edificios 0,25, contenidos 0,5), total de celda conservado.
Orden: F2 publicado → mezcla FSp → forma de motor_1 (Σ√μ fijo) → factores (género; teletrabajo y segmento en contenidos)
→ razón de nivel local (mezcla + factores, años con igual peso) × nivel creíble por celda.
Uso: python construir_F4.py [--w=m,b,c] [--sin-factores]"""
import sys, json, hashlib
argv = sys.argv[:]; sys.argv = ["x", "--prueba"]
import numpy as np, pandas as pd
exec(open("exp26/lib26.py").read())
exec(open("construir_final.py").read())
exec(open("exp26/fsp_blend_eval.py").read().split("SEQ = [")[0].split('exec(open("__RUN_ROOT__/m017/exp26/lib26.py").read())')[1])
sys.argv = argv
wa = [a for a in sys.argv if a.startswith("--w=")]
WM, WB, WC = [float(x) for x in wa[0][4:].split(",")] if wa else (0.25, 0.0, 0.5)
WL = {"motor": WM, "buildings": WB, "contents": WC}
SIN_F = "--sin-factores" in sys.argv
SEQ = [] if SIN_F else [("gender", ["motor", "contents"]), ("works_from_home", ["contents"]), ("marketing_segment", ["contents"])]
FUGA = json.load(open("exp26/fuga_celdas.json"))
A_M1 = 1 + SIG["motor_1"]["Z_tau0.25"] * (FUGA["motor_1"]["sin_nivel"]["vertice"] - 1)
cels = ["motor_0", "motor_1", "buildings_0", "buildings_1", "contents_0", "contents_1"]
# --- factores: estimados en el pool local (F2 + mezcla FSp fuera de tiempo + forma)
KAP_DEV = {2023: {l: kappas(l, [2024]) for l in WL}, 2024: {l: kappas(l, [2023]) for l in WL}}   # κ del otro año
def blend_local(t, m, mbase=None):
    """mezcla geométrica con la FSp conservando el multiplicador de F2 (m / mbase)"""
    mbase = t.m.values if mbase is None else mbase
    out = m.copy(); yr = t.year.values.astype(int)
    for y_ in (2023, 2024):
        for l in WL:
            i = (yr == y_) & (t.line.values == l)
            if not i.any() or WL[l] == 0: continue
            r = fsp_pred(t.index[i], l, KAP_DEV[y_][l]); r = r * mbase[i].sum() / r.sum()
            out[i] = m[i] * (r / mbase[i]) ** WL[l]
    c_ = celda(t)
    for cc in np.unique(c_): j = c_ == cc; out[j] *= m[j].sum() / out[j].sum()
    return out
def forma(t, m):
    tt = t.copy(); tt["m"] = m; return aplicar(tt, {"motor_1": A_M1})
pool = pd.concat([sets["DEV23_T2b"], sets["HOLD24_T2b"]]).copy()
pool["m"] = forma(pool, blend_local(pool, transformar(pool)))
FAC = []
for v, lines in SEQ:
    cl = (lambda vv: (lambda t: col(t, vv)))(v); f = fac_tw(pool, cl, lines, 300)
    pool["m"] = aplica_tw(pool, f, cl, pool.m.values); FAC.append((v, cl, f))
def factores(t, m):
    for v, cl, f in FAC: m = aplica_tw(t, f, cl, m)
    return m
# --- razón de nivel local de mezcla + factores (antes: F2 + forma; después: F2 + mezcla + forma + factores)
raz = {}
for k_ in ["DEV23_T2b", "HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]:
    s_ = sets[k_].copy(); m2 = transformar(s_)
    m0 = forma(s_, m2); m1 = factores(s_, forma(s_, m2)); c_ = celda(s_)   # razón solo de los factores (la de la mezcla es ruido, revisor K)
    raz[k_] = {cc: kopt(s_.y.values[c_ == cc], m1[c_ == cc]) / kopt(s_.y.values[c_ == cc], m0[c_ == cc]) for cc in cels}
R4 = {cc: 0.5 * raz["DEV23_T2b"][cc] + 0.5 * np.mean([raz[k][cc] for k in ["HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]]) for cc in cels}
# --- nivel creíble
sc = {}
for r in json.load(open("recibos_reales_20261026.json")):
    if r["sha256"] in nombre_sha:
        tipo, cual = nombre_sha[r["sha256"]]; sc[tipo if cual is None else (tipo, cual)] = float(r["d2"])
lv = resolver(sc)
# --- producción 2025: FSp de producción (desarrollado 2021-2024), κ con 2023+2024, coste último (cp = 1)
FP = {}
for l in WL:
    f = pd.read_parquet(f"{T}/pred/fsp/{l}_prod.parquet").set_index("row_id"); per = [c[2:] for c in f.columns if c.startswith("f_")]
    FP[l] = (f, per, FT[l][2], FT[l][3])
KAP_P = {}
for l in WL:
    FT_save = FT[l]; FT[l] = FP[l]; KAP_P[l] = kappas(l, [2023, 2024]); FT[l] = FT_save
F2 = pd.read_csv("F2_ncb_previos_2025.csv", dtype={"row_id": str}); pass
t = b.loc[F2.row_id].copy(); t["m"] = F2.prediction.values; t["c_fila"] = 1.0; c = celda(t)
Fb = pd.read_csv("in/final_sonda_global_2025.csv", dtype={"row_id": str}); assert (Fb.row_id.values == F2.row_id.values).all()
m = F2.prediction.values.copy()
for l in WL:
    i = (t.line == l).values
    if WL[l] == 0: continue
    f, per, _, _ = FP[l]; idx = t.index[i]
    r = sum(KAP_P[l][p] * f.loc[idx, f"f_{p}"].values * f.loc[idx, f"s_{p}"].values for p in per)
    mb = Fb.prediction.values[i]; r = r * mb.sum() / r.sum(); m[i] = m[i] * (r / mb) ** WL[l]
for cc in cels: j = c == cc; m[j] *= F2.prediction.values[j].sum() / m[j].sum()
i = c == "motor_1"; ms = forma(t, m); m[i] = ms[i] * (np.sqrt(m[i]).sum() / np.sqrt(ms[i]).sum()) ** 2
m = factores(t, m)
for cc in cels: m[c == cc] *= R4[cc] * lv["cred"][cc] / lv["g"]
out = F2.copy(); out["prediction"] = m
assert len(out) == 209063 and np.isfinite(m).all() and (m > 0).all()
tag = "F4b" + ("_sinfac" if SIN_F else "") + ("" if not wa else "_w" + wa[0][4:].replace(",", "-"))
p = f"exp26/{tag}_2025.csv"; out.to_csv(p, index=False, float_format="%.6f", lineterminator="\\n")
info = {"archivo": p, "sha256": hashlib.sha256(open(p, "rb").read()).hexdigest(), "pesos_fsp": WL, "a_motor_1": A_M1, "seq": [v for v, _ in SEQ],
        "kappa_prod": {l: {p_: round(v, 3) for p_, v in d.items()} for l, d in KAP_P.items()},
        "razon_nivel": {k: round(v, 4) for k, v in R4.items()}, "razon_por_serie": {k: {c_: round(v, 4) for c_, v in d.items()} for k, d in raz.items()},
        "nivel_cred_rel_F": {k: round(v / lv["g"], 4) for k, v in lv["cred"].items()},
        "total": round(float(m.sum()), 2), "total_celda": {cc: round(float(m[c == cc].sum()), 2) for cc in cels},
        "corr_log_F2": {cc: round(float(np.corrcoef(np.log(m[c == cc] / t.exposure.values[c == cc]), np.log(F2.prediction.values[c == cc] / t.exposure.values[c == cc]))[0, 1]), 4) for cc in cels}}
json.dump(info, open(f"exp26/{tag}_info.json", "w"), indent=1, ensure_ascii=False)
print(json.dumps({k: v for k, v in info.items() if k != "razon_por_serie"}, indent=1, ensure_ascii=False))
'''


### Embedded source: `m017/construir_F3c.py`

Historical source: `src/semana2/lineas/mision017_20261025/exp26_20261026_v2/m017/construir_F3c.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/construir_F3c.py'] = '''"""F3c / F4c (candidatos, sin enviar): sobre final_creible (SHA 1a003d86…, público medido 0,16519, enviado por Codex el 26-10)
se aplican solo los pasos nuevos, para que la diferencia pública mida exactamente su aporte:
- F3c = final_creible × factores residuales validados (género en autos y contenidos; teletrabajo y segmento en contenidos), total de celda
  conservado, × razón de nivel local de los factores (años con igual peso). Factores y razones idénticos a F3b.
- F4c = final_creible × mezcla FSp (autos 0,25, contenidos 0,5; conservando el multiplicador de F2: m·(FSp/F)^w) → factores (los de F4b) → razón.
Uso: python construir_F3c.py F3c|F4c"""
import sys, json, hashlib
QUE = sys.argv[1]
import numpy as np, pandas as pd
src = open("construir_F3b.py" if QUE == "F3c" else "construir_F4b.py").read()
src = src.split("# nivel creíble")[0] if QUE == "F3c" else src.split("# --- nivel creíble")[0]
sys.argv = ["x"]
exec(src)
R = Rf if QUE == "F3c" else R4
FC = "revision/J_final_creible_2025.csv"; pass
fc = pd.read_csv(FC, dtype={"row_id": str})
F2 = pd.read_csv("F2_ncb_previos_2025.csv", dtype={"row_id": str}); assert (fc.row_id.values == F2.row_id.values).all()
t = b.loc[fc.row_id].copy(); t["m"] = fc.prediction.values; t["c_fila"] = 1.0; c = celda(t)
m = fc.prediction.values.copy()
if QUE == "F4c":
    Fb = pd.read_csv("in/final_sonda_global_2025.csv", dtype={"row_id": str}); assert (Fb.row_id.values == fc.row_id.values).all()
    FP = {}
    for l in WL:
        f = pd.read_parquet(f"{T}/pred/fsp/{l}_prod.parquet").set_index("row_id"); per = [c_[2:] for c_ in f.columns if c_.startswith("f_")]
        FP[l] = (f, per, FT[l][2], FT[l][3])
    KAP_P = {}
    for l in WL:
        sv = FT[l]; FT[l] = FP[l]; KAP_P[l] = kappas(l, [2023, 2024]); FT[l] = sv
    for l in WL:
        i = (t.line == l).values
        if WL[l] == 0: continue
        f, per, _, _ = FP[l]; idx = t.index[i]
        r = sum(KAP_P[l][p] * f.loc[idx, f"f_{p}"].values * f.loc[idx, f"s_{p}"].values for p in per)
        mb = Fb.prediction.values[i]; r = r * mb.sum() / r.sum(); m[i] = m[i] * (r / mb) ** WL[l]
    for cc in cels: j = c == cc; m[j] *= fc.prediction.values[j].sum() / m[j].sum()
m = factores(t, m)
for cc in cels: m[c == cc] *= R[cc]
out = fc.copy(); out["prediction"] = m
assert len(out) == 209063 and np.isfinite(m).all() and (m > 0).all()
p = f"exp26/{QUE}_2025.csv"; out.to_csv(p, index=False, float_format="%.6f", lineterminator="\\n")
info = {"archivo": p, "sha256": hashlib.sha256(open(p, "rb").read()).hexdigest(), "base": "final_creible 1a003d86… (público 0,16519)",
        "pasos": ["mezcla FSp (autos 0,25, contenidos 0,5)"] * (QUE == "F4c") + ["factores " + ",".join(v for v, _, _ in FAC)],
        "razon_nivel_factores": {k: round(v, 4) for k, v in R.items()}, "total": round(float(m.sum()), 2),
        "total_celda": {cc: round(float(m[c == cc].sum()), 2) for cc in cels},
        "total_celda_base": {cc: round(float(fc.prediction.values[c == cc].sum()), 2) for cc in cels}}
json.dump(info, open(f"exp26/{QUE}_info.json", "w"), indent=1, ensure_ascii=False); print(json.dumps(info, indent=1, ensure_ascii=False))
'''


### Embedded source: `m017/construir_F3d.py`

Historical source: `src/semana2/lineas/mision017_20261025/exp26_20261026_v3/m017/construir_F3d.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/construir_F3d.py'] = '''"""F3d (candidato, sin enviar): como F3c (final_creible × factores residuales) pero con menos contracción en contenidos.
Motivo, fijado antes de construir: fuera de tiempo, en contenidos la ganancia crece al bajar K (K30 +0,0078/+0,0085,
K60 +0,0076/+0,0083, K300 +0,0046/+0,0048 en D² del ramo; exp26/k_eval.py); en autos K casi no importa (Z≈0,9).
F3c midió 0,16914 (+0,00395 sobre final_creible, más de lo previsto), coherente con factores más fuertes.
Pasos: género en autos con K=300 (igual que F3c); género, teletrabajo y segmento en contenidos con K=60; razón de nivel
local de los factores con los dos años pesados por igual. Uso: python construir_F3d.py [Kcont]"""
import sys, json, hashlib
KCONT = float(sys.argv[1]) if len(sys.argv) > 1 else 60.0
import numpy as np, pandas as pd
sys.argv = ["construir_F3b.py", "--sin-factores"]
exec(open("construir_F3b.py").read().split("# nivel creíble")[0])
SEQK = [("gender", ["motor"], 300.0), ("gender", ["contents"], KCONT), ("works_from_home", ["contents"], KCONT), ("marketing_segment", ["contents"], KCONT)]
FAC = []
for v, lines, k in SEQK:
    cl = (lambda vv: (lambda t: col(t, vv)))(v); f = fac_tw(pool, cl, lines, k)
    pool["m"] = aplica_tw(pool, f, cl, pool.m.values); FAC.append((v, cl, f))
def factores(t, m):
    for v, cl, f in FAC: m = aplica_tw(t, f, cl, m)
    return m
raz = {}
for k_ in ["DEV23_T2b", "HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]:
    s_ = sets[k_].copy(); tt = s_.copy(); tt["m"] = transformar(s_); m0 = aplicar(tt, {"motor_1": A_M1}); m1 = factores(s_, m0); c_ = celda(s_)
    raz[k_] = {cc: kopt(s_.y.values[c_ == cc], m1[c_ == cc]) / kopt(s_.y.values[c_ == cc], m0[c_ == cc]) for cc in cels}
R = {cc: 0.5 * raz["DEV23_T2b"][cc] + 0.5 * np.mean([raz[k][cc] for k in ["HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]]) for cc in cels}
FC = "revision/J_final_creible_2025.csv"; pass
fc = pd.read_csv(FC, dtype={"row_id": str}); t = b.loc[fc.row_id].copy(); t["m"] = fc.prediction.values; t["c_fila"] = 1.0; c = celda(t)
m = factores(t, fc.prediction.values.copy())
for cc in cels: m[c == cc] *= R[cc]
out = fc.copy(); out["prediction"] = m; assert len(out) == 209063 and np.isfinite(m).all() and (m > 0).all()
tag = "F3d" + ("" if KCONT == 60 else f"_Kc{int(KCONT)}")
p = f"exp26/{tag}_2025.csv"; out.to_csv(p, index=False, float_format="%.6f", lineterminator="\\n")
f3c = pd.read_csv("exp26/F3c_2025.csv", dtype={"row_id": str})
info = {"archivo": p, "sha256": hashlib.sha256(open(p, "rb").read()).hexdigest(), "K_contenidos": KCONT, "K_autos": 300,
        "factores": {f"{v}|{','.join(sorted(set(a for (a, g_, vv) in f)))}": {f"{a}|{vv}": round(x, 4) for (a, g_, vv), x in f.items()} for v, cl, f in FAC},
        "razon_nivel": {k: round(v, 4) for k, v in R.items()}, "total": round(float(m.sum()), 2),
        "total_celda": {cc: round(float(m[c == cc].sum()), 2) for cc in cels},
        "corr_log_F3c_contenidos": round(float(np.corrcoef(np.log(m[t.line.values == "contents"]), np.log(f3c.prediction.values[t.line.values == "contents"]))[0, 1]), 4),
        "igual_F3c_fuera_de_contenidos": bool(np.allclose(m[t.line.values != "contents"], f3c.prediction.values[t.line.values != "contents"], rtol=2e-6))}
json.dump(info, open(f"exp26/{tag}_info.json", "w"), indent=1, ensure_ascii=False); print(json.dumps(info, indent=1, ensure_ascii=False))
'''


### Embedded source: `aux/reproduccion_cadena.py`

Historical source: `src/semana2/reproduccion_cadena.py`. This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['aux/reproduccion_cadena.py'] = '''"""Pasos de enlace para la reproducción histórica; sin búsquedas ni puntuación.

Las fórmulas se transcriben de las recetas originales de Claude. Las dos
restauraciones documentadas (curva global y cruce anterior a ID01) se contrastan
con salidas históricas fuera del proceso de cálculo.
"""
from __future__ import annotations
import json
import os
import shutil
import sys
from pathlib import Path
import numpy as np
import pandas as pd

WORK = Path(os.environ['RETRAIN_WORK'])
N = WORK / 'noche'
M = WORK / 'ml'
F = WORK / 'm017'
LINES = ('motor', 'buildings', 'contents')


def dump(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\\n', encoding='utf-8')


def curve():
    # Conf_insumos original, retirando únicamente la restricción de ocurrencia
    # <=2023 para la curva global descrita en REP-N2 y la bitácora.
    import reporte as RP
    claims = pd.concat([RP.load('s1'), RP.load('s2')], ignore_index=True)
    tmax = int(claims.window.max())
    curves = {}
    for peril, g in claims.groupby('peril'):
        weights = np.minimum(g.net.values, g.net.quantile(.995))
        db = (g.delay.values // RP.B).astype(int)
        wb = (g.window.values // RP.B).astype(int)
        nb = tmax // RP.B + 1
        d = np.bincount(db, weights=weights, minlength=nb)[:nb]
        start = np.bincount(db, weights=weights, minlength=nb + 1)[:nb]
        end = np.bincount(np.minimum(wb, nb - 1) + 1, weights=weights, minlength=nb + 1)[:nb]
        risk = np.cumsum(start) - np.cumsum(end)
        h = np.where(risk > 0, d / np.maximum(risk, 1e-9), 0)
        survival = np.cumprod((1 - h)[::-1])[::-1]
        curves[peril] = np.concatenate([survival[1:], [1.0]]).tolist()
    dump(N / 'salidas/reporte_F.json', {'bucket_days': RP.B, 'F_coste': curves,
         'provenance': 'Restauración Codex de REP-N2 con la expresión de conf_insumos de Claude, sin filtro anual.'})


def ml_c1():
    import receta
    params = receta.parametros('pcv', ['T2b'], 'ramo')
    t, values = receta.aplicar('prod', params)
    sample = pd.read_parquet(M / 'datos/emerging/sample_submission.parquet')
    out = sample[['row_id']].merge(pd.DataFrame({'row_id': t.row_id.values, 'prediction': values}), on='row_id', how='left')
    assert len(out) == 209063 and out.prediction.notna().all() and out.row_id.is_unique
    out.to_csv(M / 'privado/prediccion_2025_mlind.csv', index=False, float_format='%.6f', lineterminator='\\n')
    dump(M / 'privado/params_C1.json', params)


def tag(line, seed, history=False):
    if history:
        suffix = '_v2fixf1' if line == 'motor' else '_v2f1'
        return 'dev21222324' + suffix + (f's{seed}' if seed != 11 else '')
    suffix = f'_v2fixs{seed}' if line == 'motor' else ('_m02' if seed == 11 else f'_v2s{seed}')
    return 'dev21222324' + suffix


def a1_post():
    sample = pd.read_parquet(N / 'datos/s2/sample_submission.parquet').row_id
    transfer = N / 'salidas/transfer'
    def lg(line, history=False):
        return pd.concat([pd.read_parquet(transfer / f'lg_{tag(line, seed, history)}_{line}_em.parquet')
                          .set_index('row_id').lg_dev for seed in (11, 12, 13)], axis=1).mean(axis=1)
    def production(history):
        parts = []
        for line in LINES:
            b = pd.read_parquet(N / f'salidas/s2/base_{line}.parquet')
            g = np.exp(b.row_id.map(lg(line, history)).values)
            train = (b.split == 'train').values
            test = (b.split == 'test').values
            level = b.t0.values[train].sum() / (b.eef_coste.values[train] * g[train]).sum()
            parts.append(pd.DataFrame({'row_id': b.row_id.values[test], 'prediction': b.exposure_model.values[test] * g[test] * level}))
        return pd.concat(parts).set_index('row_id').loc[sample].reset_index()
    # El original lee la referencia sin historia ya redondeada a seis decimales.
    baseline = production(False)
    path = N / 'salidas/correccion/A1fixb3_2025.csv'
    baseline.to_csv(path, index=False, float_format='%.6f', lineterminator='\\n')
    baseline = pd.read_csv(path, dtype={'row_id': str})
    result = production(True)
    prefix = result.row_id.str[0].values
    for key in 'MBC':
        mask = prefix == key
        result.loc[mask, 'prediction'] = result.prediction.values[mask] * baseline.prediction.values[mask].sum() / result.prediction.values[mask].sum()
    result.to_csv(N / 'salidas/correccion/A1fixF1b3n_2025.csv', index=False, float_format='%.6f', lineterminator='\\n')
    # Solo A1 y R0 se consumen en forma_local. No ajustar A2/A3 ajenos a la cadena.
    held = []
    for line in LINES:
        b = pd.read_parquet(N / f'salidas/conf/base_s2_{line}.parquet')
        g = pd.read_parquet(transfer / f'lg_dev212223_v2conf_{line}_em.parquet').set_index('row_id').lg_dev
        risk = np.exp(b.row_id.map(g).values)
        fit = (b.year == 2023).values
        test = (b.year == 2024).values
        level = b.t0.values[fit].sum() / (b.eef_coste.values[fit] * risk[fit]).sum()
        held.append(pd.DataFrame({'row_id': b.row_id.values[test], 'A1': b.eef_coste.values[test] * risk[test] * level,
                     'R0': b.eef_coste.values[test] * b.t0.values[fit].sum() / b.eef_coste.values[fit].sum()}))
    pd.concat(held, ignore_index=True).to_parquet(N / 'salidas/correccion/predicciones_2024.parquet', index=False)


def m017_inputs():
    # Todos los insumos por fila provienen de esta corrida, nunca de las salidas históricas.
    for phase in ('dev', 'hold'):
        for line in LINES:
            target = F / f'oof/pred/{phase}/T2b_{line}.parquet'
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(M / f'trabajo/pred/{phase}/T2b_{line}.parquet', target)
    shutil.copyfile(N / 'salidas/correccion/predicciones_2024.parquet', F / 'in/predicciones_2024.parquet')
    for name in ('customers', 'addresses', 'weather', *(f'policies_{l}' for l in LINES), *(f'test_{l}' for l in LINES)):
        target = F / f'datos/emerging/{name}.parquet'
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(M / f'datos/emerging/{name}.parquet', target)
    columns = ['row_id', 'line', 'conjunto', 'year', 'exposure', 'new_business', 'start_month', 'customer_id', 'y', 'c_fila', 'written_premium']
    labels = pd.concat([pd.read_parquet(M / f'trabajo/base_emerging_{line}.parquet', columns=columns) for line in LINES], ignore_index=True)
    labels.to_parquet(F / 'base_etiquetas.parquet', index=False)
    a = pd.read_csv(N / 'salidas/correccion/A1fixF1b3n_2025.csv', dtype={'row_id': str})
    b = pd.read_csv(M / 'privado/prediccion_2025_mlind.csv', dtype={'row_id': str})
    assert a.row_id.equals(b.row_id)
    blend = a.assign(prediction=.5 * a.prediction.values + .5 * b.prediction.values)
    p = F / 'in/mezcla50.csv'
    blend.to_csv(p, index=False, float_format='%.6f', lineterminator='\\n')
    # Constante pública congelada del recibo original; no nueva sonda ni aprendizaje público.
    blend = pd.read_csv(p, dtype={'row_id': str})
    g = json.loads((F / 'in/constantes_publicas.json').read_text(encoding='utf-8'))['factor_global']
    blend['prediction'] = blend.prediction.values * g
    blend.to_csv(F / 'in/final_sonda_global_2025.csv', index=False, float_format='%.6f', lineterminator='\\n')


def final_alias():
    shutil.copyfile(F / 'final_creible_2025.csv', F / 'revision/J_final_creible_2025.csv')


if __name__ == '__main__':
    {'curve': curve, 'ml_c1': ml_c1, 'a1_post': a1_post, 'm017_inputs': m017_inputs, 'final_alias': final_alias}[sys.argv[1]]()
'''


## Build the isolated workspace

I copy the official inputs into the new run and verify their hashes. The embedded stages run in isolated child processes whose Python audit hook blocks socket connections and child process creation. They can read the run and Python environment; they cannot write to the original inputs. Historical per-row predictions are never used as inputs. Each completed stage has a receipt and log.


In [ ]:
CONFIG = {'m017/in/preinscripcion.json': {'id': 'CLA-S2-SONDA-SEGMENTO-20261025', 'autor': 'Claude (Cowork), sesión de validación nocturna', 'pedido': 'El participante eligió «Final + diseñar sonda por segmento» el 25-10 tras la sonda global.', 'base': {'archivo': 'Repositorio_IAA2026/privado/semana2/claude_validacion_nocturna_20261023/sonda_global/final_sonda_global_2025.csv', 'sha256': '42659dbb4101d4f9a4ab00c287d77b2bd21d42be9734d2ca4a4074eb57c9add2', 'g': 1.187658238648521, 'd2_publico_previsto': 0.15948, 'mezcla': {'sha256': '12831ef4a05c1773824abe5fa139d9503dd2989b4874e7d93cfc000406cfdb4a', 'd2_publico': 0.1539}, 'beta_A': 0.6213270716132047, 'beta_B': 0.5079950966862582, 'c_optimo_global': 1.2230965922038046}, 'celdas': {'contents_0': {'cuota_raiz_mu': 0.03863, 'filas': 15806, 'Z': 0.534, 'sigma': 0.1869, 'sigma_agrupado_simulacion': 0.1859}, 'buildings_0': {'cuota_raiz_mu': 0.067202, 'filas': 12579, 'Z': 0.36, 'sigma': 0.2666, 'sigma_agrupado_simulacion': 0.2646}, 'motor_0': {'cuota_raiz_mu': 0.112814, 'filas': 21698, 'Z': 0.81, 'sigma': 0.0969, 'sigma_agrupado_simulacion': 0.0941}, 'contents_1': {'cuota_raiz_mu': 0.128667, 'filas': 50441, 'Z': 0.736, 'sigma': 0.1198, 'sigma_agrupado_simulacion': 0.1115}, 'buildings_1': {'cuota_raiz_mu': 0.221179, 'filas': 39716, 'Z': 0.724, 'sigma': 0.1236, 'sigma_agrupado_simulacion': 0.1196}, 'motor_1': {'cuota_raiz_mu': 0.431507, 'filas': 68823, 'Z': 0.866, 'sigma': 0.0786, 'sigma_agrupado_simulacion': 0.0666}}, 'd2_F_intervalo_redondeo': [0.1594743, 0.1594849], 'celda_deducida': 'motor_1', 'sondas': {'sonda_seg_contents_0.csv': {'celda': 'contents_0', 'K': 1.2, 'filas_escaladas': 15806, 'sha256': 'fb568cefdd6cf8908b5a53ed37ed211b5a984bfaea65f6d9031eea0cc9183168', 'total': 82241464.19}, 'sonda_seg_buildings_0.csv': {'celda': 'buildings_0', 'K': 1.2, 'filas_escaladas': 12579, 'sha256': '62373a316abbb97b869c9fb315bf257b1e69fa94dcb29ee91a2515ce63efe5fb', 'total': 83110161.16}, 'sonda_seg_motor_0.csv': {'celda': 'motor_0', 'K': 1.2, 'filas_escaladas': 21698, 'sha256': 'f76c3a6c53057008b0fdf242b4cb36312822b1a6721e6100dfaee961bc966d76', 'total': 83762225.51}, 'sonda_seg_contents_1.csv': {'celda': 'contents_1', 'K': 1.2, 'filas_escaladas': 50441, 'sha256': '131716eede0d7a652063b21b86018622853f224026583450170b7d9eda3455a1', 'total': 82952117.21}, 'sonda_seg_buildings_1.csv': {'celda': 'buildings_1', 'K': 1.2, 'filas_escaladas': 39716, 'sha256': '28a61e3a5281bb56bae77836f0edc40ad625f1a104ac6b1b4678ee0d6980de7f', 'total': 85852081.51}}, 'metodo': 'b_s = βB × cuota_s (Σμ^½ de la celda en el test); a_s = [D2(F) − D2(sonda_s) − b_s((gK)^½ − g^½)] / ((gK)^−½ − g^−½) para las cinco celdas sondeadas, y a_deducida = βA − Σ a_s. c*_s = a_s / b_s. Factor final por celda: g × acotar((c*_s / c*_g)^Z_s, 0,75, 1,33). Archivo final = mezcla × factor de su celda.', 'credibilidad': {'tau': 0.2, 'tau_motivo': 'Desviación logarítmica de cada celda frente al nivel global: en 2024, con el nivel de 2023, los ramos se desviaron +0,22, −0,36 y +0,08 del global (RMS 0,25), y el cociente nuevo negocio / renovación osciló entre 0,79 y 1,70 por ramo y año.', 'sigma': 'desviación típica de log(c*_s/c*_g) público menos privado, remuestreo de 2023 y 2024 al tamaño y composición de 2025 (150 réplicas por año)'}, 'orden_de_envio': 'Primero F solo; si su D2 público es compatible con el intervalo de redondeo, después las cinco sondas. Así una parada en F no gasta las sondas.', 'entrada_de_puntuaciones': 'Por nombre de archivo (o por recibo con SHA-256), nunca por nombre de celda, para que una puntuación no se asigne a la celda equivocada.', 'paradas': ['Si el intervalo [D2(F) observado ± 0,000005] no toca el intervalo previsto por el redondeo de las puntuaciones globales, no se resuelve: la métrica no tendría la forma supuesta.', 'Si a_s o b_s ≤ 0 en alguna celda, no se resuelve: el modelo implica a_s > 0, así que un valor no positivo indica un supuesto roto (público no aleatorio u otra métrica).', 'Si |log(c*_s / c*_g)| > max(0,7; 3 σ_s) en alguna celda, el resolutor se detiene sin escribir el archivo final y se consulta al participante.'], 'nota_Z': 'Z usa el mayor σ de los dos años de la simulación (sigma_por_pool.json), más conservador que el σ agrupado con que se simularon las ganancias de diseno.json.', 'nota_previa': 'El escenario «previa» sortea seis desviaciones independientes con el mismo τ que fija Z; por construcción favorece a las segmentaciones con más celdas.', 'simulacion_privado': {'nulo_2024': {'media': -0.00048, 'p05_p50_p95': [-0.00149, -0.0004, 0.00041], 'prob_ganar': 0.22, 'techo_oraculo_media': 0.00118}, 'nulo_2023': {'media': -0.00062, 'p05_p50_p95': [-0.00203, -0.00049, 0.00037], 'prob_ganar': 0.193, 'techo_oraculo_media': 0.00128}, 'previa_2024': {'media': 0.0026, 'p05_p50_p95': [-0.0005, 0.00212, 0.00636], 'prob_ganar': 0.88, 'techo_oraculo_media': 0.00472}, 'previa_2023': {'media': 0.00222, 'p05_p50_p95': [-0.00099, 0.00182, 0.00678], 'prob_ganar': 0.813, 'techo_oraculo_media': 0.00462}, 'como_2024': {'media': 0.01125, 'p05_p50_p95': [0.00893, 0.01112, 0.01418], 'prob_ganar': 1.0, 'techo_oraculo_media': 0.01552}, 'como_2023': {'media': 0.01753, 'p05_p50_p95': [0.01271, 0.01757, 0.02366], 'prob_ganar': 1.0, 'techo_oraculo_media': 0.02114}}, 'alternativas_simuladas': {'nb': {'2024_nulo': -0.00017, '2024_como_el_año': 0.0001, '2024_previa': 0.00062, '2023_nulo': -0.00025, '2023_como_el_año': 0.00526, '2023_previa': 0.00056}, 'ramo': {'2024_nulo': -0.00038, '2024_como_el_año': 0.0112, '2024_previa': 0.0011, '2023_nulo': -0.00042, '2023_como_el_año': 0.01104, '2023_previa': 0.00095}}, 'coste': 'el archivo final global, cinco sondas y el archivo final por celda: siete envíos; quedan ocho hoy y diez por día hasta el 30 de octubre', 'limites': 'Supone un público de un tercio al azar por filas; b_s aproximado por cuotas del test; σ medido con 2023 y 2024 remuestreados (el nuevo negocio de esos años tiene pocos siniestros grandes). Solo ajusta un nivel por celda, no el orden dentro de la celda. El envío lo hace Codex con autorización expresa del participante; la selección final es del participante.'}, 'm017/sigma_forma.json': {'motor_1': {'sigma': 0.07, 'a_opt_media_remuestreo': 1.092, 'bandas_2025': {'l': 0.821, 'c': 0.102, 'm': 0.077}, 'Z_tau0.25': 0.927}, 'buildings_1': {'sigma': 0.51, 'a_opt_media_remuestreo': 0.976, 'bandas_2025': {'l': 0.882, 'c': 0.067, 'm': 0.052}, 'Z_tau0.25': 0.194}, 'contents_1': {'sigma': 0.081, 'a_opt_media_remuestreo': 1.339, 'bandas_2025': {'l': 0.855, 'c': 0.082, 'm': 0.063}, 'Z_tau0.25': 0.905}}, 'm017/sondas_F2_info.json': {'base': {'archivo': 'F2_ncb_previos_2025.csv', 'sha256': '119627d93003a98933d261e773ffaf6e29a7125346367f716a0a411b97dca11a'}, 'sondas': {'sonda_F2_motor_1_a06.csv': {'celda': 'motor_1', 'a': 0.6, 'razon_nivel': 0.9466, 'sha256': '66fb61402bacd3a3f5c6bd2aba9a0979c244d553694b65515d68bb2453f78a78', 'total_celda': 37803870.18, 'total_celda_F2': 39935317.58}, 'sonda_F2_motor_1_a08.csv': {'celda': 'motor_1', 'a': 0.8, 'razon_nivel': 0.9679, 'sha256': 'b393dd709f5ef6ed208d353726cc40de8738852b74911fcc66525769f9a918eb', 'total_celda': 38654638.65, 'total_celda_F2': 39935317.58}, 'sonda_F2_buildings_1_a07.csv': {'celda': 'buildings_1', 'a': 0.7, 'razon_nivel': 0.9131, 'sha256': '451ac68bae3148da9d898fec483f45ea994a822e7aa9febbdce91a66a420a8fc', 'total_celda': 17856249.55, 'total_celda_F2': 19554638.55}, 'sonda_F2_buildings_1_a13.csv': {'celda': 'buildings_1', 'a': 1.3, 'razon_nivel': 1.1652, 'sha256': '2dff0eb76100a4116d5b4517402d0219d492adf7e708d21d08df259e92f44e01', 'total_celda': 22784970.13, 'total_celda_F2': 19554638.55}, 'sonda_F2_contents_1_a07.csv': {'celda': 'contents_1', 'a': 0.7, 'razon_nivel': 0.9525, 'sha256': 'f2401cda8e7a9c0373148995c4ee16210174011b229c2c6802c09d8b76e020b4', 'total_celda': 4785276.96, 'total_celda_F2': 5023843.53}, 'sonda_F2_contents_1_a13.csv': {'celda': 'contents_1', 'a': 1.3, 'razon_nivel': 1.0965, 'sha256': '859e2226240f682330aa7e04a8bcaaa40a101590b61beb22736496171439c4ce', 'total_celda': 5508762.58, 'total_celda_F2': 5023843.53}}}, 'm017/recibos_reales_20261026.json': [{'sha256': '42659dbb4101d4f9a4ab00c287d77b2bd21d42be9734d2ca4a4074eb57c9add2', 'd2': 0.15948}, {'sha256': 'fb568cefdd6cf8908b5a53ed37ed211b5a984bfaea65f6d9031eea0cc9183168', 'd2': 0.15888}, {'sha256': '62373a316abbb97b869c9fb315bf257b1e69fa94dcb29ee91a2515ce63efe5fb', 'd2': 0.15804}, {'sha256': 'f76c3a6c53057008b0fdf242b4cb36312822b1a6721e6100dfaee961bc966d76', 'd2': 0.15898}, {'sha256': '131716eede0d7a652063b21b86018622853f224026583450170b7d9eda3455a1', 'd2': 0.1583}, {'sha256': '28a61e3a5281bb56bae77836f0edc40ad625f1a104ac6b1b4678ee0d6980de7f', 'd2': 0.15858}, {'sha256': '119627d93003a98933d261e773ffaf6e29a7125346367f716a0a411b97dca11a', 'd2': 0.16033}, {'sha256': '66fb61402bacd3a3f5c6bd2aba9a0979c244d553694b65515d68bb2453f78a78', 'd2': 0.15246}, {'sha256': 'b393dd709f5ef6ed208d353726cc40de8738852b74911fcc66525769f9a918eb', 'd2': 0.15705}, {'sha256': '451ac68bae3148da9d898fec483f45ea994a822e7aa9febbdce91a66a420a8fc', 'd2': 0.15894}, {'sha256': '2dff0eb76100a4116d5b4517402d0219d492adf7e708d21d08df259e92f44e01', 'd2': 0.1591}, {'sha256': 'f2401cda8e7a9c0373148995c4ee16210174011b229c2c6802c09d8b76e020b4', 'd2': 0.15958}, {'sha256': '859e2226240f682330aa7e04a8bcaaa40a101590b61beb22736496171439c4ce', 'd2': 0.16001}], 'm017/exp26/fuga_celdas.json': {'motor_1': {'K_estrella_F2': 1.185914884205038, 'a0.6': {'raiz_ratio': 0.99164, 'd2_nivel': -0.000388}, 'a0.8': {'raiz_ratio': 0.99437, 'd2_nivel': -0.000257}, 'original': {'vertice': 1.4008, 'curv': -0.01637, 'ganancia': 0.00263}, 'sin_nivel': {'vertice': 1.3212, 'curv': -0.01794, 'ganancia': 0.001851}}, 'buildings_1': {'K_estrella_F2': 1.0114807998102306, 'a0.7': {'raiz_ratio': 0.98962, 'd2_nivel': -2.8e-05}, 'a1.3': {'raiz_ratio': 1.02326, 'd2_nivel': -3.3e-05}, 'original': {'vertice': 1.0092, 'curv': -0.01456, 'ganancia': 1e-06}, 'sin_nivel': {'vertice': 1.0097, 'curv': -0.01422, 'ganancia': 1e-06}}, 'contents_1': {'K_estrella_F2': 0.910477421323969, 'a0.7': {'raiz_ratio': 1.00059, 'd2_nivel': -4e-06}, 'a1.3': {'raiz_ratio': 1.00836, 'd2_nivel': -5.8e-05}, 'original': {'vertice': 1.0603, 'curv': -0.00594, 'ganancia': 2.2e-05}, 'sin_nivel': {'vertice': 1.072, 'curv': -0.0056, 'ganancia': 2.9e-05}}}, 'm017/in/constantes_publicas.json': {'factor_global': 1.1876582386485208}}
INITIAL_STEPS = [{'id': 'ml_preparar', 'argv': ['preparar_datos.py']}, {'id': 'ml_historia', 'argv': ['historia.py']}, {'id': 't2b_dev_motor', 'argv': ['gbm.py', 'dev', 'ninguno', 'motor']}, {'id': 't2b_dev_buildings', 'argv': ['gbm.py', 'dev', 'ninguno', 'buildings']}, {'id': 't2b_dev_contents', 'argv': ['gbm.py', 'dev', 'ninguno', 'contents']}, {'id': 't2b_hold_motor', 'argv': ['gbm.py', 'hold', 'ninguno', 'motor']}, {'id': 't2b_hold_buildings', 'argv': ['gbm.py', 'hold', 'ninguno', 'buildings']}, {'id': 't2b_hold_contents', 'argv': ['gbm.py', 'hold', 'ninguno', 'contents']}, {'id': 't2b_pcv_motor', 'argv': ['gbm.py', 'pcv', 'ninguno', 'motor']}, {'id': 't2b_pcv_buildings', 'argv': ['gbm.py', 'pcv', 'ninguno', 'buildings']}, {'id': 't2b_pcv_contents', 'argv': ['gbm.py', 'pcv', 'ninguno', 'contents']}, {'id': 't2b_prod_motor', 'argv': ['gbm.py', 'prod', 'ninguno', 'motor']}, {'id': 't2b_prod_buildings', 'argv': ['gbm.py', 'prod', 'ninguno', 'buildings']}, {'id': 't2b_prod_contents', 'argv': ['gbm.py', 'prod', 'ninguno', 'contents']}]
CHAIN_STEPS = [{'id': 'ml_C1', 'family': 'ml', 'argv': ['@aux', 'ml_c1']}, {'id': 'noche_base_s1', 'family': 'noche', 'argv': ['base_mercado.py', 's1']}, {'id': 'noche_base_s2', 'family': 'noche', 'argv': ['base_mercado.py', 's2']}, {'id': 'noche_curva_global', 'family': 'noche', 'argv': ['@aux', 'curve']}, {'id': 'noche_ultimo_s1', 'family': 'noche', 'argv': ['ultimo_mercado.py', 's1']}, {'id': 'noche_ultimo_s2', 'family': 'noche', 'argv': ['ultimo_mercado.py', 's2']}, {'id': 'noche_conf_insumos', 'family': 'noche', 'argv': ['conf_insumos.py']}, {'id': 'A1_full_motor_11_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'motor', '--sufijo', '_v2fixs11', '--lr', '.02', '--pat', '400', '--seed', '11', '--sin', 'make_model']}, {'id': 'A1_full_buildings_11_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'buildings', '--sufijo', '_m02', '--lr', '.02', '--pat', '400', '--seed', '11']}, {'id': 'A1_full_contents_11_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'contents', '--sufijo', '_m02', '--lr', '.02', '--pat', '400', '--seed', '11']}, {'id': 'A1_full_motor_12_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'motor', '--sufijo', '_v2fixs12', '--lr', '.02', '--pat', '400', '--seed', '12', '--sin', 'make_model']}, {'id': 'A1_full_buildings_12_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'buildings', '--sufijo', '_v2s12', '--lr', '.02', '--pat', '400', '--seed', '12']}, {'id': 'A1_full_contents_12_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'contents', '--sufijo', '_v2s12', '--lr', '.02', '--pat', '400', '--seed', '12']}, {'id': 'A1_full_motor_13_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'motor', '--sufijo', '_v2fixs13', '--lr', '.02', '--pat', '400', '--seed', '13', '--sin', 'make_model']}, {'id': 'A1_full_buildings_13_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'buildings', '--sufijo', '_v2s13', '--lr', '.02', '--pat', '400', '--seed', '13']}, {'id': 'A1_full_contents_13_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'contents', '--sufijo', '_v2s13', '--lr', '.02', '--pat', '400', '--seed', '13']}, {'id': 'A1_full_motor_11_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'motor', '--sufijo', '_v2fixf1', '--lr', '.02', '--pat', '400', '--seed', '11', '--sin', 'make_model', '--f1']}, {'id': 'A1_full_buildings_11_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'buildings', '--sufijo', '_v2f1', '--lr', '.02', '--pat', '400', '--seed', '11', '--f1']}, {'id': 'A1_full_contents_11_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'contents', '--sufijo', '_v2f1', '--lr', '.02', '--pat', '400', '--seed', '11', '--f1']}, {'id': 'A1_full_motor_12_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'motor', '--sufijo', '_v2fixf1s12', '--lr', '.02', '--pat', '400', '--seed', '12', '--sin', 'make_model', '--f1']}, {'id': 'A1_full_buildings_12_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'buildings', '--sufijo', '_v2f1s12', '--lr', '.02', '--pat', '400', '--seed', '12', '--f1']}, {'id': 'A1_full_contents_12_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'contents', '--sufijo', '_v2f1s12', '--lr', '.02', '--pat', '400', '--seed', '12', '--f1']}, {'id': 'A1_full_motor_13_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'motor', '--sufijo', '_v2fixf1s13', '--lr', '.02', '--pat', '400', '--seed', '13', '--sin', 'make_model', '--f1']}, {'id': 'A1_full_buildings_13_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'buildings', '--sufijo', '_v2f1s13', '--lr', '.02', '--pat', '400', '--seed', '13', '--f1']}, {'id': 'A1_full_contents_13_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'contents', '--sufijo', '_v2f1s13', '--lr', '.02', '--pat', '400', '--seed', '13', '--f1']}, {'id': 'A1_conf_historico_motor', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'motor', '--años', '2021,2022,2023', '--sufijo', '_v2conf', '--lr', '.02', '--pat', '400', '--conf', '--seed', '11', '--sin', 'make_model'], 'legacy_id01': True}, {'id': 'A1_conf_historico_buildings', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'buildings', '--años', '2021,2022,2023', '--sufijo', '_v2conf', '--lr', '.02', '--pat', '400', '--conf', '--seed', '11'], 'legacy_id01': False}, {'id': 'A1_conf_historico_contents', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'contents', '--años', '2021,2022,2023', '--sufijo', '_v2conf', '--lr', '.02', '--pat', '400', '--conf', '--seed', '11'], 'legacy_id01': False}, {'id': 'A1_produccion_y_hold', 'family': 'noche', 'argv': ['@aux', 'a1_post']}, {'id': 'm017_insumos_generados', 'family': 'ml', 'argv': ['@aux', 'm017_inputs']}, {'id': 'm017_F2', 'family': 'm017', 'argv': ['construir_F2.py']}, {'id': 'm017_final_creible', 'family': 'm017', 'argv': ['construir_final.py', 'recibos_reales_20261026.json']}, {'id': 'm017_alias_J', 'family': 'm017', 'argv': ['@aux', 'final_alias']}, {'id': 'm017_F3c', 'family': 'm017', 'argv': ['construir_F3c.py', 'F3c']}]
BOOTSTRAP1 = "\nimport os,sys,runpy,json\nfrom pathlib import Path\nwork=Path(sys.argv[1]).resolve()\ncode=work/'ml/codigo'\nscript=code/sys.argv[2]\nsys.path.insert(0,str(code))\nallowed=[work,Path(sys.prefix).resolve(),Path(sys.base_prefix).resolve()]\ndef inside(p,root):return p==root or root in p.parents\ndef audit(event,args):\n    if event in ('socket.connect','socket.bind','socket.sendto','subprocess.Popen'):\n        raise PermissionError('Etapa sin red ni subprocesos: '+event)\n    if event=='open' and isinstance(args[0],(str,bytes,os.PathLike)):\n        path=Path(os.fsdecode(args[0])).resolve()\n        if not any(inside(path,root) for root in allowed):\n            raise PermissionError('Lectura externa al entorno limpio: '+str(path))\n        mode=args[1] or ''\n        flags=args[2] or 0\n        writing=any(c in str(mode) for c in 'wax+') or flags & (os.O_WRONLY|os.O_RDWR|os.O_CREAT|os.O_TRUNC)\n        if writing and (not inside(path,work) or inside(path,work/'ml/datos')):\n            raise PermissionError('Escritura fuera de salidas permitidas: '+str(path))\nsys.addaudithook(audit)\nsys.argv=[str(script),*sys.argv[3:]]\nrunpy.run_path(str(script),run_name='__main__')\n"
BOOTSTRAP2 = "\nimport sys,os,runpy\nfrom pathlib import Path\nwork=Path(sys.argv[1]).resolve();family=sys.argv[2]\ncode=work/'ml/codigo' if family=='ml' else work/family/'codigo' if family=='noche' else work/'m017'\nsys.path.insert(0,str(code))\nname=sys.argv[3];script=work/'aux/reproduccion_cadena.py' if name=='@aux' else code/name\nallowed=[work,Path(sys.prefix).resolve(),Path(sys.base_prefix).resolve()]\ndef inside(p,r):return p==r or r in p.parents\ndef hook(event,args):\n    if event in ('socket.connect','socket.bind','socket.sendto','subprocess.Popen'):raise PermissionError('Etapa sin red/subprocesos: '+event)\n    if event=='open' and isinstance(args[0],(str,bytes,os.PathLike)):\n        p=Path(os.fsdecode(args[0])).resolve()\n        if not any(inside(p,r) for r in allowed):raise PermissionError('Lectura ajena a la corrida limpia: '+str(p))\n        mode=args[1] or '';flags=args[2] or 0\n        wr=any(c in str(mode) for c in 'wax+') or flags&(os.O_WRONLY|os.O_RDWR|os.O_CREAT|os.O_TRUNC)\n        if wr and (not inside(p,work) or inside(p,work/'ml/datos') or inside(p,work/'noche/datos')):raise PermissionError('Escritura no permitida: '+str(p))\nsys.addaudithook(hook)\nsys.argv=[str(script),*sys.argv[4:]]\nrunpy.run_path(str(script),run_name='__main__')\n"


### Inspect the execution plan

I show the stage identifiers and arguments before running. T2b builds the developed-market backbone first; the continuation then builds A1 and the F3c calibration. Each identifier becomes a separate log and receipt. The plan contains no historical row-level predictions.


In [ ]:
plan_rows = [{"stage": step["id"], "family": "T2b", "arguments": " ".join(step["argv"])} for step in INITIAL_STEPS]
plan_rows += [{"stage": step["id"], "family": step["family"], "arguments": " ".join(step["argv"])} for step in CHAIN_STEPS]
print(f"Stages through F3c: {len(plan_rows)}")
display(pd.DataFrame(plan_rows))


In [ ]:
def digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for part in iter(lambda: f.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

for rel, source in SOURCES.items():
    dest = RUN / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    source = source.replace("__RUN_ROOT__", RUN.as_posix())
    compile(source, str(dest), "exec")
    dest.write_text(source, encoding="utf-8", newline="\n")
for rel, value in CONFIG.items():
    write_json(RUN / rel, value)
for market, folder, names in (("developed", developed, BASE_FILES), ("emerging", emerging, emerging_files)):
    for name in names:
        dest = RUN / "ml/datos" / market / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(folder / name, dest)
        assert digest(dest) == digest(folder / name)
        dest2 = RUN / "noche/datos" / ("s1" if market == "developed" else "s2") / name
        dest2.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(dest, dest2)
        assert digest(dest) == digest(dest2)
for directory in ("tmp", "logs", "receipts", "chain_logs", "chain_receipts", "ml/trabajo/modelos/dev", "ml/trabajo/modelos/prod", "ml/entrega", "ml/privado", "noche/salidas/conf", "noche/salidas/transfer", "noche/salidas/correccion", "m017/in", "m017/revision"):
    (RUN / directory).mkdir(parents=True, exist_ok=True)
(RUN / "ejecutar_etapa.py").write_text(BOOTSTRAP1, encoding="utf-8")
(RUN / "ejecutar_bloque.py").write_text(BOOTSTRAP2, encoding="utf-8")
write_json(RUN / "notebook_run.json", {"python": sys.version, "packages": installed, "input_directories": {"developed": str(developed), "emerging": str(emerging)}, "source_count": len(SOURCES), "utc": datetime.now(timezone.utc).isoformat()})
print("Workspace prepared with", len(SOURCES), "embedded sources")


## Run the frozen stages

I train the six T2b source models and their internal round-selection fits, then reconstruct A1 and the calibration chain through F3c. This is the chain needed for the selected F3c reference. A failure stops the notebook and points to the stage log; it is not silently converted to success.

This notebook starts a fresh run and does not automatically resume a stage after interruption. Preserve the logs and receipt for diagnosis. Any transfer of intermediate data or checkpoints is a separate operation subject to the competition's data rules.


In [ ]:
env = {k: v for k, v in os.environ.items() if k.upper() in ("SYSTEMROOT", "WINDIR", "PATH", "COMSPEC", "PATHEXT", "NUMBER_OF_PROCESSORS")}
env.update(RETRAIN_WORK=str(RUN), MLIND_RAIZ=str(RUN / "ml"), MLIND_BACKBONE="T2b", PYTHONUTF8="1", PYTHONDONTWRITEBYTECODE="1", TEMP=str(RUN / "tmp"), TMP=str(RUN / "tmp"), OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2")

def run_stage(name, command, cwd, log_dir, receipt_dir, extra_env=None):
    log = RUN / log_dir / (name + ".log")
    receipt = RUN / receipt_dir / (name + ".json")
    if receipt.exists():
        raise RuntimeError(f"Stage already has a receipt: {receipt}; preserve this run and start a fresh one")
    stage_env = dict(env, **(extra_env or {}))
    start = time.monotonic()
    print("Starting", name, flush=True)
    with log.open("wb") as stream:
        result = subprocess.run(command, cwd=cwd, env=stage_env, stdout=stream, stderr=subprocess.STDOUT)
    write_json(receipt, {"stage": name, "exit_code": result.returncode, "seconds": time.monotonic() - start, "log_sha256": digest(log)})
    if result.returncode:
        print(log.read_text(encoding="utf-8", errors="replace")[-8000:])
        raise RuntimeError(f"Stage {name} failed; see {log}")
    print("Completed", name, flush=True)

for step in INITIAL_STEPS:
    run_stage(step["id"], [sys.executable, "-I", "-X", "utf8", "-u", str(RUN / "ejecutar_etapa.py"), str(RUN), *step["argv"]], RUN / "ml/codigo", "logs", "receipts")
for step in CHAIN_STEPS:
    family = step["family"]
    cwd = RUN / ("ml/codigo" if family == "ml" else "noche/codigo" if family == "noche" else "m017")
    extra = {"A1_PRE_ID01": "1"} if step.get("legacy_id01") else {}
    run_stage(step["id"], [sys.executable, "-I", "-X", "utf8", "-u", str(RUN / "ejecutar_bloque.py"), str(RUN), family, *step["argv"]], cwd, "chain_logs", "chain_receipts", extra)


## Check and save the risk-premium CSV

I compare the newly generated F3c file with the historical SHA-256 and check the submission schema and IDs. A different hash can reflect environment or numerical differences; the checks do not establish equivalence on their own.


In [ ]:
f3c = RUN / "m017/exp26/F3c_2025.csv"
assert f3c.is_file(), f"Expected output missing: {f3c}"
pred = pd.read_csv(f3c, dtype={"row_id": str})
expected_ids = pd.read_parquet(emerging / "sample_submission.parquet", columns=["row_id"]).row_id.astype(str)
assert list(pred.columns) == ["row_id", "prediction"]
assert pred.row_id.is_unique and pred.row_id.equals(expected_ids)
assert np.isfinite(pred.prediction).all() and (pred.prediction > 0).all()
historical_sha = "5ab5e4e34290cf6ed6a7a792f59e1e55a4f056400bbbdf8401933afd236b3276"
actual_sha = digest(f3c)
print("F3c SHA-256:", actual_sha, "matches historical:", actual_sha == historical_sha)
if actual_sha != historical_sha:
    raise RuntimeError(f"F3c differs from the historical CSV. Preserve and investigate {f3c} before treating this as an exact reproduction.")
output = EXPORT / "submission_week2_F3c.csv"
shutil.copyfile(f3c, output)
print("Risk premium output:", output)


## From risk premium to commercial premium

For the twenty official reference policies, I add catastrophe reinsurance for home lines and the fixed expense per policy, then divide by one minus variable expenses, commission, premium tax and target margin. The risk prediction is already for the covered period, so I do not multiply it by exposure again. I use the literal fixed expense per policy as the main interpretation and show annual proration as a sensitivity.


In [ ]:
from decimal import Decimal, localcontext, ROUND_HALF_UP
reference = pd.read_parquet(emerging / "reference_policies.parquet")
expenses = pd.read_parquet(emerging / "expenses.parquet").set_index("item").emerging
refs = reference.merge(pred, on="row_id", how="left", validate="one_to_one")
assert len(refs) == 20 and refs.prediction.notna().all()
D = lambda x: Decimal(str(x))
E = lambda k: D(expenses[k])
commercial = []
with localcontext() as context:
    context.prec = 45
    for row in refs.itertuples(index=False):
        risk, exposure = D(row.prediction), D(row.exposure)
        commission = E("commission_" + row.channel)
        cat_rate = E("cat_reinsurance") if row.line in ("buildings", "contents") else D(0)
        denominator = 1 - E("variable_expense") - commission - E("premium_tax") - E("target_profit_margin")
        assert denominator > 0 and risk >= 0 and exposure > 0
        fixed = E("fixed_expense_per_policy")
        price = (risk * (1 + cat_rate) + fixed) / denominator
        prorated = (risk * (1 + cat_rate) + fixed * exposure) / denominator
        assert abs(price - (risk * (1 + cat_rate) + fixed + price * (1 - denominator))) < D("1e-30")
        commercial.append({"row_id": row.row_id, "line": row.line, "channel": row.channel, "exposure": float(exposure), "risk_premium": float(risk), "fixed_expense": float(fixed), "commercial_premium": str(price.quantize(D("0.01"), rounding=ROUND_HALF_UP)), "commercial_premium_prorated_fixed_sensitivity": str(prorated.quantize(D("0.01"), rounding=ROUND_HALF_UP))})
commercial_path = EXPORT / "commercial_reference_premiums.csv"
pd.DataFrame(commercial).to_csv(commercial_path, index=False)
print("Commercial reference calculations:", commercial_path)


## Outputs and troubleshooting

After the final checks pass, I find `submission_week2_F3c.csv` and `commercial_reference_premiums.csv` in the `EXPORT` directory printed near the beginning. Without an override, Kaggle exports to a run-specific subdirectory of `/kaggle/working`; local execution exports beneath the new run directory. I can set `IAA_NOTEBOOK_EXPORT` before starting to choose a persistent private destination. I keep row-level outputs private.

| If execution stops | What I check |
| --- | --- |
| Missing or ambiguous input directory | The explicit market path, required filenames and attached competition data |
| Missing library or version mismatch | The optional installation cell, kernel restart and reference environment |
| Failed model stage | The stage's log and receipt under the printed run directory |
| Final fingerprint mismatch | Environment and generated artifacts; I retain the file and investigate the difference |

For Week 2, a failed stage stops execution and leaves its log and receipt. This notebook does not automatically resume the chain. I preserve the failed run before starting a fresh one or investigating a separate recovery.

I do not bypass a failed fingerprint check or interpret a resumed run as an uninterrupted execution. A schema-valid CSV and an exact historical file match answer different questions.


## License and materials

My project code and documentation are covered by the MIT License below. Competition data and third-party packages retain their own terms and are not included in that license.

MIT License

Copyright (c) 2026 Manuel Rodríguez

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
